# BondX Studio 3.0: Python scripts of the paper

This notebook holds every Python script used for the manuscript *BondX Studio: Screening Geometric Halogen Contacts with Force-Field-Consistent QM Models, NBO Analysis, and ALMO-EDA* and its Supporting Information.

**How to use it**

1. Put the notebook in the paper folder (the folder that contains `data/`, `results/`, `qm_package/`, and `qm_extra/`).
2. Run the cells of Part 1. Each cell writes one script into `Scripts/` (`%%writefile`), so the files are identical to the repository versions.
3. Run the cells of Part 2 in order. The steps that read the QM outputs need `qm_package/` and `qm_extra/` (about 400 MB, not in the repository); all later steps work from the parsed results in `results/`.

The JavaScript parts (the BondX engine `engine.js`, `bondx_cli.js`, `hiyd_bondx.js`, `hiyd_nbo.js`, and scripts 18, 19, 20, and 27 in `.js`) are not Python and stay in `Scripts/`; Part 2 calls them with Node.js.

Requirements: Python 3.10 or newer with numpy, pandas, scipy, matplotlib, and Pillow; Node.js 18 or newer; Playwright with Chromium for the screenshots. Figures use Arial (Liberation Sans, which has the same metrics, where Arial is not installed).

## Part 1: the scripts

### `bxutil.py`

bxutil.py - small helpers shared by the BondX paper scripts (geometry of gas-phase dimers, running bondx_cli.js).

In [ ]:
%%writefile Scripts/bxutil.py
"""bxutil.py - small helpers shared by the BondX paper scripts (geometry of gas-phase dimers, running bondx_cli.js)."""
import json, os, subprocess
import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
COV = {'H': 0.31, 'C': 0.76, 'N': 0.71, 'O': 0.66, 'F': 0.57, 'P': 1.07, 'S': 1.05, 'CL': 1.02, 'BR': 1.20, 'I': 1.39, 'LI': 1.28}
HAL = ('CL', 'BR', 'I')

def read_xyz(fn):
    L = open(fn).read().split('\n'); n = int(L[0].split()[0]); el, X = [], []
    for l in L[2:2 + n]:
        p = l.split(); el.append(p[0].upper()); X.append([float(v) for v in p[1:4]])
    return el, np.array(X)

def fragments(el, X, tol=0.45):
    n = len(el); adj = [[] for _ in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            if np.linalg.norm(X[i] - X[j]) < COV[el[i]] + COV[el[j]] + tol: adj[i].append(j); adj[j].append(i)
    lab = [-1] * n; k = 0
    for i in range(n):
        if lab[i] >= 0: continue
        st = [i]; lab[i] = k
        while st:
            a = st.pop()
            for b in adj[a]:
                if lab[b] < 0: lab[b] = k; st.append(b)
        k += 1
    return lab, adj

def six_ring(c, adj, el):
    """Return the carbon six-membered ring containing atom c, or None."""
    def dfs(path):
        if len(path) == 6: return path if path[0] in adj[path[-1]] else None
        for b in adj[path[-1]]:
            if el[b] == 'C' and b not in path:
                r = dfs(path + [b])
                if r: return r
        return None
    return dfs([c])

def analyse(el, X, monoA=None):
    # Strong halogen bonds (e.g. H3N...ClF) are short; lower the bond tolerance until the dimer splits into
    # two fragments whose sizes match the monomer files.
    nA = len(monoA[0]) if monoA is not None else None
    for tol in (0.45, 0.35, 0.25, 0.15, 0.05):
        lab, adj = fragments(el, X, tol)
        sizes = sorted(lab.count(k) for k in set(lab))
        if len(sizes) == 2 and (nA is None or nA in sizes): break
    if len(set(lab)) != 2: raise ValueError('expected two fragments, got %d' % len(set(lab)))
    best = None
    for prefer in (HAL, ('F',)):
        for x in range(len(el)):
            if el[x] not in prefer: continue
            for y in range(len(el)):
                if lab[y] == lab[x] or el[y] == 'H': continue
                d = np.linalg.norm(X[x] - X[y])
                if best is None or d < best[0]: best = (d, x, y)
        if best: break
    d, x, y = best
    a = min(adj[x], key=lambda j: np.linalg.norm(X[j] - X[x]))
    ring = six_ring(y, adj, el) if el[y] == 'C' else None
    if ring:
        cen = X[ring].mean(axis=0); d = np.linalg.norm(X[x] - cen)
        v = X[x] - cen; nrm = np.cross(X[ring[1]] - X[ring[0]], X[ring[2]] - X[ring[0]]); nrm /= np.linalg.norm(nrm)
        alpha = np.degrees(np.arccos(abs(v @ nrm) / np.linalg.norm(v)))
        yv = cen
    else:
        alpha = None; yv = X[y]
    u, w = X[a] - X[x], yv - X[x]
    th1 = angle_deg(u, w)
    return dict(X=x + 1, A=a + 1, Y='' if ring else y + 1, ring=','.join(str(i + 1) for i in ring) if ring else '', halogen=el[x], acc_el='pi' if ring else el[y],
                d=round(float(d), 4), theta1=round(float(th1), 2), alpha=round(float(alpha), 1) if alpha is not None else '', n_atoms=len(el),
                frag_donor=sum(1 for i in lab if i == lab[x]), frag_acceptor=sum(1 for i in lab if i != lab[x]))


def angle_deg(u, w):
    c = float(u @ w / np.linalg.norm(u) / np.linalg.norm(w))
    return float(np.degrees(np.arccos(max(-1.0, min(1.0, c)))))

def write_xyz(fn, el, X, comment=''):
    with open(fn, 'w') as f:
        f.write('%d\n%s\n' % (len(el), comment))
        for e, c in zip(el, X): f.write('%-3s %14.8f %14.8f %14.8f\n' % (e[0] + e[1:].lower(), *c))

def cli(*args):
    """Run bondx_cli.js and return its JSON output."""
    r = subprocess.run(['node', os.path.join(HERE, 'bondx_cli.js'), *[str(a) for a in args]], capture_output=True, text=True)
    if r.returncode: raise RuntimeError(r.stderr)
    return json.loads(r.stdout)


### `bxstyle.py`

Figure style of the first manuscript version (used by 08 and 10); 22-25 use bxstyle3.py.

bxstyle.py - one figure style for every figure of the BondX Studio paper.
Large labels, no legend frames, a validated categorical palette (blue, orange, aqua, yellow, magenta, green, violet, red),
PDF (vector) and PNG (300 dpi) output.

In [ ]:
%%writefile Scripts/bxstyle.py
"""bxstyle.py - one figure style for every figure of the BondX Studio paper.
Large labels, no legend frames, a validated categorical palette (blue, orange, aqua, yellow, magenta, green, violet, red),
PDF (vector) and PNG (300 dpi) output."""
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

C = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948']
HALC = {'CL': C[2], 'BR': C[1], 'I': C[0], 'F': '#8a8a86'}
INK, INK2, GRID, MUTE = '#0b0b0b', '#45443f', '#e4e3df', '#9a9993'
FS = 10.5          # base font size (pt) at the printed width
RC = {
    'font.family': 'DejaVu Sans', 'font.size': FS, 'axes.labelsize': FS + 0.5, 'axes.titlesize': FS + 1.0,
    'xtick.labelsize': FS - 0.5, 'ytick.labelsize': FS - 0.5, 'legend.fontsize': FS - 1.0, 'legend.frameon': False,
    'axes.titleweight': 'bold', 'axes.titlelocation': 'left', 'axes.edgecolor': '#8a8983', 'axes.linewidth': 0.9,
    'axes.labelcolor': INK, 'xtick.color': INK2, 'ytick.color': INK2, 'xtick.major.width': 0.9, 'ytick.major.width': 0.9,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.7, 'axes.axisbelow': True,
    'axes.spines.top': False, 'axes.spines.right': False, 'lines.linewidth': 1.8, 'lines.markersize': 5.5,
    'savefig.dpi': 300, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'figure.dpi': 100, 'mathtext.fontset': 'dejavusans'}


def apply():
    plt.rcParams.update(RC)


apply()


def save(fig, folder, name):
    os.makedirs(folder, exist_ok=True)
    for ax in fig.axes:
        leg = ax.get_legend()
        if leg is not None: leg.set_frame_on(False)
    for leg in fig.legends: leg.set_frame_on(False)
    fig.savefig(os.path.join(folder, name + '.pdf'), bbox_inches='tight')
    fig.savefig(os.path.join(folder, name + '.png'), bbox_inches='tight', dpi=300)
    plt.close(fig)
    print('wrote', name)


### `bxstyle3.py`

bxstyle3.py - figure style of the V3 paper: Arial (Liberation Sans, the metric-identical Arial clone, where Arial is not
installed), large labels, legends without frames (frameon=False), validated categorical palette, PDF (fonts embedded) + PNG.

In [ ]:
%%writefile Scripts/bxstyle3.py
"""bxstyle3.py - figure style of the V3 paper: Arial (Liberation Sans, the metric-identical Arial clone, where Arial is not
installed), large labels, legends without frames (frameon=False), validated categorical palette, PDF (fonts embedded) + PNG."""
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager

FONT = 'Arial' if any(f.name == 'Arial' for f in font_manager.fontManager.ttflist) else 'Liberation Sans'
C = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948']
HALC = {'CL': C[2], 'BR': C[1], 'I': C[0], 'F': '#8a8a86'}
VCOL = {'supported': '#2a78d6', 'inconclusive': '#eda100', 'not supported': '#e34948'}
GREY = '#f3e3b5'   # grey-zone band 0.3-0.8 kcal/mol
INK, INK2, GRID, MUTE = '#111111', '#3d3c38', '#e6e5e1', '#9a9993'
FS = 11
RC = {'font.family': 'sans-serif', 'font.sans-serif': [FONT, 'Arial', 'Liberation Sans'], 'mathtext.fontset': 'custom',
      'mathtext.rm': FONT, 'mathtext.it': FONT + ':italic', 'mathtext.bf': FONT + ':bold', 'mathtext.sf': FONT,
      'font.size': FS, 'axes.labelsize': FS + 0.5, 'axes.titlesize': FS + 1, 'xtick.labelsize': FS - 0.5, 'ytick.labelsize': FS - 0.5,
      'legend.fontsize': FS - 1, 'legend.frameon': False, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
      'axes.edgecolor': '#8a8983', 'axes.linewidth': 0.9, 'axes.labelcolor': INK, 'xtick.color': INK2, 'ytick.color': INK2,
      'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.7, 'axes.axisbelow': True, 'axes.spines.top': False,
      'axes.spines.right': False, 'lines.linewidth': 1.8, 'lines.markersize': 6, 'savefig.dpi': 300, 'pdf.fonttype': 42, 'ps.fonttype': 42}
plt.rcParams.update(RC)


def save(fig, folder, name):
    os.makedirs(folder, exist_ok=True)
    for ax in fig.axes:
        leg = ax.get_legend()
        if leg is not None: leg.set_frame_on(False)
    for leg in fig.legends: leg.set_frame_on(False)
    fig.savefig(os.path.join(folder, name + '.pdf'), bbox_inches='tight')
    fig.savefig(os.path.join(folder, name + '.png'), bbox_inches='tight', dpi=300)
    plt.close(fig); print('wrote', name)


def grey(ax, axis='y', lo=0.3, hi=0.8, label=True):
    (ax.axhspan if axis == 'y' else ax.axvspan)(lo, hi, color=GREY, zorder=0, lw=0, label='inconclusive (0.3–0.8)' if label else None)


### `00_pubmed_xb_counts.py`

pubmed_xb_counts.py -- year-wise counts of halogen-bond papers in PubMed
(reproduces Figure 1 of Manuscript_V1 and Table S1 of ESI_V1).

Uses NCBI E-utilities (esearch, rettype=count) once per year and query.
Requirements: Python 3.8+, requests   (pip install requests)

  python pubmed_xb_counts.py                     # 2003 to current year
  python pubmed_xb_counts.py --start 2000 --end 2026 --email you@uni.edu

Output: pubmed_xb_counts.csv  (year, Q1_all, Q2_bio)
Note: per-year counts use the publication-date field [dp]; they can differ
slightly from the PubMed web "Results by year" timeline.

In [ ]:
%%writefile Scripts/00_pubmed_xb_counts.py
#!/usr/bin/env python3
"""
pubmed_xb_counts.py -- year-wise counts of halogen-bond papers in PubMed
(reproduces Figure 1 of Manuscript_V1 and Table S1 of ESI_V1).

Uses NCBI E-utilities (esearch, rettype=count) once per year and query.
Requirements: Python 3.8+, requests   (pip install requests)

  python pubmed_xb_counts.py                     # 2003 to current year
  python pubmed_xb_counts.py --start 2000 --end 2026 --email you@uni.edu

Output: pubmed_xb_counts.csv  (year, Q1_all, Q2_bio)
Note: per-year counts use the publication-date field [dp]; they can differ
slightly from the PubMed web "Results by year" timeline.
"""
import argparse, csv, datetime, time
import requests

EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
Q1 = '("halogen bond" OR "halogen bonding" OR "halogen bonds")'
Q2 = f'{Q1} AND (protein OR drug OR ligand)'

def count(term, year, email, api_key):
    params = {"db": "pubmed", "term": f"{term} AND {year}[dp]", "rettype": "count",
              "retmode": "json", "tool": "bondx_counts", "email": email}
    if api_key:
        params["api_key"] = api_key
    for attempt in range(5):
        r = requests.get(EUTILS, params=params, timeout=30)
        if r.status_code == 200:
            return int(r.json()["esearchresult"]["count"])
        time.sleep(2 ** attempt)
    r.raise_for_status()

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--start", type=int, default=2003)
    ap.add_argument("--end", type=int, default=datetime.date.today().year)
    ap.add_argument("--email", default="your.name@example.org")
    ap.add_argument("--api-key", default=None, help="NCBI API key (optional, faster)")
    ap.add_argument("--out", default="pubmed_xb_counts.csv")
    a = ap.parse_args()
    pause = 0.11 if a.api_key else 0.35          # NCBI limit: 10 or 3 requests/s
    rows = []
    for y in range(a.start, a.end + 1):
        n1 = count(Q1, y, a.email, a.api_key); time.sleep(pause)
        n2 = count(Q2, y, a.email, a.api_key); time.sleep(pause)
        rows.append((y, n1, n2)); print(f"{y}  all={n1:5d}  bio={n2:5d}")
    with open(a.out, "w", newline="") as f:
        w = csv.writer(f); w.writerow(["year", "Q1_all", "Q2_bio"]); w.writerows(rows)
    t1 = sum(r[1] for r in rows); t2 = sum(r[2] for r in rows)
    print(f"\nTotal {t1} (all), {t2} (bio), bio share {100*t2/max(t1,1):.1f}%  ->  {a.out}")
    print(f"Search date: {datetime.date.today().isoformat()}")

if __name__ == "__main__":
    main()


### `01_prepare_hal59.py`

01_prepare_hal59.py
Builds the small-molecule benchmark set from the HAL59 subset of GMTKN55 (Goerigk et al. 2017).
HAL59 = 45 XB51 dimers (Kozuch & Martin 2013) + 14 X40 dimers (Rezac, Riley & Hobza 2012) with CCSD(T)/CBS
binding energies (kcal/mol, positive = bound).
Input : a clone of https://github.com/grimme-lab/GMTKN55 (folder HAL59)
Output: data/hal59/<name>.xyz and data/hal59/hal59.csv with the halogen-bond atoms X, A and Y (1-based) or the
        acceptor ring, the acceptor element and the reference energy.
Usage : python3 01_prepare_hal59.py /path/to/GMTKN55/HAL59 data/hal59

In [ ]:
%%writefile Scripts/01_prepare_hal59.py
#!/usr/bin/env python3
"""01_prepare_hal59.py
Builds the small-molecule benchmark set from the HAL59 subset of GMTKN55 (Goerigk et al. 2017).
HAL59 = 45 XB51 dimers (Kozuch & Martin 2013) + 14 X40 dimers (Rezac, Riley & Hobza 2012) with CCSD(T)/CBS
binding energies (kcal/mol, positive = bound).
Input : a clone of https://github.com/grimme-lab/GMTKN55 (folder HAL59)
Output: data/hal59/<name>.xyz and data/hal59/hal59.csv with the halogen-bond atoms X, A and Y (1-based) or the
        acceptor ring, the acceptor element and the reference energy.
Usage : python3 01_prepare_hal59.py /path/to/GMTKN55/HAL59 data/hal59
"""
import os, re, sys, csv
import numpy as np
from bxutil import read_xyz, analyse, write_xyz

def main(src, dst):
    os.makedirs(dst, exist_ok=True); rows = []
    for line in open(os.path.join(src, '.res')):
        if not line.startswith('$tmer'): continue
        p = line.split(); ref = float(p[-1]); spec = p[1]
        m = re.match(r'\{([^,}]+),', spec)
        if m: name = m.group(1); subset = 'XB51'; monoA = re.match(r'\{[^,]+,([^,]+),', spec).group(1)
        else:
            m = re.match(r'(\S+)\{A,B,AB\}', spec); name = m.group(1) + 'AB'; subset = 'X40'; monoA = m.group(1) + 'A'
        el, X = read_xyz(os.path.join(src, name, 'struc.xyz'))
        info = analyse(el, X, read_xyz(os.path.join(src, monoA, 'struc.xyz')))
        short = name[:-2] if subset == 'X40' else name
        out = os.path.join(dst, short + '.xyz')
        write_xyz(out, el, X, 'HAL59 %s %s ref_CCSD(T)/CBS = %.2f kcal/mol' % (subset, short, ref))
        if info['halogen'] == 'F': role = 'control: F-arene stacking, no sigma-hole donor'
        elif info['theta1'] < 90: role = 'control: halogen points away from the acceptor'
        else: role = 'halogen bond'
        rows.append(dict(name=short, subset=subset, ref_kcal=ref, role=role, **info))
    keys = list(rows[0].keys())
    with open(os.path.join(dst, 'hal59.csv'), 'w', newline='') as f:
        w = csv.DictWriter(f, keys); w.writeheader(); w.writerows(rows)
    print('%d dimers written to %s' % (len(rows), dst))

if __name__ == '__main__':
    main(sys.argv[1] if len(sys.argv) > 1 else '/tmp/gmtkn/HAL59', sys.argv[2] if len(sys.argv) > 2 else 'data/hal59')


### `02_make_scans.py`

02_make_scans.py
Rigid scans of halogen-bond geometry for a subset of HAL59 dimers.
  distance scan: the acceptor molecule moves along the X...Y line; d = s * d_eq, s = 0.90 ... 1.50 (8 points)
  angle scan   : the acceptor molecule rotates about X; theta1(A-X...Y) = 180, 170, ..., 120 deg (7 points).
                 The rotation axis lies in the donor ring plane and is perpendicular to A-X, so the acceptor
                 orientation relative to the X...Y line (theta2) does not change.
Output: data/scans/<name>_d<s>.xyz, data/scans/<name>_a<theta>.xyz and data/scans/scans.csv
Usage : python3 02_make_scans.py data/hal59 data/scans

In [ ]:
%%writefile Scripts/02_make_scans.py
#!/usr/bin/env python3
"""02_make_scans.py
Rigid scans of halogen-bond geometry for a subset of HAL59 dimers.
  distance scan: the acceptor molecule moves along the X...Y line; d = s * d_eq, s = 0.90 ... 1.50 (8 points)
  angle scan   : the acceptor molecule rotates about X; theta1(A-X...Y) = 180, 170, ..., 120 deg (7 points).
                 The rotation axis lies in the donor ring plane and is perpendicular to A-X, so the acceptor
                 orientation relative to the X...Y line (theta2) does not change.
Output: data/scans/<name>_d<s>.xyz, data/scans/<name>_a<theta>.xyz and data/scans/scans.csv
Usage : python3 02_make_scans.py data/hal59 data/scans
"""
import os, sys, csv
import numpy as np
from bxutil import read_xyz, analyse, write_xyz, fragments, angle_deg

DIST_SET = ['19_benCl-acetone', '20_benBr-acetone', '21_benI-acetone', '22_benCl-Nm3', '23_benBr-Nm3', '24_benI-Nm3',
            '26_benI-mSH', '28_CH3I-ben', 'NH3_PhI', 'MeI_pyr', 'FI_pyr', 'NH3_FBr']
ANGLE_SET = ['19_benCl-acetone', '20_benBr-acetone', '21_benI-acetone', '22_benCl-Nm3', '23_benBr-Nm3', '24_benI-Nm3']
SCALES = [0.90, 0.95, 1.00, 1.05, 1.10, 1.20, 1.35, 1.50]
ANGLES = [180, 170, 160, 150, 140, 130, 120]

def rot(axis, phi):
    a = axis / np.linalg.norm(axis); K = np.array([[0, -a[2], a[1]], [a[2], 0, -a[0]], [-a[1], a[0], 0]])
    return np.eye(3) + np.sin(phi) * K + (1 - np.cos(phi)) * K @ K

def split(el, X, info):
    lab = None
    for tol in (0.45, 0.35, 0.25, 0.15, 0.05):
        lab, _ = fragments(el, X, tol)
        if len(set(lab)) == 2 and lab.count(lab[int(info['X']) - 1]) == int(info['frag_donor']): break
    don = [i for i in range(len(el)) if lab[i] == lab[int(info['X']) - 1]]
    acc = [i for i in range(len(el)) if lab[i] != lab[int(info['X']) - 1]]
    return don, acc

def target(X, info):
    if info['ring']: return X[[int(i) - 1 for i in str(info['ring']).split(',')]].mean(axis=0)
    return X[int(info['Y']) - 1]

def min_contact(X, don, acc):
    return float(min(np.linalg.norm(X[i] - X[j]) for i in don for j in acc))

def main(src, dst):
    os.makedirs(dst, exist_ok=True); rows = []
    meta = {r['name']: r for r in csv.DictReader(open(os.path.join(src, 'hal59.csv')))}
    for name in DIST_SET:
        el, X0 = read_xyz(os.path.join(src, name + '.xyz')); info = meta[name]
        don, acc = split(el, X0, info); x = X0[int(info['X']) - 1]; y = target(X0, info)
        u = (y - x) / np.linalg.norm(y - x); d0 = np.linalg.norm(y - x)
        for s in SCALES:
            X = X0.copy(); X[acc] += u * (s - 1.0) * d0
            fn = '%s_d%03d.xyz' % (name, round(s * 100)); write_xyz(os.path.join(dst, fn), el, X, '%s distance scan s=%.2f' % (name, s))
            a = dict(d=float(np.linalg.norm(target(X, info) - X[int(info['X']) - 1])), theta1=angle_deg(X[int(info['A']) - 1] - X[int(info['X']) - 1], target(X, info) - X[int(info['X']) - 1]))
            rows.append(dict(file=fn, base=name, scan='distance', s=s, theta_target='', d=round(a['d'], 4), theta1=round(a['theta1'], 2),
                             min_contact=round(min_contact(X, don, acc), 3), X=info['X'], A=info['A'], Y=info['Y'], ring=info['ring'], halogen=info['halogen'], acc_el=info['acc_el']))
    for name in ANGLE_SET:
        el, X0 = read_xyz(os.path.join(src, name + '.xyz')); info = meta[name]
        don, acc = split(el, X0, info); xi, ai = int(info['X']) - 1, int(info['A']) - 1; x = X0[xi]
        ring = [i for i in don if el[i] == 'C'][:6]
        nrm = np.cross(X0[ring[1]] - X0[ring[0]], X0[ring[2]] - X0[ring[0]]); ax = np.cross(X0[ai] - x, nrm)
        th_eq = angle_deg(X0[ai] - x, target(X0, info) - x)
        for th in ANGLES:
            R = rot(ax, np.radians(th_eq - th)); X = X0.copy(); X[acc] = (X0[acc] - x) @ R.T + x
            th_now = angle_deg(X[ai] - x, target(X, info) - x)
            if abs(th_now - th) > 0.5:      # rotate the other way if needed
                R = rot(ax, -np.radians(th_eq - th)); X = X0.copy(); X[acc] = (X0[acc] - x) @ R.T + x; th_now = angle_deg(X[ai] - x, target(X, info) - x)
            fn = '%s_a%03d.xyz' % (name, th); write_xyz(os.path.join(dst, fn), el, X, '%s angle scan theta1=%d' % (name, th))
            rows.append(dict(file=fn, base=name, scan='angle', s=1.0, theta_target=th, d=round(float(np.linalg.norm(target(X, info) - x)), 4), theta1=round(th_now, 2),
                             min_contact=round(min_contact(X, don, acc), 3), X=info['X'], A=info['A'], Y=info['Y'], ring=info['ring'], halogen=info['halogen'], acc_el=info['acc_el']))
    with open(os.path.join(dst, 'scans.csv'), 'w', newline='') as f:
        w = csv.DictWriter(f, list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    print('%d scan geometries written to %s' % (len(rows), dst))

if __name__ == '__main__':
    main(sys.argv[1] if len(sys.argv) > 1 else 'data/hal59', sys.argv[2] if len(sys.argv) > 2 else 'data/scans')


### `03_t1_benchmark.py`

03_t1_benchmark.py
Tier T1 (geometry) of BondX on the HAL59 dimers and on the rigid scans.
For every geometry: BondX geometric score G (bondx_cli.js gscore, same function as the app),
pass/fail of the default criteria, and the materials-path detection (bondx_cli.js detect-xyz).
theta2 = X...Y-R with R the atom bonded to Y that is closest to Y (lone-pair acceptors only).
Output: results/t1_hal59.csv, results/t1_scans.csv

In [ ]:
%%writefile Scripts/03_t1_benchmark.py
#!/usr/bin/env python3
"""03_t1_benchmark.py
Tier T1 (geometry) of BondX on the HAL59 dimers and on the rigid scans.
For every geometry: BondX geometric score G (bondx_cli.js gscore, same function as the app),
pass/fail of the default criteria, and the materials-path detection (bondx_cli.js detect-xyz).
theta2 = X...Y-R with R the atom bonded to Y that is closest to Y (lone-pair acceptors only).
Output: results/t1_hal59.csv, results/t1_scans.csv
"""
import os, csv, sys
import numpy as np
from bxutil import read_xyz, cli, angle_deg, fragments

def theta2(el, X, xi, yi):
    lab, adj = fragments(el, X, 0.35)
    nb = [j for j in adj[yi] if lab[j] == lab[yi]]
    if not nb: return None
    r = min(nb, key=lambda j: np.linalg.norm(X[j] - X[yi]))
    return angle_deg(X[xi] - X[yi], X[r] - X[yi])

def score_row(xyz, r):
    el, X = read_xyz(xyz); xi = int(r['X']) - 1; ai = int(r['A']) - 1
    if r['ring']:
        ring = [int(i) - 1 for i in str(r['ring']).split(',')]; c = X[ring].mean(axis=0)
        n = np.cross(X[ring[1]] - X[ring[0]], X[ring[2]] - X[ring[0]]); v = X[xi] - c
        alpha = float(np.degrees(np.arccos(abs(v @ n) / np.linalg.norm(v) / np.linalg.norm(n))))
        d = float(np.linalg.norm(v)); t1 = angle_deg(X[ai] - X[xi], c - X[xi])
        g = cli('gscore', '--el', r['halogen'], '--acc', 'AR', '--d', d, '--t1', t1, '--alpha', alpha); t2 = None
    else:
        yi = int(float(r['Y'])) - 1; d = float(np.linalg.norm(X[yi] - X[xi])); t1 = angle_deg(X[ai] - X[xi], X[yi] - X[xi]); t2 = theta2(el, X, xi, yi); alpha = None
        args = ['gscore', '--el', r['halogen'], '--acc', r['acc_el'], '--d', d, '--t1', t1] + (['--t2', t2] if t2 is not None else [])
        g = cli(*args)
    det = cli('detect-xyz', xyz)['hits']
    mat = [h for h in det if h['X'] == int(r['X']) and h['status'] == 'XB']
    return dict(d=round(d, 4), theta1=round(t1, 2), theta2=round(t2, 1) if t2 is not None else '', alpha=round(alpha, 1) if alpha is not None else '',
                covered=g['covered'], cutoff=g['cutoff'], R=round(g['R'], 4) if g['R'] else '', gd=round(g['gd'], 4) if g['gd'] is not None else '',
                gt=round(g['gt'], 4) if g['gt'] is not None else '', g2=round(g['g2'], 4) if g['g2'] is not None else '', pass_T1=g['pass'], G=round(g['G'], 4),
                materials_path_XB=bool(mat))

def run(listfile, folder, outfile, keyname):
    rows = []
    for r in csv.DictReader(open(listfile)):
        fn = os.path.join(folder, r[keyname] if keyname == 'file' else r['name'] + '.xyz')
        res = score_row(fn, r)
        base = {k: r[k] for k in r if k not in ('d', 'theta1', 'alpha')}
        rows.append({**base, **res})
    with open(outfile, 'w', newline='') as f:
        w = csv.DictWriter(f, list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    print(outfile, len(rows), 'rows')
    return rows

if __name__ == '__main__':
    root = sys.argv[1] if len(sys.argv) > 1 else '..'
    os.makedirs(os.path.join(root, 'results'), exist_ok=True)
    run(os.path.join(root, 'data/hal59/hal59.csv'), os.path.join(root, 'data/hal59'), os.path.join(root, 'results/t1_hal59.csv'), 'name')
    run(os.path.join(root, 'data/scans/scans.csv'), os.path.join(root, 'data/scans'), os.path.join(root, 'results/t1_scans.csv'), 'file')


### `04_make_qm_inputs_small.py`

04_make_qm_inputs_small.py
QM inputs for the gas-phase benchmark. BondX builds every model (bondx_cli.js build-xyz).
  A_HAL59          59 dimers   NBO  wB97X-D/def2-TZVP   Gaussian 16, ORCA, Q-Chem
                               EDA  wB97X-V/def2-TZVPD  Q-Chem (ALMO-EDA2 with EDA_BSSE)
                               with every EDA job: counterpoise interaction energy in Gaussian 16 (wB97XD/def2TZVP)
                               and ORCA (wB97X-V/def2-TZVPD; three single points AB, A(AB basis), B(AB basis))
  B_scan_distance  96 points   NBO (3 programs) + EDA (Q-Chem), same levels
  C_scan_angle     42 points   NBO (3 programs) + EDA (Q-Chem), same levels
  D_method_basis   12 dimers x {wB97X-D, M06-2X, B3LYP-D3BJ, PBE0-D3BJ, HF}/def2-TZVP
                               + wB97X-D/{def2-SVP, def2-TZVPD, def2-QZVP}           NBO, 3 programs
                               (Gaussian 16 has no built-in def2-TZVPD: ORCA and Q-Chem only)
  E_eda_settings   12 dimers   EDA wB97X-V/{def2-SVPD, def2-QZVP} and def2-TZVPD without BSSE (Q-Chem)
Usage: python3 04_make_qm_inputs_small.py ../data ../qm_package

In [ ]:
%%writefile Scripts/04_make_qm_inputs_small.py
#!/usr/bin/env python3
"""04_make_qm_inputs_small.py
QM inputs for the gas-phase benchmark. BondX builds every model (bondx_cli.js build-xyz).
  A_HAL59          59 dimers   NBO  wB97X-D/def2-TZVP   Gaussian 16, ORCA, Q-Chem
                               EDA  wB97X-V/def2-TZVPD  Q-Chem (ALMO-EDA2 with EDA_BSSE)
                               with every EDA job: counterpoise interaction energy in Gaussian 16 (wB97XD/def2TZVP)
                               and ORCA (wB97X-V/def2-TZVPD; three single points AB, A(AB basis), B(AB basis))
  B_scan_distance  96 points   NBO (3 programs) + EDA (Q-Chem), same levels
  C_scan_angle     42 points   NBO (3 programs) + EDA (Q-Chem), same levels
  D_method_basis   12 dimers x {wB97X-D, M06-2X, B3LYP-D3BJ, PBE0-D3BJ, HF}/def2-TZVP
                               + wB97X-D/{def2-SVP, def2-TZVPD, def2-QZVP}           NBO, 3 programs
                               (Gaussian 16 has no built-in def2-TZVPD: ORCA and Q-Chem only)
  E_eda_settings   12 dimers   EDA wB97X-V/{def2-SVPD, def2-QZVP} and def2-TZVPD without BSSE (Q-Chem)
Usage: python3 04_make_qm_inputs_small.py ../data ../qm_package
"""
import os, sys, csv
from collections import Counter
from bxutil import cli

SENS = ['19_benCl-acetone', '20_benBr-acetone', '21_benI-acetone', '22_benCl-Nm3', '23_benBr-Nm3', '24_benI-Nm3',
        '26_benI-mSH', '28_CH3I-ben', 'NH3_PhI', 'MeI_pyr', 'FI_pyr', 'NH3_FBr']
METHODS = ['wB97X-D', 'M06-2X', 'B3LYP-D3BJ', 'PBE0-D3BJ', 'HF']
BASES = ['def2-SVP', 'def2-TZVPD', 'def2-QZVP']
RES = dict(nproc=40, mem=50000)
jobs = []


def atomargs(r):
    a = ['--X', r['X'], '--A', r['A']]
    return a + (['--ring', r['ring']] if r['ring'] else ['--Y', str(int(float(r['Y'])))])


def build(xyz, r, outdir, name, setname, extra=()):
    res = cli('build-xyz', xyz, '--out', outdir, '--name', name, '--nproc', RES['nproc'], '--mem', RES['mem'], *atomargs(r), *extra)
    for f in res['files']:
        prog = f.split('/')[0]
        if prog in ('gaussian', 'orca', 'qchem', 'qchem_eda', 'gaussian_cp', 'orca_cp') and not f.endswith('.pc'):
            jobs.append(dict(set=setname, name=name, program=prog, input=os.path.join(setname, f),
                             manifest=os.path.join(setname, 'manifests', name + '_manifest.json')))
    return res


def main(data, pkg):
    hal = list(csv.DictReader(open(os.path.join(data, 'hal59', 'hal59.csv'))))
    meta = {r['name']: r for r in hal}
    for r in hal:
        build(os.path.join(data, 'hal59', r['name'] + '.xyz'), r, os.path.join(pkg, 'A_HAL59'), r['name'], 'A_HAL59')
    for r in csv.DictReader(open(os.path.join(data, 'scans', 'scans.csv'))):
        sub = 'B_scan_distance' if r['scan'] == 'distance' else 'C_scan_angle'
        build(os.path.join(data, 'scans', r['file']), r, os.path.join(pkg, sub), r['file'][:-4], sub)
    for nm in SENS:
        r = meta[nm]; xyz = os.path.join(data, 'hal59', nm + '.xyz')
        for m in METHODS:
            build(xyz, r, os.path.join(pkg, 'D_method_basis'), f'{nm}__{m}__def2-TZVP', 'D_method_basis', ['--method', m, '--basis', 'def2-TZVP', '--noEda', '1'])
        for b in BASES:
            build(xyz, r, os.path.join(pkg, 'D_method_basis'), f'{nm}__wB97X-D__{b}', 'D_method_basis', ['--method', 'wB97X-D', '--basis', b, '--noEda', '1'])
        for b in ('def2-SVPD', 'def2-QZVP'):
            build(xyz, r, os.path.join(pkg, 'E_eda_settings'), f'{nm}__EDA__{b}', 'E_eda_settings', ['--programs', 'none', '--edaBasis', b])
        build(xyz, r, os.path.join(pkg, 'E_eda_settings'), f'{nm}__EDA__def2-TZVPD__noBSSE', 'E_eda_settings', ['--programs', 'none', '--noBSSE', '1'])
    with open(os.path.join(pkg, 'jobs_small.csv'), 'w', newline='') as f:
        w = csv.DictWriter(f, list(jobs[0].keys())); w.writeheader(); w.writerows(jobs)
    for k, v in sorted(Counter((j['set'], j['program']) for j in jobs).items()): print(k, v)


if __name__ == '__main__':
    main(sys.argv[1] if len(sys.argv) > 1 else '../data', sys.argv[2] if len(sys.argv) > 2 else '../qm_package')


### `05_protein_cases.py`

05_protein_cases.py
Protein-ligand test set. BondX T1 detection and T3a QM models (cluster and QM/MM) through bondx_cli.js.
Structures: PDB files distributed with the PLIP test suite (literature-validated halogen bonds) and 1US0 + IDD594.
For every model variant the script writes Gaussian 16, ORCA and Q-Chem NBO inputs (and a Q-Chem ALMO-EDA input for
cluster models), and it records model size, charges, electron count and MM-embedding statistics.
It also computes, from the MM point charges alone, the electrostatic potential (ESP) at the halogen X and the field
component along the A->X bond axis. These two numbers show how much the environment changes between force fields,
boundary schemes and embedding radii before any QM job is run.
Output: qm_package/F_protein/..., results/protein_models.csv, results/protein_t1.csv

In [ ]:
%%writefile Scripts/05_protein_cases.py
#!/usr/bin/env python3
"""05_protein_cases.py
Protein-ligand test set. BondX T1 detection and T3a QM models (cluster and QM/MM) through bondx_cli.js.
Structures: PDB files distributed with the PLIP test suite (literature-validated halogen bonds) and 1US0 + IDD594.
For every model variant the script writes Gaussian 16, ORCA and Q-Chem NBO inputs (and a Q-Chem ALMO-EDA input for
cluster models), and it records model size, charges, electron count and MM-embedding statistics.
It also computes, from the MM point charges alone, the electrostatic potential (ESP) at the halogen X and the field
component along the A->X bond axis. These two numbers show how much the environment changes between force fields,
boundary schemes and embedding radii before any QM job is run.
Output: qm_package/F_protein/..., results/protein_models.csv, results/protein_t1.csv
"""
import os, sys, csv, json
import numpy as np
from bxutil import cli

ROOT = sys.argv[1] if len(sys.argv) > 1 else '..'
PDB = os.path.join(ROOT, 'data', 'protein')
PKG = os.path.join(ROOT, 'qm_package', 'F_protein')
SYSTEMS = [('1us0_idd594', 'LDT:A:320'), ('2w0s', 'BVP:A:1207'), ('1p5e', 'TBS:A:301'), ('4agl', 'P84:A:400'), ('4gql', 'R47:A:306'), ('4dst', '9LI:A:201')]
REPLICAS = [('2w0s', 'BVP:B:1207'), ('1p5e', 'TBS:C:302'), ('4agl', 'P84:B:400')]
CL = {'qm': {'env': 'cluster', 'pcm': True, 'eps': 4}}
QMMM = lambda **k: {'qm': {'env': 'qmmm'}, 'mm': dict({'radius': 15, 'boundary': 'RC'}, **k.get('mm', {})), 'ff': dict({'name': 'ff14SB'}, **k.get('ff', {}))}
VARIANTS = [
    ('cluster_pcm4', CL, True),
    ('cluster_gas', {'qm': {'env': 'cluster', 'pcm': False}}, False),
    ('qmmm_ff14SB', QMMM(), False),
    ('qmmm_charmm36', QMMM(ff={'name': 'charmm36'}), False),
    ('qmmm_Z1', QMMM(mm={'boundary': 'Z1'}), False),
    ('qmmm_Z2', QMMM(mm={'boundary': 'Z2'}), False),
    ('qmmm_Z3', QMMM(mm={'boundary': 'Z3'}), False),
    ('qmmm_r10', QMMM(mm={'radius': 10}), False),
    ('qmmm_r20', QMMM(mm={'radius': 20}), False),
    ('qmmm_r25', QMMM(mm={'radius': 25}), False),
    ('qmmm_full', QMMM(mm={'radius': 200}), False),
    ('shell3', {'qm': {'env': 'cluster', 'pcm': True, 'eps': 4, 'shell': 3}}, False),
    ('shell4', {'qm': {'env': 'cluster', 'pcm': True, 'eps': 4, 'shell': 4}}, False),
    ('shell5', {'qm': {'env': 'cluster', 'pcm': True, 'eps': 4, 'shell': 5}}, False),
]
EXTRA = [('1us0_idd594', 'LDT:A:320', 'qmmm_HIS110_HID', QMMM(ff={'overrides': {'A:110': 'HID'}})),
         ('1us0_idd594', 'LDT:A:320', 'qmmm_HIS110_HIP', QMMM(ff={'overrides': {'A:110': 'HIP'}}))]
# ff19SB uses the ff14SB protein charges (it differs in backbone torsions/CMAP), so its point charges are identical.
# The script checks this for every system (results/ff19SB_check.csv) instead of writing duplicate QM jobs.
MAX_ATOMS = 260   # larger shell models are not written (too costly for NBO at def2-TZVP)
jobs, rows = [], []


def esp_field(manifest, csvfile):
    """ESP (V) at X and field component along A->X (V/A) from the MM point charges."""
    if not os.path.exists(csvfile): return None, None
    q = np.loadtxt(csvfile, delimiter=',', skiprows=1, usecols=(0, 1, 2, 3), ndmin=2)
    out = []
    for xb in manifest['xbs']:
        X = np.array([manifest['atoms'][xb['X'] - 1][k] for k in 'xyz']); A = np.array([manifest['atoms'][xb['A'] - 1][k] for k in 'xyz'])
        r = X - q[:, :3]; d = np.linalg.norm(r, axis=1); k = 14.399645  # e/(4 pi eps0) in V*A
        phi = k * np.sum(q[:, 3] / d); E = k * np.sum((q[:, 3] / d ** 3)[:, None] * r, axis=0)
        u = (X - A) / np.linalg.norm(X - A)
        out.append((phi, float(E @ u)))
    return out[0]


def record(name, sysname, lig, variant, res):
    e = res.get('embedding') or {}
    mf = json.load(open(os.path.join(PKG, 'manifests', name + '_manifest.json')))
    ef = esp_field(mf, os.path.join(PKG, 'mm_charges', name + '_mm_charges.csv')) if e else (None, None)
    rows.append(dict(model=name, system=sysname, ligand=lig, variant=variant, n_atoms=res['nAtoms'], n_heavy=res['nHeavy'], added_H=res['addedH'], link_H=res['linkH'],
                     charge_donor=res['fragCharge']['1'], charge_acceptor=res['fragCharge']['2'], total_charge=res['totalCharge'], multiplicity=res['multiplicity'], electrons=res['electrons'],
                     n_xb=len(res['xbs']), xbs=' ; '.join(x['label'] for x in res['xbs']), mm_n=e.get('n', ''), mm_net=round(e['net'], 4) if e else '', mm_M1=e.get('M1', ''),
                     mm_cut=e.get('cut', ''), boundary=e.get('boundary', ''), radius=e.get('radius', ''), esp_X_V=round(ef[0], 4) if ef[0] is not None else '', field_AX_VperA=round(ef[1], 4) if ef[1] is not None else '',
                     warnings=' | '.join(res['warnings'] + (e.get('notes') or []))))
    for f in res['files']:
        prog = f.split('/')[0]
        if prog in ('gaussian', 'orca', 'qchem', 'qchem_eda', 'gaussian_cp', 'orca_cp') and not f.endswith('.pc'):
            jobs.append(dict(set='F_protein', name=name, program=prog, input=os.path.join('F_protein', f), manifest=os.path.join('F_protein', 'manifests', name + '_manifest.json')))


def build(sysname, lig, variant, P, eda):
    name = f'{sysname}__{lig.split(":")[0]}{lig.split(":")[1]}__{variant}'
    args = ['build-pdb', os.path.join(PDB, sysname + '.pdb'), '--ligand', lig, '--out', PKG, '--name', name, '--P', json.dumps(P), '--nproc', 40, '--mem', 50000, '--cpRobust', '1']
    if not eda: args += ['--noEda', '1']
    probe = cli(*(args[:4] + ['--out', '/tmp/bx_probe', '--name', 'probe', '--P', json.dumps(P), '--programs', 'none', '--noEda', '1']))
    if probe.get('error'): print('skip', name, probe['error']); return
    if probe['nAtoms'] > MAX_ATOMS: print('skip', name, probe['nAtoms'], 'atoms'); rows.append(dict(model=name, system=sysname, ligand=lig, variant=variant, n_atoms=probe['nAtoms'], warnings='not written: more than %d atoms' % MAX_ATOMS)); return
    record(name, sysname, lig, variant, cli(*args))


def main():
    # T1 on every structure of the test set
    t1 = []
    for fn in sorted(os.listdir(PDB)):
        if not fn.endswith('.pdb'): continue
        for incF in (False, True):
            d = cli('detect-pdb', os.path.join(PDB, fn), '--P', json.dumps({'geom': {'reportNearMiss': True, 'includeF': incF}}))
            for h in d['hits']:
                t1.append(dict(pdb=fn[:-4], includeF=incF, atoms=d['atoms'], ms=round(d['msParse'] + d['msDetect'], 1), **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in h.items()}))
            if not d['hits']:
                t1.append(dict(pdb=fn[:-4], includeF=incF, atoms=d['atoms'], ms=round(d['msParse'] + d['msDetect'], 1), donor='; '.join(l['resName'] + ':' + l['chain'] + ':' + str(l['resSeq']) for l in d['halogenatedLigands']) or 'no halogenated ligand', status='none'))
    keys = sorted({k for r in t1 for k in r}, key=lambda k: (list(t1[0].keys()).index(k) if k in t1[0] else 99, k))
    with open(os.path.join(ROOT, 'results', 'protein_t1.csv'), 'w', newline='') as f:
        w = csv.DictWriter(f, keys); w.writeheader(); w.writerows(t1)
    # QM models
    for s, lig in SYSTEMS:
        for v, P, eda in VARIANTS: build(s, lig, v, P, eda)
    for s, lig in REPLICAS: build(s, lig, 'cluster_pcm4', CL, True)
    chk = []
    for s, lig in SYSTEMS:
        a = cli('build-pdb', os.path.join(PDB, s + '.pdb'), '--ligand', lig, '--out', '/tmp/bx_ff19', '--name', 'a', '--P', json.dumps(QMMM()), '--programs', 'none', '--noEda', '1')
        b = cli('build-pdb', os.path.join(PDB, s + '.pdb'), '--ligand', lig, '--out', '/tmp/bx_ff19', '--name', 'b', '--P', json.dumps(QMMM(ff={'name': 'ff19SB'})), '--programs', 'none', '--noEda', '1')
        qa = open('/tmp/bx_ff19/mm_charges/a_mm_charges.csv').read().split('\n')[1:]; qb = open('/tmp/bx_ff19/mm_charges/b_mm_charges.csv').read().split('\n')[1:]
        same = [x.split(',')[:4] for x in qa] == [x.split(',')[:4] for x in qb]   # coordinates and charges; atom types differ by design
        chk.append(dict(system=s, n_ff14SB=a['embedding']['n'], n_ff19SB=b['embedding']['n'], identical_charges=same))
    with open(os.path.join(ROOT, 'results', 'ff19SB_check.csv'), 'w', newline='') as f:
        w = csv.DictWriter(f, list(chk[0].keys())); w.writeheader(); w.writerows(chk)
    print(chk)
    for s, lig, v, P in EXTRA: build(s, lig, v, P, False)
    keys = list(dict.fromkeys(k for r in rows for k in r))
    with open(os.path.join(ROOT, 'results', 'protein_models.csv'), 'w', newline='') as f:
        w = csv.DictWriter(f, keys); w.writeheader(); w.writerows(rows)
    with open(os.path.join(ROOT, 'qm_package', 'jobs_protein.csv'), 'w', newline='') as f:
        w = csv.DictWriter(f, list(jobs[0].keys())); w.writeheader(); w.writerows(jobs)
    print(len(rows), 'models', len(jobs), 'jobs')


if __name__ == '__main__':
    main()


### `06_t1_sensitivity.py`

06_t1_sensitivity.py
Sensitivity of BondX tier T1 to its two main criteria.
  theta1Min  : 130, 135, ..., 160 deg (default 140)
  cut-offs   : all X...Y distance cut-offs shifted by -0.2 ... +0.2 A (default 0)
For each setting: number of halogen bonds in the protein test set, recovery of the four literature-validated
contacts (2W0S Asn65, 1P5E Ile10 and Leu83, 4AGL Leu145; PLIP test suite), and the number of HAL59 halogen-bond
dimers that pass.
Output: results/t1_sensitivity.csv

In [ ]:
%%writefile Scripts/06_t1_sensitivity.py
#!/usr/bin/env python3
"""06_t1_sensitivity.py
Sensitivity of BondX tier T1 to its two main criteria.
  theta1Min  : 130, 135, ..., 160 deg (default 140)
  cut-offs   : all X...Y distance cut-offs shifted by -0.2 ... +0.2 A (default 0)
For each setting: number of halogen bonds in the protein test set, recovery of the four literature-validated
contacts (2W0S Asn65, 1P5E Ile10 and Leu83, 4AGL Leu145; PLIP test suite), and the number of HAL59 halogen-bond
dimers that pass.
Output: results/t1_sensitivity.csv
"""
import os, sys, csv, json, copy
from bxutil import cli

ROOT = sys.argv[1] if len(sys.argv) > 1 else '..'
PDB = os.path.join(ROOT, 'data', 'protein')
LIT = [('2w0s', 'BVP', 'A', 65), ('1p5e', 'TBS', 'A', 10), ('1p5e', 'TBS', 'A', 83), ('4agl', 'P84', 'A', 145)]
BASE_CUT = {'I': {'N': 3.97, 'O': 3.82, 'S': 4.15, 'AR': 4.00}, 'BR': {'N': 3.844, 'O': 3.6812, 'S': 4.02, 'AR': 3.87},
            'CL': {'N': 3.772, 'O': 3.6092, 'S': 3.948, 'AR': 3.798}, 'F': {'N': 3.32, 'O': 3.29, 'S': 3.57, 'AR': 3.40}}


def main():
    pdbs = sorted(f for f in os.listdir(PDB) if f.endswith('.pdb'))
    hal = [r for r in csv.DictReader(open(os.path.join(ROOT, 'results', 't1_hal59.csv'))) if r['role'] == 'halogen bond' and r['covered'] == 'True']
    rows = []
    for th in (130, 135, 140, 145, 150, 155, 160):
        for shift in (-0.2, -0.1, 0.0, 0.1, 0.2):
            cut = {x: {y: round(v + shift, 4) for y, v in d.items()} for x, d in BASE_CUT.items()}
            P = {'geom': {'theta1Min': th, 'cutoffs': cut}}
            n_xb, found = 0, set()
            for f in pdbs:
                d = cli('detect-pdb', os.path.join(PDB, f), '--P', json.dumps(P))
                xb = [h for h in d['hits'] if h['status'] == 'XB']; n_xb += len(xb)
                for pdb, lig, ch, res in LIT:
                    if f[:-4] == pdb and any(h['donorResName'] == lig and h['accChain'] == ch and h['accResSeq'] == res for h in xb): found.add((pdb, res))
            n_hal = 0
            for r in hal:
                x, y = r['halogen'], ('AR' if r['acc_el'] == 'pi' else r['acc_el'])
                if float(r['d']) <= cut[x][y] and float(r['theta1']) >= th: n_hal += 1
            rows.append(dict(theta1Min=th, cutoff_shift=shift, protein_XB=n_xb, literature_found=len(found), literature_total=len(LIT), hal59_pass=n_hal, hal59_covered=len(hal)))
            print(rows[-1])
    with open(os.path.join(ROOT, 'results', 't1_sensitivity.csv'), 'w', newline='') as f:
        w = csv.DictWriter(f, list(rows[0].keys())); w.writeheader(); w.writerows(rows)


if __name__ == '__main__':
    main()


### `07_collect_results.py`

07_collect_results.py
Reads every finished QM output in qm_package/ with the BondX parsers (bondx_cli.js gate / eda) and writes
  results/qm_nbo.csv  one row per NBO job and halogen bond: SumE(2) terms, gate decision, NPA charges, checks
  results/qm_eda.csv  one row per ALMO-EDA job: FRZ, ELEC, PAULI, DISP, POL, CT, TOTAL (kJ/mol), chi_CT, mechanism
  results/qm_cp.csv   counterpoise-corrected interaction energies from Gaussian (Counterpoise=2) and ORCA (AB, A, B)
  results/qm_status.csv  every job with its status (ok / failed / missing)
Outputs are expected next to the inputs: Q-Chem and ORCA *.out, Gaussian *.log.
Usage: python3 07_collect_results.py ../qm_package ../results [--threshold 0.5]

In [ ]:
%%writefile Scripts/07_collect_results.py
#!/usr/bin/env python3
"""07_collect_results.py
Reads every finished QM output in qm_package/ with the BondX parsers (bondx_cli.js gate / eda) and writes
  results/qm_nbo.csv  one row per NBO job and halogen bond: SumE(2) terms, gate decision, NPA charges, checks
  results/qm_eda.csv  one row per ALMO-EDA job: FRZ, ELEC, PAULI, DISP, POL, CT, TOTAL (kJ/mol), chi_CT, mechanism
  results/qm_cp.csv   counterpoise-corrected interaction energies from Gaussian (Counterpoise=2) and ORCA (AB, A, B)
  results/qm_status.csv  every job with its status (ok / failed / missing)
Outputs are expected next to the inputs: Q-Chem and ORCA *.out, Gaussian *.log.
Usage: python3 07_collect_results.py ../qm_package ../results [--threshold 0.5]
"""
import os, sys, csv, json
from bxutil import cli

PKG = sys.argv[1] if len(sys.argv) > 1 else '../qm_package'
RES = sys.argv[2] if len(sys.argv) > 2 else '../results'
THR = sys.argv[sys.argv.index('--threshold') + 1] if '--threshold' in sys.argv else '0.5'
OK_MARK = {'qchem': 'Thank you very much for using Q-Chem', 'qchem_eda': 'Thank you very much for using Q-Chem',
           'gaussian': 'Normal termination of Gaussian', 'orca': 'ORCA TERMINATED NORMALLY',
           'gaussian_cp': 'Normal termination of Gaussian', 'orca_cp': 'ORCA TERMINATED NORMALLY'}
HARTREE_KJ = 2625.49964


def out_of(inp, prog):
    stem = os.path.splitext(inp)[0]
    return stem + ('.log' if prog.startswith('gaussian') else '.out')


def main():
    jobs = []
    for f in ('jobs_small.csv', 'jobs_protein.csv'):
        p = os.path.join(PKG, f)
        if os.path.exists(p): jobs += list(csv.DictReader(open(p)))
    import re
    nbo_rows, eda_rows, status, cp_rows, orca_e = [], [], [], [], {}
    for j in jobs:
        out = os.path.join(PKG, out_of(j['input'], j['program']))
        if not os.path.exists(out):
            status.append(dict(j, status='missing')); continue
        txt = open(out, errors='replace').read()
        finished = OK_MARK[j['program']] in txt
        if j['program'] == 'gaussian_cp':
            # Gaussian 16 (Counterpoise=2) prints the CP-corrected total energy and the sum of the fragment energies in their own basis
            mc = re.findall(r'Counterpoise corrected energy =\s+(-?\d+\.\d+)', txt); mf = re.findall(r'sum of fragments =\s+(-?\d+\.\d+)', txt)
            mk = re.findall(r'complexation energy =\s+(-?\d+\.\d+) kcal/mole \(corrected\)', txt)
            if not (mc and mf): status.append(dict(j, status='failed: no counterpoise energy')); continue
            eint = (float(mc[-1]) - float(mf[-1])) * HARTREE_KJ
            if mk: assert abs(eint / 4.184 - float(mk[-1])) < 0.006, (j['input'], eint / 4.184, mk[-1])   # agrees with the printed kcal/mol value
            cp_rows.append(dict(set=j['set'], name=j['name'], program='gaussian', level='wB97XD/def2TZVP', Eint_CP_kJ=round(eint, 3), finished=finished))
            status.append(dict(j, status='ok' if finished else 'not finished')); continue
        if j['program'] == 'orca_cp':
            m = re.findall(r'FINAL SINGLE POINT ENERGY\s+(-?\d+\.\d+)', txt)
            if not m: status.append(dict(j, status='failed: no final energy')); continue
            part = re.search(r'_cp_(AB|A|B)\.inp$', j['input']).group(1); orca_e.setdefault((j['set'], j['name']), {})[part] = float(m[-1])
            status.append(dict(j, status='ok' if finished else 'not finished')); continue
        if j['program'] == 'qchem_eda':
            r = cli('eda', out)
            if r.get('error'):
                status.append(dict(j, status='failed: ' + r['error'][:80])); continue
            d = r.get('descriptors') or {}
            eda_rows.append(dict(set=j['set'], name=j['name'], finished=finished, units=r.get('units'), frz=r.get('frz'), elec=r.get('elec'), pauli=r.get('pauli'), elec_pauli=r.get('elecPauli'), disp=r.get('disp'),
                                 pol=r.get('pol'), ct=r.get('ct'), total=r.get('total'), chi_CT=d.get('chiCT'), ct_pol_ratio=d.get('ctPolRatio'), mechanism=d.get('mechanism')))
        else:
            r = cli('gate', os.path.join(PKG, j['manifest']), out, '--threshold', THR)
            if r.get('error'):
                status.append(dict(j, status='failed: ' + r['error'][:80])); continue
            v = r['verification']
            for g in r['gate']:
                nbo_rows.append(dict(set=j['set'], name=j['name'], program=j['program'], finished=finished, xb=g['id'], label=g['label'], E2_core=round(g['E2_core'], 3),
                                     E2_total=round(g['E2_total'], 3), lp_sigma=round(g['contrib']['lp_sigma'], 3), lp_ry=round(g['contrib']['lp_ry'], 3), pi_sigma=round(g['contrib']['pi_sigma'], 3),
                                     verified=g['verified'], ionicX=g.get('ionicX'), npaX=g['npaX'], npaY=g['npaY'], ct_NPA=(r['fragNPA'] or {}).get('ct'), print_threshold=r['printThreshold'],
                                     geometry_ok=v.get('ok'), geometry_note=v.get('note')))
        status.append(dict(j, status='ok' if finished else 'not finished'))
    for (st, nm), e in orca_e.items():
        if all(k in e for k in ('AB', 'A', 'B')):
            cp_rows.append(dict(set=st, name=nm, program='orca', level='wB97X-V/def2-TZVPD', Eint_CP_kJ=round((e['AB'] - e['A'] - e['B']) * HARTREE_KJ, 3), finished=True))
    os.makedirs(RES, exist_ok=True)
    for fn, rows in (('qm_nbo.csv', nbo_rows), ('qm_eda.csv', eda_rows), ('qm_cp.csv', cp_rows), ('qm_status.csv', status)):
        if not rows: print('no rows for', fn); continue
        keys = list(dict.fromkeys(k for r in rows for k in r))
        with open(os.path.join(RES, fn), 'w', newline='') as f:
            w = csv.DictWriter(f, keys); w.writeheader(); w.writerows(rows)
        print(fn, len(rows))
    from collections import Counter
    print(Counter((s['set'], s['program'], s['status'].split(':')[0]) for s in status))


if __name__ == '__main__':
    main()


### `08_make_figures.py`

Figures of the first manuscript version; replaced by 22_v3_figures.py and kept for reproducibility of that version.

08_make_figures.py
All data figures of Manuscript_V2 and ESI_V2 (PDF + PNG). The flowchart (Fig. 2) is TikZ: Figures_V2_MS/Fig2_BondX_flowchart.tex.
Figures that need QM outputs (results/qm_nbo.csv, results/qm_eda.csv) are drawn when those files exist;
otherwise a clearly marked placeholder is written so that the LaTeX documents still compile.
Usage: python3 08_make_figures.py ..   (project root)

In [ ]:
%%writefile Scripts/08_make_figures.py
#!/usr/bin/env python3
"""08_make_figures.py
All data figures of Manuscript_V2 and ESI_V2 (PDF + PNG). The flowchart (Fig. 2) is TikZ: Figures_V2_MS/Fig2_BondX_flowchart.tex.
Figures that need QM outputs (results/qm_nbo.csv, results/qm_eda.csv) are drawn when those files exist;
otherwise a clearly marked placeholder is written so that the LaTeX documents still compile.
Usage: python3 08_make_figures.py ..   (project root)
"""
import os, sys, csv, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROOT = sys.argv[1] if len(sys.argv) > 1 else '..'
RES = os.path.join(ROOT, 'results'); MS = os.path.join(ROOT, 'Figures_V2_MS'); ESI = os.path.join(ROOT, 'Figures_V2_ESI')
SHOTS = os.path.join(ROOT, 'Figures_V2_MS', 'shots')
for d in (MS, ESI): os.makedirs(d, exist_ok=True)

# reference categorical palette (validated; see dataviz palette): blue, orange, aqua, yellow, magenta, green, violet, red
C = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948']
HAL = {'I': (C[0], 'o', 'I'), 'BR': (C[1], 's', 'Br'), 'CL': (C[2], '^', 'Cl'), 'F': ('#8a8a86', 'D', 'F')}
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e4e3df'
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8.5, 'axes.edgecolor': '#9a9993', 'axes.linewidth': 0.7, 'axes.labelcolor': INK,
                     'xtick.color': INK2, 'ytick.color': INK2, 'axes.titlesize': 9.5, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
                     'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6, 'axes.axisbelow': True, 'legend.frameon': False,
                     'axes.spines.top': False, 'axes.spines.right': False, 'savefig.dpi': 300, 'pdf.fonttype': 42})


def save(fig, folder, name):
    fig.savefig(os.path.join(folder, name + '.pdf'), bbox_inches='tight'); fig.savefig(os.path.join(folder, name + '.png'), bbox_inches='tight', dpi=220); plt.close(fig)
    print('wrote', name)


import bxstyle
from bxstyle import save
bxstyle.apply()


def placeholder(folder, name, title, lines, size=(7.0, 2.6)):
    fig, ax = plt.subplots(figsize=size); ax.axis('off')
    ax.add_patch(plt.Rectangle((0.01, 0.03), 0.98, 0.94, transform=ax.transAxes, fill=False, ls='--', lw=1.2, ec='#e34948'))
    ax.text(0.5, 0.78, title, ha='center', va='center', fontsize=11.5, weight='bold', color='#e34948', transform=ax.transAxes)
    ax.text(0.5, 0.40, '\n'.join(lines), ha='center', va='center', fontsize=11.1, color=INK2, transform=ax.transAxes)
    save(fig, folder, name)


# ----------------------------------------------------------------------------- Fig 1: literature
def fig1():
    yrs = list(range(2003, 2027))
    allxb = [11, 16, 21, 19, 30, 48, 58, 61, 95, 116, 153, 161, 200, 188, 255, 222, 255, 245, 281, 257, 270, 232, 290, 280]
    prot = [5, 3, 9, 5, 7, 11, 11, 12, 27, 33, 30, 27, 42, 46, 55, 65, 70, 62, 59, 56, 54, 54, 86, 85]
    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    ax.bar(yrs[:-1], allxb[:-1], color=C[0], width=0.78, label='All halogen-bond papers')
    ax.bar(yrs[-1], allxb[-1], color=C[0], alpha=0.4, width=0.78, label='2026 (to 18 Nov)')
    ax.plot(yrs, prot, color=C[1], lw=1.8, marker='o', ms=3.2, mec='white', mew=0.6, label='With protein, drug or ligand')
    ax.set_xlabel('Publication year'); ax.set_ylabel('Papers per year'); ax.set_ylim(0, 420); ax.grid(axis='x', visible=False)
    ax.legend(loc='upper left', handlelength=1.4, borderaxespad=0.2)
    save(fig, MS, 'Fig1_literature')


# ----------------------------------------------------------------------------- Fig 3: interface montage
def fig3():
    from PIL import Image
    sd = os.path.join(MS, 'shots')
    def main_area(f, h_css):
        im = Image.open(os.path.join(sd, f)).convert('RGB'); sc = im.width / 1440
        return im.crop((int(262 * sc), int(62 * sc), im.width - int(12 * sc), int(h_css * sc)))
    A = main_area('b_detect.png', 640); Bm = main_area('c_qm_setup.png', 660)
    Cm = Image.open(os.path.join(sd, 'd_qm_result.png')).convert('RGB'); Cm = Cm.crop((0, 0, Cm.width, min(Cm.height, int(Cm.width * 1.25))))
    ra, rb = A.height / A.width, Bm.height / Bm.width
    fig = plt.figure(figsize=(7.2, 4.65 * (ra + rb) + 0.5))
    gs = fig.add_gridspec(2, 2, width_ratios=[1.55, 1], height_ratios=[ra, rb], wspace=0.04, hspace=0.12)
    for (r, c, im, t) in ((0, 0, A, '(a) Step 2: geometric detection'), (1, 0, Bm, '(b) Step 4: QM/MM set-up'),
                          (slice(0, 2), 1, Cm, '(c) Step 4: model and charges')):
        ax = fig.add_subplot(gs[r, c]); ax.imshow(im); ax.axis('off'); ax.set_title(t, fontsize=10, loc='left', fontweight='bold')
    save(fig, MS, 'Fig3_interface')


def esi_shots():
    from PIL import Image
    files = sorted(f for f in os.listdir(SHOTS) if f[:2].isdigit())
    titles = {'01': 'Step 1: load structure', '02': 'Step 2: geometry (T1)', '03': 'Step 3: dynamics (T2)', '04': 'Step 4: QM cluster model', '05': 'Step 5: NBO gate',
              '06': 'Step 6: ALMO-EDA', '07': 'Step 7: score and report', '08': 'All parameters', '09': 'Method page', '10': 'Step 4: QM/MM model (1US0)', '11': 'Tutorial'}
    for part, chunk in enumerate([files[:6], files[6:]]):
        rows = (len(chunk) + 1) // 2
        fig, axs = plt.subplots(rows, 2, figsize=(7.2, 2.72 * rows))
        for ax in np.ravel(axs): ax.axis('off')
        for ax, f in zip(np.ravel(axs), chunk):
            im = Image.open(os.path.join(SHOTS, f)).convert('RGB'); w, h = im.size
            ax.imshow(im.crop((0, 0, w, min(h, int(w * 0.72))))); ax.set_title(titles.get(f[:2], f), fontsize=10, fontweight='bold', loc='left')
        plt.subplots_adjust(wspace=0.04, hspace=0.14)
        save(fig, ESI, 'FigS10_screens' if part == 0 else 'Extra_screens_2')


# ----------------------------------------------------------------------------- Fig 4: T1 benchmark
def fig4():
    d = pd.read_csv(os.path.join(RES, 't1_hal59.csv')); s = pd.read_csv(os.path.join(RES, 't1_sensitivity.csv'))
    xb = d[d.role == 'halogen bond']
    fig, (a, b) = plt.subplots(1, 2, figsize=(7.2, 3.5), gridspec_kw={'width_ratios': [1.2, 1]})
    acc_m = {'N': 'o', 'O': 's', 'S': 'D', 'pi': 'h'}
    for x, (col, _, lab) in HAL.items():
        if x == 'F': continue
        q = xb[(xb.halogen == x) & xb.covered]
        for acc, m in acc_m.items():
            r = q[q.acc_el == acc]
            a.scatter(r.ref_kcal, r.G, s=26, marker=m, color=col, edgecolor='white', linewidth=0.6, zorder=3)
    nc = xb[~xb.covered]
    a.scatter(nc.ref_kcal, np.full(len(nc), -0.06), s=22, marker='x', color='#8a8a86', linewidth=1.0, zorder=3)
    a.set_xscale('log'); a.set_xticks([0.5, 1, 2, 5, 10, 20]); a.set_xticklabels(['0.5', '1', '2', '5', '10', '20']); a.minorticks_off()
    a.set_xlabel('CCSD(T)/CBS binding energy (kcal/mol)'); a.set_ylabel('BondX geometric score G'); a.set_ylim(-0.12, 1.05)
    a.set_title('(a) HAL59 dimers')
    from scipy.stats import spearmanr
    cv = xb[xb.covered]; rho = spearmanr(cv.G, cv.ref_kcal).correlation
    a.text(0.03, 0.97, 'ρ = %.2f (n = %d)' % (rho, len(cv)), transform=a.transAxes, ha='left', va='top', color=INK)
    h1 = [Line2D([], [], ls='', marker='o', color=HAL[k][0], label=HAL[k][2]) for k in ('I', 'BR', 'CL')]
    h2 = [Line2D([], [], ls='', marker=m, mfc='white', mec=INK2, label=('π' if k == 'pi' else k)) for k, m in acc_m.items()]
    a.legend(handles=h1 + h2, ncol=2, loc='lower right', bbox_to_anchor=(1.0, 0.1), columnspacing=0.8, handletextpad=0.3)
    piv = s.pivot(index='theta1Min', columns='cutoff_shift', values='protein_XB'); lit = s.pivot(index='theta1Min', columns='cutoff_shift', values='literature_found')
    im = b.imshow(piv.values, cmap='Blues', aspect='auto', origin='lower', vmin=0, vmax=piv.values.max() * 1.15)
    b.set_xticks(range(piv.shape[1])); b.set_xticklabels(['%+.1f' % v for v in piv.columns]); b.set_yticks(range(piv.shape[0])); b.set_yticklabels(piv.index)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            v = piv.values[i, j]; b.text(j, i, '%d\n%d/4' % (v, lit.values[i, j]), ha='center', va='center', fontsize=9, linespacing=1.05, color='white' if v > piv.values.max() * 0.6 else INK)
    i0 = list(piv.index).index(140); j0 = list(piv.columns).index(0.0)
    b.add_patch(plt.Rectangle((j0 - 0.5, i0 - 0.5), 1, 1, fill=False, ec=C[1], lw=1.8))
    b.set_xlabel('Shift of all distance cut-offs (Å)'); b.set_ylabel('θ₁ minimum (°)'); b.grid(False)
    b.set_title('(b) Protein test set')
    plt.tight_layout(w_pad=1.5)
    save(fig, MS, 'Fig5_T1_benchmark')


# ----------------------------------------------------------------------------- Fig 5: embedding ESP
SYS_LABEL = {'1us0_idd594': '1US0 (Br···OG1 Thr113)', '2w0s': '2W0S (Br···OD1 Asn65)', '1p5e': '1P5E (Br11···O Leu83)', '4agl': '4AGL (I···O Leu145)',
             '4gql': '4GQL (Cl···O Pro232)', '4dst': '4DST (Cl···N Gly75)'}
MAIN_LIG = {'1us0_idd594': 'LDT:A:320', '2w0s': 'BVP:A:1207', '1p5e': 'TBS:A:301', '4agl': 'P84:A:400', '4gql': 'R47:A:306', '4dst': '9LI:A:201'}
VAR = [('qmmm_ff14SB', 'ff14SB'), ('qmmm_charmm36', 'CHARMM36'), ('qmmm_Z1', 'Z1'), ('qmmm_Z2', 'Z2'), ('qmmm_Z3', 'Z3'),
       ('qmmm_r10', '10 Å'), ('qmmm_r20', '20 Å'), ('qmmm_r25', '25 Å'), ('qmmm_full', 'all')]


def fig5(field=False):
    d = pd.read_csv(os.path.join(RES, 'protein_models.csv'))
    col = 'field_AX_VperA' if field else 'esp_X_V'
    fig, axs = plt.subplots(2, 3, figsize=(7.2, 5.4), sharex=True)
    groups = [C[0]] + [C[0]] + [C[1]] * 3 + [C[2]] * 4
    for ax, (sysn, lab) in zip(np.ravel(axs), SYS_LABEL.items()):
        r = d[(d.system == sysn) & (d.ligand == MAIN_LIG[sysn])].set_index('variant')
        vals = [r.loc[v, col] if v in r.index else np.nan for v, _ in VAR]
        bars = ax.bar(range(len(VAR)), vals, color=groups, width=0.72)
        bars[0].set_edgecolor(INK); bars[0].set_linewidth(1.0)
        ax.axhline(0, color='#9a9993', lw=0.7); ax.set_title(lab, fontsize=10.5)
        ax.axhline(vals[0], color=INK, lw=0.7, ls=':')
        ax.set_xticks(range(len(VAR))); ax.set_xticklabels([l for _, l in VAR], rotation=60, fontsize=9.5, ha='right'); ax.grid(axis='x', visible=False)
    for ax in axs[:, 0]: ax.set_ylabel('Field along A→X (V/Å)' if field else 'MM potential at X (V)')
    hs = [plt.Rectangle((0, 0), 1, 1, color=c) for c in (C[0], C[1], C[2])]
    fig.legend(hs, ['force field (RC, 15 Å)', 'boundary scheme (ff14SB, 15 Å)', 'embedding radius (ff14SB, RC)'], loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.04), columnspacing=1.0, handlelength=1.2)
    plt.tight_layout(rect=(0, 0, 1, 0.97))
    save(fig, MS if field else ESI, 'Fig6_embedding_field' if field else 'FigS4_embedding_ESP')


# ----------------------------------------------------------------------------- ESI figures
def figS1():
    import subprocess
    P = json.loads(subprocess.run(['node', '-e', "console.log(JSON.stringify(require('%s').defaults()))" % os.path.join(ROOT, 'Scripts', 'engine.js')], capture_output=True, text=True).stdout)
    g, sc = P['geom'], P['score']
    fig, axs = plt.subplots(2, 2, figsize=(7.0, 5.0)); axs = axs.ravel()
    R = np.linspace(0.7, 1.15, 300)
    for k, x in enumerate(('I', 'BR', 'CL')):
        s = sc['vdw'][x] + sc['vdw']['O']; Rc = g['cutoffs'][x]['O'] / s
        gd = np.clip((Rc - R) / (Rc - sc['Rbest']), 0, 1); axs[0].plot(R, gd, color=HAL[x][0], lw=1.6, label='%s···O, R$_{cut}$ %.3f' % (HAL[x][2], Rc))
    axs[0].set_xlabel('R = d / Σr$_{vdW}$'); axs[0].set_ylabel('term value'); axs[0].legend(loc='upper right'); axs[0].set_title('(a) g$_d$', fontsize=11.1)
    t = np.linspace(120, 180, 200); axs[1].plot(t, np.clip((t - g['theta1Min']) / (180 - g['theta1Min']), 0, 1), color=C[0], lw=1.6)
    axs[1].set_xlabel('θ₁ (C–X···Y, °)'); axs[1].set_title('(b) g$_θ$', fontsize=11.1)
    t2 = np.linspace(60, 180, 200); axs[2].plot(t2, np.exp(-((t2 - sc['th2Opt']) / sc['th2Width']) ** 2), color=C[1], lw=1.6)
    axs[2].set_xlabel('θ₂ (X···Y–R, °)'); axs[2].set_title('(c) g$_2$, lone-pair Y', fontsize=11.1)
    a = np.linspace(0, 90, 200); axs[3].plot(a, np.clip(1 - a / g['piNormalMax'], 0, 1), color=C[2], lw=1.6)
    axs[3].set_xlabel('α (X···centroid vs. ring normal, °)'); axs[3].set_title('(d) g$_2$, π ring', fontsize=11.1)
    for ax in axs: ax.set_ylim(-0.05, 1.08)
    axs[2].set_ylabel('term value'); plt.tight_layout(h_pad=1.2, w_pad=1.5); save(fig, ESI, 'FigS1_score_functions')


def figS2():
    d = pd.read_csv(os.path.join(RES, 't1_scans.csv'))
    fig, axs = plt.subplots(2, 2, figsize=(7.2, 5.8)); axs = axs.ravel()
    dist = d[d.scan == 'distance']
    groups = [('I', ['21_benI-acetone', '24_benI-Nm3', '26_benI-mSH', '28_CH3I-ben', 'NH3_PhI', 'MeI_pyr', 'FI_pyr']), ('BR', ['20_benBr-acetone', '23_benBr-Nm3', 'NH3_FBr']), ('CL', ['19_benCl-acetone', '22_benCl-Nm3'])]
    for ax, (x, names) in zip(axs[:3], groups):
        for k, n in enumerate(names):
            r = dist[dist.base == n].sort_values('s'); ax.plot(r.s, r.G, marker='o', ms=2.8, lw=1.3, color=C[k % 8], label=n.split('_', 1)[-1] if n[0].isdigit() else n.replace('_', '···'))
        ax.set_title('(%s) %s donors' % ('abc'[['I', 'BR', 'CL'].index(x)], HAL[x][2])); ax.set_xlabel('d / d$_{eq}$'); ax.set_ylabel('G'); ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), handlelength=1.2, fontsize=9)
    axs[0].set_ylabel('G')
    ang = d[d.scan == 'angle']
    for k, n in enumerate(sorted(ang.base.unique())):
        r = ang[ang.base == n].sort_values('theta1'); axs[3].plot(r.theta1, r.G, marker='o', ms=2.8, lw=1.3, color=C[k], label=n.split('_')[1])
    axs[3].axvline(140, color=INK2, ls=':', lw=0.8); axs[3].set_xlabel('θ₁ (°)'); axs[3].set_title('(d) Angle scans'); axs[3].set_ylabel('G'); axs[3].legend(loc='center left', bbox_to_anchor=(1.0, 0.5), handlelength=1.2, fontsize=9)
    plt.tight_layout(w_pad=0.8, h_pad=1.2); save(fig, ESI, 'FigS2_scans_T1')


def figS3():
    d = pd.read_csv(os.path.join(RES, 'protein_t1.csv')); d = d[(d.includeF == False) & (d.status != 'none')]
    cnt = d.groupby(['pdb', 'status']).size().unstack(fill_value=0)
    cnt = cnt.reindex(columns=['XB', 'near-miss'], fill_value=0).sort_values('XB')
    fig, ax = plt.subplots(figsize=(4.6, 3.8))
    y = np.arange(len(cnt)); ax.barh(y, cnt['XB'], color=C[0], height=0.62, label='halogen bond (T1)')
    ax.barh(y, cnt['near-miss'], left=cnt['XB'], color='#bdbcb6', height=0.62, label='near-miss')
    ax.set_yticks(y); ax.set_yticklabels([p.upper().replace('_IDD594', '+IDD594').replace('_PROTONATED', ' (polar H)') for p in cnt.index])
    ax.set_xlabel('Contacts (all chains)'); ax.legend(loc='lower right'); ax.grid(axis='y', visible=False)
    plt.tight_layout(); save(fig, ESI, 'Extra_protein_T1')


def figS4():
    d = pd.read_csv(os.path.join(RES, 'performance.csv')); t = d[d.test == 'T1 detect']
    fig, ax = plt.subplots(figsize=(4.6, 3.4))
    tiled = t.system.str.contains(' x'); ax.scatter(t[~tiled].atoms, t[~tiled].ms, s=16, color=C[0], edgecolor='white', lw=0.5, label='test-set structures')
    ax.scatter(t[tiled].atoms, t[tiled].ms, s=26, marker='s', color=C[1], edgecolor='white', lw=0.5, label='1P5E tiled ×2, ×4, ×8')
    xs = np.array([500, 1e5]); k = np.median(t.ms / t.atoms); ax.plot(xs, k * xs, color=INK2, ls=':', lw=0.9, label='linear scaling')
    ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlabel('Atoms'); ax.set_ylabel('Parse + T1 detection (ms)'); ax.legend(loc='upper left')
    plt.tight_layout(); save(fig, ESI, 'FigS9_performance')


def figS5():
    d = pd.read_csv(os.path.join(RES, 'protein_models.csv'))
    fig, ax = plt.subplots(figsize=(4.6, 3.4))
    for k, (sysn, lab) in enumerate(SYS_LABEL.items()):
        r = d[(d.system == sysn) & (d.ligand == MAIN_LIG[sysn])].set_index('variant'); sh = [0, 3, 4, 5]
        n = [r.loc['cluster_pcm4', 'n_atoms']] + [r.loc['shell%d' % s, 'n_atoms'] for s in sh[1:]]
        ax.plot(sh, n, marker='o', ms=3.5, lw=1.5, color=C[k], label=lab.split(' ')[0])
    ax.set_xlabel('Extra shell around X (Å)'); ax.set_ylabel('QM atoms (with H and link H)'); ax.legend(ncol=2, loc='upper left'); ax.set_xticks([0, 3, 4, 5]); ax.set_ylim(0, 290)
    plt.tight_layout(); save(fig, ESI, 'FigS3_qm_size')


def qm_figures():
    nbo = os.path.join(RES, 'qm_nbo.csv'); eda = os.path.join(RES, 'qm_eda.csv')
    if os.path.exists(os.path.join(RES, 'qm_summary.json')):
        print('Fig8, Fig9, FigS7 and FigS8 are drawn by 17_qm_nbo_eda_analysis.py'); return
    if not os.path.exists(nbo):
        placeholder(MS, 'Fig8_NBO_benchmark', 'Figure 8 is drawn from the QM outputs', [
            '(a) NBO ΣE(2) [n(Y)→σ*(C–X)] vs. CCSD(T)/CBS binding energy for the 59 HAL59 dimers (Q-Chem and Gaussian).',
            '(b) ΣE(2) along the distance scans, with the 0.5 kcal/mol gate.', '(c) ALMO-EDA terms and χ_CT for the HAL59 halogen bonds.',
            'Run run_qchem.sh and run_gaussian.sh in qm_package, then Scripts/07_collect_results.py and Scripts/08_make_figures.py.'])
        placeholder(MS, 'Fig9_protein_QM', 'Figure 9 is drawn from the QM outputs', [
            'Protein set: ΣE(2) and gate decision for each contact in cluster, PCM and QM/MM models,',
            'its change with force field, boundary scheme, embedding radius and QM-region size, and the BXQI with tier provenance.'])
        for nm, t in (('FigS5_angle_scan_QM', 'NBO and EDA along the angle scans'), ('FigS6_method_basis', 'Method, basis and program dependence of ΣE(2)')):
            placeholder(ESI, nm, t + ' (from QM outputs)', ['Generated by Scripts/08_make_figures.py after Scripts/07_collect_results.py.'])
        return
    q = pd.read_csv(nbo); hal = pd.read_csv(os.path.join(RES, 't1_hal59.csv'))
    a = q[(q.set == 'A_HAL59')].merge(hal[['name', 'ref_kcal', 'halogen', 'acc_el', 'role']], on='name')
    fig, axs = plt.subplots(1, 3, figsize=(7.1, 2.5))
    for prog, m in (('qchem', 'o'), ('gaussian', 's')):
        r = a[a.program == prog]
        axs[0].scatter(r.ref_kcal, r.E2_core.clip(lower=0.01), s=16, marker=m, c=[HAL.get(x, ('#888',))[0] for x in r.halogen], edgecolor='white', lw=0.4, label=prog)
    axs[0].set_xscale('log'); axs[0].set_yscale('log'); axs[0].axhline(0.5, color=C[7], ls='--', lw=0.9); axs[0].set_xlabel('CCSD(T)/CBS (kcal/mol)'); axs[0].set_ylabel('ΣE(2) (kcal/mol)'); axs[0].legend(fontsize=8.5)
    axs[0].set_title('(a) HAL59')
    b = q[(q.set == 'B_scan_distance') & (q.program == 'qchem')].copy(); sc = pd.read_csv(os.path.join(RES, 't1_scans.csv')); sc['name'] = sc.file.str[:-4]
    b = b.merge(sc[['name', 'base', 's']], on='name')
    for k, n in enumerate(sorted(b.base.unique())):
        r = b[b.base == n].sort_values('s'); axs[1].plot(r.s, r.E2_core.clip(lower=0.01), lw=1.1, marker='o', ms=2, color=C[k % 8])
    axs[1].set_yscale('log'); axs[1].axhline(0.5, color=C[7], ls='--', lw=0.9); axs[1].set_xlabel('d / d$_{eq}$'); axs[1].set_title('(b) distance scans (Q-Chem)')
    if os.path.exists(eda):
        e = pd.read_csv(eda); e = e[e.set == 'A_HAL59'].merge(hal[['name', 'ref_kcal', 'halogen']], on='name')
        axs[2].scatter(e.ref_kcal * 4.184, -e.total, s=16, c=[HAL.get(x, ('#888',))[0] for x in e.halogen], edgecolor='white', lw=0.4)
        lim = [0, max((e.ref_kcal * 4.184).max(), (-e.total).max()) * 1.05]; axs[2].plot(lim, lim, color=INK2, ls=':', lw=0.8)
        axs[2].set_xlabel('CCSD(T)/CBS (kJ/mol)'); axs[2].set_ylabel('−ΔE$_{int}$ ALMO (kJ/mol)'); axs[2].set_title('(c) ALMO-EDA total')
    plt.tight_layout(); save(fig, MS, 'Fig8_NBO_benchmark')


# ----------------------------------------------------------------------------- Fig 7: ORCA counterpoise benchmark
ACC_M = {'N': 'o', 'O': 's', 'S': 'D', 'pi': 'h', 'P': 'v', 'F': '<'}
def fig_cp():
    pth = os.path.join(RES, 'orca_cp_hal59.csv')
    if not os.path.exists(pth):
        placeholder(MS, 'Fig7_CP_benchmark', 'Figure 7 is drawn from the ORCA counterpoise outputs', ['Run Scripts/07_collect_results.py and Scripts/13_orca_cp_analysis.py.']); return
    h = pd.read_csv(pth); sm = json.load(open(os.path.join(RES, 'orca_cp_summary.json'))); sc = pd.read_csv(os.path.join(RES, 'orca_cp_scans.csv'))
    fig, (a, b, c) = plt.subplots(1, 3, figsize=(7.2, 3.1), gridspec_kw={'width_ratios': [1.05, 1, 1]})
    lim = [0.18, 30]; a.plot(lim, lim, color=INK2, lw=0.8, ls=':', zorder=1)
    for (X, acc), r in h.groupby(['halogen', 'acc']):
        ctl = r.role != 'halogen bond'; col = HAL[X.upper()][0]
        a.scatter(r.ref_kcal[~ctl], r.BE_kcal[~ctl], s=22, marker=ACC_M[acc], color=col, edgecolor='white', linewidth=0.5, zorder=3)
        a.scatter(r.ref_kcal[ctl], r.BE_kcal[ctl], s=26, marker=ACC_M[acc], facecolor='white', edgecolor=INK2, linewidth=0.9, zorder=3)
    a.set_xscale('log'); a.set_yscale('log'); a.set_xlim(lim); a.set_ylim(lim); a.minorticks_off()
    for ax_ in (a.xaxis, a.yaxis): ax_.set_ticks([0.3, 1, 3, 10, 30]); ax_.set_ticklabels(['0.3', '1', '3', '10', '30'])
    a.set_xlabel('CCSD(T)/CBS (kcal/mol)'); a.set_ylabel('ORCA CP (kcal/mol)'); a.set_title('(a) HAL59')
    st = sm['hal59_all']
    a.text(0.03, 0.97, 'MAE %.2f kcal/mol\nr = %.3f' % (st['MAE'], st['pearson_r']), transform=a.transAxes, ha='left', va='top', color=INK, linespacing=1.2)
    hh = [Line2D([], [], ls='', marker='o', color=HAL[k][0], label=HAL[k][2]) for k in ('I', 'BR', 'CL')] + [Line2D([], [], ls='', marker='o', mfc='white', mec=INK2, label='control')]
    a.legend(handles=hh, loc='lower right', handletextpad=0.2, borderaxespad=0.1)
    # (b) distance scans, normalised to s = 1.00; open symbols: G = 0 (outside the T1 window)
    d = sc[sc.scan == 'distance']
    for base, r in d.groupby('base'):
        r = r.sort_values('s'); col = HAL[r.halogen.iloc[0].upper()][0]
        b.plot(r.s, r.BE_rel, color=col, lw=1.0, alpha=0.9, zorder=2)
        on = r.G > 0; b.scatter(r.s[on], r.BE_rel[on], s=9, color=col, zorder=3, edgecolor='none')
        b.scatter(r.s[~on], r.BE_rel[~on], s=11, facecolor='white', edgecolor=col, linewidth=0.8, zorder=3)
    b.axhline(0, color=INK2, lw=0.6); b.set_xlabel('d / d$_{eq}$'); b.set_ylabel('BE / BE(d$_{eq}$)'); b.set_ylim(-0.05, 1.08); b.set_title('(b) Distance scans')
    b.legend(handles=[Line2D([], [], ls='', marker='o', ms=4, color=INK2, label='G > 0'), Line2D([], [], ls='', marker='o', ms=4, mfc='white', mec=INK2, label='G = 0')], loc='upper right', handletextpad=0.2)
    # (c) angle scans
    g = sc[sc.scan == 'angle']
    c.axvspan(118, 140, color='#f1efe9', lw=0, zorder=0); c.axhline(0, color=INK2, lw=0.6)
    for base, r in g.groupby('base'):
        r = r.sort_values('theta'); col = HAL[r.halogen.iloc[0].upper()][0]; ls = '-' if r.acc.iloc[0] == 'O' else '--'
        c.plot(r.theta, r.BE_rel, color=col, lw=1.1, ls=ls, marker='o', ms=2.4, zorder=2)
    c.set_xlim(118, 182); c.set_xticks([120, 140, 160, 180]); c.set_xlabel('θ$_1$ (°)'); c.set_ylabel('BE / BE(180°)'); c.set_title('(c) Angle scans')
    c.text(129, 0.75, 'G = 0', ha='center', va='center', color=INK2); c.set_ylim(-1.15, 1.62)
    c.legend(handles=[Line2D([], [], color=INK2, lw=1.1, ls='-', label='O (acetone)'), Line2D([], [], color=INK2, lw=1.1, ls='--', label='N (NMe$_3$)')], loc='upper right', handlelength=1.6, borderaxespad=0.0)
    plt.tight_layout(w_pad=1.0); save(fig, MS, 'Fig7_CP_benchmark')
    # ESI: signed error per dimer
    hs = h.sort_values(['halogen', 'ref_kcal']).reset_index(drop=True)
    nm = lambda n: n.split('_', 1)[1] if n[0].isdigit() else n.replace('_', '···')
    fig, ax = plt.subplots(figsize=(6.4, 10.5))
    y = np.arange(len(hs))[::-1]
    ax.barh(y, hs.err_kcal, color=[HAL[x.upper()][0] if r == 'halogen bond' else '#b9b7b0' for x, r in zip(hs.halogen, hs.role)], height=0.75)
    ax.axvline(0, color=INK2, lw=0.8); ax.set_yticks(y); ax.set_yticklabels([nm(n) for n in hs.name], fontsize=9); ax.set_ylim(-0.8, len(hs) - 0.2)
    ax.set_xlabel('BE(ORCA) − BE(CCSD(T)/CBS) (kcal/mol)'); ax.grid(axis='y', visible=False)
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=HAL[k][0], label=HAL[k][2]) for k in ('CL', 'BR', 'I')] + [plt.Rectangle((0, 0), 1, 1, color='#b9b7b0', label='control')], loc='lower left')
    save(fig, ESI, 'Extra_cp_errors')



if __name__ == '__main__':
    ALL = {'fig1': fig1, 'fig3': fig3, 'fig4': fig4, 'fig5': fig5, 'fig6': lambda: fig5(field=True), 'figS1': figS1, 'figS2': figS2, 'figS3': figS3,
           'figS4': figS4, 'figS5': figS5, 'shots': esi_shots, 'cp': fig_cp, 'qm': qm_figures}
    sel = [a for a in sys.argv[2:] if a in ALL] or list(ALL)
    for k in sel: ALL[k]()


### `09_performance.py`

09_performance.py
Run time of the BondX engine (Node.js, same code as the browser app).
  1. T1 parse + detection time for every protein test structure and for tiled copies of 1P5E (up to ~74 000 atoms).
  2. T2 trajectory throughput (frames per second) for 1US0+IDD594 and 1P5E with 200 synthetic frames.
  3. T3a QM-model build time (cluster and QM/MM) from results/protein_models.csv runs.
Each timing is the median of 5 repeats. Output: results/performance.csv

In [ ]:
%%writefile Scripts/09_performance.py
#!/usr/bin/env python3
"""09_performance.py
Run time of the BondX engine (Node.js, same code as the browser app).
  1. T1 parse + detection time for every protein test structure and for tiled copies of 1P5E (up to ~74 000 atoms).
  2. T2 trajectory throughput (frames per second) for 1US0+IDD594 and 1P5E with 200 synthetic frames.
  3. T3a QM-model build time (cluster and QM/MM) from results/protein_models.csv runs.
Each timing is the median of 5 repeats. Output: results/performance.csv
"""
import os, sys, csv, json, subprocess, statistics
from bxutil import cli, HERE

ROOT = sys.argv[1] if len(sys.argv) > 1 else '..'
PDB = os.path.join(ROOT, 'data', 'protein')
CHAINS = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'


def tile(src, n, out):
    """n copies of a PDB, shifted by 150 A along x, with new chain letters."""
    L = [l for l in open(src) if l.startswith(('ATOM', 'HETATM'))]
    chains = sorted({l[21] for l in L}); k = 0
    with open(out, 'w') as f:
        for c in range(n):
            for l in L:
                ch = CHAINS[(c * len(chains) + chains.index(l[21])) % len(CHAINS)]
                x = float(l[30:38]) + 150.0 * c
                f.write(l[:21] + ch + l[22:30] + '%8.3f' % x + l[38:])
            k += len(L)
        f.write('END\n')
    return k


def time_detect(path, rep=5):
    t = []
    for _ in range(rep):
        d = cli('detect-pdb', path); t.append(d['msParse'] + d['msDetect']); n = d['atoms']; nx = len([h for h in d['hits'] if h['status'] == 'XB'])
    return n, statistics.median(t), nx


TRAJ_JS = r'''
const B = require(process.argv[2]); const fs = require('fs');
const S = B.parsePDB(fs.readFileSync(process.argv[3], 'utf8')); const P = B.defaults();
const f0 = S.frames[0]; let seed = 7; const rnd = () => { seed = (seed * 16807) % 2147483647; return seed / 2147483647 - 0.5; };
const frames = []; for (let k = 0; k < 200; k++) { const f = Float32Array.from(f0); for (let i = 0; i < f.length; i++) f[i] += rnd() * 0.4; frames.push(f); }
const t = []; for (let r = 0; r < 5; r++) { const acc = B.trajAccumulator(S, P, {}); const t0 = process.hrtime.bigint(); frames.forEach((f, i) => acc.add(f, null, i)); const res = acc.finish(); t.push(Number(process.hrtime.bigint() - t0) / 1e6); }
t.sort((a, b) => a - b); console.log(JSON.stringify({ atoms: S.atoms.length, frames: 200, ms: t[2] }));
'''


def main():
    rows = []
    for f in sorted(os.listdir(PDB)):
        if f.endswith('.pdb'):
            n, ms, nx = time_detect(os.path.join(PDB, f)); rows.append(dict(test='T1 detect', system=f[:-4], atoms=n, ms=round(ms, 1), xb=nx))
    for c in (2, 4, 8):
        out = '/tmp/1p5e_x%d.pdb' % c; tile(os.path.join(PDB, '1p5e.pdb'), c, out)
        n, ms, nx = time_detect(out); rows.append(dict(test='T1 detect', system='1p5e x%d' % c, atoms=n, ms=round(ms, 1), xb=nx))
    js = '/tmp/bx_traj_bench.js'; open(js, 'w').write(TRAJ_JS)
    for s in ('1us0_idd594', '1p5e'):
        r = json.loads(subprocess.run(['node', js, os.path.join(HERE, 'engine.js'), os.path.join(PDB, s + '.pdb')], capture_output=True, text=True).stdout)
        rows.append(dict(test='T2 trajectory', system=s, atoms=r['atoms'], ms=round(r['ms'], 1), xb='', frames_per_s=round(r['frames'] / r['ms'] * 1000, 1)))
    for s, lig in (('1us0_idd594', 'LDT:A:320'), ('1p5e', 'TBS:A:301'), ('4gql', 'R47:A:306')):
        for env in ('cluster', 'qmmm'):
            t = [cli('build-pdb', os.path.join(PDB, s + '.pdb'), '--ligand', lig, '--out', '/tmp/bx_perf', '--name', 'p', '--programs', 'none', '--noEda', '1', '--P', json.dumps({'qm': {'env': env}}))['ms'] for _ in range(5)]
            rows.append(dict(test='T3a build ' + env, system=s, atoms='', ms=statistics.median(t), xb=''))
    keys = list(dict.fromkeys(k for r in rows for k in r))
    with open(os.path.join(ROOT, 'results', 'performance.csv'), 'w', newline='') as f:
        w = csv.DictWriter(f, keys); w.writeheader(); w.writerows(rows)
    for r in rows: print(r)


if __name__ == '__main__':
    main()


### `10_screenshots.py`

Screenshots of the first interface; replaced by 23_v3_screenshots.py.

10_screenshots.py
Interface screenshots for Fig. 3 and the ESI (Playwright + Chromium; the BondX Studio HTML file is opened locally).
The 3D viewer needs the 3Dmol.js library from a CDN; it is hidden when that library cannot be loaded.
Usage: python3 10_screenshots.py /path/to/BondX_Studio.html /path/to/1us0_idd594.pdb ../Figures_V2_MS/shots

In [ ]:
%%writefile Scripts/10_screenshots.py
#!/usr/bin/env python3
"""10_screenshots.py
Interface screenshots for Fig. 3 and the ESI (Playwright + Chromium; the BondX Studio HTML file is opened locally).
The 3D viewer needs the 3Dmol.js library from a CDN; it is hidden when that library cannot be loaded.
Usage: python3 10_screenshots.py /path/to/BondX_Studio.html /path/to/1us0_idd594.pdb ../Figures_V2_MS/shots
"""
import asyncio, os, sys
from playwright.async_api import async_playwright

HTML, PDB, OUT = sys.argv[1], sys.argv[2], sys.argv[3]
HIDE_VIEWER = "document.querySelectorAll('.vslot').forEach(e => { if (!window.$3Dmol) e.style.display = 'none'; });"


async def main():
    os.makedirs(OUT, exist_ok=True)
    async with async_playwright() as p:
        b = await p.chromium.launch(executable_path=os.environ.get('CHROME', '/usr/bin/google-chrome'), args=['--no-sandbox'])
        pg = await b.new_page(viewport={'width': 1440, 'height': 900}, device_scale_factor=2)
        await pg.goto('file://' + os.path.abspath(HTML)); await pg.wait_for_timeout(1200)
        nav = lambda pid: pg.click(f'#side button[data-p={pid}]')
        await pg.set_input_files('#dropPdb input', PDB); await pg.wait_for_timeout(800)
        await pg.evaluate(HIDE_VIEWER); await pg.screenshot(path=os.path.join(OUT, 'a_structure.png'))
        await nav('detect'); await pg.click('#btnDetect'); await pg.wait_for_timeout(500); await pg.evaluate(HIDE_VIEWER)
        await pg.evaluate("document.querySelector('#pg-detect .assume').open = false"); await pg.evaluate('window.scrollTo(0, 0)'); await pg.wait_for_timeout(200)
        await pg.screenshot(path=os.path.join(OUT, 'b_detect.png'))
        await pg.click('#btnToReg'); await nav('qm'); await pg.wait_for_timeout(300)
        await pg.click('#qEnvSeg button[data-env=qmmm]'); await pg.wait_for_timeout(200)
        for bx in await pg.query_selector_all('#qmReg .selc'): await bx.check()
        await pg.wait_for_timeout(500); await pg.evaluate(HIDE_VIEWER)
        await pg.evaluate("document.querySelector('#pg-qm .assume').open = false"); await pg.evaluate('window.scrollTo(0, 0)'); await pg.wait_for_timeout(200)
        await pg.screenshot(path=os.path.join(OUT, 'c_qm_setup.png'))
        await pg.click('#btnBuild'); await pg.wait_for_timeout(1500); await pg.evaluate(HIDE_VIEWER)
        await pg.evaluate("document.querySelector('header.top').style.position = 'static'")
        el = await pg.query_selector('#qOut'); await el.scroll_into_view_if_needed(); await pg.wait_for_timeout(200)
        await el.screenshot(path=os.path.join(OUT, 'd_qm_result.png'))
        await nav('params'); await pg.wait_for_timeout(300); await pg.screenshot(path=os.path.join(OUT, 'e_params.png'))
        await pg.evaluate("document.querySelector('header.top').style.position = 'sticky'")
        await pg.click('#btnTutorial'); await pg.wait_for_timeout(300); await pg.evaluate('window.scrollTo(0, 0)')
        await pg.screenshot(path=os.path.join(OUT, 'f_tutorial_1.png'))
        for i in range(4): await pg.click('#tutNext')
        await pg.wait_for_timeout(300); await pg.evaluate('window.scrollTo(0, 0)'); await pg.screenshot(path=os.path.join(OUT, 'g_tutorial_5.png'))
        await b.close()
    print('screenshots in', OUT)

asyncio.run(main())


### `11_make_tables.py`

11_make_tables.py
LaTeX tables for ESI_V2 (and two for Manuscript_V2), written from the result CSV files and from the BondX engine itself
(parameter list, assumptions, tests), so that no number is typed by hand.
Output: tables/*.tex

In [ ]:
%%writefile Scripts/11_make_tables.py
#!/usr/bin/env python3
"""11_make_tables.py
LaTeX tables for ESI_V2 (and two for Manuscript_V2), written from the result CSV files and from the BondX engine itself
(parameter list, assumptions, tests), so that no number is typed by hand.
Output: tables/*.tex
"""
import os, sys, csv, json, re, subprocess
import pandas as pd

ROOT = sys.argv[1] if len(sys.argv) > 1 else '..'
APP = sys.argv[2] if len(sys.argv) > 2 else os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '..', 'app')  # the app folder of the repository
RES = os.path.join(ROOT, 'results'); TAB = os.path.join(ROOT, 'tables'); os.makedirs(TAB, exist_ok=True)
ENG = os.path.join(ROOT, 'Scripts', 'engine.js')

U = {'θ': r'$\theta$', 'α': r'$\alpha$', 'β': r'$\beta$', 'σ': r'$\sigma$', 'π': r'$\pi$', 'χ': r'$\chi$', 'ε': r'$\varepsilon$', 'τ': r'$\tau$', 'Δ': r'$\Delta$', 'Σ': r'$\Sigma$',
     '₀': r'$_0$', '₁': r'$_1$', '₂': r'$_2$', '₃': r'$_3$', '₄': r'$_4$', '⁻': r'$^-$', '⁺': r'$^+$', '½': r'$\frac12$', '⅓': r'$\frac13$', '·': r'$\cdot$',
     '≤': r'$\le$', '≥': r'$\ge$', '×': r'$\times$', '−': r'$-$', '→': r'$\to$', '≈': r'$\approx$', '°': r'$^\circ$', 'Å': r'\AA{}', '…': r'\ldots{}',
     '“': '``', '”': "''", '’': "'", '—': '---', '–': '--', 'ᵢ': r'$_i$', 'ₖ': r'$_k$', '²': r'$^2$', '·': r'$\cdot$', 'Π': r'$\Pi$', '⇔': r'$\Leftrightarrow$', '✓': r'\checkmark'}


def tex(s):
    s = str(s)
    s = s.replace('\\', r'\textbackslash{}')
    for a, b in (('&', r'\&'), ('%', r'\%'), ('#', r'\#'), ('_', r'\_'), ('{', r'\{'), ('}', r'\}'), ('^', r'\^{}'), ('~', r'\~{}'), ('$', r'\$')):
        s = s.replace(a, b)
    s = s.replace('···', r'$\cdots$').replace('<', r'$<$').replace('>', r'$>$')
    for a, b in U.items(): s = s.replace(a, b)
    return s.replace('$$', '')


def node(js):
    return json.loads(subprocess.run(['node', '-e', js], capture_output=True, text=True).stdout)


def write(name, body):
    open(os.path.join(TAB, name + '.tex'), 'w').write(body); print('wrote', name)


def longtable(spec, head, rows, caption, label, size=r'\scriptsize'):
    out = [r'{%s' % size, r'\setlength{\tabcolsep}{3pt}', r'\begin{longtable}{%s}' % spec, r'\caption{%s}\label{%s}\\' % (caption, label), r'\toprule', head + r'\\', r'\midrule', r'\endfirsthead',
           r'\toprule', head + r'\\', r'\midrule', r'\endhead', r'\bottomrule', r'\endlastfoot']
    out += [' & '.join(r) + r'\\' for r in rows]
    out += [r'\end{longtable}}']
    return '\n'.join(out) + '\n'


def t_params():
    spec = node("const B=require('%s');const D=B.defaults();console.log(JSON.stringify(B.PARAM_SPEC.map(s=>({p:s.path,st:s.stage,l:s.label,u:s.unit,t:s.type,d:B.getPath(D,s.path),o:s.options||null}))))" % ENG)
    names = {'T0': 'Structure', 'T1': 'Geometry', 'T2': 'Dynamics', 'T3': 'QM model', 'T3b': 'NBO screen', 'T4': 'ALMO-EDA', 'T5': 'BXQI'}
    rows = []
    for s in spec:
        d = s['d']
        if isinstance(d, dict):
            if s['t'] == 'cutoffs': d = '; '.join('%s: %s' % (x, ', '.join('%s %s' % (y, v) for y, v in dd.items())) for x, dd in d.items())
            else: d = ', '.join('%s %s' % (k, v) for k, v in d.items())
        elif isinstance(d, bool): d = 'on' if d else 'off'
        elif s['o']: d = next((lab for k, lab in s['o'] if str(k) == str(d)), d)
        rows.append([names[s['st']], r'\texttt{%s}' % tex(s['p']), tex(s['l']) + (' (%s)' % tex(s['u']) if s['u'] else ''), r'\parbox[t]{5.0cm}{\raggedright %s}' % tex(d)])
    write('tab_params', longtable('llp{4.2cm}l', r'Tier & Parameter & Meaning (unit) & Default', rows, 'Every adjustable parameter of \\BX{} with its default value (%d parameters). All can be changed in the program, on each tier page, and on the \\emph{All parameters} page.' % len(spec), 'tab:params'))


def t_assumptions():
    A = node("const B=require('%s');const P=B.defaults();const o={};['T0','T1','T2','T3','T3b','T4','T5'].forEach(s=>o[s]=B.assumptions(s,P));o.QMMM=B.assumptions('T3',B.mergeParams(P,{qm:{env:'qmmm'}}));console.log(JSON.stringify(o))" % ENG)
    names = {'T0': 'Tier T0, structure input', 'T1': 'Tier T1, geometry', 'T2': 'Tier T2, persistence in dynamics', 'T3': 'Tier T3a, QM cluster model', 'QMMM': 'Tier T3a, QM/MM model (additional lines)',
             'T3b': 'Tier T3b, NBO screen', 'T4': 'Tier T4, ALMO-EDA', 'T5': 'Tier T5, ranking index (BXQI)'}
    out = []
    for k in ('T0', 'T1', 'T2', 'T3', 'QMMM', 'T3b', 'T4', 'T5'):
        items = A[k] if k != 'QMMM' else [a for a in A['QMMM'] if a not in A['T3']]
        out.append(r'\paragraph{%s.}' % names[k]); out.append(r'\begin{itemize}[nosep,leftmargin=*]')
        out += [r'\item %s' % tex(a) for a in items]; out.append(r'\end{itemize}')
    write('tab_assumptions', '\n'.join(out) + '\n')


def t_tests():
    r = subprocess.run(['node', os.path.join(APP, 'tests', 'engine.test.js')], capture_output=True, text=True, cwd=APP).stdout
    lines = [l for l in r.split('\n') if l.startswith('PASS ') or l.startswith('FAIL ')]
    ref = [l.strip()[6:] for l in r.split('\n') if l.strip().startswith('PASS  ')]
    rows = [[str(i + 1), tex(l[5:])] for i, l in enumerate(lines)]
    write('tab_tests', longtable('rp{14.5cm}', r'\# & Automated test (all passed)', rows, 'The %d automated engine tests (\\texttt{node tests/engine.test.js}). The last test runs the independent force-field comparison listed in Table~\\ref{tab:ffref}.' % len(rows), 'tab:tests'))
    rr = []
    for l in ref:
        m = re.match(r'(\S+)\s+(.+?): (\d+) residues, (\d+) heavy atoms compared, max \|Δq\| = (\S+) e(.*)', l)
        if m: rr.append([m.group(1), tex(m.group(2).replace('AMBER', 'ff14SB vs. AMBER (PDB2PQR)', 1) if m.group(2).startswith('AMBER') else m.group(2)), m.group(3), m.group(4), tex('%.2g' % float(m.group(5))), tex(m.group(6).strip(' ()'))])
    body = r'''\begin{table}[!htbp]\centering\scriptsize\setlength{\tabcolsep}{3pt}
\caption{Force-field charges of \BX{} compared with the PDB2PQR reference files. Hydrogen charges are added to their heavy atoms on both sides.}\label{tab:ffref}
\resizebox{\textwidth}{!}{\begin{tabular}{llrrlp{5.2cm}}\toprule PDB & Force field (BondX vs. reference) & Residues & Heavy atoms & max $|\Delta q|$ (e) & Note\\\midrule
''' + '\n'.join(' & '.join(x) + r'\\' for x in rr) + r'''
\bottomrule\end{tabular}}\end{table}
'''
    write('tab_ffref', body)
    return len(rows)


def t_hal59():
    d = pd.read_csv(os.path.join(RES, 't1_hal59.csv'))
    rows = []
    for _, r in d.iterrows():
        acc = 'ring' if r.acc_el == 'pi' else r.acc_el
        rows.append([tex(r['name']), r.subset, tex(r.halogen.title() if len(r.halogen) == 2 else r.halogen), tex(acc), '%.2f' % r.ref_kcal, '%.3f' % r.d, '%.1f' % r.theta1,
                     ('%.0f' % r.theta2) if str(r.theta2) != 'nan' and r.theta2 != '' else '--', 'yes' if r.covered else 'no', 'yes' if r.pass_T1 else 'no', '%.3f' % r.G,
                     tex('control' if r.role != 'halogen bond' else '')])
    write('tab_hal59_t1', longtable('llllrrrrllrl', r'Dimer & Set & X & Y & $E_\mathrm{ref}$ & $d$ (\AA) & $\theta_1$ & $\theta_2$ & Covered & T1 & $G$ & Role', rows,
                                     'HAL59 dimers. Reference binding energies: CCSD(T)/CBS (kcal\\,mol$^{-1}$, GMTKN55). Tier T1 of \\BX{} on the reference geometries. Covered: a default cut-off exists for the X/Y pair.', 'tab:hal59'))


def t_protein():
    d = pd.read_csv(os.path.join(RES, 'protein_t1.csv')); x = d[(d.includeF == False) & (d.status == 'XB')]
    rows = [[tex(r.pdb.upper().replace('_IDD594', '+IDD594').replace('_PROTONATED', ' (polar H)')), tex(r.donor), tex(r.X), tex(r.acceptor), tex(r.accAtom), tex(r.accClass), '%.3f' % r.d, '%.1f' % r.theta1,
             '%.1f' % r.theta2, tex(r.lewis), '%.3f' % r.G] for _, r in x.iterrows()]
    write('tab_protein_t1', longtable('lllllp{1.9cm}rrrlr', r'PDB & Donor & X & Acceptor & Y & Class & $d$ (\AA) & $\theta_1$ & $\theta_2$/$\alpha$ & Lewis & $G$', rows,
                                       'Halogen bonds found by tier T1 with default settings in the protein test set (all chains).', 'tab:protT1'))
    m = pd.read_csv(os.path.join(RES, 'protein_models.csv'))
    rows = []
    for _, r in m.iterrows():
        rows.append([tex(r.model.split('__')[0].upper().replace('_IDD594', '')), tex(r.ligand), tex(r.variant.replace('_', ' ')), str(int(r.n_atoms)), str(int(r.added_H)) if r.added_H == r.added_H else '',
                     str(int(r.link_H)) if r.link_H == r.link_H else '', '%d/%d' % (r.charge_donor, r.charge_acceptor) if r.charge_donor == r.charge_donor else '', str(int(r.electrons)) if r.electrons == r.electrons else '',
                     str(int(r.mm_n)) if r.mm_n == r.mm_n else '--', ('%.3f' % (r.mm_net + 0.0)) if r.mm_net == r.mm_net else '--', ('%.2f' % (round(r.esp_X_V, 2) + 0.0)) if r.esp_X_V == r.esp_X_V else '--',
                     ('%.3f' % r.field_AX_VperA) if r.field_AX_VperA == r.field_AX_VperA else '--'])
    write('tab_protein_models', longtable('lllrrrlrrrrr', r'System & Ligand & Variant & Atoms & +H & Link & $q_1/q_2$ & $N_e$ & MM $n$ & MM $Q$ & $\phi_X$ (V) & $E_{AX}$ (V/\AA)', rows,
                                           'All 89 protein QM models built by \\BX{}. Atoms: QM atoms. +H: added hydrogens. Link: link hydrogens. $q_1/q_2$: fragment charges. $N_e$: electrons. MM $n$ and MM $Q$: number and net charge of the MM point charges. $\\phi_X$ and $E_{AX}$: MM potential and field component at the halogen.', 'tab:models'))


def t_perf():
    d = pd.read_csv(os.path.join(RES, 'performance.csv'))
    t = d[d.test == 'T1 detect']; tl = t[t.system.str.contains(' x')]; ts = t[~t.system.str.contains(' x')]
    rows = [['T1 detection, %d test structures' % len(ts), '%d--%d' % (ts.atoms.min(), ts.atoms.max()), '%.0f--%.0f (median %.0f)' % (ts.ms.min(), ts.ms.max(), ts.ms.median())]]
    rows += [['T1 detection, %s' % (tex(r.system.upper().split(' X')[0]) + r' $\times$ ' + r.system.split(' x')[1]), '%d' % r.atoms, '%.0f' % r.ms] for _, r in tl.iterrows()]
    for _, r in d[d.test == 'T2 trajectory'].iterrows(): rows.append(['T2, 200 frames, %s' % tex(r.system.upper().replace('_IDD594', '')), '%d' % r.atoms, '%.0f (%.0f frames/s)' % (r.ms, r.frames_per_s)])
    for _, r in d[d.test.str.startswith('T3a')].iterrows(): rows.append(['%s model, %s' % (tex(r.test.split()[-1].replace('qmmm', 'QM/MM')), tex(r.system.upper().replace('_IDD594', ''))), '', '%.0f' % r.ms])
    body = r'''\begin{table}[!htbp]\centering\footnotesize
\caption{Run time of the \BX{} engine (Node.js 22, one core of an Intel Xeon at 2.3 GHz, median of five runs). Times include file parsing.}\label{tab:perf}
\begin{tabular}{lrl}\toprule Task & Atoms & Time (ms)\\\midrule
''' + '\n'.join(' & '.join(r) + r'\\' for r in rows) + r'''
\bottomrule\end{tabular}\end{table}
'''
    write('tab_performance', body)


def t_jobs():
    pkg = os.path.join(ROOT, 'qm_package') if os.path.isdir(os.path.join(ROOT, 'qm_package')) else os.path.join(ROOT, 'qm_inputs')  # qm_inputs/ in the GitHub copy
    j = pd.concat([pd.read_csv(os.path.join(pkg, f)) for f in ('jobs_small.csv', 'jobs_protein.csv')])
    c = j.groupby(['set', 'program']).size().unstack(fill_value=0)
    desc = {'A_HAL59': '59 HAL59 dimers', 'B_scan_distance': '12 dimers $\\times$ 8 distances', 'C_scan_angle': '6 dimers $\\times$ 7 angles', 'D_method_basis': '12 dimers $\\times$ 8 levels', 'E_eda_settings': '12 dimers $\\times$ 3 EDA settings', 'F_protein': '89 protein models'}
    P = ('gaussian', 'orca', 'qchem', 'qchem_eda', 'gaussian_cp', 'orca_cp')
    rows = [[tex(s), desc[s]] + [str(int(c.loc[s].get(p, 0))) for p in P] for s in c.index]
    rows.append([r'\textbf{Total}', ''] + [r'\textbf{%d}' % c[p].sum() for p in P])
    body = r'''\begin{table}[!htbp]\centering\footnotesize
\caption{QM jobs written by \BX{} for this work (%d input files). NBO jobs exist for Gaussian~16, ORCA, and Q-Chem. ALMO-EDA runs in Q-Chem. Each EDA job of sets A--C and F has a counterpoise interaction-energy partner in Gaussian (one job) and ORCA (three single points).}\label{tab:jobs}
\begin{tabular}{llrrrrrr}\toprule Set & Content & G16 NBO & ORCA NBO & Q-Chem NBO & Q-Chem EDA & G16 CP & ORCA CP\\\midrule
''' % len(j) + '\n'.join(' & '.join(r) + r'\\' for r in rows) + r'''
\bottomrule\end{tabular}\end{table}
'''
    write('tab_jobs', body)
    return len(j)


def t_scans():
    d = pd.read_csv(os.path.join(RES, 't1_scans.csv'))
    rows = []
    for b, g in d[d.scan == 'distance'].groupby('base', sort=False):
        g = g.sort_values('s'); rows.append([tex(b), 'distance'] + ['%.2f' % v for v in g.G])
    write('tab_scans_dist', longtable('ll' + 'r' * 8, r'Dimer & Scan & ' + ' & '.join('%.2f' % s for s in sorted(d[d.scan == 'distance'].s.unique())), rows,
                                       'Geometric score $G$ along the rigid distance scans ($d/d_\\mathrm{eq}$ in the header).', 'tab:scand'))
    rows = []
    for b, g in d[d.scan == 'angle'].groupby('base', sort=False):
        g = g.sort_values('theta1', ascending=False); rows.append([tex(b)] + ['%.1f / %.2f' % (t, v) for t, v in zip(g.theta1, g.G)])
    write('tab_scans_angle', longtable('l' + 'r' * 7, r'Dimer & \multicolumn{7}{c}{$\theta_1$ ($^\circ$) / $G$ for target angles 180, 170, \ldots, 120$^\circ$}', rows,
                                        'Geometric score along the rigid angle scans. The first number is the actual $\\theta_1$ after rotation.', 'tab:scana'))


def mnum(fmt, v):
    t = fmt % v
    return '$-$' + t[1:] if t.startswith('-') else t


def t_cp():
    """ORCA counterpoise tables (results/orca_cp_*.csv from 13_orca_cp_analysis.py)."""
    pth = os.path.join(RES, 'orca_cp_hal59.csv')
    if not os.path.exists(pth): return
    h = pd.read_csv(pth); sm = json.load(open(os.path.join(RES, 'orca_cp_summary.json')))
    rows = [[tex(r['name']), r['subset'], r['halogen'], ('$\\pi$' if r['acc'] == 'pi' else r['acc']), '%.2f' % r['ref_kcal'], '%.2f' % r['BE_kcal'], mnum('%+.2f', r['err_kcal']),
             ('%.2f' % r['G']) if r['covered'] else '--', ('XB' if r['role'] == 'halogen bond' else 'control')] for _, r in h.iterrows()]
    write('tab_cp_hal59', longtable('lllrrrrrl', r'Dimer & Set & X & Y & $E_\mathrm{ref}$ & BE(ORCA) & Error & $G$ & Role', rows,
          'Counterpoise-corrected binding energies of the HAL59 dimers (kcal\\,mol$^{-1}$, positive = bound). ORCA~5.0.1, $\\omega$B97X-V/def2-TZVPD, RIJCOSX def2/J, gas phase. $E_\\mathrm{ref}$: CCSD(T)/CBS. Error = BE(ORCA) $-$ $E_\\mathrm{ref}$. $G$: geometric score. --: no default cut-off (P and F acceptors).', 'tab:cphal'))
    lab = [('all', 'All 59 dimers'), ('XB51', 'XB51 subset'), ('X40', 'X40 subset'), ('Cl', 'Cl donors'), ('Br', 'Br donors'), ('I', 'I donors'),
           ('XB_dimers', 'Halogen bonds'), ('controls', 'Controls'), ('covered', 'Covered by T1')]
    rows = []
    for k, t in lab:
        st = sm['hal59_' + k]
        rows.append([t, str(st['n']), mnum('%+.2f', st['MSE']), '%.2f' % st['MAE'], '%.2f' % st['RMSD'], '%.2f' % st['MaxAE'], '%.3f' % st['pearson_r'], '%.3f' % st['spearman_rho']])
    head = ('\\begin{table}[!htbp]\\centering\\footnotesize\n'
            '\\caption{Error statistics of the ORCA counterpoise binding energies against CCSD(T)/CBS (kcal\\,mol$^{-1}$). MSE: mean signed error. '
            'MAE: mean absolute error. MaxAE: largest absolute error. The I row includes the control FI$\\cdots$FCCH. The two fluoroarene controls enter only the first and the last-but-one rows.}\\label{tab:cpstat}\n'
            '\\begin{tabular}{lrrrrrrr}\\toprule Group & $n$ & MSE & MAE & RMSD & MaxAE & $r$ & $\\rho$\\\\\\midrule\n')
    write('tab_cp_stats', head + '\n'.join(' & '.join(r) + '\\\\' for r in rows) + '\n\\bottomrule\\end{tabular}\\end{table}\n')
    sc = pd.read_csv(os.path.join(RES, 'orca_cp_scans.csv'))
    d = sc[sc.scan == 'distance']; ss = sorted(d.s.unique())
    rows = [[tex(b)] + ['%.2f' % v for v in g.sort_values('s').BE_kcal] for b, g in d.groupby('base')]
    write('tab_cp_dist', longtable('l' + 'r' * len(ss), r'Dimer & ' + ' & '.join('%.2f' % v for v in ss), rows,
          'ORCA counterpoise binding energies (kcal\\,mol$^{-1}$) along the rigid distance scans. The header gives $d/d_\\mathrm{eq}$. The column 1.00 equals the HAL59 value.', 'tab:cpdist'))
    a = sc[sc.scan == 'angle']; tt = sorted(a.theta.unique(), reverse=True)
    rows = [[tex(b)] + [mnum('%.2f', v) for v in g.sort_values('theta', ascending=False).BE_kcal] for b, g in a.groupby('base')]
    write('tab_cp_angle', longtable('l' + 'r' * len(tt), r'Dimer & ' + ' & '.join(r'%d$^\circ$' % v for v in tt), rows,
          'ORCA counterpoise binding energies (kcal\\,mol$^{-1}$) along the rigid angle scans (target $\\theta_1$ in the header). Negative values are repulsive.', 'tab:cpangle'))


def t_cp_protein():
    """ORCA counterpoise energies of the protein clusters (results/orca_cp_protein.csv)."""
    pth = os.path.join(RES, 'orca_cp_protein.csv')
    if not os.path.exists(pth): return
    p = pd.read_csv(pth); t = pd.read_csv(os.path.join(RES, 'orca_runtime.csv')); t = t[t.set == 'F_protein']
    t['model'] = t.input.str.split('/').str[-1].str.replace(r'_cp_(AB|A|B)\.inp$', '', regex=True)
    wall = t.groupby('model').seconds.sum() / 60; nbf = t.groupby('model').basis_functions.max()
    rows = []
    for _, r in p.iterrows():
        sysn = r['system'].split('_')[0].upper(); lig, ch = r['ligand'].split(':')[0], r['ligand'].split(':')[1]
        acc = '; '.join(x.split('···')[1].strip().split('@')[0].replace(':' + ch, '') for x in r['xbs'].split(';'))
        rows.append([sysn, '%s (%s)' % ({'LDT': 'IDD594'}.get(lig, lig), ch), tex(acc), str(int(r['n_xb'])), str(int(r['n_atoms'])), str(int(nbf[r['model']])), mnum('%+d', int(r['charge_donor'])).replace('+0', '0'),
                     mnum('%+d', int(r['charge_acceptor'])).replace('+0', '0'), mnum('%+.2f', r['Eint_kcal']), '%.0f' % wall[r['model']]])
    head = ('\\begin{table}[!htbp]\\centering\\footnotesize\n'
            '\\caption{ORCA counterpoise interaction energies of the protein clusters ($\\omega$B97X-V/def2-TZVPD, gas phase, DefGrid3, Sthresh $10^{-6}$). '
            'Fragment 1 is the ligand. Fragment 2 holds the acceptor environment. XB: halogen bonds in the model. XB acceptors: their acceptor residues. $q_1$, $q_2$: fragment charges. $N_\\mathrm{bf}$: basis functions. '
            '$E_\\mathrm{int}$ covers all contacts between the fragments (kcal\\,mol$^{-1}$). Time: wall time of the AB, A, and B jobs on 40 cores (min).}\\label{tab:cpprot}\n'
            '\\begin{tabular}{lllrrrrrrr}\\toprule System & Ligand & XB acceptors & XB & Atoms & $N_\\mathrm{bf}$ & $q_1$ & $q_2$ & $E_\\mathrm{int}$ & Time\\\\\\midrule\n')
    write('tab_cp_protein', head + '\n'.join(' & '.join(r) + '\\\\' for r in rows) + '\n\\bottomrule\\end{tabular}\\end{table}\n')


if __name__ == '__main__':
    t_params(); t_assumptions(); n = t_tests(); t_hal59(); t_protein(); t_perf(); nj = t_jobs(); t_scans(); t_cp(); t_cp_protein()
    print('tests', n, 'jobs', nj)


### `12_parameter_figure.py`

Figure: every geometric parameter of BondX tier T1 in one picture.

Panel a: atom acceptor  A-X...Y-R  (d, theta1, theta2, r_X, r_Y, R, sigma-hole)
Panel b: ring acceptor  A-X...centroid  (d, theta1, alpha, ring normal, pseudo-radius)
Panel c: the three score terms g_d, g_theta, g_2 and the score G (default values)

Usage:  python 12_parameter_figure.py [output_dir]
All default values are the BondX 2.0 defaults (engine.js, DEFAULTS.geom and DEFAULTS.score).

In [ ]:
%%writefile Scripts/12_parameter_figure.py
#!/usr/bin/env python3
"""Figure: every geometric parameter of BondX tier T1 in one picture.

Panel a: atom acceptor  A-X...Y-R  (d, theta1, theta2, r_X, r_Y, R, sigma-hole)
Panel b: ring acceptor  A-X...centroid  (d, theta1, alpha, ring normal, pseudo-radius)
Panel c: the three score terms g_d, g_theta, g_2 and the score G (default values)

Usage:  python 12_parameter_figure.py [output_dir]
All default values are the BondX 2.0 defaults (engine.js, DEFAULTS.geom and DEFAULTS.score).
"""
import sys, os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Wedge, Polygon, FancyArrowPatch, Arc, Ellipse

OUT = sys.argv[1] if len(sys.argv) > 1 else os.path.join(os.path.dirname(__file__), "..", "MS_Figures_V3")

# ---- BondX defaults (Br...O example) -------------------------------------------------------
TH1_MIN, TH1_MAX = 140.0, 180.0
TH2_OPT, TH2_W = 120.0, 40.0
TH2_WIN = (90.0, 150.0)          # optional theta2 filter (off by default)
ALPHA_MAX = 60.0
R_BEST = 0.85
R_BR, R_O, R_RING = 1.85, 1.52, 1.77
DCUT_BR_O = 3.68
DCUT_BR_AR = 3.87
RCUT = DCUT_BR_O / (R_BR + R_O)

# ---- palette ----------------------------------------------------------------------------------
BLUE, ORANGE, GREEN, AMBER, PINK, DGREEN, VIOLET, RED = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948")
INK, INK2, GRID = "#1f2328", "#57606a", "#d0d7de"
ATOM = {"C": "#8a8f96", "Br": "#a0522d", "O": "#e34948", "H": "#f4f4f4", "X": "#a0522d"}

from matplotlib import font_manager as _fm
_F = 'Arial' if any(f.name == 'Arial' for f in _fm.fontManager.ttflist) else 'Liberation Sans'
plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": [_F, "Arial", "Liberation Sans"], "font.size": 10, "axes.linewidth": 0.8,
    "mathtext.fontset": "custom", "mathtext.rm": _F, "mathtext.it": _F + ":italic", "mathtext.bf": _F + ":bold", "axes.edgecolor": INK2, "legend.frameon": False, "pdf.fonttype": 42,
})


def unit(deg):
    a = np.radians(deg)
    return np.array([np.cos(a), np.sin(a)])


def atom(ax, p, el, r=0.30, label=None, z=5, lc=INK, fs=10.3):
    ax.add_patch(Circle(p, r, fc=ATOM.get(el, "#999"), ec=INK, lw=0.8, zorder=z))
    # small highlight for a 3-D look
    ax.add_patch(Circle(p + np.array([-0.30, 0.32]) * r, 0.33 * r, fc="white", ec="none", alpha=0.45, zorder=z + 0.1))
    if label:
        ax.text(*p, label, ha="center", va="center", fontsize=fs, color=lc, weight="bold", zorder=z + 0.2)


def bond(ax, p, q, r1=0.30, r2=0.30, lw=3.0, color=INK2, dashed=False, z=3, double=False):
    v = q - p; L = np.linalg.norm(v); u = v / L
    a, b = p + u * r1 * 0.8, q - u * r2 * 0.8
    if dashed:
        ax.plot(*zip(a, b), ls=(0, (1.5, 2.2)), lw=1.6, color=color, zorder=z, solid_capstyle="round")
    elif double:
        n = np.array([-u[1], u[0]]) * 0.085
        for s in (1, -1):
            ax.plot(*zip(a + s * n, b + s * n), lw=lw * 0.55, color=color, zorder=z, solid_capstyle="round")
    else:
        ax.plot(*zip(a, b), lw=lw, color=color, zorder=z, solid_capstyle="round")


def angle_arc(ax, c, a1, a2, r, color, label, lr=None, lw=1.4, fs=11.5, ha="center"):
    lo, hi = min(a1, a2), max(a1, a2)
    if hi - lo > 180:
        lo, hi = hi, lo + 360
    ax.add_patch(Arc(c, 2 * r, 2 * r, theta1=lo, theta2=hi, color=color, lw=lw, zorder=6))
    mid = 0.5 * (lo + hi)
    lr = lr or r + 0.28
    ax.text(*(c + unit(mid) * lr), label, color=color, fontsize=fs, ha=ha, va="center", zorder=7, weight="bold")


def dim_line(ax, p, q, off, color, label, fs=10.9, lpos=0.5, loff=0.22):
    v = q - p; u = v / np.linalg.norm(v); n = np.array([-u[1], u[0]])
    a, b = p + n * off, q + n * off
    for e in (p, q):
        ax.plot(*zip(e + n * off * 0.25, e + n * off * 1.12), color=color, lw=0.7, zorder=4)
    ax.add_patch(FancyArrowPatch(a, b, arrowstyle="<|-|>", mutation_scale=8, color=color, lw=1.0, zorder=4))
    m = a + (b - a) * lpos + n * np.sign(off) * loff
    rot = np.degrees(np.arctan2(u[1], u[0]))
    ax.text(*m, label, color=color, fontsize=fs, ha="center", va="center", rotation=rot, zorder=7)


def sigma_hole(ax, X, direction, rX=0.42):
    """Blue positive cap along the C-X axis and a negative belt around X."""
    ax.add_patch(Wedge(X, rX + 0.20, direction - 32, direction + 32, width=0.20, fc=BLUE, ec="none", alpha=0.85, zorder=4))
    for c in (direction + 90, direction - 90):
        ax.add_patch(Wedge(X, rX + 0.16, c - 42, c + 42, width=0.14, fc=RED, ec="none", alpha=0.55, zorder=4))


def panel_label(ax, s):
    ax.text(0.0, 1.0, s, transform=ax.transAxes, fontsize=13.8, weight="bold", va="top", ha="left", color=INK)


# ==============================================================================================
fig = plt.figure(figsize=(7.2, 7.0))
gs = fig.add_gridspec(2, 2, height_ratios=[1.45, 1.0], width_ratios=[1.38, 1.0], hspace=0.20, wspace=0.04,
                      left=0.01, right=0.99, top=0.985, bottom=0.075)

# ---------------------------------------------------------------- (a) atom acceptor
ax = fig.add_subplot(gs[0, 0]); ax.set_aspect("equal"); ax.axis("off")
panel_label(ax, "a")
ax.text(0.06, 0.995, "Atom acceptor   A–X···Y–R", transform=ax.transAxes, fontsize=10.9, va="top", color=INK, weight="bold")

th1, th2, d = 166.0, 120.0, 3.05
X = np.array([0.0, 0.0])
dirXY = 0.0                                   # direction X -> Y
dirXA = dirXY + th1                            # theta1 = angle(A, X, Y)
A = X + unit(dirXA) * 1.95
Y = X + unit(dirXY) * d
dirYX = dirXY + 180.0
dirYR = dirYX - th2                            # theta2 = angle(X, Y, R); R above-right of Y
R = Y + unit(dirYR) * 1.25

# substituents
H1 = A + unit(dirXA + 70) * 1.0; H2 = A + unit(dirXA - 70) * 1.0; H3 = A + unit(dirXA + 180 + 180) * 0  # placeholder
ring_dir = dirXA
Cs = [A + unit(dirXA + 60) * 1.25, A + unit(dirXA - 60) * 1.25]
R2 = R + unit(dirYR + 55) * 1.05; R3 = R + unit(dirYR - 72) * 1.0

# van der Waals spheres (dashed) and overlap
ax.add_patch(Circle(X, R_BR, fc=ORANGE, alpha=0.06, ec=ORANGE, ls=(0, (3, 2)), lw=0.9, zorder=1))
ax.add_patch(Circle(Y, R_O, fc=RED, alpha=0.06, ec=RED, ls=(0, (3, 2)), lw=0.9, zorder=1))

for c in Cs:
    bond(ax, A, c); atom(ax, c, "C", r=0.22)
bond(ax, R, R2); bond(ax, R, R3); atom(ax, R2, "C", r=0.22); atom(ax, R3, "C", r=0.22)
bond(ax, A, X, r1=0.30, r2=0.42)
bond(ax, Y, R, r1=0.30, r2=0.30, double=True, lw=3.4)
bond(ax, X, Y, r1=0.42, r2=0.30, dashed=True, color=INK)
sigma_hole(ax, X, dirXY)
atom(ax, A, "C", label="A")
atom(ax, X, "X", r=0.42, label="X", lc="white", fs=12.6)
atom(ax, Y, "O", r=0.32, label="Y", lc="white")
atom(ax, R, "C", label="R")

# lone pair on Y pointing at X (sp2, 120 deg)
for s in (+1,):
    lp = Y + unit(dirYX) * 0.0
ax.text(*(X + unit(dirXY + 40) * 0.80), "σ-hole (+)", color=BLUE, fontsize=9.2, ha="left", va="center", weight="bold")
ax.text(*(X + unit(dirXY - 90) * 0.70), "belt (−)", color=RED, fontsize=9.2, ha="center", va="top")

# angles
angle_arc(ax, X, dirXA, dirXY, 0.80, BLUE, "θ₁", lr=1.12)
angle_arc(ax, Y, dirYR, dirYX, 0.62, GREEN, "θ₂", lr=0.95)

# distance
dim_line(ax, X, Y, -1.25, INK, "d (X···Y)", lpos=0.55)

# radii
ax.annotate("", xy=X + unit(-162) * R_BR, xytext=X, arrowprops=dict(arrowstyle="-|>", color=ORANGE, lw=1.0, mutation_scale=8), zorder=6)
ax.text(*(X + unit(-162) * R_BR * 0.62 + np.array([0.0, -0.30])), "r$_X$", color=ORANGE, fontsize=10.9, ha="center", va="center")
ax.annotate("", xy=Y + unit(-35) * R_O, xytext=Y, arrowprops=dict(arrowstyle="-|>", color=RED, lw=1.0, mutation_scale=8), zorder=6)
ax.text(*(Y + unit(-35) * R_O * 0.55 + np.array([0.12, -0.30])), "r$_Y$", color=RED, fontsize=10.9, ha="center", va="center")

# rule box
txt = (r"$d \leq d_\mathrm{cut}$(X,Y)" "\n"
       r"$140° \leq \theta_1 \leq 180°$" "\n"
       r"$\theta_2$: score optimum $120°$" "\n"
       r"   (filter $90$–$150°$, optional)" "\n"
       r"$R = d\,/\,(r_X + r_Y)$")
ax.text(0.985, 0.012, txt, transform=ax.transAxes, fontsize=9.2, ha="right", va="bottom", color=INK, linespacing=1.45,
        bbox=dict(boxstyle="round,pad=0.45", fc="#f6f8fa", ec=GRID, lw=0.8))
ax.set_xlim(-3.15, 4.95); ax.set_ylim(-3.75, 2.95)

# ---------------------------------------------------------------- (b) ring acceptor
ax = fig.add_subplot(gs[0, 1]); ax.set_aspect("equal"); ax.axis("off")
panel_label(ax, "b")
ax.text(0.09, 0.995, "π acceptor\nA–X···Ct", transform=ax.transAxes, fontsize=10.9, va="top", color=INK, weight="bold", linespacing=1.3)

Ct = np.array([0.0, -1.35]); sq = 0.36; rr = 1.40
ring = [Ct + np.array([rr * np.cos(np.radians(a)), sq * rr * np.sin(np.radians(a))]) for a in range(0, 360, 60)]
ax.add_patch(Ellipse(Ct, 2 * R_RING * 1.0, 2 * R_RING * sq, fc=VIOLET, alpha=0.06, ec=VIOLET, ls=(0, (3, 2)), lw=0.9, zorder=1))
ax.add_patch(Polygon(ring, closed=True, fc="#e9e6fb", ec=INK2, lw=1.6, zorder=2, alpha=0.95))
ax.add_patch(Ellipse(Ct, 1.25 * rr, 1.25 * rr * sq, fc="none", ec=VIOLET, lw=0.9, zorder=2.5))  # aromatic circle
for p in ring:
    ax.add_patch(Circle(p, 0.15, fc=ATOM["C"], ec=INK, lw=0.6, zorder=3))
ax.add_patch(Circle(Ct, 0.08, fc=INK, ec="none", zorder=5))
ax.text(*(Ct + np.array([0.18, 0.17])), "Ct", fontsize=9.8, color=INK, ha="left", va="center")

# normal
ax.add_patch(FancyArrowPatch(Ct, Ct + np.array([0, 3.55]), arrowstyle="-|>", mutation_scale=9, color=INK2, lw=1.0, ls="--", zorder=3))
ax.text(*(Ct + np.array([-0.12, 3.58])), "n", fontsize=11.5, color=INK2, ha="right", va="bottom", style="italic", weight="bold")

alpha = 22.0; dpi = 3.15
dirCX = 90 - alpha
Xp = Ct + unit(dirCX) * dpi
dirXA2 = (dirCX + 180 + (180 - 168)) % 360  # theta1 = 168 deg
dirXA2 = dirCX + 12.0                       # A roughly continues the Ct->X line
Ap = Xp + unit(dirXA2) * 1.70
bond(ax, Ap, Xp, r1=0.28, r2=0.40)
bond(ax, Xp, Ct, r1=0.40, r2=0.05, dashed=True, color=INK)
sigma_hole(ax, Xp, dirCX + 180)
atom(ax, Ap, "C", r=0.28, label="A")
atom(ax, Xp, "X", r=0.40, label="X", lc="white", fs=12.6)
angle_arc(ax, Ct, 90, dirCX, 1.05, VIOLET, "α", lr=1.32)
angle_arc(ax, Xp, dirXA2, dirCX + 180, 0.82, BLUE, "θ₁", lr=1.10)
dim_line(ax, Ct, Xp, -0.62, INK, "d (X···Ct)", lpos=0.42, loff=0.25)
tip = Ct + np.array([R_RING * np.cos(np.radians(235)), R_RING * sq * np.sin(np.radians(235))])
ax.annotate("", xy=tip, xytext=Ct, arrowprops=dict(arrowstyle="-|>", color=VIOLET, lw=1.0, mutation_scale=8), zorder=6)
ax.text(*(Ct + np.array([-1.05, -0.95])), "r$_\\mathrm{ring}$ = 1.77 Å", color=VIOLET, fontsize=9.2, ha="center")

txt = (r"$d \leq d_\mathrm{cut}$(X, ring)" "\n"
       r"$140° \leq \theta_1 \leq 180°$" "\n"
       r"$\alpha \leq 60°$ (to ring normal)")
ax.text(0.03, 0.012, txt, transform=ax.transAxes, fontsize=9.2, ha="left", va="bottom", color=INK, linespacing=1.45,
        bbox=dict(boxstyle="round,pad=0.45", fc="#f6f8fa", ec=GRID, lw=0.8))
ax.set_xlim(-2.45, 2.75); ax.set_ylim(-4.6, 3.85)

# ---------------------------------------------------------------- (c) score terms
sub = gs[1, :].subgridspec(1, 4, width_ratios=[1, 1, 1, 1.45], wspace=0.40)
axc = [fig.add_subplot(sub[0, i]) for i in range(3)]
for a in axc:                       # leave room for the y label of the first plot
    b = a.get_position(); a.set_position([b.x0 + 0.075 - 0.018 * axc.index(a), b.y0, b.width * 0.90, b.height])
axc[0].text(-0.40, 1.16, "c", transform=axc[0].transAxes, fontsize=13.8, weight="bold", va="top", color=INK)


def style(a, xl, yl=True):
    a.set_ylim(-0.04, 1.1); a.set_xlabel(xl, fontsize=9.8, color=INK, labelpad=2)
    a.tick_params(labelsize=7.5, colors=INK2, length=2.5)
    a.grid(color=GRID, lw=0.5); a.set_axisbelow(True)
    for s in ("top", "right"):
        a.spines[s].set_visible(False)
    if not yl:
        a.set_yticklabels([])


# g_d(R)
Rv = np.linspace(0.7, 1.2, 400)
gd = np.clip((RCUT - Rv) / (RCUT - R_BEST), 0, 1)
a = axc[0]; a.plot(Rv, gd, color=ORANGE, lw=2)
a.axvline(R_BEST, color=INK2, lw=0.7, ls=":"); a.axvline(RCUT, color=INK2, lw=0.7, ls=":")
a.text(R_BEST - 0.012, 0.42, "0.85", rotation=90, fontsize=8, color=INK2, ha="right", va="center")
a.text(RCUT + 0.012, 0.42, f"$R_\\mathrm{{cut}}$ = {RCUT:.2f}", rotation=90, fontsize=8, color=INK2, ha="left", va="center")
a.set_title("distance term  $g_d$", fontsize=9.8, color=ORANGE, pad=3, weight="bold")
style(a, "R = d / (r$_X$ + r$_Y$)   [Br···O]"); a.set_ylabel("term value", fontsize=9.8, color=INK)

# g_theta(theta1)
t = np.linspace(120, 180, 300)
gt = np.clip((t - TH1_MIN) / (180 - TH1_MIN), 0, 1)
a = axc[1]; a.plot(t, gt, color=BLUE, lw=2)
a.axvspan(TH1_MIN, TH1_MAX, color=BLUE, alpha=0.07, lw=0)
a.text(160, 1.04, "XB window", fontsize=8, color=BLUE, ha="center", va="bottom")
a.set_title("σ-hole angle term  $g_\\theta$", fontsize=9.8, color=BLUE, pad=3, weight="bold")
a.set_xticks([120, 140, 160, 180]); style(a, "θ₁ (°)", yl=False)

# g_2(theta2) and ring form
t2 = np.linspace(60, 180, 300)
g2 = np.exp(-((t2 - TH2_OPT) / TH2_W) ** 2)
a = axc[2]; a.plot(t2, g2, color=GREEN, lw=2, label="atom: θ₂")
al = np.linspace(0, 90, 200)
a2 = a.twiny()
a2.plot(al, np.clip(1 - al / ALPHA_MAX, 0, 1), color=VIOLET, lw=1.8, ls="--", label="ring: α")
a2.set_xlim(0, 90); a2.tick_params(labelsize=7.5, colors=VIOLET, length=2.5, pad=1)
a2.set_xticks([0, 30, 60, 90]); a2.set_xlabel("α (°)", fontsize=9.8, color=VIOLET, labelpad=2)
for s in ("right",):
    a2.spines[s].set_visible(False)
a2.spines["top"].set_color(VIOLET)
a.set_xticks([60, 90, 120, 150, 180]); style(a, "θ₂ (°)", yl=False)
a.xaxis.label.set_color(GREEN); a.tick_params(axis="x", colors=GREEN)
a.text(152, 0.86, "atom", fontsize=8.6, color=GREEN, ha="left", weight="bold")
a.text(50, 0.18, "1 − α/60°", fontsize=8, color=VIOLET, ha="left", transform=a.transData) if False else \
    a2.text(2, 0.30, "ring", fontsize=8.6, color=VIOLET, ha="left", weight="bold")
a.set_title("acceptor term  $g_2$", fontsize=9.8, color=GREEN, pad=15, weight="bold")

# formula / summary box
ab = fig.add_subplot(sub[0, 3]); ab.axis("off")
bb = ab.get_position(); ab.set_position([bb.x0 + 0.01, bb.y0, bb.width * 0.92, bb.height])
ab.add_patch(plt.Rectangle((-0.05, -0.04), 1.04, 1.14, transform=ab.transAxes, fc="#f6f8fa", ec=GRID, lw=0.8, clip_on=False))
formulas = (
    r"$g_d = \mathrm{clip}[(R_\mathrm{cut}-R)/(R_\mathrm{cut}-0.85)]$" "\n"
    r"$g_\theta = \mathrm{clip}[(\theta_1-140°)/40°]$" "\n"
    r"$g_2 = e^{-[(\theta_2-120°)/40°]^2}$  (atom)" "\n"
    r"$g_2 = 1-\alpha/60°$  (ring)" "\n"
    r"$G = [\,g_d\, g_\theta\, (\frac{1}{2}+\frac{1}{2}g_2)\,]^{1/3}$")
ab.text(-0.02, 1.07, formulas, transform=ab.transAxes, fontsize=8.1, va="top", ha="left", color=INK, linespacing=1.55)
table = ("Cut-off d$_\\mathrm{cut}$ (Å)\n"
         "    N     O     S     π\n"
         "Cl 3.77  3.61  3.95  3.80\n"
         "Br 3.84  3.68  4.02  3.87\n"
         "I  3.97  3.82  4.15  4.00")
ab.text(0.0, 0.47, table.split("\n")[0], transform=ab.transAxes, fontsize=8.5, va="bottom", ha="left", color=INK)
# the cut-off table in Arial: one text item per cell, right-aligned in fixed columns
_rows = [["", "N", "O", "S", "π"], ["Cl", "3.77", "3.61", "3.95", "3.80"], ["Br", "3.84", "3.68", "4.02", "3.87"], ["I", "3.97", "3.82", "4.15", "4.00"]]
_xs = [0.0, 0.22, 0.40, 0.58, 0.76]
for _i, _r in enumerate(_rows):
    for _j, _c in enumerate(_r):
        ab.text(_xs[_j] + (0.0 if _j == 0 else 0.12), 0.45 - _i * 0.085, _c, transform=ab.transAxes, fontsize=7.9, va="top",
                ha="left" if _j == 0 else "right", color=INK, fontweight="bold" if (_i == 0 or _j == 0) else "normal")
ab.text(0.96, -0.01, "all values editable", transform=ab.transAxes, fontsize=7.8, va="bottom", ha="right", color=INK2, style="italic")

os.makedirs(OUT, exist_ok=True)
for ext in ("pdf", "png"):
    fig.savefig(os.path.join(OUT, f"Fig3_T1_model.{ext}"), dpi=400)
print("wrote", os.path.join(OUT, "Fig3_T1_model.pdf/png"))


### `13_orca_cp_analysis.py`

13_orca_cp_analysis.py
Analysis of the ORCA 5.0.1 counterpoise jobs (wB97X-V/def2-TZVPD, RIJCOSX def2/J, gas phase).
E_int = E_AB - E_A(AB basis) - E_B(AB basis). Binding energy BE = -E_int (positive = bound).

Reads   results/qm_cp.csv (07_collect_results.py), data/hal59/hal59.csv, results/t1_hal59.csv,
        results/t1_scans.csv, results/protein_models.csv, results/qm_status.csv and the ORCA outputs.
Writes  results/orca_cp_hal59.csv, results/orca_cp_scans.csv, results/orca_cp_protein.csv,
        results/orca_runtime.csv, results/orca_cp_summary.json
Usage   python3 13_orca_cp_analysis.py ..

In [ ]:
%%writefile Scripts/13_orca_cp_analysis.py
#!/usr/bin/env python3
"""13_orca_cp_analysis.py
Analysis of the ORCA 5.0.1 counterpoise jobs (wB97X-V/def2-TZVPD, RIJCOSX def2/J, gas phase).
E_int = E_AB - E_A(AB basis) - E_B(AB basis). Binding energy BE = -E_int (positive = bound).

Reads   results/qm_cp.csv (07_collect_results.py), data/hal59/hal59.csv, results/t1_hal59.csv,
        results/t1_scans.csv, results/protein_models.csv, results/qm_status.csv and the ORCA outputs.
Writes  results/orca_cp_hal59.csv, results/orca_cp_scans.csv, results/orca_cp_protein.csv,
        results/orca_runtime.csv, results/orca_cp_summary.json
Usage   python3 13_orca_cp_analysis.py ..
"""
import os, sys, csv, json, re, glob
import numpy as np
from scipy import stats

ROOT = sys.argv[1] if len(sys.argv) > 1 else '..'
RES = os.path.join(ROOT, 'results'); PKG = os.path.join(ROOT, 'qm_package')
KJ_PER_KCAL = 4.184


def rd(p): return list(csv.DictReader(open(p)))


def wr(p, rows):
    with open(p, 'w', newline='') as f:
        w = csv.DictWriter(f, list(rows[0])); w.writeheader(); w.writerows(rows)


def errstats(x, y):
    """x = method, y = reference (kcal/mol)."""
    x, y = np.asarray(x, float), np.asarray(y, float); e = x - y
    return dict(n=int(len(e)), MSE=float(e.mean()), MAE=float(np.abs(e).mean()), RMSD=float(np.sqrt((e ** 2).mean())),
                MaxAE=float(np.abs(e).max()), pearson_r=float(stats.pearsonr(x, y)[0]) if len(e) > 2 else None,
                spearman_rho=float(stats.spearmanr(x, y)[0]) if len(e) > 2 else None,
                MRAE_pct=float(np.mean(np.abs(e) / np.abs(y)) * 100))


cp = {(r['set'], r['name']): float(r['Eint_CP_kJ']) / KJ_PER_KCAL for r in rd(os.path.join(RES, 'qm_cp.csv')) if r['program'] == 'orca'}
summary = {'level': 'wB97X-V/def2-TZVPD (ORCA 5.0.1, RIJCOSX def2/J, TightSCF, gas phase), counterpoise corrected', 'n_systems': len(cp)}

# ---------------- HAL59 ----------------
hal = {r['name']: r for r in rd(os.path.join(ROOT, 'data', 'hal59', 'hal59.csv'))}
t1 = {r['name']: r for r in rd(os.path.join(RES, 't1_hal59.csv'))}
H = []
for n, r in hal.items():
    e = cp[('A_HAL59', n)]
    H.append(dict(name=n, subset=r['subset'], role=r['role'], halogen=r['halogen'].capitalize(), acc=r['acc_el'] if r['acc_el'] else 'ring',
                  d=r['d'], theta1=r['theta1'], ref_kcal=float(r['ref_kcal']), Eint_kcal=round(e, 3), BE_kcal=round(-e, 3),
                  err_kcal=round(-e - float(r['ref_kcal']), 3), G=t1[n]['G'], covered=t1[n]['covered'], pass_T1=t1[n]['pass_T1']))
wr(os.path.join(RES, 'orca_cp_hal59.csv'), H)
be = [h['BE_kcal'] for h in H]; ref = [h['ref_kcal'] for h in H]
summary['hal59_all'] = errstats(be, ref)
for key, sel in (('XB51', lambda h: h['subset'] == 'XB51'), ('X40', lambda h: h['subset'] != 'XB51'),
                 ('Cl', lambda h: h['halogen'] == 'Cl'), ('Br', lambda h: h['halogen'] == 'Br'), ('I', lambda h: h['halogen'] == 'I'),
                 ('XB_dimers', lambda h: h['role'] == 'halogen bond'), ('controls', lambda h: h['role'] != 'halogen bond'),
                 ('covered', lambda h: h['covered'] == 'True')):
    s = [h for h in H if sel(h)]
    if len(s) >= 2: summary['hal59_' + key] = errstats([h['BE_kcal'] for h in s], [h['ref_kcal'] for h in s])
summary['hal59_subsets'] = {k: sum(1 for h in H if h['subset'] == k) for k in sorted({h['subset'] for h in H})}
summary['hal59_roles'] = {k: sum(1 for h in H if h['role'] == k) for k in sorted({h['role'] for h in H})}
big = sorted(H, key=lambda h: -abs(h['err_kcal']))[:5]
summary['hal59_largest_errors'] = [(h['name'], h['ref_kcal'], h['BE_kcal'], h['err_kcal']) for h in big]
cov = [h for h in H if h['covered'] == 'True']
summary['G_vs_BE_spearman_covered'] = float(stats.spearmanr([float(h['G']) for h in cov], [h['BE_kcal'] for h in cov])[0])
summary['G_vs_ref_spearman_covered'] = float(stats.spearmanr([float(h['G']) for h in cov], [h['ref_kcal'] for h in cov])[0])
summary['n_covered'] = len(cov)
summary['controls'] = [(h['name'], h['ref_kcal'], h['BE_kcal'], h['pass_T1']) for h in H if h['role'] != 'halogen bond']

# ---------------- scans ----------------
sc = rd(os.path.join(RES, 't1_scans.csv')); S = []
for r in sc:
    st = 'B_scan_distance' if r['scan'] == 'distance' else 'C_scan_angle'
    nm = r['file'][:-4]; e = cp[(st, nm)]
    S.append(dict(base=r['base'], scan=r['scan'], s=r['s'], theta=r['theta_target'], d=r['d'], theta1=r['theta1'], halogen=r['halogen'].capitalize(),
                  acc=r['acc_el'] or 'ring', Eint_kcal=round(e, 3), BE_kcal=round(-e, 3), G=r['G'], pass_T1=r['pass_T1']))
# relative values
for base in {x['base'] for x in S}:
    for scan, key, ref0 in (('distance', 's', '1.0'), ('angle', 'theta', '180')):
        grp = [x for x in S if x['base'] == base and x['scan'] == scan]
        if not grp: continue
        r0 = [x for x in grp if float(x[key]) == float(ref0)][0]['BE_kcal']
        for x in grp: x['BE_rel'] = round(x['BE_kcal'] / r0, 4) if r0 else None
wr(os.path.join(RES, 'orca_cp_scans.csv'), S)
# consistency: s = 1.00 equals the HAL59 geometry
dd = [abs(x['BE_kcal'] - [h['BE_kcal'] for h in H if h['name'] == x['base']][0]) for x in S if x['scan'] == 'distance' and float(x['s']) == 1.0]
summary['scan_s100_vs_hal59_maxdiff_kcal'] = max(dd)
dist = {}
for base in sorted({x['base'] for x in S if x['scan'] == 'distance'}):
    g = sorted([x for x in S if x['base'] == base and x['scan'] == 'distance'], key=lambda x: float(x['s']))
    smin = max(g, key=lambda x: x['BE_kcal'])
    dist[base] = dict(s_at_max_BE=float(smin['s']), BE_max=smin['BE_kcal'], BE_s100=[x['BE_kcal'] for x in g if float(x['s']) == 1.0][0],
                      BE_s090=[x['BE_kcal'] for x in g if float(x['s']) == 0.9][0], BE_s150=[x['BE_kcal'] for x in g if float(x['s']) == 1.5][0],
                      frac_s120=[x['BE_rel'] for x in g if float(x['s']) == 1.2][0], frac_s150=[x['BE_rel'] for x in g if float(x['s']) == 1.5][0],
                      G_zero_from_s=min([float(x['s']) for x in g if float(x['G']) == 0] or [None]))
summary['distance_scans'] = dist
ang = {}
for base in sorted({x['base'] for x in S if x['scan'] == 'angle'}):
    g = sorted([x for x in S if x['base'] == base and x['scan'] == 'angle'], key=lambda x: -float(x['theta']))
    ang[base] = dict(BE_180=g[0]['BE_kcal'], **{f'frac_{int(float(x["theta"]))}': x['BE_rel'] for x in g},
                     monotonic=all(g[i]['BE_kcal'] >= g[i + 1]['BE_kcal'] - 1e-6 for i in range(len(g) - 1)))
summary['angle_scans'] = ang
f140 = [v['frac_140'] for v in ang.values()]; f120 = [v['frac_120'] for v in ang.values()]
summary['angle_frac140_range'] = [min(f140), max(f140)]; summary['angle_frac120_range'] = [min(f120), max(f120)]
# Spearman between G and BE along each distance scan, for points inside the window (G > 0)
rhos = []
for base in dist:
    g = [x for x in S if x['base'] == base and x['scan'] == 'distance']
    rhos.append(float(stats.spearmanr([float(x['G']) for x in g], [x['BE_kcal'] for x in g])[0]))
summary['distance_scan_G_BE_spearman_range'] = [min(rhos), max(rhos)]

# ---------------- protein clusters ----------------
pm = {r['model']: r for r in rd(os.path.join(RES, 'protein_models.csv'))}
P = []
for (st, nm), e in sorted(cp.items()):
    if st != 'F_protein': continue
    m = pm[nm]
    P.append(dict(model=nm, system=m['system'], ligand=m['ligand'], n_atoms=int(m['n_atoms']), charge_donor=m['charge_donor'], charge_acceptor=m['charge_acceptor'],
                  n_xb=m['n_xb'], xbs=m['xbs'], Eint_kJ=round(e * KJ_PER_KCAL, 2), Eint_kcal=round(e, 2)))
if P: wr(os.path.join(RES, 'orca_cp_protein.csv'), P)
elif os.path.exists(os.path.join(RES, 'orca_cp_protein.csv')): os.remove(os.path.join(RES, 'orca_cp_protein.csv'))
summary['protein'] = {p['model']: p['Eint_kcal'] for p in P} if P else 'none'
if P:
    rep = {}
    for a_, b_ in (('1p5e__TBSA__cluster_pcm4', '1p5e__TBSC__cluster_pcm4'), ('2w0s__BVPA__cluster_pcm4', '2w0s__BVPB__cluster_pcm4'), ('4agl__P84A__cluster_pcm4', '4agl__P84B__cluster_pcm4')):
        d = {p['model']: p['Eint_kcal'] for p in P}
        if a_ in d and b_ in d: rep[a_.split('__')[0]] = round(abs(d[a_] - d[b_]), 2)
    summary['protein_replica_absdiff_kcal'] = rep

# ---------------- run times ----------------
T = []
for r in rd(os.path.join(RES, 'qm_status.csv')):
    if r['program'] != 'orca_cp' or r['status'] != 'ok': continue
    out = os.path.join(PKG, os.path.splitext(r['input'])[0] + '.out'); t = open(out, errors='replace').read()
    m = re.search(r'TOTAL RUN TIME:\s*(\d+) days (\d+) hours (\d+) minutes (\d+) seconds (\d+) msec', t)
    sec = int(m[1]) * 86400 + int(m[2]) * 3600 + int(m[3]) * 60 + int(m[4]) + int(m[5]) / 1000
    nb = re.search(r'Number of basis functions\s+\.\.\.\s+(\d+)', t)
    it = open(os.path.join(PKG, r['input'])).read(); mm = re.search(r'^\* xyz.*$', it, re.M)
    nat = len(it[mm.end():it.index('\n*', mm.end())].strip().split('\n'))
    T.append(dict(set=r['set'], input=r['input'], atoms=nat, basis_functions=int(nb[1]) if nb else '', seconds=round(sec, 1)))
wr(os.path.join(RES, 'orca_runtime.csv'), T)
tot = sum(x['seconds'] for x in T)
summary['runtime'] = dict(jobs=len(T), wall_hours=round(tot / 3600, 1), core_hours_40=round(tot / 3600 * 40, 0),
                          by_set={st: dict(jobs=sum(1 for x in T if x['set'] == st), wall_h=round(sum(x['seconds'] for x in T if x['set'] == st) / 3600, 2),
                                           median_s=float(np.median([x['seconds'] for x in T if x['set'] == st])),
                                           max_s=max(x['seconds'] for x in T if x['set'] == st)) for st in sorted({x['set'] for x in T})},
                          atoms_range=[min(x['atoms'] for x in T), max(x['atoms'] for x in T)],
                          nbf_range=[min(x['basis_functions'] for x in T), max(x['basis_functions'] for x in T)])
json.dump(summary, open(os.path.join(RES, 'orca_cp_summary.json'), 'w'), indent=1)
print(json.dumps(summary, indent=1))


### `14_ligand_check.py`

14_ligand_check.py
Checks the ligand protonation of BondX (one heavy-atom structure for hydrogens and charge) against the PDB
FORMUL record of every ligand residue in the protein test set (56 files).
Expected H = FORMUL H + (model charge - FORMUL charge) - (covalent links to other residues).
Writes results/ligand_check.csv and prints a summary.
Usage: python3 14_ligand_check.py ..

In [ ]:
%%writefile Scripts/14_ligand_check.py
#!/usr/bin/env python3
"""14_ligand_check.py
Checks the ligand protonation of BondX (one heavy-atom structure for hydrogens and charge) against the PDB
FORMUL record of every ligand residue in the protein test set (56 files).
Expected H = FORMUL H + (model charge - FORMUL charge) - (covalent links to other residues).
Writes results/ligand_check.csv and prints a summary.
Usage: python3 14_ligand_check.py ..
"""
import os, sys, csv, json
from bxutil import cli

ROOT = sys.argv[1] if len(sys.argv) > 1 else '..'
PDB = os.path.join(ROOT, 'data', 'protein'); RES = os.path.join(ROOT, 'results')
rows = []
for fn in sorted(f for f in os.listdir(PDB) if f.endswith('.pdb')):
    d = cli('ligand-check', os.path.join(PDB, fn))
    for r in d['ligands']: rows.append(dict(pdb=fn[:-4], **r))
with open(os.path.join(RES, 'ligand_check.csv'), 'w', newline='') as f:
    w = csv.DictWriter(f, list(rows[0])); w.writeheader(); w.writerows(rows)
metal = [r for r in rows if r['metal']]; nof = [r for r in rows if not r['metal'] and not r['formul']]
inc = [r for r in rows if r['formul'] and not r['complete']]; comp = [r for r in rows if r['formul'] and r['complete']]
ok = [r for r in comp if r['hOK']]
summary = dict(files=len({r['pdb'] for r in rows}), ligand_residues=len(rows), metal=len(metal), no_formul=len(nof), incomplete=len(inc),
               complete=len(comp), H_matches=len(ok), mismatches=sorted({f"{r['pdb']} {r['resName']}" for r in comp if not r['hOK']}))
json.dump(summary, open(os.path.join(RES, 'ligand_check_summary.json'), 'w'), indent=1)
print(json.dumps(summary, indent=1))


### `15_hiyd_analysis.py`

15_hiyd_analysis.py
BondX re-analysis of the hIYD I-Tyr unbinding data (Karmakar & Mishra, J. Chem. Inf. Model. 2026, 66, 11191).
No new MD or QM calculation is done. Inputs (hIYD_for_BondX):
  US/C1..C3/traj/*.pdb,*.dcd   stripped umbrella-sampling windows (36 per pathway, water removed)
  US/C1..C3/halo.out           published frame-wise structural screen (first 10 ns = 500 frames per window)
  US/C1..C3/qm-halo/*-nbo.log  published QM-cluster NBO outputs (Gaussian 16), one representative pose per residue
  US/C1..C3/qm-halo/role-halo.dat  window, frame, d, theta and the published NBO value per residue
  collect.log                  frame stride of every stripped trajectory
Steps
  1. node hiyd_bondx.js  -> results/hiyd/bondx_frames.jsonl (T1 contacts of every frame)
  2. node hiyd_nbo.js    -> results/hiyd/nbo_clusters.json (BondX NBO parser and gate on the 56 cluster outputs)
  3. exact check: BondX T1 contacts vs the published screen on the frames present in both
  4. occupancy per window and per residue (all 25 ns); acceptor classes; strict BondX filter
  5. NBO re-analysis with a second, independent parser (sections 'within unit' / 'from unit')
Usage: python3 15_hiyd_analysis.py <project root> <hIYD_for_BondX root>
Pathway C1, C2, C3 = Pathway A, B, C of the paper. Residues: MD numbers 1-440 (1-220 chain A = PDB 71-290; 221-440 chain B = PDB 71*-290*), 441/443 FRA/FRB, 442/444 MIA/MIB.
Chain A PDB = n + 70, chain B PDB = n - 150 (*).
The published screen (MDTraj on the parm7) writes 0-based residue indices: ARG93 there is Amber residue 94 = R164.

In [ ]:
%%writefile Scripts/15_hiyd_analysis.py
#!/usr/bin/env python3
"""15_hiyd_analysis.py
BondX re-analysis of the hIYD I-Tyr unbinding data (Karmakar & Mishra, J. Chem. Inf. Model. 2026, 66, 11191).
No new MD or QM calculation is done. Inputs (hIYD_for_BondX):
  US/C1..C3/traj/*.pdb,*.dcd   stripped umbrella-sampling windows (36 per pathway, water removed)
  US/C1..C3/halo.out           published frame-wise structural screen (first 10 ns = 500 frames per window)
  US/C1..C3/qm-halo/*-nbo.log  published QM-cluster NBO outputs (Gaussian 16), one representative pose per residue
  US/C1..C3/qm-halo/role-halo.dat  window, frame, d, theta and the published NBO value per residue
  collect.log                  frame stride of every stripped trajectory
Steps
  1. node hiyd_bondx.js  -> results/hiyd/bondx_frames.jsonl (T1 contacts of every frame)
  2. node hiyd_nbo.js    -> results/hiyd/nbo_clusters.json (BondX NBO parser and gate on the 56 cluster outputs)
  3. exact check: BondX T1 contacts vs the published screen on the frames present in both
  4. occupancy per window and per residue (all 25 ns); acceptor classes; strict BondX filter
  5. NBO re-analysis with a second, independent parser (sections 'within unit' / 'from unit')
Usage: python3 15_hiyd_analysis.py <project root> <hIYD_for_BondX root>
Pathway C1, C2, C3 = Pathway A, B, C of the paper. Residues: MD numbers 1-440 (1-220 chain A = PDB 71-290; 221-440 chain B = PDB 71*-290*), 441/443 FRA/FRB, 442/444 MIA/MIB.
Chain A PDB = n + 70, chain B PDB = n - 150 (*).
The published screen (MDTraj on the parm7) writes 0-based residue indices: ARG93 there is Amber residue 94 = R164.
"""
import os, sys, re, json, csv, math, subprocess
from collections import defaultdict, Counter
import numpy as np

ROOT = sys.argv[1] if len(sys.argv) > 1 else '..'
DATA = sys.argv[2] if len(sys.argv) > 2 else 'hIYD_for_BondX'  # folder with the hIYD trajectories (give it as the second argument)
HERE = os.path.dirname(os.path.abspath(__file__))
OUT = os.path.join(ROOT, 'results', 'hiyd'); os.makedirs(OUT, exist_ok=True)
PATH = {'C1': 'A', 'C2': 'B', 'C3': 'C'}
WINS = [round(3.8 + 0.2 * i, 1) for i in range(26)] + [9.8 + i for i in range(10)]
WINS = [round(w, 1) for w in WINS]
CUT = {'N': 3.97, 'O': 3.82, 'S': 4.15, 'AR': 4.00}

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode: sys.exit(r.stderr)
    return r.stdout.strip()

def pdbnum(res, n):
    if res in ('FRA', 'FRB'): return 'FMN'
    if res in ('MIA', 'MIB'): return 'I-Tyr(2)'
    return f'{n + 70}' if n <= 220 else f'{n - 150}*'

ONE = {'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C', 'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'HIE': 'H', 'HID': 'H', 'HIP': 'H',
       'ILE': 'I', 'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P', 'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'}
def label(res, n):
    if res in ('FRA', 'FRB'): return 'FMN'
    return ONE.get(res, res) + pdbnum(res, n)
def amber_name(res):  # halo.out writes HIE/HID as HIS
    return 'HIS' if res in ('HIE', 'HID', 'HIP') else res

# BondX Lewis classes (engine LEWIS table); everything else 'strong'
LEWIS_WEAK = {('_BB', 'N'), ('ARG', 'NE'), ('ARG', 'NH1'), ('ARG', 'NH2'), ('LYS', 'NZ'), ('TRP', 'NE1'), ('ASN', 'ND2'), ('GLN', 'NE2'), ('PRO', 'N')}
def acc_class(res, atom, typ):
    if typ in ('aromatic', 'pi'): return 'aromatic ring'
    if res in ('FRA', 'FRB', 'MIB'): return 'FMN ' + atom[0]
    if atom == 'N': return 'backbone N-H'
    if atom in ('O', 'OXT'): return 'backbone C=O'
    if (res, atom) in LEWIS_WEAK or (res == 'HIS' and False): return 'side-chain N-H (no lone pair)'
    if atom[0] == 'N': return 'side-chain N (lone pair)'
    if atom[0] == 'S': return 'thioether S'
    return 'side-chain O'
def lewis(res, atom, typ):
    if typ in ('aromatic', 'pi'): return 'strong'
    if atom == 'N' and res not in ('FRA', 'FRB', 'MIB'): return 'weak'
    return 'weak' if (amber_name(res), atom) in LEWIS_WEAK else 'strong'

# ------------------------------------------------------------------ 1-2. BondX runs
if not (len(sys.argv) > 3 and sys.argv[3] == '--no-node'):
    print(run(['node', os.path.join(HERE, 'hiyd_bondx.js'), DATA, os.path.join(OUT, 'bondx_frames.jsonl')]))
    print(run(['node', os.path.join(HERE, 'hiyd_nbo.js'), DATA, os.path.join(OUT, 'nbo_clusters.json')]))

# ------------------------------------------------------------------ strides from collect.log
stride, nraw = {}, {}
for l in open(os.path.join(DATA, 'collect.log')):
    m = re.match(r'\s+(C\d)_US-([\d.]+): (\d+) frames, stride (\d+)', l)
    if m: stride[(m.group(1), float(m.group(2)))] = int(m.group(4)); nraw[(m.group(1), float(m.group(2)))] = int(m.group(3))
assert len(stride) == 108, len(stride)

# ------------------------------------------------------------------ BondX frames
frames = defaultdict(dict)   # (C,w) -> k -> hits
nfr = {}
for l in open(os.path.join(OUT, 'bondx_frames.jsonl')):
    r = json.loads(l); frames[(r['C'], r['w'])][r['k']] = r['hits']; nfr[(r['C'], r['w'])] = r['n']
assert len(frames) == 108
for key, n in nfr.items():
    assert n == math.ceil(nraw[key] / stride[key]) and len(frames[key]) == n, (key, n)

# ------------------------------------------------------------------ published screen (halo.out)
pub = defaultdict(lambda: defaultdict(list))  # (C,w) -> frame -> [(res,num,type,atom,d_A,theta)]
LINE = re.compile(r'^\s+([A-Z0-9+\-]{2,4}?)(\d+) \((N|O|S|aromatic), ([^)]*)\), Distance: ([\d.]+) nm, Donor Pos: \[[^\]]*\], Angle: ([\d.]+)')
for C in PATH:
    w = fr = None
    txt = open(os.path.join(DATA, 'US', C, 'halo.out')).read()
    txt = re.sub(r'\[[^\]]*\]', '[]', txt, flags=re.S)   # numpy may wrap the position over two lines
    for l in txt.split('\n'):
        m = re.match(r'For trajectory ([\d.]+)', l)
        if m: w = float(m.group(1)); fr = None; continue
        m = re.match(r'Frame (\d+):', l)
        if m: fr = int(m.group(1)); continue
        m = LINE.match(l)
        if m:
            res, num, typ, atom = m.group(1), int(m.group(2)), m.group(3), m.group(4).strip()
            pub[(C, w)][fr].append((res, num + 1, typ, 'ring' if typ == 'aromatic' else atom, float(m.group(5)) * 10, float(m.group(6))))
npub_lines = sum(len(v) for d in pub.values() for v in d.values())

# ------------------------------------------------------------------ 3. exact frame check
def bx_key(h):
    res = amber_name(h['res'])
    return (res, h['seq'], 'aromatic' if h['type'] == 'pi' else h['el'], 'ring' if h['type'] == 'pi' else h['atom'])
chk = Counter(); dmax = tmax = 0.0; missing, extra = [], []
for (C, w), byk in frames.items():
    s = stride[(C, w)]
    for k, hits in byk.items():
        f = k * s
        if f > 499: continue
        chk['frames'] += 1
        P = [p for p in pub[(C, w)].get(f, []) if p[0] != 'HOH']
        chk['water_lines'] += len(pub[(C, w)].get(f, [])) - len(P)
        Bk = defaultdict(list)
        for h in hits: Bk[bx_key(h)].append(h)
        used = set()
        for p in P:
            key = (p[0], p[1], p[2], p[3]); chk['pub_lines'] += 1
            cand = [h for h in Bk.get(key, []) if id(h) not in used]
            if not cand: missing.append((C, w, f, p)); continue
            h = min(cand, key=lambda h: abs(h['d'] - p[4]) + abs(h['t1'] - p[5]) / 100)
            used.add(id(h)); chk['matched'] += 1
            dmax = max(dmax, abs(h['d'] - p[4])); tmax = max(tmax, abs(h['t1'] - p[5]))
        for key, hs in Bk.items():
            for h in hs:
                if id(h) not in used: extra.append((C, w, f, key, h['d'], h['t1']))
chk.update({'missing': len(missing), 'extra': len(extra)})
check = {'framesCompared': chk['frames'], 'publishedLinesNoWater': chk['pub_lines'], 'waterLinesSkipped': chk['water_lines'], 'matched': chk['matched'],
         'missing': chk['missing'], 'extra': chk['extra'], 'maxAbsDeltaD_A': round(dmax, 4), 'maxAbsDeltaTheta_deg': round(tmax, 3),
         'missingList': [list(map(str, m)) for m in missing[:20]], 'extraList': [list(map(str, e)) for e in extra[:20]]}
print('frame check', json.dumps({k: v for k, v in check.items() if not k.endswith('List')}))

# ------------------------------------------------------------------ 4. occupancy per window
def strict(h):  # BondX strict: Lewis-base acceptor; ring contacts within 60 deg of the ring normal
    lw = lewis(h['res'], h['atom'], h['type'])
    if h['type'] == 'pi': return h['t2'] is not None and h['t2'] <= 60
    return lw == 'strong'
rows = []
for C in PATH:
    for w in WINS:
        byk = frames[(C, w)]; n = len(byk); s = stride[(C, w)]
        anyA = sum(1 for hs in byk.values() if hs); anyS = sum(1 for hs in byk.values() if any(strict(h) for h in hs))
        k10 = [k for k in byk if k * s <= 499]
        any10 = sum(1 for k in k10 if byk[k])
        P = pub[(C, w)]
        pubAll = sum(1 for f, v in P.items() if v); pubNoW = sum(1 for f, v in P.items() if any(p[0] != 'HOH' for p in v))
        rows.append({'pathway': PATH[C], 'C': C, 'xi': w, 'stride': s, 'frames_25ns': n, 'bondx_T1_frac': anyA / n, 'bondx_strict_frac': anyS / n,
                     'frames_10ns_in_dcd': len(k10), 'bondx_T1_frac_10ns': any10 / len(k10), 'pub_frac_10ns': pubAll / 500, 'pub_frac_10ns_nowater': pubNoW / 500})
with open(os.path.join(OUT, 'hiyd_window_occupancy.csv'), 'w', newline='') as fh:
    wr = csv.DictWriter(fh, fieldnames=list(rows[0])); wr.writeheader(); wr.writerows(rows)

# ------------------------------------------------------------------ acceptor classes and residues (all frames)
cls_count = {C: Counter() for C in PATH}; cls_strict = {C: Counter() for C in PATH}
res_frames = {C: defaultdict(set) for C in PATH}; res_frames_strict = {C: defaultdict(set) for C in PATH}
nframes_C = {C: 0 for C in PATH}; xi_by_res = {C: defaultdict(set) for C in PATH}
G_all, G_strict = [], []
for (C, w), byk in frames.items():
    nframes_C[C] += len(byk)
    for k, hs in byk.items():
        for h in hs:
            typ = 'pi' if h['type'] == 'pi' else h['el']
            c = acc_class(amber_name(h['res']), h['atom'], typ); cls_count[C][c] += 1
            lab = label(h['res'], h['seq']); res_frames[C][lab].add((w, k)); xi_by_res[C][lab].add(w); G_all.append(h['G'])
            if strict(h): cls_strict[C][c] += 1; res_frames_strict[C][lab].add((w, k)); G_strict.append(h['G'])
classes = sorted({c for C in PATH for c in cls_count[C]}, key=lambda c: (-sum(cls_count[C][c] for C in PATH), c))
with open(os.path.join(OUT, 'hiyd_acceptor_classes.csv'), 'w', newline='') as fh:
    wr = csv.writer(fh); wr.writerow(['class'] + [f'contacts_{PATH[C]}' for C in PATH] + ['total', 'kept_by_strict_filter'])
    for c in classes: wr.writerow([c] + [cls_count[C][c] for C in PATH] + [sum(cls_count[C][c] for C in PATH), sum(cls_strict[C][c] for C in PATH)])
res_rows = []
for C in PATH:
    for lab, fs in res_frames[C].items():
        res_rows.append({'pathway': PATH[C], 'residue': lab, 'frames_T1': len(fs), 'frames_strict': len(res_frames_strict[C].get(lab, ())),
                         'windows': len({w for w, k in fs}), 'xi_min': min(w for w, k in fs), 'xi_max': max(w for w, k in fs)})
res_rows.sort(key=lambda r: (r['pathway'], -r['frames_T1']))
with open(os.path.join(OUT, 'hiyd_residue_contacts.csv'), 'w', newline='') as fh:
    wr = csv.DictWriter(fh, fieldnames=list(res_rows[0])); wr.writeheader(); wr.writerows(res_rows)

# ------------------------------------------------------------------ 5. NBO re-analysis
NBL = re.compile(r'^\s*(\d+|\*+)\.\s*(BD\*?|LP\*?|CR|RY\*?|LV|3C\*?)\s*\(\s*(\d+)\s*\)\s*([A-Z][a-z]?)\s*(\d+)(?:\s*-\s*([A-Z][a-z]?)\s*(\d+))?(?:\s*-\s*([A-Z][a-z]?)\s*(\d+))?\s*/\s*(\d+|\*+)\.\s*(BD\*?|LP\*?|CR|RY\*?|LV|3C\*?)\s*\(\s*(\d+)\s*\)\s*([A-Z][a-z]?)\s*(\d+)(?:\s*-\s*([A-Z][a-z]?)\s*(\d+))?(?:\s*-\s*([A-Z][a-z]?)\s*(\d+))?\s+(\d+\.\d+)\s+(-?\d+\.\d+)\s+(-?\d+\.\d+)')
def py_nbo(fn, iI, iC):
    """Independent parser: LP -> sigma*(C-I) entries with their NBO section ('within unit u' or 'from unit u to unit v')."""
    L = open(fn).read().split('\n')
    st = max(i for i, l in enumerate(L) if 'SECOND ORDER PERTURBATION THEORY ANALYSIS' in l.upper())
    sec, out = None, []
    for l in L[st:]:
        if 'NATURAL BOND ORBITALS (Summary)' in l: break
        m = re.match(r'\s*within unit\s+(\d+)', l)
        if m: sec = ('within', int(m.group(1))); continue
        m = re.match(r'\s*from unit\s+(\d+) to unit\s+(\d+)', l)
        if m: sec = ('from', int(m.group(1)), int(m.group(2))); continue
        m = NBL.match(l)
        if not m: continue
        g = m.groups()
        acc_atoms = {int(g[13])} | ({int(g[15])} if g[15] else set())
        if g[10] == 'BD*' and acc_atoms == {iI, iC} and g[1] == 'LP':
            out.append({'donorEl': g[3], 'donor': int(g[4]), 'e2': float(g[18]), 'sec': sec})
    return out

clusters = json.load(open(os.path.join(OUT, 'nbo_clusters.json')))
role = {}
for C in PATH:
    for l in open(os.path.join(DATA, 'US', C, 'qm-halo', 'role-halo.dat')):
        p = l.split()
        if p: role[(C, p[0])] = {'xi': float(p[1]), 'frame': int(p[2]), 'd_nm': float(p[3]), 'theta': float(p[4]), 'E_pub': float(p[7])}
nbo_rows = []
for r in clusters:
    C, resid = r['C'], r['residue']; ro = role[(C, resid)]
    m = re.match(r'([A-Z]{3})(\d+)', resid); rn, num = m.group(1), int(m.group(2)) + 1   # 0-based index in the published files
    # structural acceptor from the published screen: same window, frame, residue, distance and angle
    hits = [p for p in pub[(C, ro['xi'])].get(ro['frame'], []) if p[0] == rn and p[1] == num and abs(p[4] - ro['d_nm'] * 10) < 0.006 and abs(p[5] - ro['theta']) < 0.006]
    assert len(hits) >= 1, (C, resid, ro)
    p = hits[0]; typ = p[2]; atom = p[3]
    # cluster T1 pair(s)
    pairs = [(g, ga) for g, ga in zip(r['geo'], r['gate']) if g['dok'] and g['t1ok']]
    want = 'AR' if typ == 'aromatic' else typ
    pm = [(g, ga) for g, ga in pairs if g['el'] == want and abs(g['d'] - p[4]) < 0.02 and abs(g['t1'] - p[5]) < 0.2]
    assert len(pm) == 1, (C, resid, typ, p, [(g['el'], g['d'], g['t1']) for g, ga in pairs])
    g, ga = pm[0]
    # second parser
    ent = py_nbo(os.path.join(DATA, 'US', C, 'qm-halo', resid + '-nbo.log'), r['I'], r['Cx'])
    unitI = None
    allsec = [e['sec'] for e in ent]
    # the unit that holds the iodine is the target of every 'from' section and the unit of 'within' entries of fragment-1 donors
    intra_py = sum(e['e2'] for e in ent if e['sec'][0] == 'within')
    inter_py = sum(e['e2'] for e in ent if e['sec'][0] == 'from')
    fr1 = {x['donor']: x['frag'] for x in r['lpRows']}
    node_intra = sum(x['e2'] for x in r['lpRows'] if x['frag'] == 1); node_inter = sum(x['e2'] for x in r['lpRows'] if x['frag'] == 2)
    assert abs(intra_py - node_intra) < 1e-9 and abs(inter_py - node_inter) < 1e-9, (C, resid, intra_py, node_intra, inter_py, node_inter)
    ons = [e for e in ent if e['donorEl'] in ('O', 'N', 'S')]
    emax = max([e['e2'] for e in ons], default=0.0)
    src = sorted({('intra' if e['sec'][0] == 'within' else 'inter') for e in ons if e['e2'] == emax}) or ['none']
    lw = lewis(rn, atom, typ)
    nbo_rows.append({'pathway': PATH[C], 'residue_amber': resid, 'residue': label(rn, num), 'xi': ro['xi'], 'frame': ro['frame'],
                     'acceptor_atom': atom, 'acceptor_type': typ, 'acceptor_class': acc_class(rn, atom, typ), 'lewis': lw,
                     'd_A': round(p[4], 2), 'theta1': round(p[5], 2), 'E_pub': ro['E_pub'], 'E_max_line': round(emax, 2), 'E_pub_source': '+'.join(src),
                     'E2_intra_LP': round(node_intra, 2), 'E2_intra_phenolateO': round(sum(x['e2'] for x in r['lpRows'] if x['frag'] == 1 and x['donor'][0] == 'O'), 2), 'E2_inter_LP': round(node_inter, 2), 'E2_bondx_pair': round(ga['E2_core'], 2), 'E2_RY_pair': round(ga.get('E2_RY') or 0, 2), 'hb_LPI_to_YH': round(ga.get('hbY') or 0, 2), 'LP_on_acceptor': ga.get('lpOnY'), 'verdict_v3': ga.get('verdict'),
                     'bondx_verified': ga['E2_core'] >= 0.5, 'bondx_verified_lewis': ga['E2_core'] >= 0.5 and lw == 'strong', 'nAtoms': r['nAtoms']})
with open(os.path.join(OUT, 'hiyd_nbo_reanalysis.csv'), 'w', newline='') as fh:
    wr = csv.DictWriter(fh, fieldnames=list(nbo_rows[0])); wr.writeheader(); wr.writerows(nbo_rows)

# ------------------------------------------------------------------ summary
def q(x): return round(float(x), 3)
S = {'windows': len(frames), 'frames_total': sum(len(v) for v in frames.values()), 'frames_per_window_min': min(nfr.values()), 'frames_per_window_max': max(nfr.values()),
     'stride_values': sorted(set(stride.values())), 'pub_lines_total': npub_lines, 'check': {k: v for k, v in check.items() if not k.endswith('List')}}
for C in PATH:
    R = [r for r in rows if r['C'] == C]
    S[f'occ_{PATH[C]}'] = {'windows_with_T1': sum(1 for r in R if r['bondx_T1_frac'] > 0), 'windows_with_strict': sum(1 for r in R if r['bondx_strict_frac'] > 0),
        'mean_T1_xi_le_8_8': q(np.mean([r['bondx_T1_frac'] for r in R if r['xi'] <= 8.8])), 'mean_T1_xi_ge_9_8': q(np.mean([r['bondx_T1_frac'] for r in R if r['xi'] >= 9.8])),
        'mean_strict_xi_le_8_8': q(np.mean([r['bondx_strict_frac'] for r in R if r['xi'] <= 8.8])), 'mean_strict_xi_ge_9_8': q(np.mean([r['bondx_strict_frac'] for r in R if r['xi'] >= 9.8])),
        'max_T1': q(max(r['bondx_T1_frac'] for r in R)), 'xi_max_T1': max(R, key=lambda r: r['bondx_T1_frac'])['xi'],
        'max_strict': q(max(r['bondx_strict_frac'] for r in R)), 'xi_max_strict': max(R, key=lambda r: r['bondx_strict_frac'])['xi'],
        'first_xi_strict_ge_0_1': next((r['xi'] for r in R if r['bondx_strict_frac'] >= 0.1), None),
        'contacts_T1': sum(cls_count[C].values()), 'contacts_strict': sum(cls_strict[C].values()), 'frames': nframes_C[C],
        'residues_T1': len(res_frames[C]), 'residues_strict': len(res_frames_strict[C]),
        'top_strict': sorted(((lab, len(v)) for lab, v in res_frames_strict[C].items()), key=lambda x: -x[1])[:6]}
tot = Counter(); tots = Counter()
for C in PATH: tot.update(cls_count[C]); tots.update(cls_strict[C])
S['classes'] = {c: [tot[c], tots[c]] for c in classes}
S['contacts_total'] = sum(tot.values()); S['contacts_strict_total'] = sum(tots.values())
S['G_median_all'] = q(np.median(G_all)); S['G_median_strict'] = q(np.median(G_strict))
weakN = sum(tot[c] for c in classes if c in ('backbone N-H', 'side-chain N-H (no lone pair)'))
S['frac_contacts_no_lone_pair'] = q(weakN / sum(tot.values()))
N = len(nbo_rows)
S['nbo'] = {'clusters': N, 'per_pathway': dict(Counter(r['pathway'] for r in nbo_rows)), 'all_terminated': all(r['terminated'] for r in clusters),
    'pub_equals_max_line': sum(1 for r in nbo_rows if abs(r['E_pub'] - r['E_max_line']) < 0.006),
    'pub_source_intra': sum(1 for r in nbo_rows if r['E_pub_source'] == 'intra'), 'pub_source_inter': sum(1 for r in nbo_rows if r['E_pub_source'] == 'inter'),
    'pub_source_none': sum(1 for r in nbo_rows if r['E_pub_source'] == 'none'),
    'pub_ge_0_5': sum(1 for r in nbo_rows if r['E_pub'] >= 0.5),
    'bondx_verified': sum(1 for r in nbo_rows if r['bondx_verified']), 'bondx_verified_lewis': sum(1 for r in nbo_rows if r['bondx_verified_lewis']),
    'acceptor_weak': sum(1 for r in nbo_rows if r['lewis'] == 'weak'), 'acceptor_backbone_NH': sum(1 for r in nbo_rows if r['acceptor_class'] == 'backbone N-H'),
    'verified_weak': sum(1 for r in nbo_rows if r['bondx_verified'] and r['lewis'] == 'weak'),
    'intra_min': q(min(r['E2_intra_LP'] for r in nbo_rows if r['E2_intra_LP'] > 0)), 'intra_max': q(max(r['E2_intra_LP'] for r in nbo_rows)),
    'pair_E2_range_verified_lewis': [q(min(r['E2_bondx_pair'] for r in nbo_rows if r['bondx_verified_lewis'])), q(max(r['E2_bondx_pair'] for r in nbo_rows if r['bondx_verified_lewis']))],
    'verified_lewis_list': [(r['pathway'], r['residue'], r['acceptor_atom'], r['E2_bondx_pair']) for r in nbo_rows if r['bondx_verified_lewis']],
    'verified_weak_list': [(r['pathway'], r['residue'], r['acceptor_atom'], r['E2_bondx_pair']) for r in nbo_rows if r['bondx_verified'] and r['lewis'] == 'weak']}
io = [max(x['e2'] for x in r['lpRows'] if x['frag'] == 1 and x['donor'][0] == 'O') for r in clusters if any(x['frag'] == 1 and x['donor'][0] == 'O' for x in r['lpRows'])]
orth = all(x['bondsToC'] == 2 for r in clusters for x in r['lpRows'] if x['frag'] == 1 and x['donor'][0] == 'O')
S['nbo'].update({'intra_O_clusters': len(io), 'intra_O_min': q(min(io)), 'intra_O_max': q(max(io)), 'intra_O_median': q(np.median(io)), 'intra_O_all_ortho_phenolate': orth,
                 'intra_O_ge_0_5': sum(1 for v in io if v >= 0.5)})
# ortho phenolate O term: donor in fragment 1 that is an O bonded (2 bonds away) to the C-I carbon is the dominant intramolecular term
json.dump(S, open(os.path.join(OUT, 'hiyd_summary.json'), 'w'), indent=1)
json.dump(check, open(os.path.join(OUT, 'hiyd_frame_check.json'), 'w'), indent=1)
print(json.dumps(S, indent=1))


### `16_hiyd_figures_tables.py`

16_hiyd_figures_tables.py
Figures and LaTeX tables of the hIYD case study from the CSV/JSON files of 15_hiyd_analysis.py.
  Figures_V2_MS/Fig10_hIYD.pdf          occupancy along the pathways, acceptor classes, NBO re-analysis
  Figures_V2_ESI/FigS7_hiyd_residues.pdf  residues with T1 contacts per pathway (all vs strict)
  Figures_V2_ESI/FigS8_hiyd_nbo.pdf       per-cluster NBO terms (intramolecular vs the T1 pair)
  tables/tab_hiyd_classes.tex, tab_hiyd_nbo.tex, tab_hiyd_residues.tex
Usage: python3 16_hiyd_figures_tables.py ..

In [ ]:
%%writefile Scripts/16_hiyd_figures_tables.py
#!/usr/bin/env python3
"""16_hiyd_figures_tables.py
Figures and LaTeX tables of the hIYD case study from the CSV/JSON files of 15_hiyd_analysis.py.
  Figures_V2_MS/Fig10_hIYD.pdf          occupancy along the pathways, acceptor classes, NBO re-analysis
  Figures_V2_ESI/FigS7_hiyd_residues.pdf  residues with T1 contacts per pathway (all vs strict)
  Figures_V2_ESI/FigS8_hiyd_nbo.pdf       per-cluster NBO terms (intramolecular vs the T1 pair)
  tables/tab_hiyd_classes.tex, tab_hiyd_nbo.tex, tab_hiyd_residues.tex
Usage: python3 16_hiyd_figures_tables.py ..
"""
import os, sys, json
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

ROOT = sys.argv[1] if len(sys.argv) > 1 else '..'
RES = os.path.join(ROOT, 'results', 'hiyd'); MS = os.path.join(ROOT, 'Figures_V2_MS'); ESI = os.path.join(ROOT, 'Figures_V2_ESI'); TAB = os.path.join(ROOT, 'tables')
C = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948']
INK, INK2, GRID, MUTE = '#0b0b0b', '#52514e', '#e4e3df', '#9a9993'
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8.5, 'axes.edgecolor': MUTE, 'axes.linewidth': 0.7, 'axes.labelcolor': INK,
                     'xtick.color': INK2, 'ytick.color': INK2, 'axes.titlesize': 9.5, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
                     'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6, 'axes.axisbelow': True, 'legend.frameon': False,
                     'axes.spines.top': False, 'axes.spines.right': False, 'savefig.dpi': 300, 'pdf.fonttype': 42})
def save(fig, folder, name):
    fig.savefig(os.path.join(folder, name + '.pdf'), bbox_inches='tight'); fig.savefig(os.path.join(folder, name + '.png'), bbox_inches='tight', dpi=220); plt.close(fig); print('wrote', name)

import bxstyle
from bxstyle import save
bxstyle.apply()


occ = pd.read_csv(os.path.join(RES, 'hiyd_window_occupancy.csv'))
cls = pd.read_csv(os.path.join(RES, 'hiyd_acceptor_classes.csv'))
nbo = pd.read_csv(os.path.join(RES, 'hiyd_nbo_reanalysis.csv'))
resc = pd.read_csv(os.path.join(RES, 'hiyd_residue_contacts.csv'))
S = json.load(open(os.path.join(RES, 'hiyd_summary.json')))

# ------------------------------------------------------------------ Fig 10
fig = plt.figure(figsize=(7.2, 6.6))
gs = fig.add_gridspec(2, 3, height_ratios=[1, 1.25], hspace=0.48, wspace=0.18, top=0.88)
for j, P in enumerate('ABC'):
    ax = fig.add_subplot(gs[0, j]); d = occ[occ.pathway == P].sort_values('xi')
    ax.axvspan(9.3, 19.3, color='#f3f2ee', zorder=0, lw=0)
    ax.plot(d.xi, 100 * d.pub_frac_10ns, ls='none', marker='o', ms=3.4, mfc='white', mec=MUTE, mew=0.8, label='Published screen (first 10 ns)')
    ax.plot(d.xi, 100 * d.bondx_T1_frac, color=C[0], lw=1.5, marker='o', ms=2.6, mec='white', mew=0.4, label='BondX T1 (25 ns)')
    ax.plot(d.xi, 100 * d.bondx_strict_frac, color=C[1], lw=1.5, marker='s', ms=2.4, mec='white', mew=0.4, label='BondX T1, Lewis bases only')
    ax.set_title(f'({"abc"[j]}) Pathway {P}'); ax.set_xlabel(r'$\xi$ (Å)'); ax.set_xlim(3.3, 19.3); ax.set_ylim(-2, 52)
    ax.set_xticks([4, 8, 12, 16])
    if j == 0: ax.set_ylabel('Frames with contact (%)')
    else: ax.set_yticklabels([])
    if j == 0: fig.legend(*ax.get_legend_handles_labels(), loc='upper center', ncol=2, bbox_to_anchor=(0.5, 1.0), handlelength=1.8, columnspacing=1.2)
# (d) acceptor classes
gs2 = gs[1, :].subgridspec(1, 2, width_ratios=[1.45, 1], wspace=0.55)
ax = fig.add_subplot(gs2[0])
order = cls.sort_values('total', ascending=True)
names = {'backbone C=O': 'Backbone C=O', 'backbone N-H': 'Backbone N–H', 'side-chain O': 'Side-chain O', 'aromatic ring': 'Aromatic ring (π)',
         'side-chain N-H (no lone pair)': 'Side-chain N–H', 'side-chain N (lone pair)': 'His N', 'FMN N': 'FMN N5', 'thioether S': 'Met SD'}
y = np.arange(len(order))
ax.barh(y, order.kept_by_strict_filter, color=C[0], height=0.62, label='Lewis base (kept)')
rem = order.total - order.kept_by_strict_filter; m = rem > 0
ax.barh(y[m], rem[m], left=order.kept_by_strict_filter[m], color='white', edgecolor=C[7], hatch='////', lw=0.8, height=0.62, label='No lone pair (flagged weak)')
for yi, t in zip(y, order.total): ax.text(t + 3, yi, str(t), va='center', color=INK)
ax.set_yticks(y); ax.set_yticklabels([names.get(c, c) for c in order['class']]); ax.grid(axis='y', visible=False)
ax.set_xlabel('T1 contacts (11 361 frames)'); ax.set_xlim(0, max(order.total) * 1.18)
ax.set_title('(d) Acceptor classes'); ax.legend(loc='lower right')
# (e) NBO
ax = fig.add_subplot(gs2[1])
col = {'intra': C[1], 'inter': C[0], 'none': MUTE}
for _, r in nbo.iterrows():
    filled = r.lewis == 'strong'
    ax.plot(r.E_pub, r.E2_bondx_pair, ls='none', marker='o' if filled else 'D', ms=4.2 if filled else 3.6,
            mfc=col[r.E_pub_source] if filled else 'white', mec=col[r.E_pub_source], mew=0.9, alpha=0.95)
ax.axhline(0.5, color=INK2, ls=':', lw=0.8); ax.axvline(0.5, color=INK2, ls=':', lw=0.8)
ax.plot([0, 2.8], [0, 2.8], color=GRID, lw=0.8, zorder=0)
ax.set_xlim(-0.1, 2.8); ax.set_ylim(-0.1, 2.8)
ax.set_xlabel('Published NBO (kcal/mol)'); ax.set_ylabel(r'BondX $\Sigma E^{(2)}$ (kcal/mol)')
ax.set_title('(e) NBO, 56 clusters'); ax.set_xticks([0, 1, 2]); ax.set_yticks([0, 1, 2])
h = [Line2D([], [], ls='none', marker='o', mfc=C[1], mec=C[1], ms=4, label='Largest: intra'),
     Line2D([], [], ls='none', marker='o', mfc=C[0], mec=C[0], ms=4, label='Largest: inter'),
     Line2D([], [], ls='none', marker='D', mfc='white', mec=INK2, ms=3.6, label='No lone pair')]
ax.legend(handles=h, loc='upper left', bbox_to_anchor=(0.17, 1.0), handletextpad=0.2, borderaxespad=0.2, fontsize=9.2)
save(fig, MS, 'Fig10_hIYD')

# ------------------------------------------------------------------ ESI Fig S10: residues
fig, axs = plt.subplots(1, 3, figsize=(7.2, 4.6))
for ax, P in zip(axs, 'ABC'):
    d = resc[resc.pathway == P].sort_values('frames_T1', ascending=True)
    y = np.arange(len(d))
    ax.barh(y, d.frames_strict, color=C[0], height=0.65)
    rem = (d.frames_T1 - d.frames_strict).values; m = rem > 0
    ax.barh(y[m], rem[m], left=d.frames_strict.values[m], color='white', edgecolor=C[7], hatch='////', lw=0.7, height=0.65)
    ax.set_yticks(y); ax.set_yticklabels(d.residue, fontsize=9.6); ax.grid(axis='y', visible=False); ax.set_ylim(-0.6, len(d) - 0.4)
    ax.set_title(f'({"abc"["ABC".index(P)]}) Pathway {P}'); ax.set_xlabel('Frames'); ax.set_xlim(0, 55)
fig.tight_layout(rect=(0, 0, 1, 0.93)); fig.legend(handles=[Patch(color=C[0], label='Lewis-base acceptor'), Patch(facecolor='white', edgecolor=C[7], hatch='////', label='No lone pair')], loc='upper center', ncol=2, bbox_to_anchor=(0.5, 1.0)); save(fig, ESI, 'FigS7_hiyd_residues')

# ------------------------------------------------------------------ ESI Fig S11: per-cluster terms
d = nbo.copy(); d['lab'] = d.pathway + ': ' + d.residue + ' ' + d.acceptor_atom
fig, axs = plt.subplots(1, 3, figsize=(7.2, 6.0), sharex=True)
for ax, P in zip(axs, 'ABC'):
    e = d[d.pathway == P].reset_index(drop=True); y = np.arange(len(e))[::-1]
    ax.barh(y + 0.19, e.E2_intra_phenolateO, height=0.36, color=C[1], label='LP(phenolate O of I-Tyr) → σ*(C–I)')
    ax.barh(y - 0.19, e.E2_bondx_pair, height=0.36, color=[C[0] if l == 'strong' else 'white' for l in e.lewis], edgecolor=C[0], lw=0.8, label='T1 pair (BondX gate)')
    ax.plot(e.E_pub, y, ls='none', marker='|', ms=7, mew=1.4, color=INK, label='Published value')
    ax.axvline(0.5, color=INK2, ls=':', lw=0.8)
    ax.set_yticks(y); ax.set_yticklabels(e.residue + ' ' + e.acceptor_atom, fontsize=9.2); ax.grid(axis='y', visible=False)
    ax.set_title(f'({"abc"["ABC".index(P)]}) Pathway {P}'); ax.set_xlabel(r'$E^{(2)}$ (kcal/mol)')
hh = [Patch(color=C[1], label='LP(phenolate O of I-Tyr) → σ*(C–I)'), Patch(color=C[0], label='T1 pair, Lewis-base acceptor'),
      Patch(facecolor='white', edgecolor=C[0], label='T1 pair, acceptor without lone pair'), Line2D([], [], ls='none', marker='|', ms=7, mew=1.4, color=INK, label='Published value')]
fig.tight_layout(rect=(0, 0, 1, 0.915)); fig.legend(handles=hh, loc='upper center', ncol=2, bbox_to_anchor=(0.5, 1.0), columnspacing=1.0)
save(fig, ESI, 'FigS8_hiyd_nbo')

# ------------------------------------------------------------------ tables
def tex(s): return str(s).replace('*', '$^{*}$').replace('_', r'\_')
with open(os.path.join(TAB, 'tab_hiyd_classes.tex'), 'w') as f:
    f.write('\\begin{table}[!htbp]\n\\caption{Acceptor classes of the T1 contacts of the iodine of I-Tyr in all 108 umbrella-sampling windows (11\\,361 frames, 25~ns per window). Kept: contacts with a Lewis-base acceptor (strict filter of \\BX{}).}\n\\label{tab:hiydclasses}\n\\centering\\small\n\\begin{tabular}{lrrrrr}\n\\toprule\nAcceptor class & A & B & C & Total & Kept \\\\\n\\midrule\n')
    for _, r in cls.iterrows():
        f.write(f"{names.get(r['class'], r['class']).replace('π', '$\\pi$')} & {r.contacts_A} & {r.contacts_B} & {r.contacts_C} & {r.total} & {r.kept_by_strict_filter} \\\\\n")
    f.write(f"\\midrule\nAll & {cls.contacts_A.sum()} & {cls.contacts_B.sum()} & {cls.contacts_C.sum()} & {cls.total.sum()} & {cls.kept_by_strict_filter.sum()} \\\\\n\\bottomrule\n\\end{{tabular}}\n\\end{{table}}\n")
with open(os.path.join(TAB, 'tab_hiyd_nbo.tex'), 'w') as f:
    f.write('{\\footnotesize\\setlength{\\tabcolsep}{3pt}\n\\begin{longtable}{llrlrrrrrrl}\n')
    f.write('\\caption{NBO re-analysis of the 56 published QM-cluster outputs (B3LYP-D3, CPCM $\\varepsilon=4$). $\\xi$: window (\\AA). $d$ (\\AA) and $\\theta_1$ (deg): T1 geometry of the published pose. Pub.: published value. Intra: LP of the phenolate O of I-Tyr $\\to\\sigma^*$(C--I). Pair: gate value of \\BX{}, LP of the T1 acceptor (or ring bonds) $\\to\\sigma^*$(C--I). Source: donor of the largest single term. L: Lewis class of the acceptor in the protein (s strong, w no lone pair). Energies in kcal\\,mol$^{-1}$.}\\label{tab:hiydnbo}\\\\\n')
    hdr = 'Path & Residue & $\\xi$ & Acc. & $d$ & $\\theta_1$ & Pub. & Intra & Pair & L & Source \\\\\n'
    f.write('\\toprule\n' + hdr + '\\midrule\n\\endfirsthead\n\\toprule\n' + hdr + '\\midrule\n\\endhead\n\\bottomrule\n\\endfoot\n')
    for _, r in nbo.iterrows():
        acc = 'ring' if r.acceptor_type == 'aromatic' else r.acceptor_atom
        pair = f'\\textbf{{{r.E2_bondx_pair:.2f}}}' if r.bondx_verified_lewis else f'{r.E2_bondx_pair:.2f}'
        f.write(f"{r.pathway} & {tex(r.residue)} & {r.xi:.1f} & {acc} & {r.d_A:.2f} & {r.theta1:.1f} & {r.E_pub:.2f} & {r.E2_intra_phenolateO:.2f} & {pair} & {'s' if r.lewis == 'strong' else 'w'} & {r.E_pub_source} \\\\\n")
    f.write('\\end{longtable}}\n')
with open(os.path.join(TAB, 'tab_hiyd_residues.tex'), 'w') as f:
    f.write('\\begin{table}[!htbp]\n\\caption{Residues with T1 contacts to the iodine of I-Tyr. Frames: number of frames with a contact (all T1 contacts / Lewis-base contacts only). $\\xi$: range of windows with a contact (\\AA). $^{*}$Chain B.}\n\\label{tab:hiydres}\n\\centering\\footnotesize\\setlength{\\tabcolsep}{3pt}\n')
    f.write('\\begin{tabular}{llrrr|llrrr}\n\\toprule\nPath & Residue & All & Lewis & $\\xi$ & Path & Residue & All & Lewis & $\\xi$ \\\\\n\\midrule\n')
    rows = [r for _, r in resc.iterrows()]; half = (len(rows) + 1) // 2
    for i in range(half):
        a = rows[i]; s = f"{a.pathway} & {tex(a.residue)} & {a.frames_T1} & {a.frames_strict} & {a.xi_min:.1f}--{a.xi_max:.1f}"
        if i + half < len(rows):
            b = rows[i + half]; s += f" & {b.pathway} & {tex(b.residue)} & {b.frames_T1} & {b.frames_strict} & {b.xi_min:.1f}--{b.xi_max:.1f}"
        else: s += ' & & & & &'
        f.write(s + ' \\\\\n')
    f.write('\\bottomrule\n\\end{tabular}\n\\end{table}\n')
print('tables written')


### `17_qm_nbo_eda_analysis.py`

Counts the Rydberg terms; its NBO decisions are superseded by script 18, and it still supplies the EDA and counterpoise columns.

17_qm_nbo_eda_analysis.py
Analysis of the Gaussian 16 NBO, Gaussian counterpoise and Q-Chem ALMO-EDA outputs (read by 07_collect_results.py).
Inputs : results/qm_nbo.csv, qm_eda.csv, qm_cp.csv, qm_status.csv, t1_hal59.csv, t1_scans.csv, orca_cp_hal59.csv, orca_cp_scans.csv,
         orca_cp_protein.csv, protein_t1.csv
Outputs: results/qm_hal59.csv, qm_scans.csv, qm_method_basis.csv, qm_eda_settings.csv, qm_protein.csv, qm_summary.json
         Figures_V2_MS/Fig8_NBO_benchmark, Fig9_protein_QM; Figures_V2_ESI/FigS5_angle_scan_QM, FigS6_method_basis
         tables/tab_qm_hal59.tex, tab_qm_eda.tex, tab_qm_prot.tex
Usage: python3 17_qm_nbo_eda_analysis.py ..
Energies: NBO E(2) in kcal/mol; EDA terms in kJ/mol in the tables (1 kcal = 4.184 kJ).

In [ ]:
%%writefile Scripts/17_qm_nbo_eda_analysis.py
#!/usr/bin/env python3
"""17_qm_nbo_eda_analysis.py
Analysis of the Gaussian 16 NBO, Gaussian counterpoise and Q-Chem ALMO-EDA outputs (read by 07_collect_results.py).
Inputs : results/qm_nbo.csv, qm_eda.csv, qm_cp.csv, qm_status.csv, t1_hal59.csv, t1_scans.csv, orca_cp_hal59.csv, orca_cp_scans.csv,
         orca_cp_protein.csv, protein_t1.csv
Outputs: results/qm_hal59.csv, qm_scans.csv, qm_method_basis.csv, qm_eda_settings.csv, qm_protein.csv, qm_summary.json
         Figures_V2_MS/Fig8_NBO_benchmark, Fig9_protein_QM; Figures_V2_ESI/FigS5_angle_scan_QM, FigS6_method_basis
         tables/tab_qm_hal59.tex, tab_qm_eda.tex, tab_qm_prot.tex
Usage: python3 17_qm_nbo_eda_analysis.py ..
Energies: NBO E(2) in kcal/mol; EDA terms in kJ/mol in the tables (1 kcal = 4.184 kJ).
"""
import os, sys, json, math
import numpy as np, pandas as pd
from scipy.stats import spearmanr, pearsonr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from bxutil import cli

ROOT = sys.argv[1] if len(sys.argv) > 1 else '..'
RES = os.path.join(ROOT, 'results'); MS = os.path.join(ROOT, 'Figures_V2_MS'); ESI = os.path.join(ROOT, 'Figures_V2_ESI'); TAB = os.path.join(ROOT, 'tables')
KJ = 4.184; GATE = 0.5
C = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948']
HALC = {'CL': C[2], 'BR': C[1], 'I': C[0], 'F': '#8a8a86'}
ACCM = {'N': 'o', 'O': 's', 'S': 'D', 'pi': '^', 'P': 'v', 'F': '<'}
INK, INK2, GRID, MUTE = '#0b0b0b', '#52514e', '#e4e3df', '#9a9993'
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8.5, 'axes.edgecolor': MUTE, 'axes.linewidth': 0.7, 'axes.labelcolor': INK,
                     'xtick.color': INK2, 'ytick.color': INK2, 'axes.titlesize': 9.5, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
                     'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6, 'axes.axisbelow': True, 'legend.frameon': False,
                     'axes.spines.top': False, 'axes.spines.right': False, 'savefig.dpi': 300, 'pdf.fonttype': 42})
def save(fig, folder, name):
    fig.savefig(os.path.join(folder, name + '.pdf'), bbox_inches='tight'); fig.savefig(os.path.join(folder, name + '.png'), bbox_inches='tight', dpi=220); plt.close(fig); print('wrote', name)
import bxstyle
from bxstyle import save
bxstyle.apply()


def r3(x): return None if x is None or (isinstance(x, float) and math.isnan(x)) else round(float(x), 3)
def mnum(fmt, v):
    s = fmt % v
    return s.replace('-', '$-$') if s.startswith('-') else s

nbo = pd.read_csv(os.path.join(RES, 'qm_nbo.csv')); eda = pd.read_csv(os.path.join(RES, 'qm_eda.csv')); cp = pd.read_csv(os.path.join(RES, 'qm_cp.csv'))
st = pd.read_csv(os.path.join(RES, 'qm_status.csv')); hal = pd.read_csv(os.path.join(RES, 't1_hal59.csv'))
sc = pd.read_csv(os.path.join(RES, 't1_scans.csv')); sc['name'] = sc.file.str[:-4]
ocp = pd.read_csv(os.path.join(RES, 'orca_cp_hal59.csv')); ocs = pd.read_csv(os.path.join(RES, 'orca_cp_scans.csv'))
for d in (nbo, eda, cp): assert d.finished.all()
S = {}

# ------------------------------------------------------------------ job status
gq = st[~st.program.str.startswith('orca')].copy(); gq['ok'] = gq.status == 'ok'
S['status'] = {f'{s}|{p}': {'n': int(len(g)), 'ok': int(g.ok.sum())} for (s, p), g in gq.groupby(['set', 'program'])}
qn = gq[gq.program == 'qchem']; S['qchem_nbo_no_E2_table'] = int(qn.status.str.contains('SECOND ORDER').sum())

# ------------------------------------------------------------------ HAL59
a = nbo[nbo.set == 'A_HAL59'].merge(hal[['name', 'subset', 'ref_kcal', 'role', 'halogen', 'acc_el', 'covered', 'pass_T1', 'G']], on='name', how='left')
assert len(a) == 59 and a.name.is_unique
e = eda[eda.set == 'A_HAL59'].set_index('name'); g = cp[(cp.set == 'A_HAL59') & (cp.program == 'gaussian')].set_index('name')
a['eda_total_kJ'] = a.name.map(e.total); a['eda_frz_kJ'] = a.name.map(e.frz); a['eda_pol_kJ'] = a.name.map(e.pol); a['eda_ct_kJ'] = a.name.map(e.ct)
a['eda_ep_kJ'] = a.name.map(e.elec_pauli); a['eda_disp_kJ'] = a.name.map(e.disp); a['chi_CT'] = a.name.map(e.chi_CT); a['mechanism'] = a.name.map(e.mechanism)
a['g16_cp_kcal'] = a.name.map(g.Eint_CP_kJ) / KJ; a['orca_cp_kcal'] = a.name.map(ocp.set_index('name').Eint_kcal)
a['eda_total_kcal'] = a.eda_total_kJ / KJ
a['control'] = a.role.str.startswith('control')
assert a[['eda_total_kJ', 'g16_cp_kcal', 'orca_cp_kcal']].notna().all().all()
# BXQI from G, E and C (dimers: the EDA total is the contact energy)
bx = []
for _, r in a.iterrows():
    j = {'G': float(r.G), 'E2': float(r.E2_core), 'eda': {'total': float(r.eda_total_kJ)}}
    q = cli('bxqi', '--json', json.dumps(j)); bx.append((q.get('score'), q.get('grade', '')))
a['BXQI'] = [b[0] for b in bx]; a['BXQI_grade'] = [b[1] for b in bx]
a.to_csv(os.path.join(RES, 'qm_hal59.csv'), index=False)
xb = a[~a.control]; cov = xb[xb.covered]
rho_all = spearmanr(xb.ref_kcal, xb.E2_core); rho_cov = spearmanr(cov.ref_kcal, cov.E2_core)
rho_cov_noion = spearmanr(cov[cov.ionicX != True].ref_kcal, cov[cov.ionicX != True].E2_core)
ct_e2 = spearmanr(-xb.eda_ct_kJ, xb.E2_core)
def err(x, ref):
    d = x - ref; return {'MAE': r3(d.abs().mean()), 'RMSD': r3(np.sqrt((d ** 2).mean())), 'MSE': r3(d.mean()), 'MaxAE': r3(d.abs().max()), 'r': r3(pearsonr(x, ref)[0])}
be_eda = -a.eda_total_kcal; be_g16 = -a.g16_cp_kcal; be_orca = -a.orca_cp_kcal
S['hal59'] = {
    'n': 59, 'n_xb': int(len(xb)), 'n_covered': int(len(cov)), 'controls': a[a.control][['name', 'E2_core', 'verified']].values.tolist(),
    'controls_all_rejected': bool((~a[a.control].verified).all()),
    'covered_all_verified': bool(cov.verified.all()), 'xb_verified': int(xb.verified.sum()), 'xb_rejected': xb[~xb.verified][['name', 'acc_el', 'E2_core', 'ref_kcal']].values.tolist(),
    'rho_E2_ref_covered': r3(rho_cov.statistic), 'rho_E2_ref_all_xb': r3(rho_all.statistic), 'rho_E2_ref_covered_without_ionic': r3(rho_cov_noion.statistic),
    'rho_E2_negCT': r3(ct_e2.statistic), 'rho_G_E2_covered': r3(spearmanr(cov.G, cov.E2_core).statistic),
    'E2_min_covered': r3(cov.E2_core.min()), 'E2_max_covered': r3(cov.E2_core.max()),
    'E2_over_BE_median_covered': r3((cov.E2_core / cov.ref_kcal).median()),
    'ionic': a[a.ionicX == True][['name', 'E2_core', 'lp_sigma', 'lp_ry']].values.tolist(),
    'ry_share_median': r3((xb.lp_ry / xb.E2_core.where(xb.E2_core > 0)).median()), 'ry_share_max': r3((xb.lp_ry / xb.E2_core.where(xb.E2_core > 0)).max()),
    'npa_ct_max': r3(xb.ct_NPA.max()), 'rho_E2_npaCT': r3(spearmanr(xb.E2_core, xb.ct_NPA).statistic),
    'err_eda_vs_ref': err(be_eda, a.ref_kcal), 'err_g16cp_vs_ref': err(be_g16, a.ref_kcal), 'err_orca_vs_ref': err(be_orca, a.ref_kcal),
    'eda_minus_orca_maxabs_kcal': r3((a.eda_total_kcal - a.orca_cp_kcal).abs().max()), 'eda_minus_orca_mean_kcal': r3((a.eda_total_kcal - a.orca_cp_kcal).mean()),
    'chi_ct_range_xb': [r3(xb.chi_CT.min()), r3(xb.chi_CT.max())], 'chi_ct_median_xb': r3(xb.chi_CT.median()),
    'mechanism_counts_xb': xb.mechanism.value_counts().to_dict(),
    'ct_share_of_attraction_median': r3((xb.eda_ct_kJ / (xb.eda_pol_kJ + xb.eda_ct_kJ + xb.eda_disp_kJ)).median()),
    'frz_positive_xb': int((xb.eda_frz_kJ > 0).sum()),
    'halogen_series': {k: xb[xb.name.isin(v)].set_index('name').loc[v, 'E2_core'].round(2).tolist() for k, v in
                       {'acetone Cl/Br/I': ['19_benCl-acetone', '20_benBr-acetone', '21_benI-acetone'], 'NMe3 Cl/Br/I': ['22_benCl-Nm3', '23_benBr-Nm3', '24_benI-Nm3']}.items()},
    'bxqi_covered_min_max': [r3(cov.BXQI.min()), r3(cov.BXQI.max())],
    'grades_covered': cov.BXQI_grade.str[0].value_counts().to_dict(),
}

# ------------------------------------------------------------------ distance and angle scans
b = nbo[nbo.set.isin(['B_scan_distance', 'C_scan_angle'])].merge(sc[['name', 'base', 'scan', 's', 'theta_target', 'G', 'halogen', 'acc_el']], on='name', how='left')
eb = eda[eda.set.isin(['B_scan_distance', 'C_scan_angle'])].set_index('name')
for k in ('total', 'ct', 'pol', 'frz', 'chi_CT'): b['eda_' + k] = b.name.map(eb[k])
assert b[['base', 'eda_total']].notna().all().all()
ocs['key'] = ocs.apply(lambda r: (r.base, 'distance', round(r.s, 2)) if r.scan == 'distance' else (r.base, 'angle', int(r.theta)), axis=1)
ok_ = dict(zip(ocs.key, ocs.BE_kcal))
b['key'] = b.apply(lambda r: (r.base, 'distance', round(r.s, 2)) if r.scan == 'distance' else (r.base, 'angle', int(r.theta_target)), axis=1)
b['orca_BE'] = b.key.map(ok_); assert b.orca_BE.notna().all()
def rel(df, ref_sel, col):
    out = {}
    for base, g_ in df.groupby('base'):
        r0 = g_[ref_sel(g_)][col].iloc[0]; out.update({i: v / r0 for i, v in zip(g_.index, g_[col])})
    return pd.Series(out)
dist = b[b.scan == 'distance'].copy(); ang = b[b.scan == 'angle'].copy()
dist['E2_rel'] = rel(dist, lambda g_: g_.s == 1.0, 'E2_core'); dist['CT_rel'] = rel(dist, lambda g_: g_.s == 1.0, 'eda_ct'); dist['BE_rel'] = rel(dist, lambda g_: g_.s == 1.0, 'orca_BE')
ang['E2_rel'] = rel(ang, lambda g_: g_.theta_target == 180, 'E2_core'); ang['CT_rel'] = rel(ang, lambda g_: g_.theta_target == 180, 'eda_ct'); ang['BE_rel'] = rel(ang, lambda g_: g_.theta_target == 180, 'orca_BE')
pd.concat([dist, ang]).drop(columns=['key']).to_csv(os.path.join(RES, 'qm_scans.csv'), index=False)
def rng(df, sel, col): v = df[sel(df)][col]; return [r3(v.min()), r3(v.max())]
first_fail = {}
for base, g_ in dist.groupby('base'):
    f = g_[(g_.s > 1.0) & (g_.E2_core < GATE)].s; first_fail[base] = r3(f.min()) if len(f) else None
first_fail_ang = {}
for base, g_ in ang.groupby('base'):
    f = g_[g_.E2_core < GATE].theta_target; first_fail_ang[base] = int(f.max()) if len(f) else None
S['scans'] = {
    'dist_E2_rel_1_2': rng(dist, lambda d: d.s == 1.2, 'E2_rel'), 'dist_E2_rel_1_5': rng(dist, lambda d: d.s == 1.5, 'E2_rel'),
    'dist_CT_rel_1_2': rng(dist, lambda d: d.s == 1.2, 'CT_rel'), 'dist_CT_rel_1_5': rng(dist, lambda d: d.s == 1.5, 'CT_rel'),
    'dist_BE_rel_1_2': rng(dist, lambda d: d.s == 1.2, 'BE_rel'), 'dist_E2_rel_0_9': rng(dist, lambda d: d.s == 0.9, 'E2_rel'),
    'dist_E2_monotonic_all': bool(all((g_.sort_values('s').E2_core.diff().dropna() <= 1e-9).all() for _, g_ in dist.groupby('base'))),
    'dist_gate_first_fail': first_fail, 'dist_gate_fail_count_at_1_5': int((dist[dist.s == 1.5].E2_core < GATE).sum()), 'dist_n_dimers': int(dist.base.nunique()),
    'dist_G0_but_verified': int(((dist.G == 0) & dist.verified).sum()), 'dist_G0_points': int((dist.G == 0).sum()),
    'ang_E2_rel_150': rng(ang, lambda d: d.theta_target == 150, 'E2_rel'), 'ang_E2_rel_140': rng(ang, lambda d: d.theta_target == 140, 'E2_rel'),
    'ang_E2_rel_130': rng(ang, lambda d: d.theta_target == 130, 'E2_rel'), 'ang_E2_rel_120': rng(ang, lambda d: d.theta_target == 120, 'E2_rel'),
    'ang_CT_rel_150': rng(ang, lambda d: d.theta_target == 150, 'CT_rel'), 'ang_CT_rel_120': rng(ang, lambda d: d.theta_target == 120, 'CT_rel'),
    'ang_gate_largest_failing_theta': first_fail_ang, 'ang_verified_at_130': int(ang[ang.theta_target == 130].verified.sum()), 'ang_verified_at_120': int(ang[ang.theta_target == 120].verified.sum()),
    'ang_n_dimers': int(ang.base.nunique()), 'ang_E2_monotonic_all': bool(all((g_.sort_values('theta_target').E2_core.diff().dropna() >= -1e-9).all() for _, g_ in ang.groupby('base'))),
    'ang_ct_attractive_at_120': int((ang[ang.theta_target == 120].eda_ct < 0).sum()), 'ang_orca_repulsive_at_130': int((ang[ang.theta_target == 130].orca_BE < 0).sum()),
}

# ------------------------------------------------------------------ method and basis (Gaussian)
d = nbo[nbo.set == 'D_method_basis'].copy(); d['base'] = d.name.str.split('__').str[0]; d['level'] = d.name.str.split('__', n=1).str[1]
d.to_csv(os.path.join(RES, 'qm_method_basis.csv'), index=False)
pv = d.pivot_table(index='base', columns='level', values='E2_core')
fun = ['wB97X-D__def2-TZVP', 'M06-2X__def2-TZVP', 'B3LYP-D3BJ__def2-TZVP', 'PBE0-D3BJ__def2-TZVP']
rel_f = pv[fun].div(pv['wB97X-D__def2-TZVP'], axis=0)
S['method_basis'] = {'n_dimers': int(pv.shape[0]), 'levels': list(pv.columns),
    'dft_rel_range': [r3(rel_f.values.min()), r3(rel_f.values.max())], 'hf_rel_range': [r3((pv['HF__def2-TZVP'] / pv['wB97X-D__def2-TZVP']).min()), r3((pv['HF__def2-TZVP'] / pv['wB97X-D__def2-TZVP']).max())],
    'svp_rel_range': [r3((pv['wB97X-D__def2-SVP'] / pv['wB97X-D__def2-TZVP']).min()), r3((pv['wB97X-D__def2-SVP'] / pv['wB97X-D__def2-TZVP']).max())],
    'qzvp_rel_range': [r3((pv['wB97X-D__def2-QZVP'] / pv['wB97X-D__def2-TZVP']).min()), r3((pv['wB97X-D__def2-QZVP'] / pv['wB97X-D__def2-TZVP']).max())],
    'gate_changes': int((d.groupby('base').verified.nunique() > 1).sum()),
    'qzvp_rel_sigma_only': [r3(x) for x in (lambda q: [q.min(), q.max()])((d.assign(sig=d.lp_sigma + d.pi_sigma).pivot_table(index='base', columns='level', values='sig')).pipe(lambda t: t['wB97X-D__def2-QZVP'] / t['wB97X-D__def2-TZVP']))],
    'svp_rel_sigma_only': [r3(x) for x in (lambda q: [q.min(), q.max()])((d.assign(sig=d.lp_sigma + d.pi_sigma).pivot_table(index='base', columns='level', values='sig')).pipe(lambda t: t['wB97X-D__def2-SVP'] / t['wB97X-D__def2-TZVP']))],
    'ry_qzvp_max': r3(d[d.level == 'wB97X-D__def2-QZVP'].lp_ry.max()), 'ry_tzvp_max_nonionic': r3(d[(d.level == 'wB97X-D__def2-TZVP') & (d.ionicX != True)].lp_ry.max()),
    'rank_rho_min_vs_wb97xd_tzvp': r3(min(spearmanr(pv[c], pv['wB97X-D__def2-TZVP']).statistic for c in pv.columns))}

# ------------------------------------------------------------------ EDA settings
es = eda[eda.set == 'E_eda_settings'].copy(); es['base'] = es.name.str.split('__').str[0]; es['setting'] = es.name.str.split('__').str[-1]
ref = eda[eda.set == 'A_HAL59'].set_index('name')
es['ref_total'] = es.base.map(ref.total); es['ref_ct'] = es.base.map(ref.ct)
assert es.ref_total.notna().all()
es.to_csv(os.path.join(RES, 'qm_eda_settings.csv'), index=False)
S['eda_settings'] = {s_: {'dTotal_kJ_range': [r3((g_.total - g_.ref_total).min()), r3((g_.total - g_.ref_total).max())],
                          'dCT_kJ_range': [r3((g_.ct - g_.ref_ct).min()), r3((g_.ct - g_.ref_ct).max())],
                          'CT_ratio_range': [r3((g_.ct / g_.ref_ct).min()), r3((g_.ct / g_.ref_ct).max())], 'n': int(len(g_))}
                     for s_, g_ in es.groupby('setting')}

# ------------------------------------------------------------------ protein
f = nbo[nbo.set == 'F_protein'].copy()
f['sys'] = f.name.str.split('__').str[0]; f['lig'] = f.name.str.split('__').str[1]; f['variant'] = f.name.str.split('__').str[2]
pt = pd.read_csv(os.path.join(RES, 'protein_t1.csv')); pt = pt[pt.status == 'XB'].drop_duplicates(['pdb', 'donor', 'X', 'acceptor', 'accAtom'])
def gkey(lbl):
    dn, ac = [x.strip() for x in lbl.split('···')]; return dn.split('@')[0], dn.split('@')[1], ac.split('@')[0], ac.split('@')[1]
gmap = {(r.pdb, r.donor, r.X, r.acceptor, r.accAtom): r.G for r in pt.itertuples()}
f['G'] = [gmap.get((s_,) + gkey(l)) for s_, l in zip(f.sys, f.label)]
assert f.G.notna().all(), f[f.G.isna()].label.unique()
f['contact'] = f.label.str.replace(' ··· ', '···')
f.to_csv(os.path.join(RES, 'qm_protein.csv'), index=False)
main = f[f.lig.str.endswith('A')]
pv2 = main.pivot_table(index='contact', columns='variant', values='E2_core')
S['protein'] = {'models': int(f.name.nunique()), 'contacts': int(f.contact.nunique()), 'rows': int(len(f)),
    'default_cluster': {r.contact: [r3(r.E2_core), bool(r.verified)] for r in f[f.variant == 'cluster_pcm4'].itertuples()},
    'range_over_variants': {c: [r3(pv2.loc[c].min()), r3(pv2.loc[c].max()), int(pv2.loc[c].notna().sum())] for c in pv2.index},
    'gate_flips': [c for c in pv2.index if (pv2.loc[c].dropna() >= GATE).nunique() > 1],
    'pcm_minus_gas': {c: r3(pv2.loc[c, 'cluster_pcm4'] - pv2.loc[c, 'cluster_gas']) for c in pv2.index},
    'qmmm_vs_pcm': {c: r3(pv2.loc[c, 'qmmm_ff14SB'] - pv2.loc[c, 'cluster_pcm4']) for c in pv2.index},
    'ff_charmm_minus_ff14': {c: r3(pv2.loc[c, 'qmmm_charmm36'] - pv2.loc[c, 'qmmm_ff14SB']) for c in pv2.index},
    'boundary_range': {c: [r3(pv2.loc[c, ['qmmm_ff14SB', 'qmmm_Z1', 'qmmm_Z2', 'qmmm_Z3']].min()), r3(pv2.loc[c, ['qmmm_ff14SB', 'qmmm_Z1', 'qmmm_Z2', 'qmmm_Z3']].max())] for c in pv2.index},
    'radius_range': {c: [r3(pv2.loc[c, ['qmmm_r10', 'qmmm_ff14SB', 'qmmm_r20', 'qmmm_r25', 'qmmm_full']].min()), r3(pv2.loc[c, ['qmmm_r10', 'qmmm_ff14SB', 'qmmm_r20', 'qmmm_r25', 'qmmm_full']].max())] for c in pv2.index},
    'shell_range': {c: [r3(pv2.loc[c, ['cluster_pcm4', 'shell3', 'shell4', 'shell5']].min()), r3(pv2.loc[c, ['cluster_pcm4', 'shell3', 'shell4', 'shell5']].max())] for c in pv2.index},
    'his110': {k: r3(v) for k, v in pv2.loc[[c for c in pv2.index if 'THR113' in c][0], ['qmmm_HIS110_HID', 'qmmm_ff14SB', 'qmmm_HIS110_HIP']].items()},
}
# replicas (chain B / C) at the default cluster model
rep = f[f.variant == 'cluster_pcm4'].copy(); rep['res_pair'] = rep.contact.str.replace(r':[A-Z]@', '@', regex=True).str.replace(r'\d{3,4}', '', regex=True)
S['protein']['replica_pairs'] = {k: [r3(x) for x in g_.E2_core] for k, g_ in rep.groupby(rep.sys + '|' + rep.contact.str.split('@').str[1].str.split('···').str[0] + '|' + rep.contact.str.split('@').str[-1]) if len(g_) > 1}
# EDA and counterpoise of the protein clusters
pe = eda[eda.set == 'F_protein'].copy(); pe['sys'] = pe.name.str.split('__').str[0]; pe['lig'] = pe.name.str.split('__').str[1]
gc = cp[(cp.set == 'F_protein') & (cp.program == 'gaussian')].set_index('name'); oc = cp[(cp.set == 'F_protein') & (cp.program == 'orca')].set_index('name')
pe['g16_cp_kJ'] = pe.name.str.replace('_eda', '').map(gc.Eint_CP_kJ); pe['orca_cp_kJ'] = pe.name.str.replace('_eda', '').map(oc.Eint_CP_kJ)
pe.to_csv(os.path.join(RES, 'qm_protein_eda.csv'), index=False)
S['protein']['eda'] = {r.name.replace('__cluster_pcm4_eda', ''): {'frz': r3(r.frz), 'pol': r3(r.pol), 'ct': r3(r.ct), 'total': r3(r.total), 'chi': r3(r.chi_CT), 'orca_cp': r3(r.orca_cp_kJ), 'g16_cp': r3(r.g16_cp_kJ)} for r in pe.itertuples()}
S['protein']['eda_minus_orca_kJ'] = r3((pe.total - pe.orca_cp_kJ).abs().max())
S['protein']['g16cp'] = {k.replace('__cluster_pcm4', ''): r3(v) for k, v in gc.Eint_CP_kJ.items()}
# BXQI of the protein contacts (G and E; the cluster EDA covers every contact of the two fragments)
bq = []
for r in f[f.variant == 'cluster_pcm4'].itertuples():
    q = cli('bxqi', '--json', json.dumps({'G': float(r.G), 'E2': float(r.E2_core)})); bq.append({'contact': r.contact, 'sys': r.sys, 'G': r3(r.G), 'E2': r3(r.E2_core), 'BXQI': r3(q['score']), 'grade': q['grade'], 'provenance': q['provenance']})
bq = pd.DataFrame(bq); bq.to_csv(os.path.join(RES, 'qm_protein_bxqi.csv'), index=False)
S['protein']['bxqi'] = bq[['contact', 'BXQI', 'grade']].values.tolist()
# ------------------------------------------------------------------ wall times (from the outputs)
import re, glob
PKG = os.path.join(ROOT, 'qm_package')
def wall(fn):
    t = open(fn, errors='replace').read()
    if fn.endswith('.log'):
        m = re.findall(r'Elapsed time:\s+(\d+) days\s+(\d+) hours\s+(\d+) minutes\s+([\d.]+) seconds', t)
        return sum(int(a) * 86400 + int(b) * 3600 + int(c) * 60 + float(dd) for a, b, c, dd in m) if 'Normal termination' in t else None
    m = re.findall(r'Total job time:\s+([\d.]+)s\(wall\)', t); return float(m[-1]) if m and 'Thank you very much' in t else None
tw = {}
for prog, pat in (('gaussian', '*/gaussian/*.log'), ('gaussian_cp', '*/gaussian_cp/*.log'), ('qchem_eda', '*/qchem_eda/*.out')):
    v = [w for w in (wall(x) for x in glob.glob(os.path.join(PKG, pat))) if w]; tw[prog] = {'n': len(v), 'hours': r3(sum(v) / 3600)}
pe_t = {os.path.basename(x).replace('__cluster_pcm4_eda.out', ''): wall(x) for x in glob.glob(os.path.join(PKG, 'F_protein', 'qchem_eda', '*.out'))}
pe_t = {k: v for k, v in pe_t.items() if v}
rej = [k for k, v in S['protein']['eda'].items()]
gate_fail_clusters = sorted({r.name.replace('__cluster_pcm4', '') for r in f[f.variant == 'cluster_pcm4'].itertuples()} - {r.name.replace('__cluster_pcm4', '') for r in f[(f.variant == 'cluster_pcm4') & f.verified].itertuples()})
tw['protein_eda_hours'] = r3(sum(pe_t.values()) / 3600); tw['protein_eda_n'] = len(pe_t)
tw['protein_clusters_all_rejected'] = gate_fail_clusters
tw['protein_eda_hours_rejected_clusters'] = r3(sum(v for k, v in pe_t.items() if k in gate_fail_clusters) / 3600)
tw['protein_gaussian_nbo_hours'] = r3(sum(w for w in (wall(x) for x in glob.glob(os.path.join(PKG, 'F_protein', 'gaussian', '*.log'))) if w) / 3600)
S['timing'] = tw
if tw['gaussian']['n'] == 0 and os.path.exists(os.path.join(RES, 'qm_summary.json')):  # no QM outputs here (e.g. the GitHub copy): keep the stored wall times
    S['timing'] = json.load(open(os.path.join(RES, 'qm_summary.json'))).get('timing', tw)
json.dump(S, open(os.path.join(RES, 'qm_summary.json'), 'w'), indent=1, default=str)

# ================================================================== figures
# Fig 8: HAL59 NBO, scans, EDA
fig, axs = plt.subplots(1, 3, figsize=(7.2, 3.3)); fig.subplots_adjust(wspace=0.55)
ax = axs[0]
for _, r in a.iterrows():
    acc = 'pi' if r.acc_el == 'pi' else r.acc_el; mk = ACCM.get(acc, 'o'); col = HALC.get(r.halogen, MUTE)
    y = max(r.E2_core, 0.012)
    if r.control: ax.plot(r.ref_kcal, y, marker=mk, ms=4.5, mfc='white', mec=INK2, mew=0.9, ls='none')
    else: ax.plot(r.ref_kcal, y, marker=mk, ms=4.3 if acc in ('N', 'O', 'S', 'pi') else 4.0, mfc=col if r.covered else 'white', mec=col, mew=0.9, ls='none')
ax.axhline(GATE, color=C[7], ls='--', lw=0.9); ax.set_xscale('log'); ax.set_yscale('log'); ax.set_ylim(0.01, 150)
ax.set_xlabel('CCSD(T)/CBS (kcal/mol)'); ax.set_ylabel(r'$\Sigma E^{(2)}$ (kcal/mol)'); ax.set_title('(a) HAL59')
ax.text(0.03, 0.97, 'ρ = %.2f' % S['hal59']['rho_E2_ref_covered'], transform=ax.transAxes, va='top')
hh = [Line2D([], [], ls='none', marker='o', mfc=HALC[k], mec=HALC[k], ms=4, label=l) for k, l in (('CL', 'Cl'), ('BR', 'Br'), ('I', 'I'))] + \
     [Line2D([], [], ls='none', marker='o', mfc='white', mec=INK2, ms=4, label='control / no cut-off')]
fig.legend(handles=hh, loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.04), handletextpad=0.2)
ax = axs[1]
for base, g_ in dist.groupby('base'):
    g_ = g_.sort_values('s'); col = HALC.get(g_.halogen.iloc[0], MUTE)
    ax.plot(g_.s, g_.E2_core.clip(lower=0.01), color=col, lw=1.1, marker='o', ms=2.4, alpha=0.9)
ax.axhline(GATE, color=C[7], ls='--', lw=0.9); ax.set_yscale('log'); ax.set_xlabel(r'$d/d_\mathrm{eq}$'); ax.set_ylabel(r'$\Sigma E^{(2)}$ (kcal/mol)'); ax.set_title('(b) Distance scans')
ax = axs[2]
for _, r in a.iterrows():
    acc = 'pi' if r.acc_el == 'pi' else r.acc_el; mk = ACCM.get(acc, 'o'); col = HALC.get(r.halogen, MUTE)
    ax.plot(r.ref_kcal, -r.eda_total_kcal, marker=mk, ms=4, mfc='white' if r.control else col, mec=INK2 if r.control else col, mew=0.8, ls='none')
lim = [0, 22]; ax.plot(lim, lim, color=INK2, ls=':', lw=0.8); ax.set_xlim(lim); ax.set_ylim(lim); ax.set_xticks([0, 5, 10, 15, 20]); ax.set_yticks([0, 5, 10, 15, 20])
ax.set_xlabel('CCSD(T)/CBS (kcal/mol)'); ax.set_ylabel(r'$-\Delta E_\mathrm{int}$, EDA (kcal/mol)'); ax.set_title('(c) ALMO-EDA')
ax.text(0.03, 0.97, 'MAE %.2f\nkcal/mol' % S['hal59']['err_eda_vs_ref']['MAE'], transform=ax.transAxes, va='top')
save(fig, MS, 'Fig8_NBO_benchmark')

# Fig 9: protein
VORD = [('cluster_gas', 'gas'), ('cluster_pcm4', 'PCM ε=4'), ('shell3', '+3 Å'), ('shell4', '+4 Å'), ('shell5', '+5 Å'), ('qmmm_ff14SB', 'ff14SB'), ('qmmm_charmm36', 'CHARMM36'),
        ('qmmm_Z1', 'Z1'), ('qmmm_Z2', 'Z2'), ('qmmm_Z3', 'Z3'), ('qmmm_r10', '10 Å'), ('qmmm_r20', '20 Å'), ('qmmm_r25', '25 Å'), ('qmmm_full', 'all MM')]
CORD = ['BVP1207:A@BR···ASN65:A@OD1', 'TBS301:A@BR11···LEU83:A@O', 'P84400:A@I15···LEU145:A@O', 'TBS301:A@BR12···GLU81:A@O', 'R47306:A@CL1···PRO232:A@O',
        'TBS301:A@BR13···PHE80:A@π(ring)', 'LDT320:A@BR8···THR113:A@OG1', 'TBS301:A@BR10···ILE10:A@O', '9LI201:A@CL2···GLY75:A@N']
CLAB = {'BVP1207:A@BR···ASN65:A@OD1': '2W0S Br···OD1 Asn65', 'TBS301:A@BR11···LEU83:A@O': '1P5E Br11···O Leu83', 'P84400:A@I15···LEU145:A@O': '4AGL I···O Leu145',
        'TBS301:A@BR12···GLU81:A@O': '1P5E Br12···O Glu81', 'R47306:A@CL1···PRO232:A@O': '4GQL Cl···O Pro232', 'TBS301:A@BR13···PHE80:A@π(ring)': '1P5E Br13···π Phe80',
        'LDT320:A@BR8···THR113:A@OG1': '1US0 Br···OG1 Thr113', 'TBS301:A@BR10···ILE10:A@O': '1P5E Br10···O Ile10', '9LI201:A@CL2···GLY75:A@N': '4DST Cl···N Gly75'}
assert set(CORD) == set(pv2.index), set(pv2.index) ^ set(CORD)
M = pv2.loc[CORD, [v for v, _ in VORD]].values
fig = plt.figure(figsize=(7.4, 7.6)); gs = fig.add_gridspec(2, 2, height_ratios=[1.35, 1], width_ratios=[4.6, 1], hspace=0.42, wspace=0.06)
ax = fig.add_subplot(gs[0, 0])
from matplotlib.colors import LinearSegmentedColormap
cm = LinearSegmentedColormap.from_list('b', ['#f4f7fb', '#a9c8ef', C[0], '#163f73'])
ax.imshow(np.log10(np.clip(M, 0.03, None)), cmap=cm, vmin=np.log10(0.03), vmax=np.log10(6), aspect='auto')
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        v = M[i, j]
        ax.text(j, i, '%.2f' % v, ha='center', va='center', fontsize=8.6, color='white' if v > 1.5 else INK, weight='bold' if v < GATE else 'normal')
        if v < GATE: ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec=C[7], lw=1.2))
ax.set_xticks(range(len(VORD))); ax.set_xticklabels([l for _, l in VORD], fontsize=9.5, rotation=50, ha='right'); ax.set_yticks(range(len(CORD))); ax.set_yticklabels([CLAB[c] for c in CORD], fontsize=9.8)
ax.grid(False); ax.set_title(r'(a) $\Sigma E^{(2)}$ (kcal/mol) per contact and model', pad=22)
for x in (1.5, 4.5, 6.5, 9.5): ax.axvline(x, color='white', lw=2.5)
for xc, t in ((0.5, 'cluster'), (3.0, 'QM size'), (5.5, 'force field'), (8.0, 'boundary'), (11.5, 'MM radius')):
    ax.text(xc, -0.75, t, ha='center', va='bottom', fontsize=9.5, color=INK2)
# (b) BXQI, same rows as (a)
axq = fig.add_subplot(gs[0, 1], sharey=ax)
bb = bq.set_index('contact').loc[CORD]
cols = [C[7] if 'rejected' in g_ else (C[0] if g_[0] in 'AB' else C[3]) for g_ in bb.grade]
axq.barh(range(len(CORD)), bb.BXQI, color=cols, height=0.62)
for yi, (v, g_) in enumerate(zip(bb.BXQI, bb.grade)): axq.text(v + 3, yi, g_.split(' ')[0], va='center', fontsize=9.5)
axq.set_xlim(0, 120); axq.set_xticks([0, 50, 100]); axq.tick_params(axis='y', left=False, labelleft=False); axq.grid(axis='y', visible=False)
axq.set_xlabel('BXQI'); axq.set_title('(b) BXQI', pad=22)
# (c) EDA
ax = fig.add_subplot(gs[1, :])
pe_ = pe.copy(); pe_['lab'] = pe_.sys.str.upper().str.replace('_IDD594', '') + ' ' + pe_.lig.str[:-1] + ' (chain ' + pe_.lig.str[-1] + ')'
pe_ = pe_.sort_values('total'); y = np.arange(len(pe_))
for k, (col, lab) in enumerate((('frz', 'FRZ'), ('pol', 'POL'), ('ct', 'CT'))):
    ax.barh(y + (k - 1) * 0.26, pe_[col], height=0.25, color=[MUTE, C[2], C[0]][k], label=lab)
ax.plot(pe_.total, y, ls='none', marker='D', ms=6, mfc=INK, mec='white', label='total')
ax.axvline(0, color=INK2, lw=0.8); ax.set_yticks(y); ax.set_yticklabels(pe_.lab); ax.grid(axis='y', visible=False)
ax.set_xlabel('ALMO-EDA term (kJ/mol, gas phase)'); ax.set_title('(c) ALMO-EDA of six clusters'); ax.legend(loc='lower right', ncol=4, handlelength=1.2)
ax.set_xlim(-50, 35); ax.set_ylim(-1.3, len(pe_) - 0.5)
save(fig, MS, 'Fig9_protein_QM')

# Fig S7: angle scans (NBO, CT, ORCA)
fig, axs = plt.subplots(1, 3, figsize=(7.2, 3.3), sharey=True); fig.subplots_adjust(wspace=0.12, top=0.80)
for ax, col, lab in zip(axs, ('E2_rel', 'CT_rel', 'BE_rel'), ('Value relative to 180°', None, None)):
    for base, g_ in ang.groupby('base'):
        g_ = g_.sort_values('theta_target', ascending=False); ax.plot(g_.theta_target, g_[col], marker='o', ms=2.6, lw=1.2, color=HALC.get(g_.halogen.iloc[0], MUTE))
    ax.axvspan(118, 140, color='#f3f2ee', zorder=0, lw=0); ax.set_xlim(182, 118); ax.axhline(0, color=INK2, lw=0.6); ax.set_xlabel(r'$\theta_1$ (deg)'); ax.set_xticks([180, 160, 140, 120]); ax.set_ylim(-1.15, 1.1)
    if lab: ax.set_ylabel(lab)
axs[0].set_title(r'(a) NBO $\Sigma E^{(2)}$'); axs[1].set_title('(b) EDA CT'); axs[2].set_title('(c) ORCA binding')
fig.legend(handles=[Line2D([], [], color=HALC[h], marker='o', ms=4, lw=1.5, label=h.title()) for h in HALC if h in set(ang.halogen)], loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.0))
save(fig, ESI, 'FigS5_angle_scan_QM')

# Fig S8: method / basis
LV = [('wB97X-D__def2-SVP', 'ωB97X-D/SVP'), ('wB97X-D__def2-TZVP', 'ωB97X-D/TZVP'), ('wB97X-D__def2-QZVP', 'ωB97X-D/QZVP'), ('M06-2X__def2-TZVP', 'M06-2X'),
      ('B3LYP-D3BJ__def2-TZVP', 'B3LYP-D3(BJ)'), ('PBE0-D3BJ__def2-TZVP', 'PBE0-D3(BJ)'), ('HF__def2-TZVP', 'HF')]
fig, ax = plt.subplots(figsize=(7.2, 4.4))
hx = d.drop_duplicates('base').set_index('base').name
bh = sc.drop_duplicates('base').set_index('base').halogen
order = pv.loc[:, 'wB97X-D__def2-TZVP'].sort_values().index
for k, (lv, lab) in enumerate(LV):
    ax.plot(np.arange(len(order)) + (k - 3) * 0.1, pv.loc[order, lv], ls='none', marker='o^sDvP<'[k], ms=3.8, color=[C[6], C[0], '#163f73', C[1], C[3], C[2], INK2][k], label=lab)
ax.axhline(GATE, color=C[7], ls='--', lw=0.9); ax.set_yscale('log'); ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=50, ha='right', fontsize=9.2)
ax.set_ylabel(r'$\Sigma E^{(2)}$ (kcal/mol)'); ax.legend(ncol=4, loc='lower center', bbox_to_anchor=(0.5, 1.0), columnspacing=1.0, handletextpad=0.2); ax.grid(axis='x', visible=False)
save(fig, ESI, 'FigS6_method_basis')

# ================================================================== tables
def tex(s): return str(s).replace('_', r'\_').replace('···', r'$\cdots$').replace('π', r'$\pi$').replace('#', r'\#')
def lt(spec, head, rows, cap, lab, size=r'\scriptsize'):
    o = [r'{%s' % size, r'\setlength{\tabcolsep}{3pt}', r'\begin{longtable}{%s}' % spec, r'\caption{%s}\label{%s}\\' % (cap, lab), r'\toprule', head + r'\\', r'\midrule', r'\endfirsthead',
         r'\toprule', head + r'\\', r'\midrule', r'\endhead', r'\bottomrule', r'\endlastfoot'] + [' & '.join(r) + r'\\' for r in rows] + [r'\end{longtable}}']
    return '\n'.join(o) + '\n'
rows = []
for _, r in a.iterrows():
    acc = r'$\pi$' if r.acc_el == 'pi' else r.acc_el
    gate = 'pass' if r.verified else 'fail'
    rows.append([tex(r['name']) + (r'$^c$' if r.control else '') + (r'$^i$' if r.ionicX == True else ''), r.halogen.title(), acc, '%.2f' % r.ref_kcal, '%.2f' % r.G, '%.2f' % r.E2_core, '%.2f' % r.lp_ry if r.acc_el != 'pi' else '--',
                 mnum('%.3f', r.ct_NPA), gate, mnum('%.2f', -r.g16_cp_kcal), mnum('%.2f', -r.eda_total_kcal), '%.0f' % r.BXQI if r.BXQI is not None else '--'])
open(os.path.join(TAB, 'tab_qm_hal59.tex'), 'w').write(lt('lllrrrrrlrrr', r'Dimer & X & Y & $E_\mathrm{ref}$ & $G$ & $\Sigma\ET$ & RY & $\Delta q_\mathrm{CT}$ & Gate & BE(G16) & BE(EDA) & BXQI',
    rows, r'NBO gate on the 59 HAL59 dimers. Gaussian~16, $\omega$B97X-D/def2-TZVP, gas phase. $E_\mathrm{ref}$: CCSD(T)/CBS binding energy (kcal\,mol$^{-1}$). $\Sigma\ET$: gate sum (kcal\,mol$^{-1}$) with the Rydberg term RY. $\Delta q_\mathrm{CT}$: NPA charge transfer to fragment 2 (e). Gate: $\Sigma\ET \ge 0.5$. BE(G16): counterpoise-corrected binding energy from Gaussian ($\omega$B97X-D/def2-TZVP). BE(EDA): $-\Delta E_\mathrm{int}$ from ALMO-EDA ($\omega$B97X-V/def2-TZVPD). BXQI from $G$, $\Sigma\ET$ and the EDA energy. $^c$Control. $^i$Ionic NBO Lewis structure: donation into the vacant LP$^*$ of the halogen.', 'tab:qmhal'))
rows = []
for _, r in a.iterrows():
    rows.append([tex(r['name']) + (r'$^c$' if r.control else ''), mnum('%.2f', r.eda_frz_kJ), mnum('%.2f', r.eda_ep_kJ), mnum('%.2f', r.eda_disp_kJ), mnum('%.2f', r.eda_pol_kJ), mnum('%.2f', r.eda_ct_kJ), mnum('%.2f', r.eda_total_kJ),
                 '%.2f' % r.chi_CT if not math.isnan(r.chi_CT) else '--'])
open(os.path.join(TAB, 'tab_qm_eda.tex'), 'w').write(lt('lrrrrrrr', r'Dimer & FRZ & ELEC+PAULI & DISP & POL & CT & Total & $\chi_\mathrm{CT}$',
    rows, r'ALMO-EDA of the 59 HAL59 dimers (Q-Chem~6.2, $\omega$B97X-V/def2-TZVPD, gas phase, kJ\,mol$^{-1}$). FRZ = ELEC+PAULI+DISP. Q-Chem joins ELEC and PAULI for systems with an ECP. CT is counterpoise-corrected. $\chi_\mathrm{CT}=E_\mathrm{CT}/(E_\mathrm{POL}+E_\mathrm{CT})$. $^c$Control.', 'tab:qmeda'))
VV = [('cluster_gas', 'Cluster, gas'), ('cluster_pcm4', 'Cluster, PCM'), ('shell3', 'PCM, +3 Å'), ('shell4', 'PCM, +4 Å'), ('shell5', 'PCM, +5 Å'), ('qmmm_ff14SB', 'QM/MM ff14SB'), ('qmmm_charmm36', 'QM/MM CHARMM36'),
      ('qmmm_Z1', 'QM/MM Z1'), ('qmmm_Z2', 'QM/MM Z2'), ('qmmm_Z3', 'QM/MM Z3'), ('qmmm_r10', 'QM/MM 10 Å'), ('qmmm_r20', 'QM/MM 20 Å'), ('qmmm_r25', 'QM/MM 25 Å'), ('qmmm_full', 'QM/MM all'),
      ('qmmm_HIS110_HID', 'QM/MM His110 HID'), ('qmmm_HIS110_HIP', 'QM/MM His110 HIP')]
rows = []
for c in CORD:
    for v, lab in VV:
        if v in pv2.columns and not math.isnan(pv2.loc[c, v]):
            rr = f[(f.contact == c) & (f.variant == v)].iloc[0]
            rows.append([tex(CLAB[c]).replace('Å', r'\AA{}'), lab.replace('Å', r'\AA{}'), '%.2f' % rr.E2_core, '%.2f' % rr.lp_ry, mnum('%.3f', rr.ct_NPA), 'pass' if rr.verified else 'fail'])
for r in f[~f.lig.str.endswith('A')].itertuples():
    rows.append([tex(r.contact.replace('@', ' ')), 'Cluster, PCM (replica)', '%.2f' % r.E2_core, '%.2f' % r.lp_ry, mnum('%.3f', r.ct_NPA), 'pass' if r.verified else 'fail'])
open(os.path.join(TAB, 'tab_qm_prot.tex'), 'w').write(lt('llrrrl', r'Contact & Model & $\Sigma\ET$ & RY & $\Delta q_\mathrm{CT}$ & Gate', rows,
    r'NBO gate for the protein contacts (Gaussian~16, $\omega$B97X-D/def2-TZVP). PCM: $\varepsilon=4$. QM/MM: RC boundary and \SI{15}{\angstrom} radius by default. The model name gives any other choice. $\Sigma\ET$ and RY in kcal\,mol$^{-1}$. $\Delta q_\mathrm{CT}$: NPA charge transfer between the fragments (e).', 'tab:qmprot'))
print(json.dumps(S, indent=1, default=str)[:6000])


### `21_v3_analysis.py`

21_v3_analysis.py - every number of the revised paper (V3) that comes from the existing outputs.
Usage: python3 21_v3_analysis.py ..
Needs results/*.csv of V2 (scripts 03-17) and results/v3/*.csv of scripts 18-20. Writes results/v3/summary_v3.json and
results/v3/*_table.csv. No QM job is run. Spearman confidence intervals: 10000 bootstrap resamples (seed 20261126).

In [ ]:
%%writefile Scripts/21_v3_analysis.py
#!/usr/bin/env python3
"""21_v3_analysis.py - every number of the revised paper (V3) that comes from the existing outputs.
Usage: python3 21_v3_analysis.py ..
Needs results/*.csv of V2 (scripts 03-17) and results/v3/*.csv of scripts 18-20. Writes results/v3/summary_v3.json and
results/v3/*_table.csv. No QM job is run. Spearman confidence intervals: 10000 bootstrap resamples (seed 20261126)."""
import os, sys, json
import numpy as np, pandas as pd
from scipy.stats import spearmanr, pearsonr

ROOT = sys.argv[1] if len(sys.argv) > 1 else '..'
R = os.path.join(ROOT, 'results'); V = os.path.join(R, 'v3')
KCAL = 4.184
LO, HI = 0.3, 0.8
rng = np.random.default_rng(20261126)
S = {}

def verdict(e): return 'supported' if e >= HI else ('inconclusive' if e >= LO else 'not supported')
def boot_rho(x, y, n=10000):
    x, y = np.asarray(x), np.asarray(y); m = len(x); r = []
    for _ in range(n):
        i = rng.integers(0, m, m); rr = spearmanr(x[i], y[i]).correlation
        if np.isfinite(rr): r.append(rr)
    return [round(float(np.percentile(r, 2.5)), 2), round(float(np.percentile(r, 97.5)), 2)]
def rho(x, y): return round(float(spearmanr(x, y).correlation), 2)
r2 = lambda v: round(float(v), 2)

nbo = pd.read_csv(os.path.join(V, 'nbo_v3.csv'))
# ------------------------------------------------------------------ HAL59
h = pd.read_csv(os.path.join(R, 'qm_hal59.csv'))
hv = nbo[nbo.set == 'A_HAL59'][['name', 'E2_valence', 'E2_RY', 'E2_with_RY', 'hbY']]
h = h.merge(hv, on='name')
h['verdict'] = h.E2_valence.map(verdict)
xb = h[h.role == 'halogen bond']; cov = xb[xb.covered]; pf = xb[~xb.covered]; ctl = h[h.control]
S['hal59'] = dict(n=len(h), n_xb=len(xb), n_cov=len(cov), n_pf=len(pf),
    cov_verdicts=cov.verdict.value_counts().to_dict(), pf_verdicts=pf.verdict.value_counts().to_dict(), xb_verdicts=xb.verdict.value_counts().to_dict(),
    cov_val_range=[r2(cov.E2_valence.min()), r2(cov.E2_valence.max())], cov_inconclusive=cov[cov.verdict != 'supported'][['name', 'E2_valence', 'ref_kcal']].values.tolist(),
    pf_not_supported=pf[pf.verdict != 'supported'][['name', 'E2_valence', 'ref_kcal']].values.tolist(),
    controls=ctl[['name', 'E2_valence', 'verdict']].values.tolist(),
    rho_G_cov=rho(cov.G, cov.ref_kcal), rho_G_cov_CI=boot_rho(cov.G, cov.ref_kcal),
    rho_E_cov=rho(cov.E2_valence, cov.ref_kcal), rho_E_cov_CI=boot_rho(cov.E2_valence, cov.ref_kcal),
    rho_E_all=rho(xb.E2_valence, xb.ref_kcal), rho_E_all_CI=boot_rho(xb.E2_valence, xb.ref_kcal),
    rho_E_NPA=rho(xb.E2_valence, xb.ct_NPA), rho_E_CT=rho(xb.E2_valence, -xb.eda_ct_kJ),
    median_ratio_cov=r2((cov.E2_valence / cov.ref_kcal).median()), median_ratio_all=r2((xb.E2_valence / xb.ref_kcal).median()),
    RY_share_median=r2((xb.E2_RY / xb.E2_with_RY.replace(0, np.nan)).median() * 100), RY_share_max=r2((xb.E2_RY / xb.E2_with_RY.replace(0, np.nan)).max() * 100),
    rho_G_I=rho(cov[cov.halogen == 'I'].G, cov[cov.halogen == 'I'].ref_kcal), rho_G_Br=rho(cov[cov.halogen == 'BR'].G, cov[cov.halogen == 'BR'].ref_kcal))
for acc, nm in [('acetone', 'acetone'), ('Nm3', 'NMe3')]:
    S['hal59']['series_' + nm] = [r2(h[h.name == f'{i}_ben{x}-{acc}'].E2_valence.iloc[0]) for i, x in [(19, 'Cl'), (20, 'Br'), (21, 'I')]] if acc == 'acetone' else [r2(h[h.name == f'{i}_ben{x}-{acc}'].E2_valence.iloc[0]) for i, x in [(22, 'Cl'), (23, 'Br'), (24, 'I')]]
S['hal59']['FI_pyr'] = r2(h[h.name == 'FI_pyr'].E2_valence.iloc[0])
# EDA in kcal/mol
eda = xb.eda_total_kJ / KCAL; orca = xb.orca_cp_kcal; ref = -xb.ref_kcal
S['hal59']['eda_MAE'] = r2((eda - ref).abs().mean()); S['hal59']['eda_vs_orca_max'] = r2((eda - orca).abs().max())
alleda = h.eda_total_kJ / KCAL; S['hal59']['eda_vs_orca_max_all59'] = r2((alleda - h.orca_cp_kcal).abs().max())
S['hal59']['eda_MAE_all59'] = r2((alleda + h.ref_kcal).abs().mean())
S['hal59']['chi_range'] = [r2(xb.chi_CT.min()), r2(xb.chi_CT.max())]; S['hal59']['frz_repulsive'] = int((xb.eda_frz_kJ > 0).sum())
S['hal59']['rho_CT_E'] = rho(-xb.eda_ct_kJ, xb.E2_valence)
# ------------------------------------------------------------------ BXQI (new definition) on HAL59: external check against CCSD(T)/CBS
def bxqi(G, E, C, w=dict(G=0.25, E=0.30, C=0.25), E0=2.0, C0=2.4):
    t = {}
    if G is not None and np.isfinite(G): t['G'] = max(0, min(1, G))
    if E is not None: t['E'] = 1 - np.exp(-max(0, E) / E0)
    if C is not None: t['C'] = 1 - np.exp(-max(0, -C) / C0)
    if any(v <= 0 for v in t.values()): return 0.0
    ws = sum(w[k] for k in t); return float(100 * np.exp(sum(w[k] / ws * np.log(t[k]) for k in t)))
xb = xb.copy()
xb['BXQI'] = [bxqi(r.G if r.covered else None, r.E2_valence, r.eda_total_kJ / KCAL) for r in xb.itertuples()]
xb['BXQI_GE'] = [bxqi(r.G if r.covered else None, r.E2_valence, None) for r in xb.itertuples()]
xb['BXQI_old'] = xb.BXQI_x if 'BXQI_x' in xb else h.set_index('name').loc[xb.name, 'BXQI'].values
S['bxqi_hal59'] = dict(rho_all=rho(xb.BXQI, xb.ref_kcal), rho_all_CI=boot_rho(xb.BXQI, xb.ref_kcal), rho_cov=rho(xb[xb.covered].BXQI, xb[xb.covered].ref_kcal),
    rho_GE_all=rho(xb.BXQI_GE, xb.ref_kcal), rho_old_all=rho(xb.BXQI_old, xb.ref_kcal),
    PF_examples=xb[xb.name.isin(['FI_PCH', 'FI_FMe'])][['name', 'BXQI_old', 'BXQI', 'ref_kcal']].round(1).values.tolist(), range_all=[r2(xb.BXQI.min()), r2(xb.BXQI.max())])
xb[['name', 'ref_kcal', 'covered', 'G', 'E2_valence', 'eda_total_kJ', 'BXQI_old', 'BXQI', 'BXQI_GE']].to_csv(os.path.join(V, 'bxqi_hal59_table.csv'), index=False)
# ------------------------------------------------------------------ scans
sc = pd.read_csv(os.path.join(R, 'qm_scans.csv')).merge(nbo[nbo.set.isin(['B_scan_distance', 'C_scan_angle'])][['name', 'E2_valence', 'E2_RY']], on='name')
dsc = sc[sc.scan == 'distance'].copy(); asc = sc[sc.scan == 'angle'].copy()
ref_d = dsc[dsc.s == 1.0].set_index('base').E2_valence; dsc['rel'] = dsc.E2_valence / dsc.base.map(ref_d)
ref_a = asc[asc.theta_target == 180].set_index('base').E2_valence; asc['rel'] = asc.E2_valence / asc.base.map(ref_a)
S['scans'] = dict(d120_rel=[r2(dsc[dsc.s == 1.2].rel.min() * 100), r2(dsc[dsc.s == 1.2].rel.max() * 100)], d090_rel=[r2(dsc[dsc.s == 0.9].rel.min()), r2(dsc[dsc.s == 0.9].rel.max())],
    d150_verdicts=dsc[dsc.s == 1.5].E2_valence.map(verdict).value_counts().to_dict(), monotone_d=bool(all((g.sort_values('s').E2_valence.diff().dropna() <= 1e-9).all() for _, g in dsc.groupby('base'))),
    a140_rel=[r2(asc[asc.theta_target == 140].rel.min() * 100), r2(asc[asc.theta_target == 140].rel.max() * 100)], a120_rel=[r2(asc[asc.theta_target == 120].rel.min() * 100), r2(asc[asc.theta_target == 120].rel.max() * 100)],
    a120_verdicts=asc[asc.theta_target == 120].E2_valence.map(verdict).value_counts().to_dict(), a130_verdicts=asc[asc.theta_target == 130].E2_valence.map(verdict).value_counts().to_dict(),
    a120_RYshare=[r2((asc[asc.theta_target == 120].E2_RY / (asc[asc.theta_target == 120].E2_RY + asc[asc.theta_target == 120].E2_valence)).min() * 100), r2((asc[asc.theta_target == 120].E2_RY / (asc[asc.theta_target == 120].E2_RY + asc[asc.theta_target == 120].E2_valence)).max() * 100)],
    a180_RYshare_max=r2((asc[asc.theta_target == 180].E2_RY / (asc[asc.theta_target == 180].E2_RY + asc[asc.theta_target == 180].E2_valence)).max() * 100))
# T1-positive benchmark geometries with a repulsive counterpoise energy (specificity against an energy criterion)
t1pos = pd.concat([sc[sc.G > 0][['name', 'scan', 's', 'theta_target', 'G', 'E2_valence', 'orca_BE']]])
rep = t1pos[t1pos.orca_BE < 0]
S['scans']['t1pos_n'] = int(len(t1pos)); S['scans']['t1pos_repulsive'] = rep[['name', 'G', 'E2_valence', 'orca_BE']].round(3).values.tolist()
S['scans']['t1pos_repulsive_verdicts'] = rep.E2_valence.map(verdict).value_counts().to_dict()
S['scans']['t1pos_attractive_verdicts'] = t1pos[t1pos.orca_BE >= 0].E2_valence.map(verdict).value_counts().to_dict()
# ------------------------------------------------------------------ level of theory (Set D) and grey zone
mb = pd.read_csv(os.path.join(R, 'qm_method_basis.csv')).merge(nbo[nbo.set == 'D_method_basis'][['name', 'E2_valence']], on='name')
r0 = mb[mb.level == 'wB97X-D__def2-TZVP'].set_index('base').E2_valence; mb['rel'] = mb.E2_valence / mb.base.map(r0)
lv = mb.groupby('level').rel.agg(['min', 'max']).round(2)
S['levels'] = dict(rel=lv.to_dict(orient='index'), all_range=[r2(mb[mb.level != 'wB97X-D__def2-TZVP'].rel.min()), r2(mb[mb.level != 'wB97X-D__def2-TZVP'].rel.max())],
    verdict_changes=int(sum(len(set(g.E2_valence.map(verdict))) > 1 for _, g in mb.groupby('base'))), min_val=r2(mb.E2_valence.min()),
    rho_min=r2(min(spearmanr(mb[mb.level == l].sort_values('base').E2_valence, mb[mb.level == 'wB97X-D__def2-TZVP'].sort_values('base').E2_valence).correlation for l in mb.level.unique())))
pr = pd.read_csv(os.path.join(R, 'qm_protein.csv')).merge(nbo[nbo.set == 'F_protein'][['name', 'label', 'E2_valence', 'E2_RY', 'E2_with_RY', 'hbY', 'hbFrag2', 'lpOnY']], on=['name', 'label'])
mr = []
for c, d in pr.groupby('contact'):
    if len(d) < 5: continue
    v0 = d[d.variant == 'cluster_pcm4'].E2_valence.iloc[0]
    if v0 < 0.2: continue
    mr += [(d.E2_valence / v0).min(), (d.E2_valence / v0).max()]
lo, hi = min(mr), max(mr); clo, chi = S['levels']['all_range'][0] * lo, S['levels']['all_range'][1] * hi
S['greyzone'] = dict(model_ratio=[r2(lo), r2(hi)], level_ratio=S['levels']['all_range'], combined=[r2(clo), r2(chi)], zone=[r2(0.5 / chi), r2(0.5 / clo)], used=[LO, HI])
# ------------------------------------------------------------------ protein contacts
rows = []
for c, d in pr.groupby('contact'):
    d0 = d[d.variant == 'cluster_pcm4']
    rows.append(dict(contact=c, sys=d.sys.iloc[0], n=len(d), G=r2(d.G.iloc[0]), val_default=r2(d0.E2_valence.iloc[0]) if len(d0) else None, val_min=r2(d.E2_valence.min()), val_max=r2(d.E2_valence.max()),
        RY_default=r2(d0.E2_RY.iloc[0]) if len(d0) else None, withRY_min=r2(d.E2_with_RY.min()), withRY_max=r2(d.E2_with_RY.max()), hbY_max=r2(d.hbY.max()), hbFrag2_max=r2(d.hbFrag2.max()),
        verdicts=sorted(set(d.E2_valence.map(verdict))), verdicts_withRY_vs_05=sorted(set((d.E2_with_RY >= 0.5).map({True: 'pass', False: 'fail'})))))
pt = pd.DataFrame(rows); pt.to_csv(os.path.join(V, 'protein_contacts_table.csv'), index=False)
S['protein'] = dict(contacts=pt.to_dict(orient='records'))
# QM size and environment effects (valence)
def span(d, vs): x = d[d.variant.isin(vs)]; return x.groupby('contact').E2_valence.agg(lambda s: s.max() - s.min()).max()
S['protein']['size_span'] = r2(span(pr, ['cluster_pcm4', 'shell3', 'shell4', 'shell5'])); S['protein']['ff_span'] = r2(span(pr, ['qmmm_ff14SB', 'qmmm_charmm36']))
S['protein']['boundary_span'] = r2(span(pr, ['qmmm_ff14SB', 'qmmm_Z1', 'qmmm_Z2', 'qmmm_Z3'])); S['protein']['radius_span'] = r2(span(pr, ['qmmm_r10', 'qmmm_ff14SB', 'qmmm_r20', 'qmmm_r25', 'qmmm_full']))
S['protein']['pcm_gas_span'] = r2(span(pr, ['cluster_pcm4', 'cluster_gas']))
# M12: change cluster -> QM/MM against the MM field E_AX
pm = pd.read_csv(os.path.join(R, 'protein_models.csv'))
m12 = []
for c, d in pr.groupby('contact'):
    if len(d) < 10: continue
    cl = d[d.variant == 'cluster_pcm4'].E2_valence.iloc[0]; cg = d[d.variant == 'cluster_gas'].E2_valence.iloc[0]
    for v in ['qmmm_ff14SB', 'qmmm_charmm36', 'qmmm_Z1', 'qmmm_Z2', 'qmmm_Z3', 'qmmm_r10', 'qmmm_r20', 'qmmm_r25', 'qmmm_full']:
        q = d[d.variant == v]
        if not len(q): continue
        f = pm[pm.model == q.name.iloc[0]].field_AX_VperA
        m12.append(dict(contact=c, variant=v, E_AX=float(f.iloc[0]) if len(f) else np.nan, dE_vs_gas=q.E2_valence.iloc[0] - cg, dE_vs_pcm=q.E2_valence.iloc[0] - cl, val=q.E2_valence.iloc[0]))
m12 = pd.DataFrame(m12)
# note: the field is computed at the first contact of each model; use only rows whose contact is that first contact
first = {r.model: r.xbs.split(' ; ')[0].replace(' ', '') for r in pm.itertuples() if isinstance(r.xbs, str)}
m12['first'] = [first.get(pr[(pr.contact == r.contact) & (pr.variant == r.variant)].name.iloc[0], '') == r.contact.replace(' ', '') for r in m12.itertuples()]
m12f = m12[m12['first'] & m12.E_AX.notna()]
d15 = m12f[m12f.variant == 'qmmm_ff14SB']
S['m12'] = dict(n_contacts=int(len(d15)), rho_across_default=rho(d15.E_AX, d15.dE_vs_gas), rows_default=d15[['contact', 'E_AX', 'dE_vs_gas', 'dE_vs_pcm']].round(3).values.tolist(),
    rho_all_variants=rho(m12f.E_AX, m12f.dE_vs_gas), n_all=int(len(m12f)),
    within=[(c, rho(g.E_AX, g.val), int(len(g))) for c, g in m12f.groupby('contact') if len(g) >= 5])
m12.to_csv(os.path.join(V, 'm12_field_vs_dE.csv'), index=False)
# ------------------------------------------------------------------ in-cluster pairs and amide lone pairs
pp = pd.read_csv(os.path.join(V, 'pairs_v3.csv')); am = pd.read_csv(os.path.join(V, 'amideN_v3.csv'))
S['pairs'] = dict(n=int(len(pp)), t1pass=int(pp.pass_T1.sum()), t1fail=int((~pp.pass_T1).sum()), fail_max_val=r2(pp[~pp.pass_T1].E2_valence.max()), fail_hb_max=r2(pp[~pp.pass_T1].hbY.max()),
    fail_rows=pp[~pp.pass_T1][['name', 'X', 'Y', 'd', 'theta1', 'E2_valence', 'hbY']].round(2).values.tolist(), thr113_hbY=r2(pp[pp.Y.str.contains('THR113')].hbY.max()))
S['amide'] = dict(n=int(len(am)), with_LP=int(am.lp_N.sum()), max_to_CX=r2(am.max_LP_N_to_sigmaCX.max()))
# ------------------------------------------------------------------ field (no QM)
fd = pd.read_csv(os.path.join(V, 'field_v3.csv')); fd['radius'] = fd.radius.astype(str)
def fval(**k):
    q = fd.copy()
    for a, b in k.items(): q = q[q[a].astype(str) == str(b)]
    return q.groupby('system').E_AX.first()
base15 = fval(ff='ff14SB', boundary='RC', radius=15, explicitH=0)
S['field'] = dict(explicitH_shift={s: r2(v) for s, v in (fval(ff='ff14SB', boundary='RC', radius=15, explicitH=6) - base15).items()},
    RCD_shift={s: r2(v) for s, v in (fval(ff='ff14SB', boundary='RCD', radius=15, explicitH=0) - base15).items()},
    r15_vs_full={s: r2(v) for s, v in (base15 - fval(ff='ff14SB', boundary='RC', radius='full', explicitH=0)).items()},
    r25_vs_full={s: r2(v) for s, v in (fval(ff='ff14SB', boundary='RC', radius=25, explicitH=0) - fval(ff='ff14SB', boundary='RC', radius='full', explicitH=0)).items()},
    r30_vs_full={s: r2(v) for s, v in (fval(ff='ff14SB', boundary='RC', radius=30, explicitH=0) - fval(ff='ff14SB', boundary='RC', radius='full', explicitH=0)).items()},
    r40_vs_full={s: r2(v) for s, v in (fval(ff='ff14SB', boundary='RC', radius=40, explicitH=0) - fval(ff='ff14SB', boundary='RC', radius='full', explicitH=0)).items()},
    nZero=fd[(fd.radius == '15') & (fd.boundary == 'RC') & (fd.explicitH == 0)].groupby(['system', 'ff'])[['nMM', 'nZero', 'nCharges', 'net']].first().reset_index().values.tolist(),
    full_default={s: r2(v) for s, v in fval(ff='ff14SB', boundary='RC', radius='full', explicitH=6).items()})
# ------------------------------------------------------------------ protein BXQI (G and valence E; cluster EDA not used)
pb = []
for r in pt.itertuples():
    if r.val_default is None or not np.isfinite(r.val_default): continue
    pb.append(dict(contact=r.contact, G=r.G, E=r.val_default, verdict=verdict(r.val_default), BXQI=round(bxqi(r.G, r.val_default, None), 1)))
pb = pd.DataFrame(pb); order = {'supported': 0, 'inconclusive': 1, 'not supported': 2}
pb = pb.sort_values(by=['verdict', 'BXQI'], key=lambda s: s.map(order) if s.name == 'verdict' else -s); pb.to_csv(os.path.join(V, 'bxqi_protein_table.csv'), index=False)
S['bxqi_protein'] = pb.values.tolist()
# ------------------------------------------------------------------ protein EDA and counterpoise in kcal/mol
pe = pd.read_csv(os.path.join(R, 'qm_protein_eda.csv'))
S['protein_eda'] = [[r.sys, r.lig, r2(r.frz / KCAL), r2(r.pol / KCAL), r2(r.ct / KCAL), r2(r.total / KCAL), r2(r.chi_CT), r2(r.g16_cp_kJ / KCAL), r2(r.orca_cp_kJ / KCAL)] for r in pe.itertuples()]
S['protein_eda_orca_maxdiff_kcal'] = r2(max(abs(r.total - r.orca_cp_kJ) for r in pe.itertuples()) / KCAL)
cpp = pd.read_csv(os.path.join(R, 'orca_cp_protein.csv')) if os.path.exists(os.path.join(R, 'orca_cp_protein.csv')) else None
S['protein_cp_cols'] = list(cpp.columns) if cpp is not None else None
# ------------------------------------------------------------------ hIYD
hn = pd.read_csv(os.path.join(R, 'hiyd', 'hiyd_nbo_reanalysis.csv')); hn['v3'] = hn.E2_bondx_pair.map(verdict)
S['hiyd_nbo'] = dict(crosstab=pd.crosstab(hn.lewis, hn.v3).to_dict(), supported_lewis=hn[(hn.lewis == 'strong') & (hn.v3 == 'supported')][['pathway', 'residue', 'acceptor_atom', 'E2_bondx_pair', 'hb_LPI_to_YH']].values.tolist(),
    supported_poor=hn[(hn.lewis == 'weak') & (hn.v3 == 'supported')][['pathway', 'residue', 'acceptor_atom', 'E2_bondx_pair']].values.tolist(),
    backbone_N_LP=int(hn[hn.acceptor_class.str.startswith('backbone N')].LP_on_acceptor.sum()), backbone_N_n=int(hn.acceptor_class.str.startswith('backbone N').sum()),
    fmn_hb=r2(hn[hn.residue == 'FMN'].hb_LPI_to_YH.max()), ge05=int((hn.E2_bondx_pair >= 0.5).sum()), ge05_lewis=int(((hn.E2_bondx_pair >= 0.5) & (hn.lewis == 'strong')).sum()))
t2 = pd.read_csv(os.path.join(V, 'hiyd_t2_v3.csv'))
S['hiyd_t2'] = dict(rows=int(len(t2)), windows_with_contact=int(t2.groupby(['pathway', 'xi']).ngroups), occ_max=r2(t2.occupancy.max()), n_ge_010=int((t2.occupancy >= 0.10).sum()),
    pocket_occ_max=r2(t2[t2.xi <= 8.8].occupancy.max()), tau_range=[r2(t2.tau_ps.min()), r2(t2.tau_ps.max())], tau_median=r2(t2.tau_ps.median()), tau_max=r2(t2.tau_max_ps.max()),
    top=t2.sort_values('D', ascending=False).head(10)[['pathway', 'xi', 'acceptor', 'atom', 'lewis', 'occupancy', 'tau_ps', 'D', 'G_best']].values.tolist(), dt_ps=sorted(t2.dt_ps.unique().tolist()))
# T5 on the trajectory: best persistent Lewis-base contacts with a published NBO pose of the same residue, atom and pathway
def amber(acc): return acc.split(':')[0]
hn['key'] = hn.pathway + '|' + hn.residue_amber.str.extract(r'([A-Z]+)(\d+)').apply(lambda r: r[0] + str(int(r[1]) + 1), axis=1) + '|' + hn.acceptor_atom
t2['key'] = t2.pathway + '|' + t2.acceptor.str.split(':').str[0] + '|' + t2.atom
best = t2.sort_values('D', ascending=False).drop_duplicates('key').merge(hn[['key', 'E2_bondx_pair', 'xi']].rename(columns={'xi': 'xi_pub'}), on='key')
best['BXQI_GDE'] = [round(bxqi(r.G_best, r.E2_bondx_pair, None, w=dict(G=0.25, D=0.20, E=0.30)) if False else 0, 1) for r in best.itertuples()]
def bx3(G, D, E):
    t = dict(G=G, D=D, E=1 - np.exp(-max(0, E) / 2.0)); w = dict(G=0.25, D=0.20, E=0.30)
    if any(v <= 0 for v in t.values()): return 0.0
    return float(100 * np.exp(sum(w[k] / sum(w.values()) * np.log(t[k]) for k in t)))
best['BXQI_GDE'] = [round(bx3(r.G_best, r.D, r.E2_bondx_pair), 1) for r in best.itertuples()]; best['verdict'] = best.E2_bondx_pair.map(verdict)
best[['key', 'xi', 'occupancy', 'tau_ps', 'D', 'G_best', 'xi_pub', 'E2_bondx_pair', 'verdict', 'BXQI_GDE']].to_csv(os.path.join(V, 'hiyd_t5_table.csv'), index=False)
S['hiyd_t5'] = best[['key', 'xi', 'occupancy', 'D', 'G_best', 'E2_bondx_pair', 'verdict', 'BXQI_GDE']].round(3).values.tolist()
json.dump(S, open(os.path.join(V, 'summary_v3.json'), 'w'), indent=1, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
print(json.dumps(S, indent=1, default=lambda o: o.item() if hasattr(o, 'item') else str(o))[:200])


### `22_v3_figures.py`

22_v3_figures.py - every data figure of Manuscript_V3 and ESI_V3 (Arial style of bxstyle3.py, legends without frames).
Usage: python3 22_v3_figures.py ..      Output: MS_Figures_V3/ and ESI_Figures_V3/ (PDF + PNG).
The workflow chart (Fig. 1) and the TOC graphic are TikZ files; Fig. 2 and Fig. S13 are screenshots (script 23);
Fig. 3 comes from 12_parameter_figure.py.

In [ ]:
%%writefile Scripts/22_v3_figures.py
#!/usr/bin/env python3
"""22_v3_figures.py - every data figure of Manuscript_V3 and ESI_V3 (Arial style of bxstyle3.py, legends without frames).
Usage: python3 22_v3_figures.py ..      Output: MS_Figures_V3/ and ESI_Figures_V3/ (PDF + PNG).
The workflow chart (Fig. 1) and the TOC graphic are TikZ files; Fig. 2 and Fig. S13 are screenshots (script 23);
Fig. 3 comes from 12_parameter_figure.py."""
import os, sys, re, json, math
import numpy as np, pandas as pd
from scipy.stats import spearmanr
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from bxstyle3 import plt, C, HALC, VCOL, GREY, INK, INK2, MUTE, save, grey
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle

ROOT = sys.argv[1] if len(sys.argv) > 1 else '..'
R = os.path.join(ROOT, 'results'); V = os.path.join(R, 'v3')
MS = os.path.join(ROOT, 'MS_Figures_V3'); ESI = os.path.join(ROOT, 'ESI_Figures_V3')
KC = 4.184; LO, HI = 0.3, 0.8
S = json.load(open(os.path.join(V, 'summary_v3.json')))
verdict = lambda e: 'supported' if e >= HI else ('inconclusive' if e >= LO else 'not supported')
SHAPE = {'N': 'o', 'O': 's', 'S': 'D', 'AR': '^', 'π': '^', 'P': 'v', 'F': '<'}
HNAME = {'CL': 'Cl', 'BR': 'Br', 'I': 'I'}
nbo = pd.read_csv(os.path.join(V, 'nbo_v3.csv'))
h = pd.read_csv(os.path.join(R, 'qm_hal59.csv')).merge(nbo[nbo.set == 'A_HAL59'][['name', 'E2_valence', 'E2_RY']], on='name')
t1 = pd.read_csv(os.path.join(R, 't1_hal59.csv'))

def hal_legend(ax, loc='upper left', extra=()):
    hs = [Line2D([], [], marker='o', ls='', color=HALC[x], label=HNAME[x]) for x in ['I', 'BR', 'CL']]
    hs += [Line2D([], [], marker=SHAPE[a], ls='', color=MUTE, mfc='white' if a in 'PF' else MUTE, label=l) for a, l in [('N', 'N'), ('O', 'O'), ('S', 'S'), ('AR', 'ring')]]
    ax.legend(handles=hs + list(extra), loc=loc, ncol=2, handletextpad=0.3, columnspacing=0.8, fontsize=9.5)

# ---------------------------------------------------------------- Fig. 4: T1 on HAL59 and parameter sensitivity
def fig4():
    fig, (a, b) = plt.subplots(1, 2, figsize=(10.4, 4.0), gridspec_kw=dict(width_ratios=[1.15, 1]))
    for r in t1.itertuples():
        if r.role != 'halogen bond': continue
        acc = 'AR' if isinstance(r.ring, str) or (isinstance(r.ring, float) and not math.isnan(r.ring)) else r.acc_el
        if r.covered: a.scatter(r.ref_kcal, r.G, c=HALC[r.halogen], marker=SHAPE.get(acc, 'o'), s=40, edgecolor='white', lw=0.5, zorder=3)
        else: a.scatter(r.ref_kcal, -0.04, c=HALC[r.halogen], marker='x', s=30, lw=1.2, zorder=3)
    a.set_xscale('log'); a.set_xlabel('CCSD(T)/CBS binding energy (kcal/mol)'); a.set_ylabel('geometric score $G$'); a.set_ylim(-0.08, 1.04)
    a.set_title('(a) HAL59 halogen bonds')
    hal_legend(a, 'lower right', [Line2D([], [], marker='x', ls='', color=MUTE, label='P, F acceptor (no cut-off)')])
    a.text(0.03, 0.40, r'Spearman $\rho$ = %.2f [%.2f, %.2f], $n$ = 41' % (S['hal59']['rho_G_cov'], *S['hal59']['rho_G_cov_CI']), transform=a.transAxes, va='top', fontsize=10)
    sens = pd.read_csv(os.path.join(R, 't1_sensitivity.csv'))
    piv = sens.pivot(index='theta1Min', columns='cutoff_shift', values='protein_XB'); rec = sens.pivot(index='theta1Min', columns='cutoff_shift', values='literature_found')
    im = b.imshow(piv.values, cmap='Blues', aspect='auto', origin='lower', vmin=0, vmax=piv.values.max() * 1.25)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            b.text(j, i, f'{piv.values[i, j]}\n{rec.values[i, j]}/4', ha='center', va='center', fontsize=9, color='white' if piv.values[i, j] > 14 else INK)
    b.set_xticks(range(piv.shape[1])); b.set_xticklabels([f'{x:+.1f}' for x in piv.columns]); b.set_yticks(range(piv.shape[0])); b.set_yticklabels([f'{x}°' for x in piv.index])
    j0 = list(piv.columns).index(0.0); i0 = list(piv.index).index(140); b.add_patch(Rectangle((j0 - 0.5, i0 - 0.5), 1, 1, fill=False, ec=C[1], lw=2.2))
    b.set_xlabel('shift of all distance cut-offs (Å)'); b.set_ylabel(r'lower limit of $\theta_1$'); b.grid(False); b.set_title('(b) protein test set: contacts, literature recovered')
    fig.tight_layout(); save(fig, MS, 'Fig4_T1_benchmark')

# ---------------------------------------------------------------- Fig. 5: MM field (explicit H, radius) and its link to SigmaE(2)
SYSN = {'1us0_idd594': '1US0', '2w0s': '2W0S', '1p5e': '1P5E', '4agl': '4AGL', '4gql': '4GQL', '4dst': '4DST'}
def fig5():
    fd = pd.read_csv(os.path.join(V, 'field_v3.csv')); fd['r'] = fd.radius.replace('full', 60).astype(float)
    fig, axs = plt.subplots(1, 3, figsize=(12.2, 3.9), gridspec_kw=dict(width_ratios=[1.25, 1, 1]))
    a = axs[0]
    for k, s in enumerate(SYSN):
        d = fd[(fd.system == s) & (fd.ff == 'ff14SB') & (fd.boundary == 'RC')]; d = d[d.contact == d.contact.iloc[0]]
        for eh, ls, mk in [(0, '--', 'o'), (6, '-', 's')]:
            q = d[d.explicitH == eh].sort_values('r'); a.plot(q.r, q.E_AX, ls=ls, marker=mk, ms=4, color=C[k], label=SYSN[s] if eh == 6 else None)
    a.set_xticks([10, 15, 20, 25, 30, 40, 60]); a.set_xticklabels(['10', '15', '20', '25', '30', '40', 'all'])
    a.set_xlabel('embedding radius (Å)'); a.set_ylabel(r'$E_{AX}$ (V Å$^{-1}$)'); a.set_title('(a) field along A→X')
    a.legend(ncol=3, fontsize=9, loc='upper center', bbox_to_anchor=(0.5, -0.2), handlelength=1.6); a.text(0.5, -0.43, 'solid: explicit MM hydrogens; dashed: folded hydrogens', transform=a.transAxes, ha='center', va='top', fontsize=9, color=INK2)
    b = axs[1]; keys = ['explicitH_shift', 'RCD_shift', 'r15_vs_full']; lab = ['explicit H − folded', 'RCD − RC', '15 Å − whole']
    x = np.arange(len(SYSN)); w = 0.26
    for k, (key, l) in enumerate(zip(keys, lab)): b.bar(x + (k - 1) * w, [S['field'][key][s] for s in SYSN], w, color=[C[2], C[6], C[3]][k], label=l)
    b.axhline(0, color=INK2, lw=0.8); b.set_xticks(x); b.set_xticklabels(list(SYSN.values()), rotation=40); b.set_ylabel(r'$\Delta E_{AX}$ (V Å$^{-1}$)'); b.set_title('(b) effect of each setting'); b.legend(fontsize=9, loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2)
    c = axs[2]; m = pd.read_csv(os.path.join(V, 'm12_field_vs_dE.csv')); m = m[m['first'] & m.E_AX.notna()]
    lab = {'BVP': '2W0S', 'LDT': '1US0', 'P84': '4AGL', 'R47': '4GQL', 'TBS': '1P5E', '9LI': '4DST'}
    for k, (ct, g) in enumerate(m.groupby('contact')):
        nm = lab[ct[:3]]; col = C[list(SYSN.values()).index(nm)]
        c.scatter(g.E_AX, g.val, color=col, s=26, label=nm, edgecolor='white', lw=0.4)
    c.set_xlabel(r'$E_{AX}$ (V Å$^{-1}$)'); c.set_ylabel(r'valence $\Sigma E^{(2)}$ (kcal/mol)'); c.set_yscale('log'); c.set_title('(c) QM/MM variants'); c.legend(fontsize=9, ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.2))
    c.text(0.5, -0.43, r'within a contact: $\rho$ = %.2f–%.2f' % (min(x[1] for x in S['m12']['within'] if x[1] == x[1]), max(x[1] for x in S['m12']['within'] if x[1] == x[1])) + r'; across systems: $\rho$ = %.2f' % S['m12']['rho_across_default'], transform=c.transAxes, va='top', ha='center', fontsize=9, color=INK2)
    fig.tight_layout(); save(fig, MS, 'Fig5_embedding_field')

# ---------------------------------------------------------------- Fig. 6: ORCA counterpoise benchmark
def fig6():
    cp = pd.read_csv(os.path.join(R, 'orca_cp_hal59.csv')); sc = pd.read_csv(os.path.join(R, 'orca_cp_scans.csv'))
    fig, axs = plt.subplots(1, 3, figsize=(12.2, 3.8)); a = axs[0]
    acc = {r.name: ('AR' if isinstance(r.ring, str) else r.acc_el) for r in t1.itertuples()}
    for r in cp.itertuples():
        hal = r.halogen.upper(); ctl = r.role != 'halogen bond'; mk = SHAPE.get(acc.get(r.name, r.acc), 'o')
        a.scatter(r.ref_kcal, r.BE_kcal, marker=mk, s=36, color=HALC.get(hal, MUTE), facecolors='white' if ctl else HALC.get(hal, MUTE), edgecolor=HALC.get(hal, MUTE), lw=0.9, zorder=3)
    lim = [-1, 22]; a.plot(lim, lim, ':', color=INK2, lw=1); a.set_xlim(lim); a.set_ylim(lim)
    a.set_xlabel('CCSD(T)/CBS binding energy (kcal/mol)'); a.set_ylabel('ORCA counterpoise (kcal/mol)'); a.set_title('(a) HAL59')
    a.text(0.04, 0.96, 'MAE 0.40, RMSE 0.54 kcal/mol\nmean signed error −0.07 kcal/mol', transform=a.transAxes, va='top', fontsize=9.5); hal_legend(a, 'lower right')
    for ax, scan, xcol, xl in [(axs[1], 'distance', 's', r'$d/d_\mathrm{eq}$'), (axs[2], 'angle', 'theta', r'$\theta_1$ (°)')]:
        d = sc[sc.scan == scan]
        for k, (bname, g) in enumerate(d.groupby('base')):
            g = g.sort_values(xcol); col = HALC[g.halogen.iloc[0].upper()]
            ax.plot(g[xcol], g.BE_rel, '-', color=col, lw=1.2, alpha=0.9)
            ax.scatter(g[xcol], g.BE_rel, s=18, color=col, facecolors=np.where(g.G > 0, col, 'white'), edgecolor=col, zorder=3)
        ax.axhline(0, color=INK2, lw=0.8); ax.set_xlabel(xl); ax.set_ylabel('binding energy / reference value')
        if scan == 'angle': ax.axvspan(118, 140, color='#eeeeea', zorder=0); ax.invert_xaxis(); ax.set_title(r'(c) angle scans (open: $G$ = 0)')
        else: ax.set_title(r'(b) distance scans (open: $G$ = 0)')
    axs[1].legend(handles=[Line2D([], [], color=HALC[x], label=HNAME[x]) for x in ['I', 'BR', 'CL']], loc='upper right')
    fig.tight_layout(); save(fig, MS, 'Fig6_CP_benchmark')

# ---------------------------------------------------------------- Fig. 7: NBO screen (valence) and ALMO-EDA on the benchmark
def fig7():
    sc = pd.read_csv(os.path.join(R, 'qm_scans.csv')).merge(nbo[nbo.set.isin(['B_scan_distance', 'C_scan_angle'])][['name', 'E2_valence', 'E2_RY']], on='name')
    fig, axs = plt.subplots(1, 3, figsize=(12.4, 4.0)); a = axs[0]
    acc = {r.name: ('AR' if isinstance(r.ring, str) else r.acc_el) for r in t1.itertuples()}
    grey(a)
    for r in h.itertuples():
        hal = r.halogen; open_ = (r.role != 'halogen bond') or (not r.covered); mk = SHAPE.get(acc.get(r.name, r.acc_el), 'o')
        y = max(r.E2_valence, 0.012); col = HALC.get(hal, MUTE)
        a.scatter(max(r.ref_kcal, 0.03), y, marker=mk, s=38, color=col, facecolors='white' if open_ else col, edgecolor=col, lw=0.9, zorder=3)
    a.set_xscale('log'); a.set_yscale('log'); a.set_ylim(0.01, 150); a.set_xlabel('CCSD(T)/CBS binding energy (kcal/mol)'); a.set_ylabel(r'valence $\Sigma E^{(2)}$ (kcal/mol)')
    a.set_title('(a) HAL59'); a.text(0.03, 0.97, r'$\rho$ = %.2f [%.2f, %.2f], $n$ = 41' % (S['hal59']['rho_E_cov'], *S['hal59']['rho_E_cov_CI']), transform=a.transAxes, va='top', fontsize=9.5)
    hal_legend(a, 'lower right', [Line2D([], [], marker='o', ls='', mfc='white', mec=MUTE, color=MUTE, label='open: control, P, or F')])
    b = axs[1]; grey(b, label=False); d = sc[sc.scan == 'distance']
    for bname, g in d.groupby('base'):
        g = g.sort_values('s'); col = HALC[g.halogen.iloc[0]]; b.plot(g.s, g.E2_valence.clip(lower=0.012), '-o', ms=3.5, color=col, lw=1.2)
    b.set_yscale('log'); b.set_ylim(0.01, 150); b.set_xlabel(r'$d/d_\mathrm{eq}$'); b.set_ylabel(r'valence $\Sigma E^{(2)}$ (kcal/mol)'); b.set_title('(b) distance scans')
    b.legend(handles=[Line2D([], [], color=HALC[x], label=HNAME[x]) for x in ['I', 'BR', 'CL']] + [Patch(color=GREY, label='inconclusive 0.3–0.8')], loc='upper right', fontsize=9)
    c = axs[2]; x = h[h.role == 'halogen bond']
    for r in x.itertuples():
        c.scatter(r.ref_kcal, -r.eda_total_kJ / KC, color=HALC[r.halogen], marker=SHAPE.get(acc.get(r.name, r.acc_el), 'o'), s=34, edgecolor='white', lw=0.4, zorder=3)
    lim = [-1, 22]; c.plot(lim, lim, ':', color=INK2, lw=1); c.set_xlim(lim); c.set_ylim(lim); c.set_xlabel('CCSD(T)/CBS binding energy (kcal/mol)'); c.set_ylabel('−ALMO-EDA total (kcal/mol)')
    c.set_title('(c) ALMO-EDA'); c.text(0.04, 0.96, 'MAE 0.39 kcal/mol (59 dimers)', transform=c.transAxes, va='top', fontsize=9.5)
    fig.tight_layout(); save(fig, MS, 'Fig7_NBO_benchmark')

# ---------------------------------------------------------------- Fig. 8: protein contacts
CONTACTS = [('BVP1207:A@BR···ASN65:A@OD1', '2W0S Br···OD1 Asn65'), ('TBS301:A@BR11···LEU83:A@O', '1P5E Br11···O Leu83'), ('P84400:A@I15···LEU145:A@O', '4AGL I···O Leu145'),
            ('TBS301:A@BR12···GLU81:A@O', '1P5E Br12···O Glu81'), ('R47306:A@CL1···PRO232:A@O', '4GQL Cl···O Pro232'), ('TBS301:A@BR13···PHE80:A@π(ring)', '1P5E Br13···π Phe80'),
            ('LDT320:A@BR8···THR113:A@OG1', '1US0 Br···OG1 Thr113*'), ('TBS301:A@BR10···ILE10:A@O', '1P5E Br10···O Ile10'), ('9LI201:A@CL2···GLY75:A@N', '4DST Cl···N Gly75')]
VARS = [('cluster_gas', 'gas'), ('cluster_pcm4', 'PCM ε=4'), ('shell3', '+3 Å'), ('shell4', '+4 Å'), ('shell5', '+5 Å'), ('qmmm_ff14SB', 'ff14SB'), ('qmmm_charmm36', 'CHARMM36'),
        ('qmmm_Z1', 'Z1'), ('qmmm_Z2', 'Z2'), ('qmmm_Z3', 'Z3'), ('qmmm_r10', '10 Å'), ('qmmm_r20', '20 Å'), ('qmmm_r25', '25 Å'), ('qmmm_full', 'all MM')]
def fig8():
    pr = pd.read_csv(os.path.join(R, 'qm_protein.csv')).merge(nbo[nbo.set == 'F_protein'][['name', 'label', 'E2_valence', 'E2_RY', 'hbY', 'hbFrag2']], on=['name', 'label'])
    pr['hbFlag'] = (pr[['hbY', 'hbFrag2']].max(axis=1) >= 0.5) & (pr[['hbY', 'hbFrag2']].max(axis=1) > pr.E2_valence)
    fig = plt.figure(figsize=(12.0, 9.4)); gs = fig.add_gridspec(2, 2, width_ratios=[4.3, 1.25], height_ratios=[1.2, 1], hspace=0.62, wspace=0.07)
    a = fig.add_subplot(gs[0, 0]); M = np.full((len(CONTACTS), len(VARS)), np.nan); FL = set()
    for i, (c, _) in enumerate(CONTACTS):
        for j, (v, _) in enumerate(VARS):
            q = pr[(pr.contact == c) & (pr.variant == v)]
            if len(q): M[i, j] = q.E2_valence.iloc[0]
            if len(q) and q.hbFlag.iloc[0]: FL.add((i, j))
    cmap = plt.get_cmap('Blues'); a.imshow(np.log10(np.clip(M, 0.03, 10)), cmap=cmap, vmin=-1.8, vmax=1.0, aspect='auto')
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M[i, j]; vd = verdict(v)
            a.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=9.3, color='white' if v > 1.0 else INK, fontweight='bold' if vd != 'supported' else 'normal')
            if vd != 'supported': a.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec=VCOL[vd], lw=1.8))
            if (i, j) in FL: a.text(j + 0.42, i - 0.3, '†', ha='right', va='center', fontsize=10, color='white' if v > 1.0 else INK)
    for xb in [1.5, 4.5, 6.5, 9.5]: a.axvline(xb, color='white', lw=3)
    a.set_xticks(range(len(VARS))); a.set_xticklabels([l for _, l in VARS], rotation=45, ha='right'); a.set_yticks(range(len(CONTACTS))); a.set_yticklabels([l for _, l in CONTACTS]); a.grid(False)
    for x0, l in [(0.5, 'cluster'), (3, 'QM size'), (5.5, 'force field'), (8, 'boundary'), (11.5, 'MM radius')]: a.text(x0, -0.85, l, ha='center', fontsize=10.5, color=INK2)
    a.set_title('(a) valence $\\Sigma E^{(2)}$ (kcal/mol) per contact and model', pad=22)
    a.legend(handles=[Patch(fc='white', ec=VCOL['inconclusive'], lw=1.8, label='inconclusive (0.3–0.8)'), Patch(fc='white', ec=VCOL['not supported'], lw=1.8, label='not supported (< 0.3)'), Line2D([], [], ls='none', marker='$†$', color=INK, ms=9, label='competing H-bond dominates')], loc='upper center', bbox_to_anchor=(0.5, -0.24), ncol=3)
    b = fig.add_subplot(gs[0, 1]); pb = pd.read_csv(os.path.join(V, 'bxqi_protein_table.csv'))
    lab = dict(CONTACTS); pb = pb[pb.contact.isin(lab)]; pb['lab'] = pb.contact.map(lab)
    order = [l for _, l in CONTACTS]; pb = pb.set_index('lab').loc[order].reset_index()
    b.barh(range(len(pb)), pb.BXQI, color=[VCOL[v] for v in pb.verdict], height=0.62)
    for i, r in pb.iterrows(): b.text(r.BXQI + 2, i, f'{r.BXQI:.0f}', va='center', fontsize=9.5)
    b.set_yticks(range(len(pb))); b.set_yticklabels([]); b.set_ylim(len(pb) - 0.5, -0.5); b.set_xlim(0, 112); b.set_xlabel('BXQI (G·E)'); b.set_title('(b) ranking index', pad=22)
    b.legend(handles=[Patch(color=VCOL[k], label=k) for k in ['supported', 'inconclusive', 'not supported']], loc='upper center', bbox_to_anchor=(0.45, -0.24), fontsize=9)
    c = fig.add_subplot(gs[1, :]); pe = S['protein_eda']; names = [f"{SYSN[r[0]]} {r[1][:3]} (chain {r[1][3]})" for r in pe]
    y = np.arange(len(pe)); w = 0.26
    for k, (idx, nm, col) in enumerate([(2, 'FRZ', '#9a9993'), (3, 'POL', C[2]), (4, 'CT', C[0])]): c.barh(y + (k - 1) * w, [r[idx] for r in pe], w, color=col, label=nm)
    c.scatter([r[5] for r in pe], y, marker='D', color=INK, s=34, zorder=4, label='total')
    c.axvline(0, color=INK2, lw=0.8); c.set_yticks(y); c.set_yticklabels(names); c.invert_yaxis(); c.set_xlabel('ALMO-EDA term (kcal/mol, gas phase)'); c.legend(ncol=4, loc='upper center', bbox_to_anchor=(0.5, -0.2))
    c.set_title('(c) ALMO-EDA of six clusters (every ligand–fragment contact, not the halogen bond alone)')
    save(fig, MS, 'Fig8_protein_QM')

# ---------------------------------------------------------------- Fig. 9: hIYD
def fig9():
    occ = pd.read_csv(os.path.join(R, 'hiyd', 'hiyd_window_occupancy.csv')); cls = pd.read_csv(os.path.join(R, 'hiyd', 'hiyd_acceptor_classes.csv'))
    hn = pd.read_csv(os.path.join(R, 'hiyd', 'hiyd_nbo_reanalysis.csv'))
    fig = plt.figure(figsize=(12.2, 7.6)); gs = fig.add_gridspec(2, 3, hspace=0.5, wspace=0.32)
    for k, p in enumerate('ABC'):
        a = fig.add_subplot(gs[0, k]); d = occ[occ.pathway == p].sort_values('xi')
        a.axvspan(9.3, 19.3, color='#f2f1ec', zorder=0)
        a.plot(d.xi, d.pub_frac_10ns * 100, 'o', mfc='white', mec=INK2, ms=5, label='published screen (10 ns)')
        a.plot(d.xi, d.bondx_T1_frac * 100, '-', color=C[0], label='BondX T1 (25 ns)'); a.plot(d.xi, d.bondx_strict_frac * 100, '-', color=C[1], label='Lewis-base acceptors')
        a.set_xlabel(r'$\xi$ (Å)'); a.set_ylabel('frames with a contact (%, per window)'); a.set_title(f'({"abc"[k]}) pathway {p}'); a.set_ylim(-2, 45)
        if k == 0: a.legend(loc='upper left', fontsize=9)
    a = fig.add_subplot(gs[1, 0]); cls = cls.sort_values('total'); poor = cls['class'].str.contains('N–H|N-H|no lone|side-chain N-H|backbone N', regex=True)
    cls['class'] = cls['class'].replace({'side-chain N-H (no lone pair)': 'side-chain N–H (poor base)', 'backbone N-H': 'backbone N–H (amide)', 'side-chain N (lone pair)': 'side-chain N (His)'})
    a.barh(cls['class'], cls.total, color=[C[1] if p else C[0] for p in poor], hatch=None)
    for i, (t, p) in enumerate(zip(cls.total, poor)): a.text(t + 3, i, str(t), va='center', fontsize=9.5)
    a.set_xlabel('T1 contacts (all frames)'); a.set_title('(d) acceptor classes'); a.legend(handles=[Patch(color=C[0], label='Lewis base'), Patch(color=C[1], label='poor Lewis base')], loc='lower right', fontsize=9)
    a = fig.add_subplot(gs[1, 1:]); grey(a, 'x')
    src = hn.E_pub_source.fillna('none')
    for s_, col, lab in [('intra', C[1], 'published value = intramolecular O→σ*(C–I)'), ('inter', C[0], 'published value = intermolecular'), ('none', MUTE, 'no term')]:
        q = hn[src == s_]
        for r in q.itertuples(): a.scatter(r.E2_bondx_pair, r.E_pub, marker='D' if r.lewis == 'weak' else 'o', s=36, color=col, facecolors='white' if r.lewis == 'weak' else col, edgecolor=col, lw=1, zorder=3)
        a.scatter([], [], color=col, label=lab)
    a.scatter([], [], marker='D', facecolors='white', edgecolor=INK2, label='poor Lewis base (published model)')
    up = pd.read_csv(os.path.join(V, 'hiyd_decisions_updated.csv')); bb = up[up.source.str.startswith('acetyl')]
    for r in bb.itertuples():
        a.annotate('', xy=(r.E2_used, r.E_pub), xytext=(r.E2_bondx_pair, r.E_pub), arrowprops=dict(arrowstyle='->', color=INK2, lw=0.7, shrinkA=3, shrinkB=2), zorder=2)
        a.scatter(r.E2_used, r.E_pub, marker='D', s=22, color=INK2, zorder=4)
    a.scatter([], [], marker='D', s=22, color=INK2, label='backbone N with the amide restored (acetyl cap)')
    a.set_xlabel(r'BondX valence $\Sigma E^{(2)}$ of the T1 pair (kcal/mol)'); a.set_ylabel('published NBO value (kcal/mol)'); a.set_title('(e) re-analysis of the 56 published NBO outputs')
    a.legend(loc='lower right', fontsize=8.6); a.set_xlim(-0.1, 2.8); a.set_ylim(0.3, 2.8)
    save(fig, MS, 'Fig9_hIYD')

# ---------------------------------------------------------------- ESI figures
def figS1():
    fig, axs = plt.subplots(1, 4, figsize=(12.2, 2.9)); R_ = np.linspace(0.6, 1.2, 300); rc = 3.68 / 3.37
    gd = np.clip((rc - R_) / (rc - 0.85), 0, 1); axs[0].plot(R_, gd, color=C[0]); axs[0].set_xlabel(r'$R = d/(r_X + r_Y)$'); axs[0].set_ylabel(r'$g_d$'); axs[0].set_title('(a) distance (Br···O)')
    t = np.linspace(120, 180, 200); axs[1].plot(t, np.clip((t - 140) / 40, 0, 1), color=C[0]); axs[1].set_xlabel(r'$\theta_1$ (°)'); axs[1].set_ylabel(r'$g_\theta$'); axs[1].set_title('(b) σ-hole angle')
    t2 = np.linspace(60, 180, 200); axs[2].plot(t2, np.exp(-((t2 - 120) / 40) ** 2), color=C[0]); axs[2].set_xlabel(r'$\theta_2$ (°)'); axs[2].set_ylabel(r'$g_2$'); axs[2].set_title('(c) lone-pair acceptor')
    al = np.linspace(0, 90, 200); axs[3].plot(al, np.clip(1 - al / 60, 0, 1), color=C[0]); axs[3].set_xlabel(r'$\alpha$ (°)'); axs[3].set_ylabel(r'$g_2$'); axs[3].set_title('(d) ring acceptor')
    fig.tight_layout(); save(fig, ESI, 'FigS1_score_functions')

def figS2():
    s = pd.read_csv(os.path.join(R, 't1_scans.csv')); fig, axs = plt.subplots(1, 4, figsize=(12.2, 3.0), sharey=True)
    for k, hal in enumerate(['CL', 'BR', 'I']):
        d = s[(s.scan == 'distance') & (s.halogen == hal)]
        for j, (b_, g) in enumerate(d.groupby('base')): g = g.sort_values('s'); axs[k].plot(g.s, g.G, '-o', ms=3.5, color=C[j % 8], label=b_.split('_', 1)[1].replace('Nm3', 'NMe3'))
        axs[k].set_xlabel(r'$d/d_\mathrm{eq}$'); axs[k].set_title(f'({"abc"[k]}) {HNAME[hal]} donors'); axs[k].legend(fontsize=7.5, loc='upper right')
    axs[0].set_ylabel('$G$'); d = s[s.scan == 'angle']
    for j, (b_, g) in enumerate(d.groupby('base')): g = g.sort_values('theta_target'); axs[3].plot(g.theta_target, g.G, '-o', ms=3.5, color=C[j % 8], label=b_.split('_', 1)[1].replace('Nm3', 'NMe3'))
    axs[3].axvline(140, ls=':', color=INK2); axs[3].set_xlabel(r'$\theta_1$ (°)'); axs[3].set_title('(d) angle scans'); axs[3].legend(fontsize=7.5); axs[3].invert_xaxis()
    fig.tight_layout(); save(fig, ESI, 'FigS2_scans_T1')

def figS3_S4():
    pm = pd.read_csv(os.path.join(R, 'protein_models.csv'))
    fig, (a, b) = plt.subplots(1, 2, figsize=(11.0, 3.6), gridspec_kw=dict(width_ratios=[1, 1.5]))
    for k, s in enumerate(SYSN):
        q = pm[(pm.system == s) & pm.variant.isin(['cluster_pcm4', 'shell3', 'shell4', 'shell5'])]
        xs = q.variant.map({'cluster_pcm4': 0, 'shell3': 3, 'shell4': 4, 'shell5': 5}); a.plot(xs, q.n_atoms, '-o', color=C[k], label=SYSN[s])
    a.set_xlabel('extra QM shell (Å)'); a.set_ylabel('QM atoms'); a.set_title('(a) QM-region size'); a.legend(ncol=2, fontsize=9)
    fd = pd.read_csv(os.path.join(V, 'field_v3.csv')); fd['r'] = fd.radius.replace('full', 60).astype(float)
    for k, s in enumerate(SYSN):
        d = fd[(fd.system == s) & (fd.ff == 'ff14SB') & (fd.boundary == 'RC') & (fd.explicitH == 6)]; d = d[d.contact == d.contact.iloc[0]].sort_values('r')
        b.plot(d.r, d.phi_X, '-o', ms=4, color=C[k], label=SYSN[s])
    b.set_xticks([10, 15, 20, 25, 30, 40, 60]); b.set_xticklabels(['10', '15', '20', '25', '30', '40', 'all']); b.set_xlabel('embedding radius (Å)'); b.set_ylabel(r'$\phi_X$ (V)'); b.set_title('(b) MM potential at the halogen'); b.legend(ncol=3, fontsize=9)
    fig.tight_layout(); save(fig, ESI, 'FigS3_size_potential')

def figS4_angles():
    sc = pd.read_csv(os.path.join(R, 'qm_scans.csv')).merge(nbo[nbo.set == 'C_scan_angle'][['name', 'E2_valence', 'E2_RY']], on='name'); d = sc[sc.scan == 'angle']
    fig, axs = plt.subplots(1, 4, figsize=(12.4, 3.1))
    for j, (b_, g) in enumerate(d.groupby('base')):
        g = g.sort_values('theta_target'); r0 = g[g.theta_target == 180].iloc[0]; col = C[j]
        axs[0].plot(g.theta_target, g.E2_valence / r0.E2_valence, '-o', ms=3.5, color=col, label=b_.split('_', 1)[1].replace('Nm3', 'NMe3'))
        axs[1].plot(g.theta_target, 100 * g.E2_RY / (g.E2_RY + g.E2_valence), '-o', ms=3.5, color=col)
        axs[2].plot(g.theta_target, g.eda_ct / r0.eda_ct, '-o', ms=3.5, color=col); axs[3].plot(g.theta_target, g.orca_BE / r0.orca_BE, '-o', ms=3.5, color=col)
    for ax, t, yl in zip(axs, ['(a) valence ΣE(2)', '(b) Rydberg share', '(c) ALMO-EDA CT', '(d) ORCA binding energy'], ['value / value at 180°', 'n→RY* share (%)', 'value / value at 180°', 'value / value at 180°']):
        ax.axvspan(118, 140, color='#eeeeea', zorder=0); ax.invert_xaxis(); ax.set_xlabel(r'$\theta_1$ (°)'); ax.set_title(t); ax.set_ylabel(yl)
    axs[3].axhline(0, color=INK2, lw=0.8); axs[0].legend(fontsize=7.5, loc='lower left'); fig.tight_layout(); save(fig, ESI, 'FigS4_angle_scan_QM')

def figS5_levels():
    mb = pd.read_csv(os.path.join(R, 'qm_method_basis.csv')).merge(nbo[nbo.set == 'D_method_basis'][['name', 'E2_valence']], on='name')
    L = [('wB97X-D__def2-TZVP', 'ωB97X-D/TZVP'), ('M06-2X__def2-TZVP', 'M06-2X/TZVP'), ('B3LYP-D3BJ__def2-TZVP', 'B3LYP-D3(BJ)/TZVP'), ('PBE0-D3BJ__def2-TZVP', 'PBE0-D3(BJ)/TZVP'), ('HF__def2-TZVP', 'HF/TZVP'), ('wB97X-D__def2-SVP', 'ωB97X-D/SVP'), ('wB97X-D__def2-QZVP', 'ωB97X-D/QZVP')]
    bases = sorted(mb.base.unique()); fig, ax = plt.subplots(figsize=(11.5, 3.6)); grey(ax); w = 0.11
    for k, (lv, lab) in enumerate(L):
        q = mb[mb.level == lv].set_index('base').reindex(bases); ax.bar(np.arange(len(bases)) + (k - 3) * w, q.E2_valence, w, color=C[k], label=lab)
    ax.set_yscale('log'); ax.set_xticks(range(len(bases))); ax.set_xticklabels([b.split('_', 1)[1] for b in bases], rotation=35, ha='right'); ax.set_ylabel(r'valence $\Sigma E^{(2)}$ (kcal/mol)')
    ax.legend(ncol=4, fontsize=9, loc='upper left'); ax.set_title('valence ΣE(2) of the 12 scan dimers at seven levels'); fig.tight_layout(); save(fig, ESI, 'FigS5_method_basis')

def figS6_calibration():
    sc = pd.read_csv(os.path.join(R, 'qm_scans.csv')).merge(nbo[nbo.set.isin(['B_scan_distance', 'C_scan_angle'])][['name', 'E2_valence']], on='name')
    fig, (a, b) = plt.subplots(1, 2, figsize=(11.6, 3.9), gridspec_kw=dict(width_ratios=[1.3, 1]))
    grey(a); x = h[h.role == 'halogen bond']; rng_ = np.random.default_rng(1)
    groups = [('HAL59 XB, covered', x[x.covered].E2_valence, C[0]), ('HAL59 XB, P/F', x[~x.covered].E2_valence, C[6]), ('controls', h[h.control].E2_valence, MUTE),
              ('T1-positive scan points,\nCP attractive', sc[(sc.G > 0) & (sc.orca_BE >= 0)].E2_valence, C[2]), ('T1-positive scan points,\nCP repulsive', sc[(sc.G > 0) & (sc.orca_BE < 0)].E2_valence, C[7]),
              ('T1-negative scan points', sc[sc.G == 0].E2_valence, C[3])]
    for k, (lab, v, col) in enumerate(groups): a.scatter(k + rng_.uniform(-0.18, 0.18, len(v)), np.clip(v, 0.012, None), s=16, color=col, alpha=0.85)
    a.set_yscale('log'); a.set_xticks(range(len(groups))); a.set_xticklabels([g[0] for g in groups], rotation=30, ha='right', fontsize=9); a.set_ylabel(r'valence $\Sigma E^{(2)}$ (kcal/mol)')
    a.set_title('(a) decision value in the benchmark sets'); a.legend(loc='upper right', fontsize=9)
    gz = S['greyzone']; rows = [('level of theory (Set D)', gz['level_ratio']), ('protein model variants', gz['model_ratio']), ('combined', gz['combined'])]
    for k, (lab, (lo, hi)) in enumerate(rows): b.plot([lo, hi], [k, k], '-', lw=6, color=C[k], solid_capstyle='butt'); b.text(hi + 0.03, k, f'{lo:.2f}–{hi:.2f}', va='center', fontsize=9.5)
    b.axvline(1, color=INK2, lw=0.8, ls=':'); b.set_yticks(range(3)); b.set_yticklabels([r[0] for r in rows]); b.set_xlim(0.4, 1.9); b.set_ylim(-0.6, 2.6); b.set_xlabel('ΣE(2) relative to the reference model')
    b.set_title('(b) spread used for the grey zone'); b.text(0.42, -0.45, f"0.5/{gz['combined'][1]:.2f} = {gz['zone'][0]:.2f} and 0.5/{gz['combined'][0]:.2f} = {gz['zone'][1]:.2f} → 0.3–0.8 kcal/mol", fontsize=9.5)
    fig.tight_layout(); save(fig, ESI, 'FigS6_gate_calibration')

def figS7_S8_hiyd():
    res = pd.read_csv(os.path.join(R, 'hiyd', 'hiyd_residue_contacts.csv')); fig, axs = plt.subplots(1, 3, figsize=(12.4, 4.2), sharex=False)
    for k, p in enumerate('ABC'):
        d = res[res.pathway == p].sort_values('frames_T1').tail(12); a = axs[k]
        a.barh(d.residue, d.frames_strict, color=C[0], label='Lewis-base contact'); a.barh(d.residue, d.frames_T1 - d.frames_strict, left=d.frames_strict, color='white', ec=C[1], hatch='///', label='poor Lewis base only')
        a.set_title(f'({"abc"[k]}) pathway {p}'); a.set_xlabel('frames with a T1 contact')
    axs[0].legend(fontsize=9, loc='lower right'); fig.tight_layout(); save(fig, ESI, 'FigS7_hiyd_residues')
    hn = pd.read_csv(os.path.join(R, 'hiyd', 'hiyd_nbo_reanalysis.csv')); hn = hn.sort_values(['pathway', 'xi']); fig, ax = plt.subplots(figsize=(12.4, 3.8)); x = np.arange(len(hn))
    ax.axhspan(LO, HI, color=GREY, zorder=0, lw=0)
    ax.bar(x - 0.2, hn.E2_intra_phenolateO, 0.4, color=C[1], label='phenolate O → σ*(C–I) (intramolecular)')
    ax.bar(x + 0.2, hn.E2_bondx_pair, 0.4, color=[C[0] if l == 'strong' else 'white' for l in hn.lewis], ec=C[0], label='BondX valence ΣE(2) of the T1 pair')
    ax.scatter(x, hn.E_pub, marker='_', s=120, color=INK, zorder=4, label='published value')
    up = pd.read_csv(os.path.join(V, 'hiyd_decisions_updated.csv')).sort_values(['pathway', 'xi']).reset_index(drop=True); bbm = up.source.str.startswith('acetyl').values
    ax.scatter(x[bbm] + 0.2, up.E2_used[bbm], marker='D', s=20, color=C[3], zorder=5, label='backbone N, amide restored (acetyl cap)')
    ax.set_xticks(x); ax.set_xticklabels([f'{r.pathway} {r.residue} {r.acceptor_atom}' for r in hn.itertuples()], rotation=90, fontsize=7.5); ax.set_ylabel('E(2) (kcal/mol)'); ax.legend(ncol=2, fontsize=9, loc='upper left')
    ax.set_title('NBO terms of the 56 published cluster outputs (open bars: poor Lewis base)'); ax.set_xlim(-0.8, len(hn) - 0.2); fig.tight_layout(); save(fig, ESI, 'FigS8_hiyd_nbo')

def figS9_t2():
    t2 = pd.read_csv(os.path.join(V, 'hiyd_t2_v3.csv')); fig, axs = plt.subplots(1, 2, figsize=(11.6, 3.7))
    for k, p in enumerate('ABC'):
        d = t2[t2.pathway == p]; axs[0].scatter(d.xi, d.occupancy * 100, s=18 + 60 * d.D / t2.D.max(), color=C[k], alpha=0.8, label=f'pathway {p}', edgecolor='white', lw=0.3)
        axs[1].scatter(d.occupancy * 100, d.tau_ps, s=22, color=C[k], alpha=0.8, label=f'pathway {p}')
    axs[0].axhline(10, ls=':', color=INK2); axs[0].text(18.9, 10.6, 'default report cut-off (10%)', fontsize=9, ha='right'); axs[0].set_xlabel(r'$\xi$ (Å)'); axs[0].set_ylabel('occupancy (%, per window, unweighted)'); axs[0].set_title('(a) T2 occupancy of each contact'); axs[0].legend(fontsize=9)
    for dt in sorted(set(t2.dt_ps)): axs[1].axhline(dt, ls=':', color=MUTE, lw=0.8)
    axs[1].text(axs[1].get_xlim()[1] * 0.98 if False else 25.5, 250, 'frame spacing 160–240 ps', fontsize=8.5, ha='right', color=INK2)
    axs[1].set_xlabel('occupancy (%)'); axs[1].set_ylabel(r'mean residence time $\tau$ (ps)'); axs[1].set_title('(b) residence time'); fig.tight_layout(); save(fig, ESI, 'FigS9_hiyd_T2')

def figS12_classes():
    fig, axs = plt.subplots(1, 2, figsize=(11.6, 5.0), gridspec_kw=dict(width_ratios=[1, 1.25]))
    xb = h[~h.control].copy(); xb['ratio'] = xb.E2_valence / xb.ref_kcal; xb['cls'] = xb.acc_el.replace({'pi': 'ring'})
    order = ['P', 'N', 'S', 'O', 'F', 'ring']; a = axs[0]; rng = np.random.default_rng(3)
    for k, c in enumerate(order):
        d = xb[xb.cls == c]; a.scatter(k + rng.uniform(-0.16, 0.16, len(d)), d.ratio, s=26, color=C[0] if c != 'ring' else C[1], edgecolor='white', lw=0.4, zorder=3)
        a.plot([k - 0.28, k + 0.28], [d.ratio.median()] * 2, color=INK, lw=2, zorder=4)
    a.set_xticks(range(len(order))); a.set_xticklabels(order); a.set_xlabel('acceptor class (HAL59)'); a.set_ylabel(r'valence $\Sigma E^{(2)}$ / binding energy'); a.set_yscale('log'); a.set_yticks([0.4, 0.6, 1, 2, 3, 4]); a.set_yticklabels(['0.4', '0.6', '1', '2', '3', '4']); a.minorticks_off(); a.set_title('(a) orbital donation per unit binding energy')
    ex = pd.read_csv(os.path.join(V, 'extra_nbo.csv')); ex['flags'] = ex['flags'].fillna(''); ex = ex[ex.E2_valence.notna()]
    ex['hb'] = ex[['hbY', 'hbFrag2']].max(axis=1); b = axs[1]
    b.axvspan(LO, HI, color=GREY, zorder=0, lw=0); b.plot([0.01, 6], [0.01, 6], ':', color=INK2, lw=0.8)
    zp = lambda v: np.maximum(v, 0.03)
    G = ex[ex.set == 'G_hard_negatives']; ctrl = G.role.str.startswith('control')
    b.scatter(zp(G[~ctrl].E2_valence), zp(G[~ctrl].hb), marker='D', s=40, facecolors='white', edgecolors=C[1], lw=1.2, label='hIYD frame, poor Lewis base (set G)', zorder=3)
    b.scatter(zp(G[ctrl].E2_valence), zp(G[ctrl].hb), marker='o', s=40, color=C[0], label='hIYD frame, Lewis base (set G)', zorder=3)
    K = ex[ex.set == 'K_hiyd_amide']; b.scatter(zp(K.E2_valence), zp(K.hb), marker='s', s=26, color=MUTE, label='published pose, backbone N with acetyl cap (set K)', zorder=2)
    J = ex[ex.set == 'J_4dst_amide']; b.scatter(zp(J.E2_valence), zp(J.hb), marker='^', s=44, color=C[3], label='4DST Gly75 N, intact amide (set J)', zorder=3)
    b.set_xscale('log'); b.set_yscale('log'); b.set_xlim(0.025, 3); b.set_ylim(0.025, 7)
    b.set_xlabel(r'valence $\Sigma E^{(2)}$ (kcal/mol)'); b.set_ylabel(r'X$\to\sigma^*$(N–H/O–H) (kcal/mol)'); b.set_title('(b) contacts that pass T1 with poor Lewis bases')
    b.text(0.04, 4.6, 'above the diagonal:\ncompeting H-bond dominates', fontsize=8.5, color=INK2, va='top'); b.legend(fontsize=8.5, loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=2)
    b.set_xticks([0.03, 0.1, 0.3, 1, 3]); b.set_xticklabels(['≤0.03', '0.1', '0.3', '1', '3']); b.set_yticks([0.03, 0.1, 0.3, 1, 3]); b.set_yticklabels(['≤0.03', '0.1', '0.3', '1', '3']); b.minorticks_off()
    fig.tight_layout(); save(fig, ESI, 'FigS12_classes_hardneg')

def figS10_perf():
    p = pd.read_csv(os.path.join(R, 'performance.csv')); d = p[p.test == 'T1 detect']; fig, ax = plt.subplots(figsize=(4.6, 3.4))
    ax.scatter(d.atoms, d.ms, s=20, color=C[0]); xs = np.array([d.atoms.min(), d.atoms.max()]); k = np.median(d.ms / d.atoms); ax.plot(xs, k * xs, ':', color=INK2)
    ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlabel('atoms'); ax.set_ylabel('parse + T1 time (ms)'); fig.tight_layout(); save(fig, ESI, 'FigS10_performance')

if __name__ == '__main__':
    for f in [fig4, fig5, fig6, fig7, fig8, fig9, figS1, figS2, figS3_S4, figS4_angles, figS5_levels, figS6_calibration, figS7_S8_hiyd, figS9_t2, figS10_perf, figS12_classes]: f()


### `23_v3_screenshots.py`

23_v3_screenshots.py - interface screenshots of BondX Studio 3.0 for Fig. 2 and Fig. S11 (Playwright + Chromium).
Usage: python3 23_v3_screenshots.py <BondX_Studio.html> <paper folder>
For the figures all page text is set in Arial (Liberation Sans where Arial is not installed); the app itself uses its own fonts.
The 3D viewer needs 3Dmol.js from a CDN; offline it is hidden. Panel (c) reads a real NBO output of the paper (1US0 cluster model
of V2, with its manifest), so it shows the real decision values.

In [ ]:
%%writefile Scripts/23_v3_screenshots.py
#!/usr/bin/env python3
"""23_v3_screenshots.py - interface screenshots of BondX Studio 3.0 for Fig. 2 and Fig. S11 (Playwright + Chromium).
Usage: python3 23_v3_screenshots.py <BondX_Studio.html> <paper folder>
For the figures all page text is set in Arial (Liberation Sans where Arial is not installed); the app itself uses its own fonts.
The 3D viewer needs 3Dmol.js from a CDN; offline it is hidden. Panel (c) reads a real NBO output of the paper (1US0 cluster model
of V2, with its manifest), so it shows the real decision values."""
import asyncio, os, sys
from playwright.async_api import async_playwright
HTML, ROOT = os.path.abspath(sys.argv[1]), os.path.abspath(sys.argv[2])
OUT = os.path.join(ROOT, 'MS_Figures_V3', 'shots'); os.makedirs(OUT, exist_ok=True)
PDB = os.path.join(ROOT, 'data', 'protein', '1us0_idd594.pdb')
MAN = os.path.join(ROOT, 'qm_package', 'F_protein', 'manifests', '1us0_idd594__LDTA__cluster_pcm4_manifest.json')
LOG = os.path.join(ROOT, 'qm_package', 'F_protein', 'gaussian', '1us0_idd594__LDTA__cluster_pcm4.log')
ARIAL = ".toast{display:none !important} *{font-family:Arial,'Liberation Sans',sans-serif !important} .pagehead h2,.brand h1,.card h2,.tuth{font-family:Arial,'Liberation Sans',sans-serif !important;font-weight:700 !important;letter-spacing:0 !important} .mono,code,pre{font-family:'Liberation Mono',monospace !important}"
HIDE = "document.querySelectorAll('.vslot').forEach(e => { if (!window.$3Dmol) e.style.display = 'none'; }); document.querySelectorAll('.assume').forEach(e => e.open = false);"

async def main():
    async with async_playwright() as p:
        b = await p.chromium.launch(); pg = await b.new_page(viewport={'width': 1600, 'height': 960}, device_scale_factor=2)
        await pg.goto('file://' + HTML); await pg.wait_for_timeout(1500)
        await pg.evaluate("localStorage.setItem('bondx-theme','light')"); await pg.reload(); await pg.wait_for_timeout(1500)
        await pg.add_style_tag(content=ARIAL + " .reveal-on .page.on .pagehead,.reveal-on .page.on .card,.reveal-on .page.on .assume{opacity:1 !important;transform:none !important}")
        nav = lambda pid: pg.click(f'#side button[data-p={pid}]')
        async def shot(name, sel=None):
            await pg.evaluate(HIDE); await pg.evaluate('window.scrollTo(0,0)'); await pg.wait_for_timeout(400)
            if sel: el = await pg.query_selector(sel); await el.screenshot(path=os.path.join(OUT, name + '.png'))
            else: await pg.screenshot(path=os.path.join(OUT, name + '.png'))
        async def clip(name, page, first, n):
            await pg.evaluate(HIDE); await pg.wait_for_timeout(300)
            bb = await pg.evaluate(f"""(() => {{ const kids = [...document.querySelector('{page}').children].filter(e => !e.classList.contains('pagehead') && !e.classList.contains('assume') && e.offsetHeight > 0);
              const sel = kids.slice({first}, {first} + {n}); const r = sel.map(e => e.getBoundingClientRect()); const top = Math.min(...r.map(x => x.top)) + scrollY, bot = Math.max(...r.map(x => x.bottom)) + scrollY;
              const left = Math.min(...r.map(x => x.left)), right = Math.max(...r.map(x => x.right)); return [left - 8, top - 8, right - left + 16, bot - top + 16]; }})()""")
            await pg.screenshot(path=os.path.join(OUT, name + '.png'), full_page=True, clip=dict(x=bb[0], y=bb[1], width=bb[2], height=bb[3]))
        await pg.set_input_files('#dropPdb input', PDB); await pg.wait_for_timeout(900); await shot('s01_structure')
        await nav('detect'); await pg.click('#btnDetect'); await pg.wait_for_timeout(600); await shot('a_detect'); await shot('s02_detect'); await clip('a_detect_clip', '#pg-detect', 0, 2)
        await pg.click('#btnToReg'); await nav('qm'); await pg.wait_for_timeout(300)
        await pg.click('#qEnvSeg button[data-env=qmmm]'); await pg.wait_for_timeout(200)
        for bx in await pg.query_selector_all('#qmReg .selc'): await bx.check()
        await pg.wait_for_timeout(600); await shot('b_qm_setup'); await clip('b_qm_clip', '#pg-qm', 0, 2)
        await pg.click('#btnBuild'); await pg.wait_for_timeout(1800)
        await pg.evaluate("document.querySelector('header.top').style.position='static'")
        el = await pg.query_selector('#qOut'); await el.scroll_into_view_if_needed(); await pg.wait_for_timeout(300); await el.screenshot(path=os.path.join(OUT, 's03_qm_result.png'))
        await pg.evaluate("document.querySelector('header.top').style.position=''")
        await nav('nbo'); await pg.wait_for_timeout(300)
        await pg.set_input_files('#dropMan input', MAN); await pg.wait_for_timeout(500)
        await pg.set_input_files('#dropNbo input', LOG); await pg.wait_for_timeout(1200); await shot('c_nbo'); await shot('s04_nbo')
        card = pg.locator('#nRes').locator('xpath=ancestor::div[contains(@class,"card")][1]'); await card.screenshot(path=os.path.join(OUT, 'c_nbo_card.png'))
        await nav('params'); await pg.wait_for_timeout(300); await shot('s05_params')
        await nav('method'); await pg.wait_for_timeout(300); await shot('s06_method')
        await pg.click('#btnTutorial'); await pg.wait_for_timeout(300); await shot('s07_tutorial1')
        for i in range(4): await pg.click('#tutNext')
        await pg.wait_for_timeout(300); await shot('s08_tutorial5')
        await b.close()
    print('screenshots in', OUT)
asyncio.run(main())


### `24_v3_screen_figures.py`

24_v3_screen_figures.py - composes Fig. 2 (interface) and Fig. S11 (screens) from the screenshots of script 23.
Usage: python3 24_v3_screen_figures.py ..

In [ ]:
%%writefile Scripts/24_v3_screen_figures.py
#!/usr/bin/env python3
"""24_v3_screen_figures.py - composes Fig. 2 (interface) and Fig. S11 (screens) from the screenshots of script 23.
Usage: python3 24_v3_screen_figures.py .."""
import os, sys
import numpy as np
from PIL import Image
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from bxstyle3 import plt, save
ROOT = sys.argv[1] if len(sys.argv) > 1 else '..'
SH = os.path.join(ROOT, 'MS_Figures_V3', 'shots'); MS = os.path.join(ROOT, 'MS_Figures_V3'); ESI = os.path.join(ROOT, 'ESI_Figures_V3')
def main_area(f, top=80, h_css=760):
    im = Image.open(os.path.join(SH, f)).convert('RGB'); sc = im.width / 1600
    return im.crop((int(262 * sc), int(top * sc), im.width - int(16 * sc), int(h_css * sc)))
A = Image.open(os.path.join(SH, 'a_detect_clip.png')).convert('RGB'); A = A.crop((0, 0, A.width, 1200)); Bm = Image.open(os.path.join(SH, 'b_qm_clip.png')).convert('RGB'); Bm = Bm.crop((0, 0, Bm.width, 1500)); Cm = Image.open(os.path.join(SH, 'c_nbo_card.png')).convert('RGB')
# compact layout: three bands stacked, each cropped to the part that the caption describes
A = A.crop((0, 0, A.width, 700)); Bm = Bm.crop((0, 0, Bm.width, 640))
fig = plt.figure(figsize=(7.2, 7.4)); gs = fig.add_gridspec(3, 1, height_ratios=[A.height / A.width, Bm.height / Bm.width, Cm.height / Cm.width], hspace=0.16)
for k, (im, t) in enumerate([(A, '(a) T1: halogen contacts of 1US0'), (Bm, '(b) T3a: candidates and QM/MM set-up'), (Cm, '(c) T3b: NBO screen of a real 1US0 output')]):
    ax = fig.add_subplot(gs[k]); ax.imshow(im); ax.axis('off'); ax.set_title(t, fontsize=9.5, loc='left', fontweight='bold')
save(fig, MS, 'Fig2_interface')
titles = {'s01': 'T0: load structure', 's02': 'T1: halogen contacts', 's03': 'T3a: QM/MM model and charges (1US0)', 's04': 'T3b: NBO screen',
          's05': 'All parameters', 's06': 'Method page', 's07': 'Tutorial, page 1', 's08': 'Tutorial, page 5'}
files = sorted(f for f in os.listdir(SH) if f[:3] in titles)
fig, axs = plt.subplots(4, 2, figsize=(7.2, 11.0))
for ax in np.ravel(axs): ax.axis('off')
for ax, f in zip(np.ravel(axs), files):
    im = Image.open(os.path.join(SH, f)).convert('RGB'); w, h = im.size
    ax.imshow(im.crop((0, 0, w, min(h, int(w * 0.66))))); ax.set_title(titles[f[:3]], fontsize=10, fontweight='bold', loc='left')
plt.subplots_adjust(wspace=0.04, hspace=0.16); save(fig, ESI, 'FigS11_screens')


### `25_v3_toc.py`

25_v3_toc.py - TOC graphic of Manuscript_V3 (ACS size 3.25 in x 1.75 in), Arial style, drawn from the paper data.
Usage: python3 25_v3_toc.py ..

In [ ]:
%%writefile Scripts/25_v3_toc.py
#!/usr/bin/env python3
"""25_v3_toc.py - TOC graphic of Manuscript_V3 (ACS size 3.25 in x 1.75 in), Arial style, drawn from the paper data.
Usage: python3 25_v3_toc.py .."""
import os, sys
import numpy as np, pandas as pd
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from bxstyle3 import plt, C, VCOL, GREY, INK, INK2, MUTE, save
from matplotlib.patches import Circle, Wedge, FancyArrowPatch, Rectangle
ROOT = sys.argv[1] if len(sys.argv) > 1 else '..'
nbo = pd.read_csv(os.path.join(ROOT, 'results', 'v3', 'nbo_v3.csv'))
h = pd.read_csv(os.path.join(ROOT, 'results', 'qm_hal59.csv')).merge(nbo[nbo.set == 'A_HAL59'][['name', 'E2_valence']], on='name')
pc = pd.read_csv(os.path.join(ROOT, 'results', 'v3', 'protein_contacts_table.csv'))
fig = plt.figure(figsize=(3.25, 1.75)); fig.patch.set_facecolor('white')
# molecule sketch -------------------------------------------------------------
a = fig.add_axes([0.0, 0.22, 0.37, 0.56]); a.set_xlim(0, 3.6); a.set_ylim(-1.1, 1.3); a.set_aspect('equal'); a.axis('off')
a.plot([0.25, 0.75], [0.55, 0.0], color='#55595c', lw=1.6, solid_capstyle='round'); a.plot([0.25, 0.75], [-0.55, 0.0], color='#55595c', lw=1.6)
a.add_patch(Circle((0.75, 0), 0.22, fc='#7b7f83', ec=INK, lw=0.6, zorder=3)); a.text(0.75, 0, 'C', color='white', fontsize=6.5, ha='center', va='center', zorder=4, fontweight='bold')
a.plot([0.97, 1.18], [0, 0], color='#55595c', lw=1.8)
a.add_patch(Wedge((1.55, 0), 0.52, -32, 32, width=0.12, fc=C[0], ec='none', zorder=2))
a.add_patch(Wedge((1.55, 0), 0.52, 60, 120, width=0.10, fc='#e8a6a2', ec='none', zorder=2)); a.add_patch(Wedge((1.55, 0), 0.52, 240, 300, width=0.10, fc='#e8a6a2', ec='none', zorder=2))
a.add_patch(Circle((1.55, 0), 0.38, fc='#a34a22', ec=INK, lw=0.6, zorder=3)); a.text(1.55, 0, 'Br', color='white', fontsize=7, ha='center', va='center', zorder=4, fontweight='bold')
a.plot([2.1, 2.78], [0, 0], ls=(0, (1.2, 1.2)), color=INK, lw=1.2)
a.add_patch(Circle((3.0, 0), 0.24, fc='#d94a43', ec=INK, lw=0.6, zorder=3)); a.text(3.0, 0, 'O', color='white', fontsize=6.5, ha='center', va='center', zorder=4, fontweight='bold')
a.plot([3.0, 3.0], [0.25, 0.75], color='#55595c', lw=1.3); a.plot([3.07, 3.07], [0.25, 0.75], color='#55595c', lw=1.3)
a.text(2.2, 0.5, 'σ-hole', color=C[0], fontsize=5.6, ha='center'); a.text(2.5, -0.62, r'$n_\mathrm{O}\!\rightarrow\!\sigma^*_\mathrm{C-Br}$', fontsize=5.6, ha='center', color=INK2)
# data strip ------------------------------------------------------------------
b = fig.add_axes([0.47, 0.36, 0.50, 0.40]); b.set_xscale('log'); b.set_xlim(0.03, 150); b.set_ylim(-0.6, 1.6)
b.axvspan(0.3, 0.8, color=GREY, lw=0, zorder=0)
x = h[(h.role == 'halogen bond')].E2_valence.values; rng = np.random.default_rng(3)
b.scatter(x, 1 + rng.uniform(-0.22, 0.22, len(x)), s=5, color=C[0], alpha=0.8, lw=0, zorder=3)
ctl = h[h.control].E2_valence.clip(lower=0.035).values; b.scatter(ctl, np.full(len(ctl), 1), s=7, marker='x', color=MUTE, lw=0.8, zorder=3)
vc = lambda v: VCOL['supported'] if v >= 0.8 else (VCOL['inconclusive'] if v >= 0.3 else VCOL['not supported'])
pv = pc.val_default.dropna().clip(lower=0.035).values
b.scatter(pv, np.zeros(len(pv)), s=16, marker='D', c=[vc(v) for v in pv], edgecolor='white', lw=0.4, zorder=3)
b.set_yticks([0, 1]); b.set_yticklabels(['protein', 'HAL59'], fontsize=5.8); b.tick_params(axis='x', labelsize=6, pad=1.5, length=2); b.tick_params(axis='y', length=0, pad=2)
b.set_xticks([0.1, 1, 10, 100]); b.set_xticklabels(['0.1', '1', '10', '100'])
b.set_xlabel(r'valence $\Sigma E^{(2)}$ (kcal/mol)', fontsize=5.8, labelpad=1); b.grid(False)
for sp in ['left', 'top', 'right']: b.spines[sp].set_visible(False)
b.text(0.49, 1.52, 'grey zone', fontsize=5.3, ha='center', color='#8a6a00'); b.text(9, 1.52, 'orbital support', fontsize=5.3, ha='center', color=VCOL['supported'])
# tier line -------------------------------------------------------------------
fig.text(0.03, 0.93, 'BondX Studio', fontsize=9, fontweight='bold', color=INK, va='top')
fig.text(0.97, 0.915, 'halogen bonds, screened in the browser', fontsize=5.8, color=INK2, va='top', ha='right')
fig.add_artist(plt.Line2D([0.03, 0.97], [0.82, 0.82], color='#d6d5d0', lw=0.6))
fig.text(0.5, 0.05, 'T1 geometry → T2 MD → T3 QM model, NBO → T4 EDA → T5 rank', fontsize=5.6, ha='center', color=INK)
os.makedirs(os.path.join(ROOT, 'MS_Figures_V3'), exist_ok=True)
fig.savefig(os.path.join(ROOT, 'MS_Figures_V3', 'TOC_graphic.pdf')); fig.savefig(os.path.join(ROOT, 'MS_Figures_V3', 'TOC_graphic.png'), dpi=600); print('wrote TOC_graphic')


### `26_v3_tables.py`

26_v3_tables.py
LaTeX tables for ESI_V3 that depend on the V3 analysis (valence NBO screen with grey zone, kcal/mol units,
redesigned BXQI, competing hydrogen bonds, field tests, hIYD T2/T5). Every number comes from the result files.
Run after 11_make_tables.py (which writes the tables that did not change) and 21_v3_analysis.py.
Usage: python3 26_v3_tables.py [ROOT] [APP]
Output: tables/*.tex

In [ ]:
%%writefile Scripts/26_v3_tables.py
#!/usr/bin/env python3
"""26_v3_tables.py
LaTeX tables for ESI_V3 that depend on the V3 analysis (valence NBO screen with grey zone, kcal/mol units,
redesigned BXQI, competing hydrogen bonds, field tests, hIYD T2/T5). Every number comes from the result files.
Run after 11_make_tables.py (which writes the tables that did not change) and 21_v3_analysis.py.
Usage: python3 26_v3_tables.py [ROOT] [APP]
Output: tables/*.tex
"""
import os, sys, json, ast, subprocess, importlib.util
import pandas as pd, numpy as np

ROOT = sys.argv[1] if len(sys.argv) > 1 else '..'
APP = sys.argv[2] if len(sys.argv) > 2 else os.path.join(ROOT, '..', 'app')
RES = os.path.join(ROOT, 'results'); V3 = os.path.join(RES, 'v3'); TAB = os.path.join(ROOT, 'tables')
spec = importlib.util.spec_from_file_location('mt', os.path.join(os.path.dirname(os.path.abspath(__file__)), '11_make_tables.py'))
mt = importlib.util.module_from_spec(spec); sys.argv = [sys.argv[0], ROOT, APP]; spec.loader.exec_module(mt)
tex, write, longtable = mt.tex, mt.write, mt.longtable
S = json.load(open(os.path.join(V3, 'summary_v3.json')))
R2 = json.load(open(os.path.join(V3, 'summary_round2.json')))
KJ = 4.184
VSHORT = {'supported': 'S', 'inconclusive': 'I', 'not supported': 'N'}
SYSN = {'1us0_idd594': '1US0', '2w0s': '2W0S', '1p5e': '1P5E', '4agl': '4AGL', '4gql': '4GQL', '4dst': '4DST'}
VARN = {'cluster_gas': 'cluster, gas', 'cluster_pcm4': r'cluster, PCM', 'shell3': r'+3~\AA{} shell', 'shell4': r'+4~\AA{} shell', 'shell5': r'+5~\AA{} shell',
        'qmmm_ff14SB': 'QM/MM ff14SB', 'qmmm_charmm36': 'QM/MM CHARMM36', 'qmmm_Z1': 'QM/MM Z1', 'qmmm_Z2': 'QM/MM Z2', 'qmmm_Z3': 'QM/MM Z3',
        'qmmm_r10': r'QM/MM 10~\AA{}', 'qmmm_r20': r'QM/MM 20~\AA{}', 'qmmm_r25': r'QM/MM 25~\AA{}', 'qmmm_full': 'QM/MM whole', 'qmmm_HIS110_HID': 'QM/MM His110 HID', 'qmmm_HIS110_HIP': 'QM/MM His110 HIP'}


def f(v, n=2):
    if v is None or (isinstance(v, float) and np.isnan(v)): return '--'
    return ('%.' + str(n) + 'f') % v


def lab(c):
    """'TBS301:A@BR11···LEU83:A@O' -> 'Br11$\\cdots$O Leu83 (A)'"""
    c = c.replace(' ··· ', '···'); d, a = c.split('···'); x = d.split('@')[1]; ch = d.split('@')[0].split(':')[1]
    r, at = a.split('@'); rn = r.split(':')[0]; res = rn[:3].title() + rn[3:]
    at = r'$\pi$' if 'π' in at else at
    return '%s$\\cdots$%s %s (%s)' % (x.title(), at, res, ch)


# ---------------------------------------------------------------- distance cut-offs
def t_cutoffs():
    B = mt.node("const B=require('%s');console.log(JSON.stringify({c:B.defaults().geom.cutoffs}))" % mt.ENG)['c']
    bondi = {'I': 1.98, 'BR': 1.85, 'CL': 1.75, 'N': 1.55, 'O': 1.52, 'S': 1.80, 'AR': 1.77}
    rows = []
    for x in ('I', 'BR', 'CL'):
        for y in ('N', 'O', 'S', 'AR'):
            s = bondi[x] + bondi[y]; c = B[x][y]
            rows.append([x.title(), 'ring' if y == 'AR' else y, '%.2f' % c, '%.2f' % s, '%.3f' % (c / s), '%+.2f' % (c - B['I'][y]) if x != 'I' else '--'])
    body = r'''\begin{table}[!htbp]\centering\small
\caption{Default X$\cdots$Y distance cut-offs of tier T1 (\AA). The iodine values come from our earlier hIYD study.\cite{Karmakar2026} The Br and Cl values follow from them by the difference in halogen radius (I$\to$Br about $-0.13$~\AA; Br$\to$Cl about $-0.07$~\AA) and are rounded to 0.01~\AA. $\Sigma r$: sum of the Bondi radii (ring centroid: 1.77~\AA). The cut-offs lie at 1.09--1.14 times $\Sigma r$ for atom acceptors and at 1.07--1.08 times $\Sigma r$ for ring centroids. Rounding changed no T1 decision in HAL59, in the scans, or in the protein test set.}\label{tab:cutoffs}
\begin{tabular}{llrrrr}\toprule X & Y & $d_\mathrm{cut}$ & $\Sigma r$ & $d_\mathrm{cut}/\Sigma r$ & shift from I \\\midrule
''' + '\n'.join(' & '.join(r) + r'\\' for r in rows) + '\n\\bottomrule\\end{tabular}\\end{table}\n'
    write('tab_cutoffs', body)
    return [float(r[4]) for r in rows]


# ---------------------------------------------------------------- force-field reference (heavy atoms and hydrogens)
def t_ffref():
    out = subprocess.run(['node', os.path.join(APP, 'tests', 'ff_reference.test.js')], capture_output=True, text=True).stdout.split('\n')
    import re
    heavy = {}; hyd = {}
    for l in out:
        m = re.match(r'(PASS|FAIL)\s+(\S+)\s+(.+?): (\d+) residues, (\d+) heavy atoms compared, max \|Δq\| = (\S+) e(.*)', l)
        if m: heavy[(m.group(2), m.group(3))] = (m.group(4), m.group(5), float(m.group(6)), m.group(7).strip(' ()'))
        m = re.match(r'\s+(\S+)\s+(.+?): (\d+) hydrogens compared one by one, max \|Δq\| = (\S+) e(.*)', l)
        if m: hyd[(m.group(1), m.group(2))] = (m.group(3), float(m.group(4)), m.group(5).strip(' ()'))
    rows = []
    for k, (nr, na, dq, note) in heavy.items():
        h = hyd.get(k, ('--', float('nan'), ''))
        ffl = 'ff14SB vs.\\ AMBER (PDB2PQR)' if k[1].startswith('AMBER') else 'CHARMM36 vs.\\ CHARMM22 (PDB2PQR)'
        nt = 'Trp only (C36 update)' if 'CHARMM' in k[1] and dq > 0 else ''
        rows.append([k[0], ffl, nr, na, '%.2g' % dq, h[0], ('%.2g' % h[1]) if h[0] != '--' else '--', nt])
    body = r'''\begin{table}[!htbp]\centering\scriptsize\setlength{\tabcolsep}{3pt}
\caption{Force-field charges of \BX{} compared with the PDB2PQR reference files.\cite{Jurrus2018} Heavy atoms: charge of each heavy atom plus its hydrogens on both sides. Hydrogens: every hydrogen of the reference file against the template charge of the same hydrogen name, without folding. Both programs read the same AMBER parameter files, so the AMBER rows test the template matching, termini, and protonation handling of \BX{}, not the charges themselves. PDB2PQR ships CHARMM22 charges; the CHARMM36 rows only show that \BX{} differs from CHARMM22 where CHARMM36 changed the parameters (tryptophan).}\label{tab:ffref}
\resizebox{\textwidth}{!}{\begin{tabular}{llrrrrrl}\toprule PDB & Force field (BondX vs.\ reference) & Residues & Heavy atoms & max $|\Delta q|$ (e) & Hydrogens & max $|\Delta q_\mathrm{H}|$ (e) & Differences\\\midrule
''' + '\n'.join(' & '.join(x) + r'\\' for x in rows) + '\n\\bottomrule\\end{tabular}}\\end{table}\n'
    write('tab_ffref', body)
    return sum(int(r[5]) for r in rows if r[5] != '--' and 'ff14SB' in r[1])


# ---------------------------------------------------------------- HAL59 NBO, EDA, BXQI
def t_hal59():
    q = pd.read_csv(os.path.join(RES, 'qm_hal59.csv'))
    n = pd.read_csv(os.path.join(V3, 'nbo_v3.csv')); n = n[n.set == 'A_HAL59'][['name', 'E2_valence', 'E2_RY', 'verdict', 'hbY']]
    b = pd.read_csv(os.path.join(V3, 'bxqi_hal59_table.csv'))[['name', 'BXQI', 'BXQI_old']]
    d = q.drop(columns=['BXQI', 'BXQI_grade']).merge(n, on='name').merge(b, on='name')
    rows = []
    for _, r in d.sort_values(['role', 'halogen', 'ref_kcal'], ascending=[False, True, True]).iterrows():
        role = 'control' if r.control else ('XB' if r.covered else 'XB, P/F')
        rows.append([tex(r['name']), r.halogen.title(), tex(r.acc_el), f(r.ref_kcal), f(r.G) if r.covered else '--', f(r.E2_valence), f(r.E2_RY),
                     VSHORT[r.verdict], f(r.ct_NPA * 1000, 1), f(r.eda_total_kcal), f(r.orca_cp_kcal), f(r.chi_CT), f(r.BXQI, 1), role])
    write('tab_qm_hal59', longtable('lllrrrrlrrrrrl', r'Dimer & X & Y & $E_\mathrm{ref}$ & $G$ & $\Sigma E^{(2)}$ & RY & D & $\Delta q_\mathrm{CT}$ & EDA & ORCA & $\chi_\mathrm{CT}$ & BXQI & Role', rows,
                                 r'NBO screen, ALMO-EDA, and BXQI for the 59 HAL59 dimers. $E_\mathrm{ref}$: CCSD(T)/CBS binding energy (kcal\,mol$^{-1}$; positive values mean binding). $G$: T1 score ("--": no default cut-off, tier not evaluated). $\Sigma E^{(2)}$: valence decision value (Gaussian~16, $\omega$B97X-D/def2-TZVP, kcal\,mol$^{-1}$). RY: $n\to$RY$^*$(X) part, reported beside the decision value and not counted. D: decision (S orbital-supported, $\ge 0.8$; I inconclusive, 0.3--0.8; N not supported, $<0.3$~kcal\,mol$^{-1}$). $\Delta q_\mathrm{CT}$: NPA charge moved from fragment~2 (acceptor) to fragment~1 (halogen donor), in $10^{-3}$~e. EDA: ALMO-EDA total ($\omega$B97X-V/def2-TZVPD). ORCA: counterpoise interaction energy at the same level (kcal\,mol$^{-1}$). BXQI: index from $G$, $\Sigma E^{(2)}$, and the EDA total, with tiers that cannot be evaluated left out.', 'tab:qmhal'))
    # EDA table in kcal/mol
    e = pd.read_csv(os.path.join(RES, 'qm_eda.csv')); e = e[e.set == 'A_HAL59'].merge(q[['name', 'ref_kcal', 'halogen', 'acc_el', 'orca_cp_kcal', 'role', 'control']], on='name')
    rows = []
    for _, r in e.sort_values(['role', 'halogen', 'ref_kcal'], ascending=[False, True, True]).iterrows():
        rows.append([tex(r['name']), f(r.frz / KJ), f(r.pol / KJ), f(r.ct / KJ), f(r.total / KJ), f(-r.ref_kcal), f(r.orca_cp_kcal), f(r.chi_CT)])
    write('tab_qm_eda', longtable('lrrrrrrr', r'Dimer & FRZ & POL & CT & Total & $-E_\mathrm{ref}$ & ORCA & $\chi_\mathrm{CT}$', rows,
                                r'ALMO-EDA of the 59 HAL59 dimers (Q-Chem~6.2.2, $\omega$B97X-V/def2-TZVPD, gas phase, counterpoise-corrected CT; kcal\,mol$^{-1}$). $-E_\mathrm{ref}$: CCSD(T)/CBS interaction energy. ORCA: counterpoise-corrected SCF interaction energy at the same level. With the same functional and basis, the EDA total is by construction this interaction energy; the residual difference (at most 0.05~kcal\,mol$^{-1}$) reflects RIJCOSX and integration grids.', 'tab:qmeda'))


# ---------------------------------------------------------------- protein contacts summary and per-model values
def t_protein():
    c = pd.read_csv(os.path.join(V3, 'protein_contacts_table.csv')); b = pd.read_csv(os.path.join(V3, 'bxqi_protein_table.csv'))
    c = c.merge(b[['contact', 'BXQI']], on='contact')
    rows = []
    for _, r in c.sort_values('BXQI', ascending=False).iterrows():
        v = ast.literal_eval(r.verdicts); vr = ast.literal_eval(r.verdicts_withRY_vs_05)
        fl = R2['protein_hbflag_counts'].get(r.contact.replace('···', ' ··· '), [0, r.n])
        rows.append([SYSN[r.sys], lab(r.contact), str(r.n), f(r.G), f(r.val_default), '%s--%s' % (f(r.val_min), f(r.val_max)), f(r.RY_default), '%s--%s' % (f(r.withRY_min), f(r.withRY_max)),
                     '/'.join(vr), f(r.hbY_max), f(r.hbFrag2_max), '%d/%d' % (fl[0], r.n), '/'.join(VSHORT[x] for x in v), f(r.BXQI, 1)])
    body = r'''\begin{table}[!htbp]\centering\scriptsize\setlength{\tabcolsep}{2.5pt}
\caption{Electronic tiers for the protein contacts (Gaussian~16, $\omega$B97X-D/def2-TZVP; kcal\,mol$^{-1}$). $n$: number of models. Default: PCM cluster ($\varepsilon=4$). Range: all models (gas, PCM, QM size, and QM/MM variants). RY: $n\to$RY$^*$(X) part in the default model. With RY: valence plus Rydberg sum, and the decision of a rule that counts the Rydberg terms against a single threshold of 0.5~kcal\,mol$^{-1}$ (pass/fail; the rule of BondX Studio 2.0). X$\to$Y--H: summed donation from the halogen lone pairs into the N--H and O--H antibonds of the acceptor residue; X$\to$frag.~2: the same into any N--H or O--H antibond of fragment~2. H-bond flag: models in which X$\to$frag.~2 exceeds $\Sigma E^{(2)}$ (and 0.5~kcal\,mol$^{-1}$). D: decisions over all models. BXQI: from $G$ and the default $\Sigma E^{(2)}$. All models were built with BondX Studio 2.0; the 1US0 models have the template rotamer of the Thr113 hydroxyl, whose hydrogen points at the bromine (see text).}\label{tab:protcontacts}
\resizebox{\textwidth}{!}{\begin{tabular}{llrrrlrlllrllr}\toprule PDB & Contact & $n$ & $G$ & $\Sigma E^{(2)}$ & range & RY & with RY & 0.5 rule & X$\to$Y--H & X$\to$frag.~2 & H-bond flag & D & BXQI\\\midrule
''' + '\n'.join(' & '.join(x) + r'\\' for x in rows) + '\n\\bottomrule\\end{tabular}}\\end{table}\n'
    write('tab_protcontacts', body)
    # per model
    n = pd.read_csv(os.path.join(V3, 'nbo_v3.csv')); n = n[n.set == 'F_protein'].copy()
    n['sys'] = n.name.str.split('__').str[0]; n['var'] = n.name.str.split('__').str[-1]
    order = list(VARN)
    n['vo'] = n['var'].map(lambda v: order.index(v) if v in order else 99)
    rows = []
    for _, r in n.sort_values(['sys', 'label', 'vo']).iterrows():
        rows.append([SYSN[r.sys], lab(r.label), VARN.get(r['var'], tex(r['var'])), f(r.E2_valence), f(r.E2_RY), f(r.hbY), f(r.hbFrag2), 'yes' if str(r.lpOnY).lower() == 'true' else 'no', f(r.ct_NPA * 1000, 1), VSHORT[r.verdict]])
    write('tab_qm_prot', longtable('lllrrrrlrl', r'PDB & Contact & Model & $\Sigma E^{(2)}$ & RY & X$\to$Y--H & X$\to$frag.~2 & LP(Y) & $\Delta q_\mathrm{CT}$ & D', rows,
                                 r'NBO screen of every protein model (Gaussian~16, $\omega$B97X-D/def2-TZVP; kcal\,mol$^{-1}$). QM/MM models use ff14SB, the RC boundary, and \SI{15}{\angstrom} unless the model name says otherwise. LP(Y): NBO prints a lone pair on the acceptor atom. $\Delta q_\mathrm{CT}$: NPA charge moved to fragment~1 ($10^{-3}$~e); in a protein cluster it sums every ligand--environment contact and is not specific to the halogen bond. D: decision as in Table~\ref{tab:qmhal}.', 'tab:qmprot'))
    return len(n)


# ---------------------------------------------------------------- in-cluster pairs (hard negatives inside the models)
def t_pairs():
    p = pd.read_csv(os.path.join(V3, 'pairs_v3.csv'))
    p['sys'] = p.name.str.split('__').str[0]
    rows = []
    for _, r in p.sort_values(['pass_T1', 'E2_valence'], ascending=[True, True]).iterrows():
        why = 'T1 pass' if r.pass_T1 else ('angle' if r.pass_dist else ('distance' if r.pass_angle else 'distance, angle'))
        rows.append([SYSN[r.sys], tex(r.X.split('@')[1].title()), tex(r.Yname) + ' ' + tex(r.Yres.split(':')[0].title()) + ' (' + r.Yres.split(':')[1] + ')', f(r.d), f(r.theta1, 1), f(r.cutoff), why,
                     'poor' if str(r.lewis_poor).lower() == 'true' else 'base', f(r.E2_valence), f(r.E2_RY), f(r.hbY)])
    body = r'''\begin{table}[!htbp]\centering\scriptsize\setlength{\tabcolsep}{3pt}
\caption{Every halogen$\cdots$N, O, or S pair within 4.5~\AA{} inside the default PCM cluster models, whether or not it passes T1 (kcal\,mol$^{-1}$). The twelve pairs that pass T1 are the target contacts and their chain replicas. The six pairs that fail T1 never reach the NBO screen in the workflow, so their small values confirm the geometric exclusion rather than test the screen on contacts that pass T1; ESI Table~\ref{tab:hardneg} gives that test. Fail: criterion of T1 that is not met. Base: Lewis base or poor Lewis base. X$\to$Y--H: halogen lone pair into an N--H or O--H antibond on the partner residue.}\label{tab:pairs}
\begin{tabular}{lllrrrllrrr}\toprule PDB & X & Y & $d$ (\AA) & $\theta_1$ & $d_\mathrm{cut}$ & T1 & Base & $\Sigma E^{(2)}$ & RY & X$\to$Y--H\\\midrule
''' + '\n'.join(' & '.join(x) + r'\\' for x in rows) + '\n\\bottomrule\\end{tabular}\\end{table}\n'
    write('tab_pairs', body)


# ---------------------------------------------------------------- MM field tests
def sgn(v):
    v = int(round(v)); return '0' if v == 0 else '%+d' % v


def t_field():
    d = pd.read_csv(os.path.join(V3, 'field_v3.csv'))
    rows = []
    for s in SYSN:
        g = d[d.system == s]
        def E(ff='ff14SB', bd='RC', rad='15', eh=0):
            q = g[(g.ff == ff) & (g.boundary == bd) & (g.radius.astype(str) == rad) & (g.explicitH == eh)]
            return q.iloc[0] if len(q) else None
        a, b, c, w, ch = E(eh=0), E(eh=6), E(bd='RCD', eh=6), E(rad='full', eh=6), E(ff='charmm36', eh=6)
        rows.append([SYSN[s], f(a.E_AX), f(b.E_AX), f(c.E_AX), f(ch.E_AX), f(E(rad='25', eh=6).E_AX), f(w.E_AX), '%d' % b.nExplicitH, '%d/%d' % (b.nZero, ch.nZero), '%d/%d' % (b.nCharges, ch.nCharges), sgn(b.net), sgn(w.net)])
    body = r'''\begin{table}[!htbp]\centering\scriptsize\setlength{\tabcolsep}{3pt}
\caption{Field component $E_{AX}$ (V\,\AA$^{-1}$) along the A$\to$X bond at the halogen from the MM point charges of the QM/MM models (no QM job), with the QM regions of the computed models. The hydrogen-bond shell of version 3.0 moves Lys105 of 2W0S into the QM region, which changes $E_{AX}$ from the MM charges of the whole structure from $-0.44$ to $+0.22$~V\,\AA$^{-1}$; the field values therefore refer to a given QM region. Positive values point from A to X and should deepen the $\sigma$-hole. Folded: hydrogen charges moved onto their heavy atoms everywhere (the setting of the QM/MM models of this work, built with BondX Studio 2.0). Explicit H: MM residues within 6~\AA{} of X carry explicit hydrogens with their template charges (default of version 3.0). $n_\mathrm{H}$: explicit MM hydrogens. Zero: MM atoms with zero charge after folding (ff14SB/CHARMM36); CHARMM36 gives zero net charge to many aliphatic CH$_n$ groups, so after folding these atoms carry no charge and are not written, which explains the smaller CHARMM36 point-charge counts at the same radius. $N_q$: written point charges (ff14SB/CHARMM36, \SI{15}{\angstrom}). $Q$: MM net charge (e) at \SI{15}{\angstrom} and for the whole structure.}\label{tab:field}
\resizebox{\textwidth}{!}{\begin{tabular}{lrrrrrrrrrrr}\toprule PDB & folded & explicit H & RCD & CHARMM36 & 25~\AA & whole & $n_\mathrm{H}$ & Zero & $N_q$ & $Q_{15}$ & $Q_\mathrm{whole}$\\\midrule
''' + '\n'.join(' & '.join(x) + r'\\' for x in rows) + '\n\\bottomrule\\end{tabular}}\\end{table}\n'
    write('tab_field', body)


def t_m12():
    m = S['m12']; rows = []
    W = {w[0]: w for w in m['within']}
    for c, e, dv, dp in m['rows_default']:
        w = W.get(c, [c, float('nan'), 0])
        rows.append([lab(c), f(e), '%+.2f' % dv, '%+.2f' % dp, f(w[1]) if w[1] == w[1] else '--', str(w[2])])
    body = r'''\begin{table}[!htbp]\centering\small
\caption{Does the MM field predict the change of $\Sigma E^{(2)}$ on embedding? $E_{AX}$: default QM/MM model (V\,\AA$^{-1}$). $\Delta\Sigma E^{(2)}$: QM/MM minus gas-phase cluster, and QM/MM minus PCM cluster (kcal\,mol$^{-1}$). $\rho$: Spearman correlation between $E_{AX}$ and $\Sigma E^{(2)}$ over the QM/MM variants of one contact ($n$ models). Across contacts, $\rho = %.2f$ ($n = %d$); over all variants pooled, $\rho = %.2f$ ($n = %d$).}\label{tab:m12}
\begin{tabular}{lrrrrr}\toprule Contact & $E_{AX}$ & vs.\ gas & vs.\ PCM & $\rho$ within & $n$\\\midrule
''' % (m['rho_across_default'], m['n_contacts'], m['rho_all_variants'], m['n_all']) + '\n'.join(' & '.join(x) + r'\\' for x in rows) + '\n\\bottomrule\\end{tabular}\\end{table}\n'
    write('tab_m12', body)


def t_levels():
    L = S['levels']['rel']; rows = []
    for k in ['HF__def2-TZVP', 'B3LYP-D3BJ__def2-TZVP', 'PBE0-D3BJ__def2-TZVP', 'M06-2X__def2-TZVP', 'wB97X-D__def2-SVP', 'wB97X-D__def2-QZVP']:
        m, b = k.split('__'); m = m.replace('wB97X-D', r'$\omega$B97X-D').replace('D3BJ', 'D3(BJ)')
        rows.append([m, b, f(L[k]['min']), f(L[k]['max'])])
    body = r'''\begin{table}[!htbp]\centering\small
\caption{Valence $\Sigma E^{(2)}$ of the 12 scan dimers at $d_\mathrm{eq}$ relative to $\omega$B97X-D/def2-TZVP (Set D, Gaussian~16). No decision changes; the smallest value at any level is %.2f~kcal\,mol$^{-1}$; the rank correlation with the default level is at least %.2f.}\label{tab:levels}
\begin{tabular}{llrr}\toprule Method & Basis & min ratio & max ratio\\\midrule
''' % (S['levels']['min_val'], S['levels']['rho_min']) + '\n'.join(' & '.join(x) + r'\\' for x in rows) + '\n\\bottomrule\\end{tabular}\\end{table}\n'
    write('tab_levels', body)


def t_protein_eda():
    rows = []
    for s, lig, frz, pol, ct, tot, chi, g16, orca in S['protein_eda']:
        rows.append([SYSN[s], tex(lig[:-1]) + ' (' + lig[-1] + ')', f(frz), f(pol), f(ct), f(tot), f(chi), f(orca), f(g16)])
    body = r'''\begin{table}[!htbp]\centering\small
\caption{ALMO-EDA of the six finished protein-cluster jobs (Q-Chem~6.2.2, $\omega$B97X-V/def2-TZVPD, gas phase; kcal\,mol$^{-1}$) with the counterpoise interaction energies from ORCA (same level) and Gaussian~16 ($\omega$B97X-D/def2-TZVP, no diffuse functions). The terms cover every contact between the ligand and the acceptor fragment. The large Gaussian--ORCA gap for 2W0S (ligand charge $-2$) shows the need for diffuse functions on anions. Three EDA jobs (2W0S chain A, 1P5E chain A, and 4GQL) stopped during the fragment calculations and gave no result; the 4GQL ORCA job was not run.}\label{tab:proteda}
\begin{tabular}{llrrrrrrr}\toprule PDB & Ligand & FRZ & POL & CT & Total & $\chi_\mathrm{CT}$ & ORCA & G16\\\midrule
''' + '\n'.join(' & '.join(x) + r'\\' for x in rows) + '\n\\bottomrule\\end{tabular}\\end{table}\n'
    write('tab_proteda', body)


# ---------------------------------------------------------------- hIYD NBO and T2/T5
def t_hiyd():
    h = pd.read_csv(os.path.join(V3, 'hiyd_decisions_updated.csv'))
    rows = []
    for _, r in h.sort_values(['pathway', 'xi']).iterrows():
        bb = r.source.startswith('acetyl')
        rows.append([r.pathway, tex(r.residue), f(r.xi, 1), tex(r.acceptor_atom), 'poor' if r.lewis == 'weak' else 'base', f(r.d_A), f(r.theta1, 1), f(r.E_pub), f(r.E2_intra_phenolateO),
                     f(r.E2_bondx_pair), VSHORT[r.verdict_v3], f(r.E2_used) if bb else '', f(r.hb_used), VSHORT[r.verdict_used] + (r'$^\dagger$' if r.hbFlag else '')])
    write('tab_hiyd_nbo', longtable('llrllrrrrrlrrl', r'Path & Residue & $\xi$ & Y & Base & $d$ & $\theta_1$ & Pub. & Intra & $\Sigma E^{(2)}$ & D & cap & X$\to$Y--H & Final', rows,
                                  r'NBO screen of the 56 published hIYD poses (B3LYP-D3/6-311++G(d,p) with LANL2DZdp on I, CPCM $\varepsilon=4$; kcal\,mol$^{-1}$).\cite{Karmakar2026} $\xi$: window (\AA). $d$ (\AA) and $\theta_1$ ($^\circ$): T1 geometry of the published pose. Base: Lewis class of the acceptor in the protein (poor: delocalized or protonated lone pair). Pub.: published value. Intra: lone pair of the phenolate O of I-Tyr into $\sigma^*$(C--I) of the same ring. $\Sigma E^{(2)}$ and D: valence decision value and decision for the published model; all 18 backbone N acceptors of the published models are amines, because a hydrogen replaced the preceding carbonyl carbon. cap: valence value after this hydrogen was rebuilt as an acetyl group (set K; same pose and level). X$\to$Y--H: halogen lone pairs into N--H or O--H antibonds (for backbone N, of the acetyl-capped model). Final: decision with the acetyl-capped models for backbone N; $^\dagger$: the competing hydrogen bond to I exceeds $\Sigma E^{(2)}$. One set-K job (A130$^*$, pathway C) stopped while writing the NBO file after the complete NBO analysis had been printed; its values are used. These outputs use a different level of theory from the one the grey zone was derived for.', 'tab:hiydnbo', size=r'\scriptsize'))
    t = pd.read_csv(os.path.join(V3, 'hiyd_t5_table.csv'))
    import re as _re
    # the stripped trajectories number residues one higher than the published cluster files (see 21_v3_analysis.py)
    amap = {(_re.match(r'([A-Z]+)(\d+)', a).group(1) + str(int(_re.match(r'([A-Z]+)(\d+)', a).group(2)) + 1)): r for a, r in zip(h.residue_amber, h.residue)}
    upd = {(x.pathway, x.residue, x.acceptor_atom, round(x.xi, 1)): x.E2_used for _, x in h.iterrows()}
    rows = []
    for _, r in t.iterrows():
        p, res, at = r.key.split('|')
        same = (r.xi_pub == r.xi_pub) and abs(r.xi_pub - r.xi) < 1e-6 and (r.E2_bondx_pair == r.E2_bondx_pair)
        rows.append([p, tex(amap.get(res, res.title()[:3] + res[3:])), at, f(r.xi, 1), f(r.occupancy * 100, 1), '%.0f' % r.tau_ps, f(r.D), f(r.D / r.occupancy), f(r.G_best),
                     f(upd.get((p, amap.get(res, ''), at, round(r.xi, 1)), r.E2_bondx_pair)) if same else '--'])
    body = r"""\begin{table}[!htbp]\centering\scriptsize\setlength{\tabcolsep}{3pt}
\caption{Tier T2 on the hIYD release windows for the contacts with the highest occupancy. Occupancy: share of the frames of one umbrella window (unweighted, \%). $\tau$: mean residence time (ps; 20~ps per MD frame times the stride of the stripped file). $D$: persistence score ($\tau_0 = 100$~ps); at the frame spacing of these windows (160--240~ps), $D/f \ge 0.95$, so $D$ is practically the occupancy. $G$: best frame. $\Sigma E^{(2)}$: valence value of the NBO output for the same residue and acceptor atom in the same window (published model; acetyl-capped model for backbone N; ``--'': no output from this window).}\label{tab:hiydt5}
\begin{tabular}{lllrrrrrrr}\toprule Path & Residue & Y & $\xi$ & Occ. & $\tau$ & $D$ & $D/f$ & $G$ & $\Sigma E^{(2)}$\\\midrule
""" + '\n'.join(' & '.join(x) + r'\\' for x in rows) + '\n\\bottomrule\\end{tabular}\\end{table}\n'
    write('tab_hiydt5', body)


# ---------------------------------------------------------------- round-2 tables
def t_classratio():
    c = pd.read_csv(os.path.join(V3, 'class_ratio.csv'), index_col=0)
    nm = {'P': 'P', 'N': 'N', 'S': 'S', 'O': 'O', 'F': 'F', 'ring': r'ring ($\pi$)'}
    rows = [[nm.get(k, k), '%d' % v['count'], f(v['median']), '%s--%s' % (f(v['min']), f(v['max'])), f(0.8 / v['median'])] for k, v in c.iterrows()]
    body = r"""\begin{table}[!htbp]\centering\small
\caption{Valence $\Sigma E^{(2)}$ per unit CCSD(T)/CBS binding energy for each acceptor class of the 56 HAL59 halogen bonds. BE$_{0.8}$: binding energy (kcal\,mol$^{-1}$) that corresponds to the upper grey-zone limit of 0.8~kcal\,mol$^{-1}$ at the median ratio. Ring acceptors are under-scored about fourfold relative to N acceptors; with only four ring dimers, no class-specific zone is fitted.}\label{tab:classratio}
\begin{tabular}{lrrlr}\toprule Acceptor & $n$ & median & range & BE$_{0.8}$\\\midrule
""" + '\n'.join(' & '.join(x) + r'\\' for x in rows) + '\n\\bottomrule\\end{tabular}\\end{table}\n'
    write('tab_classratio', body)


def t_hardneg():
    ex = pd.read_csv(os.path.join(V3, 'extra_nbo.csv')); ex['flags'] = ex['flags'].fillna('')
    G = ex[ex.set == 'G_hard_negatives'].copy()
    G['kind'] = np.where(G.role.str.startswith('control'), 'base', 'poor')
    G['res'] = G.role.str.extract(r'\(([^)]+)\)')[0]; G['what'] = G.role.str.replace(r'^(negative|control): ', '', regex=True).str.replace(r' \([^)]+\)', '', regex=True)
    G['path'] = G.name.str.extract(r'hiyd_C(\d)_')[0].map({'1': 'A', '2': 'B', '3': 'C'}); G['xi'] = G.name.str.extract(r'hiyd_C\d_([\d.]+)_')[0].astype(float)
    rows = []
    for _, r in G.sort_values(['kind', 'E2_valence'], ascending=[False, True]).iterrows():
        rows.append([tex(r.res), tex(r.what), r.kind, r.path, f(r.xi, 1), '%d' % r.frame, f(r.d), f(r.theta1, 1), f(r.G), f(r.E2_valence), f(r.E2_RY), f(max(r.hbY, r.hbFrag2)), 'yes' if str(r.lpOnY).lower() == 'true' else 'no', VSHORT[r.verdict] + (r'$^\dagger$' if 'hbond' in r['flags'] else '')])
    J = ex[ex.set == 'J_4dst_amide']
    ENVN = {'cluster_gas': 'gas', 'cluster_pcm4': 'PCM', 'qmmm_ff14SB': 'QM/MM'}
    jr = [['Gly75 (4DST, %s)' % ENVN[r['name'].split('__')[2]], 'backbone amide N', 'poor', '--', '--', '--', '3.52', '140.5', '0.12', f(r.E2_valence), f(r.E2_RY), f(max(r.hbY, r.hbFrag2)), 'yes' if str(r.lpOnY).lower() == 'true' else 'no', VSHORT[r.verdict]] for _, r in J.iterrows()]
    envs = [n.split('__')[2] for n in J.name]
    body = r"""\begin{table}[!htbp]\centering\scriptsize\setlength{\tabcolsep}{2.6pt}
\caption{Contacts that pass T1 but involve poor Lewis bases (hard negatives), with Lewis-base controls from the same simulations (kcal\,mol$^{-1}$). hIYD rows: one frame of a release window each, cluster models built with the defaults of BondX Studio 3.0 (intact amides, oriented hydroxyl hydrogens), I-Tyr phenolate charge $-1$, CPCM $\varepsilon=4$, $\omega$B97X-D/def2-TZVP (Gaussian~16, NBO~3.1); every output matched its manifest. 4DST rows: the Cl$\cdots$N Gly75 contact with the amide of Gly75 intact (""" + ', '.join(envs).replace('cluster_gas', 'gas cluster').replace('cluster_pcm4', 'PCM cluster').replace('qmmm_ff14SB', 'QM/MM') + r"""). Base: Lewis base or poor Lewis base. X$\to$H: summed donation from the iodine lone pairs into the N--H and O--H antibonds of fragment~2. D: decision; $^\dagger$: the competing hydrogen bond exceeds $\Sigma E^{(2)}$.}\label{tab:hardneg}
\resizebox{\textwidth}{!}{\begin{tabular}{lllcrrrrrrrrll}\toprule Residue & Acceptor & Base & Path & $\xi$ & Frame & $d$ & $\theta_1$ & $G$ & $\Sigma E^{(2)}$ & RY & X$\to$H & LP(Y) & D\\\midrule
""" + '\n'.join(' & '.join(x) + r'\\' for x in rows) + '\n\\midrule\n' + '\n'.join(' & '.join(x) + r'\\' for x in jr) + '\n\\bottomrule\\end{tabular}}\\end{table}\n'
    write('tab_hardneg', body)


if __name__ == '__main__':
    print('cut-off ratios', t_cutoffs())
    print('hydrogens compared (AMBER)', t_ffref())
    t_hal59(); print('protein NBO rows', t_protein()); t_pairs(); t_field(); t_m12(); t_levels(); t_protein_eda(); t_hiyd(); t_classratio(); t_hardneg()


### `27_extra_parse.py`

27_extra_parse.py
Independent parser (no BondX code) for the additional Gaussian 16 NBO jobs (sets G, J, and K).
For every output it reads the geometry and the second-order perturbation table of NBO 3.1 and computes
  E2_valence : sum of LP(Y) -> BD*(A-X) or LP*(X) terms (Y = T1 acceptor atom, X = halogen, A = its carbon)
  E2_RY      : sum of LP(Y) -> RY*(X)
  hb_Y       : sum of LP(X) -> BD*(Y-H) terms (competing hydrogen bond from the acceptor atom)
  hb_any     : sum of LP(X) -> BD*(N-H or O-H) terms to atoms outside the halogen-bearing molecule
  LP_on_Y    : NBO prints at least one lone pair on Y
Sets G and J take X, A, and Y from the BondX manifests. Set K (published hIYD inputs with a new acetyl cap) has no
manifest: X is the iodine, A its carbon, and Y the backbone N nearest to X.
Usage: python3 27_extra_parse.py ../qm_extra ../results/extra
Output: results/extra/extra_parsed.csv

In [ ]:
%%writefile Scripts/27_extra_parse.py
#!/usr/bin/env python3
"""27_extra_parse.py
Independent parser (no BondX code) for the additional Gaussian 16 NBO jobs (sets G, J, and K).
For every output it reads the geometry and the second-order perturbation table of NBO 3.1 and computes
  E2_valence : sum of LP(Y) -> BD*(A-X) or LP*(X) terms (Y = T1 acceptor atom, X = halogen, A = its carbon)
  E2_RY      : sum of LP(Y) -> RY*(X)
  hb_Y       : sum of LP(X) -> BD*(Y-H) terms (competing hydrogen bond from the acceptor atom)
  hb_any     : sum of LP(X) -> BD*(N-H or O-H) terms to atoms outside the halogen-bearing molecule
  LP_on_Y    : NBO prints at least one lone pair on Y
Sets G and J take X, A, and Y from the BondX manifests. Set K (published hIYD inputs with a new acetyl cap) has no
manifest: X is the iodine, A its carbon, and Y the backbone N nearest to X.
Usage: python3 27_extra_parse.py ../qm_extra ../results/extra
Output: results/extra/extra_parsed.csv
"""
import os, re, sys, json, glob, math
import pandas as pd

ROOT = sys.argv[1] if len(sys.argv) > 1 else '../qm_extra'
OUT = sys.argv[2] if len(sys.argv) > 2 else '../results/extra'
EL = {1: 'H', 6: 'C', 7: 'N', 8: 'O', 9: 'F', 15: 'P', 16: 'S', 17: 'Cl', 35: 'Br', 53: 'I'}
ROW = re.compile(r'^\s*(?:\d+|\*+)\.\s+(\S+)\s*\(\s*\d+\)\s*(.+?)\s+/\s*(?:\d+|\*+)\.\s+(\S+)\s*\(\s*\d+\)\s*(.+?)\s+(-?\d+\.\d+)\s+(-?\d+\.\d+)\s+(-?\d+\.\d+)\s*$')
ATOMS = re.compile(r'([A-Z][a-z]?)\s*(\d+)')


def geometry(lines):
    """Last 'Input orientation' block (nosymm jobs)."""
    start = None
    for i, l in enumerate(lines):
        if 'Input orientation:' in l or 'Standard orientation:' in l:
            start = i
    xyz = []
    for l in lines[start + 5:]:
        if l.strip().startswith('----'):
            break
        p = l.split()
        xyz.append((EL.get(int(p[1]), p[1]), float(p[3]), float(p[4]), float(p[5])))
    return xyz


def table(lines):
    """Second-order table rows: (donor type, donor atoms, acceptor type, acceptor atoms, E2)."""
    rows, on = [], False
    for l in lines:
        if 'Second Order Perturbation Theory Analysis' in l:
            on = True; rows = []; continue
        if on and 'Natural Bond Orbitals (Summary)' in l:
            on = False; continue
        if on:
            m = ROW.match(l)
            if m:
                da = [int(n) for _, n in ATOMS.findall(m.group(2))]
                aa = [int(n) for _, n in ATOMS.findall(m.group(4))]
                rows.append((m.group(1), da, m.group(3), aa, float(m.group(5))))
    return rows


def lone_pairs(lines):
    """Atoms that carry a Lewis lone pair in the NBO listing, e.g. '115. (1.99308) LP ( 1) I  23'."""
    s = set()
    for l in lines:
        m = re.match(r'^\s*(?:\d+|\*+)\.\s+\(\s*[0-9.]+\)\s+LP\s*\(\s*\d+\)\s*[A-Z][a-z]?\s*(\d+)', l)
        if m:
            s.add(int(m.group(1)))
    return s


def dist(a, b):
    return math.dist(a[1:], b[1:])


def bonded(xyz, i, j):
    lim = {'H': 0.32, 'C': 0.76, 'N': 0.71, 'O': 0.66, 'S': 1.05, 'P': 1.07, 'Cl': 1.02, 'Br': 1.20, 'I': 1.39, 'F': 0.57}
    return dist(xyz[i - 1], xyz[j - 1]) < lim.get(xyz[i - 1][0], 0.8) + lim.get(xyz[j - 1][0], 0.8) + 0.45


def molecule(xyz, start):
    seen, stack = {start}, [start]
    while stack:
        a = stack.pop()
        for b in range(1, len(xyz) + 1):
            if b not in seen and bonded(xyz, a, b):
                seen.add(b); stack.append(b)
    return seen


def analyse(log, X, A, Y):
    L = open(log, errors='ignore').read().split('\n')
    xyz = geometry(L); T = table(L); lp = lone_pairs(L)
    mol1 = molecule(xyz, X)
    val = ry = hbY = hbany = 0.0
    for dt, da, at, aa, e in T:
        if dt == 'LP' and da == [Y]:
            if at == 'BD*' and sorted(aa) == sorted([A, X]):
                val += e
            elif at == 'LP*' and aa == [X]:
                val += e
            elif at == 'RY*' and aa == [X]:
                ry += e
        if dt == 'LP' and da == [X] and at == 'BD*' and len(aa) == 2:
            heavy = [a for a in aa if xyz[a - 1][0] != 'H']; hyd = [a for a in aa if xyz[a - 1][0] == 'H']
            if len(heavy) == 1 and len(hyd) == 1 and xyz[heavy[0] - 1][0] in ('N', 'O') and heavy[0] not in mol1:
                hbany += e
                if heavy[0] == Y:
                    hbY += e
    ok = any('Normal termination' in l for l in L[-5:])
    d = dist(xyz[X - 1], xyz[Y - 1])
    v1 = [xyz[A - 1][k] - xyz[X - 1][k] for k in (1, 2, 3)]; v2 = [xyz[Y - 1][k] - xyz[X - 1][k] for k in (1, 2, 3)]
    th = math.degrees(math.acos(sum(p * q for p, q in zip(v1, v2)) / (math.hypot(*v1) * math.hypot(*v2))))
    nheavy = sum(1 for b in range(1, len(xyz) + 1) if b != Y and bonded(xyz, Y, b) and xyz[b - 1][0] != 'H')
    nh = sum(1 for b in range(1, len(xyz) + 1) if b != Y and bonded(xyz, Y, b) and xyz[b - 1][0] == 'H')
    return dict(normal=ok, nNBOrows=len(T), E2_valence=round(val, 2), E2_RY=round(ry, 2), hb_Y=round(hbY, 2), hb_any=round(hbany, 2),
                LP_on_Y=Y in lp, d=round(d, 3), theta1=round(th, 1), Y_el=xyz[Y - 1][0], Y_heavy_nb=nheavy, Y_H=nh, natoms=len(xyz))


rows = []
for s in ('G_hard_negatives', 'J_4dst_amide'):
    for man in sorted(glob.glob(os.path.join(ROOT, s, 'manifests', '*_manifest.json'))):
        m = json.load(open(man)); name = os.path.basename(man).replace('_manifest.json', '')
        log = os.path.join(ROOT, s, 'gaussian', name + '.log')
        if not os.path.exists(log):
            continue
        xb = m['xbs'][0]
        r = analyse(log, xb['X'], xb['A'], xb['Y']); r.update(set=s, name=name, contact=xb['label'], charge=m['totalCharge'])
        rows.append(r)
for log in sorted(glob.glob(os.path.join(ROOT, 'K_hiyd_amide', 'gaussian', '*.log'))):
    L = open(log, errors='ignore').read().split('\n'); xyz = geometry(L)
    X = next(i + 1 for i, a in enumerate(xyz) if a[0] == 'I')
    A = min((i + 1 for i, a in enumerate(xyz) if a[0] == 'C'), key=lambda j: dist(xyz[X - 1], xyz[j - 1]))
    cand = [i + 1 for i, a in enumerate(xyz) if a[0] == 'N' and (i + 1) not in molecule(xyz, X)]
    Y = min(cand, key=lambda j: dist(xyz[X - 1], xyz[j - 1]))
    r = analyse(log, X, A, Y); r.update(set='K_hiyd_amide', name=os.path.basename(log)[:-4], contact='I···N (nearest backbone N)', charge=None)
    rows.append(r)
df = pd.DataFrame(rows)
os.makedirs(OUT, exist_ok=True)
df.to_csv(os.path.join(OUT, 'extra_parsed.csv'), index=False)
print(df[['set', 'name', 'normal', 'E2_valence', 'E2_RY', 'hb_Y', 'hb_any', 'LP_on_Y', 'd', 'theta1', 'Y_el', 'Y_heavy_nb', 'Y_H']].to_string())


### `28_v3_round2_analysis.py`

28_v3_round2_analysis.py
Analysis added after the second review:
  - Sigma E(2) per unit binding energy for each acceptor class of HAL59 (bias of one threshold)
  - which data points set the extremes of the grey-zone spreads, and the zone without the Thr113 contact
  - the smallest supported benchmark value
  - additional job sets: hIYD hard negatives (set G), 4DST with the intact amide (set J), and the published hIYD
    backbone-N poses with an acetyl cap (set K); the updated hIYD decision counts
  - the competing-hydrogen-bond flag on all protein and hIYD contacts
  - the residence factor of D at the saved-frame spacing of the hIYD windows
Usage: python3 28_v3_round2_analysis.py [ROOT]
Output: results/v3/summary_round2.json, results/v3/class_ratio.csv, results/v3/hiyd_decisions_updated.csv

In [ ]:
%%writefile Scripts/28_v3_round2_analysis.py
#!/usr/bin/env python3
"""28_v3_round2_analysis.py
Analysis added after the second review:
  - Sigma E(2) per unit binding energy for each acceptor class of HAL59 (bias of one threshold)
  - which data points set the extremes of the grey-zone spreads, and the zone without the Thr113 contact
  - the smallest supported benchmark value
  - additional job sets: hIYD hard negatives (set G), 4DST with the intact amide (set J), and the published hIYD
    backbone-N poses with an acetyl cap (set K); the updated hIYD decision counts
  - the competing-hydrogen-bond flag on all protein and hIYD contacts
  - the residence factor of D at the saved-frame spacing of the hIYD windows
Usage: python3 28_v3_round2_analysis.py [ROOT]
Output: results/v3/summary_round2.json, results/v3/class_ratio.csv, results/v3/hiyd_decisions_updated.csv
"""
import os, sys, json
import numpy as np, pandas as pd

ROOT = sys.argv[1] if len(sys.argv) > 1 else '..'
R = os.path.join(ROOT, 'results'); V3 = os.path.join(R, 'v3')
LO, HI, HBMIN = 0.3, 0.8, 0.5
verdict = lambda v: 'supported' if v >= HI else 'inconclusive' if v >= LO else 'not supported'
r2 = lambda x: None if x is None else round(float(x), 2)
S = {}
nbo = pd.read_csv(os.path.join(V3, 'nbo_v3.csv'))

# ---------------------------------------------------------------- HAL59 acceptor classes
q = pd.read_csv(os.path.join(R, 'qm_hal59.csv')).merge(nbo[nbo.set == 'A_HAL59'][['name', 'E2_valence', 'verdict']], on='name')
xb = q[~q.control].copy(); xb['ratio'] = xb.E2_valence / xb.ref_kcal
cls = xb.groupby('acc_el').ratio.agg(['count', 'median', 'min', 'max'])
cls = cls.rename(index={'pi': 'ring'}).sort_values('median', ascending=False)
cls.round(2).to_csv(os.path.join(V3, 'class_ratio.csv'))
S['class_ratio'] = {k: dict(n=int(v['count']), median=r2(v['median']), min=r2(v['min']), max=r2(v['max'])) for k, v in cls.iterrows()}
med = cls['median']
S['class_ratio_note'] = dict(N_over_ring=r2(med['N'] / med['ring']), O_over_N=r2(med['O'] / med['N']),
                             BE_at_08_N=r2(HI / med['N']), BE_at_08_O=r2(HI / med['O']), BE_at_08_ring=r2(HI / med['ring']))
sup = xb[xb.verdict == 'supported']
S['supported_min'] = [sup.loc[sup.E2_valence.idxmin(), 'name'], r2(sup.E2_valence.min())]
S['supported_max'] = [sup.loc[sup.E2_valence.idxmax(), 'name'], r2(sup.E2_valence.max())]
ring = xb[xb.acc_el == 'pi'][['name', 'ref_kcal', 'E2_valence', 'verdict']]
S['ring_dimers'] = ring.round(2).values.tolist()

# ---------------------------------------------------------------- grey zone: which points set the extremes
mb = pd.read_csv(os.path.join(R, 'qm_method_basis.csv')).merge(nbo[nbo.set == 'D_method_basis'][['name', 'E2_valence']], on='name')
r0 = mb[mb.level == 'wB97X-D__def2-TZVP'].set_index('base').E2_valence; mb['rel'] = mb.E2_valence / mb.base.map(r0)
x = mb[mb.level != 'wB97X-D__def2-TZVP']
S['level_extremes'] = dict(min=[x.loc[x.rel.idxmin(), 'base'], x.loc[x.rel.idxmin(), 'level'], r2(x.rel.min())],
                           max=[x.loc[x.rel.idxmax(), 'base'], x.loc[x.rel.idxmax(), 'level'], r2(x.rel.max())])
pr = pd.read_csv(os.path.join(R, 'qm_protein.csv')).merge(nbo[nbo.set == 'F_protein'][['name', 'label', 'E2_valence']], on=['name', 'label'])
rows = []
for c, d in pr.groupby('contact'):
    if len(d) < 5: continue
    v0 = d[d.variant == 'cluster_pcm4'].E2_valence.iloc[0]
    if v0 < 0.2: continue
    rr = d.E2_valence / v0
    rows.append(dict(contact=c, n=len(d), v0=v0, rmin=rr.min(), vmin=d.variant.iloc[rr.values.argmin()], rmax=rr.max(), vmax=d.variant.iloc[rr.values.argmax()]))
M = pd.DataFrame(rows)
def zone(mm):
    lo, hi = mm.rmin.min(), mm.rmax.max(); clo, chi = x.rel.min() * lo, x.rel.max() * hi
    return dict(model=[r2(lo), r2(hi)], combined=[r2(clo), r2(chi)], zone=[r2(0.5 / chi), r2(0.5 / clo)])
S['greyzone_all'] = zone(M); S['greyzone_without_Thr113'] = zone(M[~M.contact.str.contains('THR113')])
S['model_extremes'] = dict(min=M.loc[M.rmin.idxmin(), ['contact', 'vmin']].tolist() + [r2(M.rmin.min())], max=M.loc[M.rmax.idxmax(), ['contact', 'vmax']].tolist() + [r2(M.rmax.max())])
mt = M[~M.contact.str.contains('THR113')]
S['model_extremes_without_Thr113'] = dict(max=mt.loc[mt.rmax.idxmax(), ['contact', 'vmax']].tolist() + [r2(mt.rmax.max())])

# ---------------------------------------------------------------- competing H-bond flag on the protein models and contacts
pn = nbo[nbo.set == 'F_protein'].copy(); pn['hbMax'] = pn[['hbY', 'hbFrag2']].max(axis=1)
pn['hbFlag'] = (pn.hbMax >= HBMIN) & (pn.hbMax > pn.E2_valence)
pn['contact'] = pn.label
fl = pn[pn.hbFlag].groupby('label').name.apply(lambda s: sorted(set(v.split('__')[-1] for v in s))).to_dict()
S['protein_hbflag_models'] = {k: v for k, v in fl.items()}
S['protein_hbflag_counts'] = {k: [int(pn[(pn.label == k)].hbFlag.sum()), int((pn.label == k).sum())] for k in fl}

# ---------------------------------------------------------------- additional job sets
ex = pd.read_csv(os.path.join(V3, 'extra_nbo.csv'))
ex['flags'] = ex['flags'].fillna('')
G = ex[ex.set == 'G_hard_negatives'].copy(); G['kind'] = np.where(G.role.str.startswith('control'), 'Lewis base', 'poor Lewis base')
S['setG'] = {k: dict(n=int(len(d)), verdicts=d.verdict.value_counts().to_dict(), hbflag=int(d['flags'].str.contains('hbond').sum()),
                     supported=d[d.verdict == 'supported'][['role', 'E2_valence']].round(2).values.tolist()) for k, d in G.groupby('kind')}
S['setG_all_geometry_ok'] = bool((G.geometry_ok.astype(str).str.lower() == 'true').all())
J = ex[ex.set == 'J_4dst_amide']; S['setJ'] = dict(n=int(len(J)), max_valence=r2(J.E2_valence.max()), max_hb=r2(J[['hbY', 'hbFrag2']].max().max()), lp=J.lpOnY.astype(str).tolist())
K = ex[ex.set == 'K_hiyd_amide'].copy()
K['pose'] = K.role.str.replace('published pose ', '').str.replace('; cap H -> C(=O)CH3', '', regex=False)
S['setK'] = dict(n=int(len(K)), verdicts=K.verdict.value_counts().to_dict(), max_valence=[K.loc[K.E2_valence.idxmax(), 'pose'], r2(K.E2_valence.max())],
                 hbflag=int(K['flags'].str.contains('hbond').sum()), hb_range=[r2(K[K['flags'].str.contains('hbond')].hbFrag2.min()), r2(K[K['flags'].str.contains('hbond')].hbFrag2.max())],
                 amine_supported_before=K[K.E_pub_gate >= HI][['pose', 'E_pub_gate', 'E2_valence']].round(2).values.tolist(),
                 incomplete_job=K[K.terminated.astype(str).str.lower() != 'true'].name.tolist())
H = ex[ex.set == 'H_marginal_levels']; S['setH'] = dict(n=int(len(H)), finished=int((H.terminated.astype(str).str.lower() == 'true').sum()))
S['setG_ranges'] = dict(intra=[r2(G['intra_O_sigmaCI'].min()), r2(G['intra_O_sigmaCI'].max())])

# ---------------------------------------------------------------- hIYD: decisions with the acetyl-capped backbone N poses
h = pd.read_csv(os.path.join(R, 'hiyd', 'hiyd_nbo_reanalysis.csv'))
h['hbFlag'] = (h.hb_LPI_to_YH >= HBMIN) & (h.hb_LPI_to_YH > h.E2_bondx_pair)
h['source'] = 'published model'; h['E2_used'] = h.E2_bondx_pair; h['verdict_used'] = h.verdict_v3; h['hb_used'] = h.hb_LPI_to_YH
bb = h.acceptor_class == 'backbone N-H'
key = K.pose.str.extract(r'^(\S+) N \(pathway (\w), xi ([\d.]+)\)')
K['residue'] = key[0]; K['pathway'] = key[1]; K['xi'] = key[2].astype(float)
for i in h[bb].index:
    m = K[(K.residue == h.at[i, 'residue']) & (K.pathway == h.at[i, 'pathway']) & (np.isclose(K.xi, h.at[i, 'xi']))]
    assert len(m) == 1, (h.at[i, 'residue'], h.at[i, 'pathway'], h.at[i, 'xi'])
    m = m.iloc[0]
    h.at[i, 'E2_used'] = m.E2_valence; h.at[i, 'verdict_used'] = m.verdict; h.at[i, 'hb_used'] = m.hbFrag2; h.at[i, 'source'] = 'acetyl cap (set K)'
    h.at[i, 'hbFlag'] = ('hbond' in m['flags'])
h.to_csv(os.path.join(V3, 'hiyd_decisions_updated.csv'), index=False)
S['hiyd_published_counts'] = h.verdict_v3.value_counts().to_dict()
S['hiyd_updated_counts'] = h.verdict_used.value_counts().to_dict()
S['hiyd_updated_supported'] = h[h.verdict_used == 'supported'][['pathway', 'residue', 'acceptor_atom', 'E2_used', 'hb_used', 'hbFlag']].round(2).values.tolist()
S['hiyd_updated_hbflag'] = h[h.hbFlag][['pathway', 'residue', 'acceptor_atom', 'E2_used', 'hb_used', 'verdict_used']].round(2).values.tolist()
S['hiyd_backbone_updated'] = h[bb].verdict_used.value_counts().to_dict()
S['hiyd_backbone_before'] = h[bb].verdict_v3.value_counts().to_dict()

# ---------------------------------------------------------------- residence factor at the hIYD frame spacing
t2 = pd.read_csv(os.path.join(V3, 'hiyd_t2_v3.csv'))
dts = sorted(set(t2.dt_ps.astype(int)))
S['hiyd_dt_with_contacts'] = dts
fac = lambda dt: (1 - np.exp(-dt / 100.0)) ** 0.25
S['residence_factor'] = {str(dt): r2(fac(dt)) for dt in [100] + dts}
S['hiyd_tau'] = dict(single_frame_share=r2((t2.tau_ps <= t2.dt_ps + 1e-6).mean()), n=int(len(t2)), median=r2(t2.tau_ps.median()), range=[r2(t2.tau_ps.min()), r2(t2.tau_ps.max())], max_run=r2(t2.tau_max_ps.max()), D_over_f_min=r2((t2.D / t2.occupancy).min()))

json.dump(S, open(os.path.join(V3, 'summary_round2.json'), 'w'), indent=1, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
print(json.dumps(S, indent=1, default=str)[:6000])


### `inline_tables.py`

inline_tables.py
Writes ESI_V3.tex from ESI_V3_src.tex. Every line \input{tables/NAME.tex} is replaced by the text of tables/NAME.tex.
The tables come from scripts 11 and 26. The result holds all tables inside one .tex file.
Usage: python3 inline_tables.py /path/to/paper_folder

In [ ]:
%%writefile Scripts/inline_tables.py
#!/usr/bin/env python3
"""inline_tables.py
Writes ESI_V3.tex from ESI_V3_src.tex. Every line \\input{tables/NAME.tex} is replaced by the text of tables/NAME.tex.
The tables come from scripts 11 and 26. The result holds all tables inside one .tex file.
Usage: python3 inline_tables.py /path/to/paper_folder
"""
import os, re, sys

ROOT = sys.argv[1] if len(sys.argv) > 1 else '..'
SRC, OUT = os.path.join(ROOT, 'ESI_V3_src.tex'), os.path.join(ROOT, 'ESI_V3.tex')
pat = re.compile(r'^\\input\{(tables/[A-Za-z0-9_]+)\.tex\}\s*$')
out, n = [], 0
for line in open(SRC, encoding='utf-8'):
    m = pat.match(line.strip())
    if not m:
        out.append(line)
        continue
    path = os.path.join(ROOT, m.group(1) + '.tex')
    body = open(path, encoding='utf-8').read().rstrip('\n')
    out.append('%% ---- table from %s.tex (generated) ----\n%s\n%% ---- end of %s.tex ----\n' % (m.group(1), body, m.group(1)))
    n += 1
text = ''.join(out)
text = text.replace('%% ESI_V3_src.tex -- source of ESI_V3.tex', '%% ESI_V3.tex -- written by Scripts/inline_tables.py from ESI_V3_src.tex', 1)
assert not any(l.lstrip().startswith('\\input{tables/') for l in text.splitlines()), 'a table input was not replaced'
open(OUT, 'w', encoding='utf-8').write(text)
print('inlined %d tables into %s' % (n, OUT))


### `compare_runs.py`

compare_runs.py  RUN_A RUN_B
Compares every CSV and JSON file of results/ between two pipeline runs.
Timing columns (ms, seconds, hours, wall time) and performance.csv are skipped. Numbers are compared exactly.

In [ ]:
%%writefile Scripts/compare_runs.py
#!/usr/bin/env python3
"""compare_runs.py  RUN_A RUN_B
Compares every CSV and JSON file of results/ between two pipeline runs.
Timing columns (ms, seconds, hours, wall time) and performance.csv are skipped. Numbers are compared exactly.
"""
import os, sys, json, pandas as pd
A, B = sys.argv[1], sys.argv[2]
SKIP_FILES = {'performance.csv', 'orca_runtime.csv'}
TIME_COLS = ('ms', 'seconds', 'hours', 'wall', 'time', 'runtime', 'created')
def strip(o):
    if isinstance(o, dict): return {k: strip(v) for k, v in o.items() if not any(t in k.lower() for t in TIME_COLS)}
    if isinstance(o, list): return [strip(v) for v in o]
    return o
bad, n = 0, 0
for root, _, files in os.walk(os.path.join(A, 'results')):
    for f in sorted(files):
        if f in SKIP_FILES or f.endswith('.jsonl') and os.path.getsize(os.path.join(root, f)) > 5e8: continue
        pa = os.path.join(root, f); pb = pa.replace(A, B, 1)
        if not os.path.exists(pb): print('MISSING in B:', pb); bad += 1; continue
        n += 1
        if f.endswith('.csv'):
            da, db = pd.read_csv(pa), pd.read_csv(pb)
            cols = [c for c in da.columns if not any(t == c.lower() or c.lower().endswith('_' + t) or c.lower().startswith(t) for t in TIME_COLS)]
            if list(da.columns) != list(db.columns) or not da[cols].equals(db[cols]): print('DIFF', pa); bad += 1
        elif f.endswith('.json'):
            if strip(json.load(open(pa))) != strip(json.load(open(pb))): print('DIFF', pa); bad += 1
        else:
            if open(pa, 'rb').read() != open(pb, 'rb').read(): print('DIFF', pa); bad += 1
print('%d files compared, %d differences' % (n, bad))


## Part 2: running the analysis

The commands are run from the `Scripts/` folder. Steps that need the raw QM outputs are marked.

In [ ]:
import os
os.makedirs('Scripts', exist_ok=True)
%cd Scripts
APP = '../../app'   # folder of the BondX Studio application (build.js, src/, tests/) for scripts 11 and 26

### Benchmark geometries, scans, and tier T1 (needs the GMTKN55 HAL59 folder for step 01)

In [ ]:
!python3 01_prepare_hal59.py ..
!python3 02_make_scans.py ..
!python3 03_t1_benchmark.py ..
!python3 06_t1_sensitivity.py ..
!python3 14_ligand_check.py ..

### QM inputs (sets A-F)

In [ ]:
!python3 04_make_qm_inputs_small.py ..
!python3 05_protein_cases.py ..

### Read the QM outputs (needs qm_package/)

In [ ]:
!python3 07_collect_results.py ..
!python3 13_orca_cp_analysis.py ..
!python3 17_qm_nbo_eda_analysis.py ..

### hIYD case study (needs the hIYD trajectories)

In [ ]:
!python3 15_hiyd_analysis.py ..
!python3 16_hiyd_figures_tables.py ..

### Screen of BondX Studio 3.0, field tests, tier T2 on hIYD, and statistics (18-20 need qm_package/ and the hIYD files)

In [ ]:
!node 18_v3_nbo_reparse.js ../qm_package ../results/v3
!node 19_v3_field.js ../data/protein ../results/v3
!node 20_v3_hiyd_t2.js ../hIYD_for_BondX ../results/v3
!python3 21_v3_analysis.py ..

### Additional job sets G, J, and K (need qm_extra/), acceptor classes, grey-zone extremes, and flags

In [ ]:
!node 27_v3_extra_nbo.js ../qm_extra ../results/v3
!python3 27_extra_parse.py ../qm_extra ../results/extra
!python3 28_v3_round2_analysis.py ..

### Tables, timings, and figures

In [ ]:
!python3 09_performance.py ..
!python3 11_make_tables.py .. {APP}
!python3 26_v3_tables.py .. {APP}
!python3 12_parameter_figure.py
!python3 22_v3_figures.py ..
!python3 25_v3_toc.py ..

### Screenshots of the application (Playwright) and the interface figures

In [ ]:
!python3 23_v3_screenshots.py ../../app/dist/BondX_Studio.html ..
!python3 24_v3_screen_figures.py ..

### Write the tables into ESI_V3.tex

In [ ]:
!python3 inline_tables.py ..